# INSPIRE Perioperative Mortality — Organ-System DNN (v3)

**ISAMBARD-AI notebook** (`RUN_PROFILE = 'isambard'`): the whole ~99,886-patient cohort on a big GPU (NVIDIA GH200). 128-number embeddings (`ENCODER_SIZE='xlarge'`), batch 1024, bf16 mixed precision, 8 data-loading workers, longer training caps, and the embedding-size sweep (16 -> 256, Part 13.9). Usually run headless instead: see `isambard/README_isambard.md`.

**What v3 is:** eight equal organ systems (kidneys, heart & circulation, lungs, metabolism
& liver, blood, brain & nerves, digestive, bones & joints) — each built from its own
measurements (where the dataset has any) and its own ICD-10 diagnosis chapter — read by
one shared organ-system network; a **whole-patient layer** that learns from every feature
and guides each system network; a **learned inter-system layer** pre-trained on every
patient by predicting a hidden organ system from the others; an **arbitration layer**
setting each system's say; and an additive output, so every prediction splits exactly
into per-system contributions. Anti-memorisation: label-free pre-training, gentle
fine-tuning, AdamW, organ-system dropout, capped SMOTENC, balanced batches, a seed
ensemble — plus diagnostics that show whether it is learning or memorising.
Every change is a `CONFIG` switch; the full list is in `src-dnn/CHANGES.md`.
Generated from `src-dnn/pipeline/` — edit there, then run `python build_notebook.py`.

Predicts 30-day post-surgical mortality using per-organ-system encoders (renal,
cardiovascular, respiratory, metabolic/hepatic, haematology, neurological, GI, MSK),
fused through a Neural Additive Model so every prediction stays explainable
system-by-system. **For the full reasoning behind every design choice below, see the
project's docs site** (`Multimodal_Notebook_Summary.md`, `roadmap_and_architecture.md`) —
this notebook is deliberately code-first.

**Tuned for a real, single Colab/Kaggle session:** default config runs on a
~10,000-patient sample (all real deaths kept, survivors capped — see Part 3), targets
under 20 minutes total, checkpoints training every few epochs, and caches the parsed data
so a session that dies mid-run costs you minutes, not a restart from zero.

**Quick start:** run every cell top to bottom. Part 2 auto-detects Colab vs. Kaggle and
mounts/finds your data. Part 12 prints exactly how long your run actually took, stage by
stage.

# Part 2 — Setup

Installs (Kaggle usually has most of these; the cell is safe to run either way), imports,
and the `CONFIG` dict that every flag from §1.7 lives in. **Change values here, then
Run All** — nothing below this cell should need editing to run an experiment.

In [ ]:
# Kaggle already ships torch, sklearn, pandas, numpy, matplotlib.
# imbalanced-learn is usually NOT preinstalled -> install if missing.
# Which notebook is this? build_notebook.py sets this line: 'subset' (your analysis runs on
# the sample) or 'full' (the whole ~99,886-patient cohort, every analysis switched on).
RUN_PROFILE = "isambard"
import importlib, subprocess, sys, os
# v4: a headless run (run_pipeline.py, e.g. on Isambard-AI) can pick the profile without
# editing this file: INSPIRE_RUN_PROFILE=isambard python run_pipeline.py ...
RUN_PROFILE = os.environ.get("INSPIRE_RUN_PROFILE", RUN_PROFILE)

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

_ensure("imbalanced-learn", "imblearn")
_ensure("pyarrow")   # needed for Part 3's Parquet caching of parsed subject data
_ensure("umap-learn", "umap")   # v4, optional: UMAP embedding maps in Part 13 (skipped, and said so, if this fails)
print("Setup check complete.")

In [ ]:
import os, glob, json, math, random, warnings, time
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer  # noqa: F401  (must precede IterativeImputer import)
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                              roc_curve, precision_recall_curve, confusion_matrix)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}  "
          f"total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("(Even the 'large' encoder preset is under ~1M parameters -- GPU memory is very unlikely "
          "to be the constraint. System RAM, from the raw data tables in Part 3, is the one "
          "actually worth watching -- see that Part's memory printout.)")

In [ ]:
# Mount Google Drive on Colab only. (The original cell mounted unconditionally, which
# crashes immediately on Kaggle or a local run; the cell below needs Drive on Colab.)
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import os, sys, zipfile, time

IN_COLAB = "google.colab" in sys.modules
COLAB_SUBJECTS_DIR = None

if IN_COLAB:
    # v2: first of these that exists on your Drive is used -- put your zip in MyDrive root,
    # or set DRIVE_ZIP_PATH yourself.
    _ZIP_CANDIDATES = (["/content/drive/MyDrive/subjects.zip"] if RUN_PROFILE == "full"
                       else ["/content/drive/MyDrive/subjects_sample.zip", "/content/drive/MyDrive/subjects.zip"])
    DRIVE_ZIP_PATH = next((p for p in _ZIP_CANDIDATES if os.path.isfile(p)), _ZIP_CANDIDATES[0])
    print(f"Using data zip: {DRIVE_ZIP_PATH}")
    COLAB_SUBJECTS_DIR = "/content/inspire_subjects_data_" + os.path.splitext(os.path.basename(DRIVE_ZIP_PATH))[0]   # local disk -- much faster than Drive

    assert os.path.isfile(DRIVE_ZIP_PATH), f"Can't find {DRIVE_ZIP_PATH} -- check the exact filename/path on your Drive."

    if not os.path.isdir(COLAB_SUBJECTS_DIR):
        print(f"Extracting {DRIVE_ZIP_PATH} -> {COLAB_SUBJECTS_DIR} ...")
        _t0 = time.time()
        with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as z:
            z.extractall(COLAB_SUBJECTS_DIR)
        print(f"Done in {time.time() - _t0:.1f}s.")
    else:
        print(f"{COLAB_SUBJECTS_DIR} already exists -- skipping. Delete it first if you want to re-extract.")

    # Handle a zip that has one extra nested folder inside it
    _contents = os.listdir(COLAB_SUBJECTS_DIR)
    if len(_contents) == 1 and os.path.isdir(os.path.join(COLAB_SUBJECTS_DIR, _contents[0])):
        COLAB_SUBJECTS_DIR = os.path.join(COLAB_SUBJECTS_DIR, _contents[0])

    n_files = sum(len(files) for _, _, files in os.walk(COLAB_SUBJECTS_DIR))
    print(f"\n{n_files} files found under {COLAB_SUBJECTS_DIR}")
    print("Top-level contents:", os.listdir(COLAB_SUBJECTS_DIR))

    assert "died" in os.listdir(COLAB_SUBJECTS_DIR) and "survived" in os.listdir(COLAB_SUBJECTS_DIR), (
        f"{COLAB_SUBJECTS_DIR} doesn't contain 'died' and 'survived' subfolders -- check the zip's structure."
    )
    print("\nLooks good.")
else:
    print("Not running on Colab -- skipping the Drive-zip extraction step. On Kaggle, "
          "attach your dataset and Part 2's auto-detection will find it under /kaggle/input.")

In [ ]:
# ---------------------------------------------------------------------------
# CONFIG — every flag documented in Part 1 §1.7. Edit here, not below.
# ---------------------------------------------------------------------------
CONFIG = {
    # --- data location (auto-detected below; override if detection is wrong) ---
    "SUBJECTS_DIR": COLAB_SUBJECTS_DIR,   # folder containing died/ and survived/ subfolders of per-patient JSON
    "CODES_DIR": None,         # optional: folder with icd10.json.gz / WHO_ATC-DDD_*.csv (finer descriptions only)

    # --- full-scale readiness ---
    # Caps patients PER FOLDER (died/survived) -- NOT a percentage sample, and it never
    # removes real death cases: the full cohort has only ~469 real deaths, always fewer
    # than this cap, so every real death is always kept; only the (much larger) survived
    # pool gets capped -- this cap IS the downsampling mechanism, and it's deliberately
    # class-aware (never touches the rare class) rather than a blind random sample of
    # everyone. Default here is 10,000 -- a ~10,469-patient run, comfortably inside a
    # 20-minute budget given the batch-size/early-stopping settings below. Set to None
    # for the true full run once you have the time budget for it (expect data loading
    # alone to take several minutes at that scale, per Part 3's own measured timing).
    "MAX_SUBJECTS_PER_CLASS": 10000,

    # --- memory safety for Part 3's parse (new) -- patients are parsed and flushed to
    # disk in chunks of this size, rather than holding every row of every table for the
    # WHOLE cohort as Python dict objects in memory at once (which is roughly 6x more
    # memory-hungry per row than the eventual optimized DataFrame -- easily enough to
    # exceed a Kaggle/Colab session's RAM at the full ~99,886-patient scale). Lower this
    # if you still see an out-of-memory restart during Part 3; raise it for speed if you
    # have RAM to spare.
    "PARSE_CHUNK_SIZE": 2000,   # v2: 5000 held a whole 4k-patient sample in RAM at once

    # --- memory/compute lever, off by default -- see Part 3's "note on memory strategy" ---
    # If set, caps the number of REAL SURVIVED patients in the TRAINING split only (never
    # died patients, never val/test) to this count, via a department-stratified sample so
    # the department mix of the training pool is preserved. Try TIME_WINDOW='pre_op' and a
    # MAX_SUBJECTS_PER_CLASS smoke test first -- this is a last resort, not a first move.
    "DOWNSAMPLE_TRAIN_SURVIVED_TO": None,

    # --- §1.6.3 time window ---
    "TIME_WINDOW": "pre_op",              # 'pre_op' | 'peri_op'
    "PRE_OP_DAYS": 5,                     # matches the source repo's existing window

    # --- §1.3 missing data ---
    "IMPUTATION_STRATEGY": "decision_tree",  # 'decision_tree' | 'median' | 'interpolate' | 'knn'
    "KNN_NEIGHBORS": 5,

    # --- §1.4 / Part C class imbalance (updated to the agreed combined pipeline) ---
    "SAMPLING_STRATEGY": "grouped_smotenc_tomek",
    # 'grouped_smotenc_tomek' (new default, Part C §6) | 'class_weight' | 'smote' | 'adasyn'
    # | 'random_oversample' | 'random_undersample' | 'none'
    "SMOTE_TARGET_RATIO": 0.10,     # positive:negative target, e.g. 0.10 = 1:10 (Part C's conservative pick)
    "SMOTE_MIN_STRATUM_MINORITY": 3,   # skip a department x ASA stratum with fewer real positives than this
    # v3: cap synthetic deaths per stratum at this multiple of its REAL deaths. Uncapped, a
    # 1:10 target made ~85 synthetic copies from 3 real deaths in low-risk strata at full scale.
    "SMOTE_MAX_AMPLIFICATION": 5,      # None = uncapped (v2 behaviour)
    # v3: balanced batches -- at ~1:200, ~29% of 256-patient batches contain no death at all.
    "BALANCED_SAMPLER": True,
    "SAMPLER_TARGET_POSITIVE_RATE": 0.10,
    # Part C §6's clinical-neighbourhood grouping; v4 adds emop, so a synthetic death is only
    # ever blended from patients with the same department, ASA AND emergency/scheduled status.
    "SMOTE_STRATA_COLS": ["department", "asa", "emop"],
    "USE_SEQUENCE_AUGMENTATION": True,      # Part C §7: jitter + time-mask for REAL minority training patients
    "SEQUENCE_AUGMENTATION_COPIES": 2,      # extra augmented copies per real positive training patient
    "JITTER_SIGMA": 0.05,                   # on the standardized (z-scored) scale, per Part C §7
    # v2: never jitter discrete/ordinal/binary items -- GCS 4 -> 3.98 or a ventilator flag of
    # 0.97 is not a real reading (caught by the NEWS2 before/after check, Part 9.1b).
    "JITTER_EXCLUDE_ITEMS": ["gcs_e", "gcs_m", "gcs_v", "crrt", "iabp", "vent", "ecmo"],
    "USE_FOCAL_LOSS": False,
    "FOCAL_GAMMA": 2.0,
    # v2: raise loudly if SMOTENC ERRORS on a stratum (vs. a legitimate "too few positives"
    # skip). The original notebook caught both with one bare `except ValueError`, which is
    # how SMOTENC silently produced zero synthetic patients on every run.
    "STRICT_SAMPLING": True,
    # v2: what time series a SMOTENC synthetic patient gets. SMOTENC only synthesises the
    # STATIC vector. 'empty' (the original behaviour) gives every synthetic -- all labelled
    # "died" -- a completely unobserved time series, which teaches the model the shortcut
    # "no data at all => died". 'nearest_real_donor' borrows the jittered series of the
    # nearest real died patient in the same training set instead.
    "SYNTHETIC_TS_MODE": "nearest_real_donor",   # 'nearest_real_donor' | 'empty'

    # --- v2: NEWS2 (Royal College of Physicians 2017), computed from pre-op ward vitals ---
    # Added to the static vector ONLY if the scoring self-test passes (Part 6.16).
    "INCLUDE_NEWS2": True,
    "NEWS2_BUCKET_MINUTES": 60,          # vitals charted within the same hour = one observation set
    "NEWS2_CARRY_FORWARD_MINUTES": 240,  # bounded LOCF for a parameter missing from a set
    "NEWS2_MIN_CORE_VITALS": 3,          # of RR/SpO2/SBP/HR/temp -- fewer => no score (not a falsely low one)
    "NEWS2_ASSUME_AIR_IF_MISSING": True, # no FiO2/vent charted => room air (flagged approximation)
    "NEWS2_ASSUME_ALERT_IF_MISSING": True,

    # --- §1.5 architecture ---
    # (a flat-vs-system-split ablation flag was considered here and dropped rather than
    # shipped unwired -- see §1.1's note. The organ-system split is not optional in this
    # notebook's architecture.)
    "FUSION_STRATEGY": "nam",              # 'nam' | 'concat'  ('gated' is superseded by the
                                           # arbitration layer below, which keeps NAM additivity)
    # v2: encoder capacity. The v1 model was 66k parameters in TOTAL, and only ~17k of that
    # was the six system encoders (~3k each) -- most of it was the reconstruction heads
    # and the static MLP. 'medium' is ~9x the encoder capacity; 'small' reproduces v1.
    "ENCODER_SIZE": "medium",              # 'small' (v1) | 'medium' | 'large' -- sets the 4 keys below
    "EMBED_DIM": None, "TRANSFORMER_HEADS": None, "TRANSFORMER_LAYERS": None, "FF_MULTIPLIER": None,
    "DROPOUT": 0.1,

    # --- v3: the eight organ systems (renal, cardiovascular, respiratory, metabolic/hepatic,
    # haematology, neurological, GI, MSK) are all built the same way and run through ONE
    # shared encoder body (per-system input adapters + a system tag). False = a separate
    # body per system (the v2 behaviour, ~8x more encoder parameters) -- for ablation.
    "SHARE_ENCODER": True,
    "SYSTEM_DROPOUT": 0.1,            # fine-tuning: hide a whole system for 10% of patients per step

    # --- v3: whole-patient layer -- reads ALL features, gently re-scales each system's
    # summary (scale-only bottleneck, 0.5x-1.5x) and adds its own whole-patient term.
    "USE_WHOLE_PATIENT_LAYER": True,

    # --- inter-system coupling layer -- learned, nothing hand-defined ---
    "COUPLING_MODE": "learned",       # 'learned' | 'none' (ablation baseline)
    "COUPLING_HEADS": 2,
    "COUPLING_PRIOR_LINKS": [],       # left empty by design: every link must be learned from data
    "COUPLING_PRIOR_STRENGTH": 1.0,
    "COUPLING_GATE_INIT": 0.1,
    # v3: masked-system pre-training -- hide one system per patient and predict it from the
    # other seven through the coupling layer. Label-free, so it learns from EVERY patient.
    "MASKED_SYSTEM_WEIGHT": 1.0,      # 0 = v2 behaviour (coupling learns from deaths only)

    # --- arbitration layer -- how much "say" each organ system's answer gets per patient ---
    "USE_ARBITRATION_LAYER": True,
    "ARBITRATION_REG": 1e-3,

    # --- v3: learn, don't memorise ---
    "WEIGHT_DECAY": 1e-2,             # AdamW
    "FINETUNE_ENCODER_LR_MULT": 0.1,  # fine-tuning moves the pre-trained encoder 10x more gently
    "N_ENSEMBLE": 3,                  # Part 11.4b: models trained from different seeds, averaged
    "RUN_LEARNING_CURVE": True,       # Part 11.4c: retrain on 25% / 50% / 100% of training patients
    "RUN_GBM_BASELINE": True,         # Part 11.1d: gradient boosting next to logistic regression
    "N_BOOTSTRAP": 1000,              # confidence intervals for AUROC / AUPRC
    "TRAIN_METRIC_SUBSAMPLE": 5000,   # real training patients scored each epoch (train vs val gap)

    # --- §1.2 ICD-10 ---
    "USE_ICD10_EMBEDDING": False,
    "INCLUDE_HFRS": True,
    "HFRS_LOOKBACK_YEARS": 2,              # published Gilbert et al. window, age 75+
    "INCLUDE_STATIC_ASA": True,

    # --- training ---
    "BATCH_SIZE": 256,        # raised again from 128 -- at 10,469 patients, 128 still means
                              # ~4,400 training iterations total, risking most of a 20-minute
                              # budget on training alone. 256 roughly halves that with no
                              # accuracy cost for a model this small (see the run-specs doc for
                              # the measured basis). Lower toward 8-16 only for the tiny
                              # 30-patient dev subset.
    "EPOCHS_PRETRAIN": 30,     # upper CAP -- early stopping (below) will normally cut this short
    "EPOCHS_FINETUNE": 60,     # upper CAP -- early stopping (below) will normally cut this short
    "EARLY_STOPPING_PATIENCE": 8,   # stop a phase if its tracked metric hasn't improved for this
                                     # many epochs -- protects both the time budget (don't burn the
                                     # full epoch cap when it's not helping) and quality (still lets
                                     # training run as long as it's genuinely improving)
    "LR": 1e-3,
    # v2: select the best fine-tuning epoch on validation AUPRC (the primary metric at
    # ~0.5-4% prevalence) rather than AUROC.
    "MODEL_SELECTION_METRIC": "auprc",   # 'auprc' | 'auroc'
    "VAL_FRACTION": 0.2,
    "TEST_FRACTION": 0.2,
    "TARGET_SEQ_LEN": 24,      # number of time points each system's series is resampled/padded to

    # --- checkpointing (matters most on Colab -- free-tier sessions can disconnect for
    # reasons unrelated to memory: idle timeout, daily usage caps) ---
    "CHECKPOINT_DIR": None,          # auto-set below: Drive if on Colab+mounted, else local
    "CHECKPOINT_EVERY_N_EPOCHS": 5,

    # --- data-parsing cache (Part 3) -- avoids re-parsing tens of thousands of JSON
    # files on every session. First run parses once and saves Parquet here; every
    # later run with the same SUBJECTS_DIR/MAX_SUBJECTS_PER_CLASS/TIME_WINDOW loads
    # from cache instead, in seconds rather than minutes. ---
    "CACHE_DIR": None,               # auto-set below: Drive if on Colab+mounted, else local
    "USE_CACHE": True,

    # --- run-time budget controls (new) -- specifically for hitting a reliable 20-30
    # minute total runtime. Both of these sections are genuinely useful analysis, not
    # dead weight -- but they're also the two most expensive OPTIONAL steps in the
    # notebook (the regression/MICE imputation benchmark in particular), so they're
    # switchable independently of the core pipeline. See the "Run Specifications" doc
    # for measured per-section timing this decision is based on.
    "SKIP_IMPUTATION_ACCURACY_BENCHMARKS": True,   # Part 7.6/7.7 -- skips the held-out masking benchmarks (rule-based AND regression/MICE)
    "SKIP_FUSION_ABLATION": True,                  # Part 11.4 -- skips retraining a second (concat-fusion) model just for comparison
    # v2 evaluation additions (cheap -- no retraining of the DNN)
    "RECALIBRATE_ON_VAL": True,     # Platt-scale on the validation set (sampling + pos_weight inflate raw probabilities)
    "RUN_SIMPLE_BASELINE": True,    # logistic regression on the same features, same split

    # ======================= v4 additions (see CHANGES.md "v4") ======================= #
    # --- 1. saving: model, figures, per-epoch snapshots ---
    "OUTPUT_DIR": None,               # auto-set below: Drive on Colab, ./inspire_outputs elsewhere
    "SAVE_FIGURES": True,             # every plt.show() also writes a PNG to OUTPUT_DIR/figures
    "SAVE_FINAL_MODEL": True,         # final model + everything needed to reload it -> OUTPUT_DIR/models
    "SNAPSHOT_N_PATIENTS": 3000,      # validation patients whose embeddings are snapshotted (every val death always kept)
    "SNAPSHOT_FINETUNE_EPOCHS": [0, 1, 2, 3, 5, 8, 12, 20, 30, 45, 60, 80],   # + always: before training, after pre-training, last, best

    # --- 2. pictures of the embeddings ---
    "EMBEDDING_METHODS": ["pca", "tsne", "umap"],   # umap is skipped (and said so) if umap-learn is missing
    "TSNE_MAX_POINTS": 3000,          # t-SNE slows down quadratically; deaths are always kept
    "KNN_ENRICHMENT_K": 15,           # "what fraction of a patient's 15 nearest neighbours died?"

    # --- 4. SHAP-style attribution inside each organ system (expected gradients) ---
    "RUN_SHAP": True,
    "SHAP_N_PATIENTS": 600,           # test patients explained (every test death always kept)
    "SHAP_N_BACKGROUND": 200,         # "typical patients" the explanation is measured against
    "SHAP_N_SAMPLES": 32,             # Monte-Carlo samples per patient (more = smoother, slower)

    # --- 5. feature upgrades ---
    "USE_PROCEDURE_CODES": True,      # ICD-10-PCS body-system of the operation -> each organ system (GI and MSK get real content)
    "USE_ATC_SYSTEM_GROUPS": True,    # medication families (ATC) -> each organ system (A=gut, M=bones/muscles, ...)
    "USE_ANTYPE": True,               # anaesthesia type (one-hot), part of the surgical context
    "USE_SURGICAL_CONTEXT_TERM": True,      # emergency/scheduled, ASA, department, anaesthesia -> their OWN risk term
    "SURGICAL_CONTEXT_EXCLUSIVE": True,     # those columns are read ONLY by that term (clean "how much did surgery add")
    "RUN_SIZE_SWEEP": False,          # Part 13.9: retrain at several embedding sizes (the "score vs hidden size" figure)
    "SIZE_SWEEP": ["small", "medium", "large", "xlarge"],
    "SIZE_SWEEP_SEEDS": 2,

    # --- compute (big-data / Isambard-AI) ---
    "NUM_WORKERS": 0,                 # DataLoader worker processes (0 = main process; 8+ on a big node)
    "MIXED_PRECISION": False,         # bf16 autocast during training on GPUs that support it (H100/GH200/A100)
}

# Run profile presets (see RUN_PROFILE at the top). 'full' switches every analysis on and
# ignores run-time budgets -- intended for a machine with plenty of RAM and time.
PROFILE_PRESETS = {
    "subset": {},
    "full": {
        "MAX_SUBJECTS_PER_CLASS": None,          # the whole cohort
        "PARSE_CHUNK_SIZE": 5000,
        "SKIP_IMPUTATION_ACCURACY_BENCHMARKS": False,
        "SKIP_FUSION_ABLATION": False,
        "EPOCHS_PRETRAIN": 40, "EPOCHS_FINETUNE": 80, "EARLY_STOPPING_PATIENCE": 10,
        "N_ENSEMBLE": 5,
        "ENCODER_SIZE": "large",                 # v4: 64-number embeddings for the full cohort
        "TSNE_MAX_POINTS": 5000, "SNAPSHOT_N_PATIENTS": 5000, "SHAP_N_PATIENTS": 1000,
    },
    # v4: the whole cohort on a big GPU node (Isambard-AI: 4x NVIDIA GH200 per node, aarch64).
    # The model is small (well under 10M parameters even at 'xxlarge'), so ONE GH200 per run is
    # plenty; the node's value is running the full cohort, the full ensemble and the
    # embedding-size sweep in hours instead of days.
    "isambard": {
        "MAX_SUBJECTS_PER_CLASS": None,
        "PARSE_CHUNK_SIZE": 10000,
        "SKIP_IMPUTATION_ACCURACY_BENCHMARKS": False,
        "SKIP_FUSION_ABLATION": False,
        "EPOCHS_PRETRAIN": 60, "EPOCHS_FINETUNE": 100, "EARLY_STOPPING_PATIENCE": 12,
        "N_ENSEMBLE": 5, "N_BOOTSTRAP": 2000,
        "ENCODER_SIZE": "xlarge",                # 128-number embeddings, 8 heads, 3 layers
        "BATCH_SIZE": 1024, "NUM_WORKERS": 8, "MIXED_PRECISION": True,
        "TSNE_MAX_POINTS": 10000, "SNAPSHOT_N_PATIENTS": 10000, "SHAP_N_PATIENTS": 2000, "SHAP_N_BACKGROUND": 500,
        "SNAPSHOT_FINETUNE_EPOCHS": [0, 1, 2, 3, 5, 8, 12, 20, 30, 45, 60, 80, 100],
        "RUN_SIZE_SWEEP": True, "SIZE_SWEEP": ["small", "medium", "large", "xlarge", "xxlarge"], "SIZE_SWEEP_SEEDS": 3,
    },
}
CONFIG.update(PROFILE_PRESETS[RUN_PROFILE])

# Optional overrides without editing this cell -- used by src-dnn/run_pipeline.py --config,
# and handy on Kaggle for quick variants: set INSPIRE_CONFIG_OVERRIDES='{"ENCODER_SIZE": "small"}'.
_overrides = json.loads(os.environ.get("INSPIRE_CONFIG_OVERRIDES", "{}") or "{}")
_unknown = sorted(set(_overrides) - set(CONFIG))
assert not _unknown, f"Unknown CONFIG keys in INSPIRE_CONFIG_OVERRIDES: {_unknown}"
CONFIG.update(_overrides)
if _overrides:
    print(f"CONFIG overrides applied: {_overrides}")

# Resolve the encoder-size preset (explicit values win over the preset).
_ENCODER_PRESETS = {"small": (16, 2, 1, 2), "medium": (32, 4, 2, 4), "large": (64, 4, 2, 4),
                    "xlarge": (128, 8, 3, 4), "xxlarge": (256, 8, 4, 4)}   # (embed, heads, layers, ff) -- same as layers.py
for _key, _val in zip(["EMBED_DIM", "TRANSFORMER_HEADS", "TRANSFORMER_LAYERS", "FF_MULTIPLIER"],
                      _ENCODER_PRESETS[CONFIG["ENCODER_SIZE"]]):
    if CONFIG[_key] is None:
        CONFIG[_key] = _val
assert CONFIG["COUPLING_MODE"] in ("learned", "none")
assert CONFIG["FUSION_STRATEGY"] in ("nam", "concat")
print(f"Architecture: ENCODER_SIZE={CONFIG['ENCODER_SIZE']!r} (embed={CONFIG['EMBED_DIM']}, "
      f"heads={CONFIG['TRANSFORMER_HEADS']}, layers={CONFIG['TRANSFORMER_LAYERS']}), "
      f"COUPLING_MODE={CONFIG['COUPLING_MODE']!r}, ARBITRATION={CONFIG['USE_ARBITRATION_LAYER']}, "
      f"WHOLE_PATIENT={CONFIG['USE_WHOLE_PATIENT_LAYER']}, SHARED_ENCODER={CONFIG['SHARE_ENCODER']}, "
      f"NEWS2={CONFIG['INCLUDE_NEWS2']}, profile={RUN_PROFILE!r}, N_ENSEMBLE={CONFIG['N_ENSEMBLE']}")

# ---------------------------------------------------------------------------
# Environment detection: Kaggle vs. Colab vs. local. Matters for both data-path
# auto-detection and where checkpoints should live (Colab's local disk is ephemeral --
# a checkpoint saved there is lost on disconnect just like everything else).
# ---------------------------------------------------------------------------
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.isdir("/kaggle/input")

if IN_COLAB:
    try:
        from google.colab import drive
        if not os.path.isdir("/content/drive/MyDrive"):
            print("Colab detected -- mounting Google Drive (needed so your data/checkpoints "
                  "survive a session disconnect; a browser auth prompt may appear)...")
            drive.mount("/content/drive")
        else:
            print("Colab detected -- Google Drive already mounted.")
    except Exception as e:
        print(f"Colab detected but Drive mount failed or was skipped ({e}). "
              f"Data auto-detection and checkpointing will fall back to local (ephemeral) storage.")

# NOTE: checks IN_COLAB first -- the "google.colab" import check is a definitive signal,
# whereas an unrelated leftover "/kaggle/input" directory (e.g. from a prior session state)
# is not, and previously caused this to misreport "Kaggle" on an actual Colab run.
print(f"Environment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'local/other'}")

# ---------------------------------------------------------------------------
# Auto-detect data location: Kaggle input dirs, Colab Drive paths, then local upload paths.
# ---------------------------------------------------------------------------
def _find_subjects_dir():
    candidates = []
    # Fast, exact-path check first -- your data is at this known Drive location, so this
    # skips the (slower, walk-based) broader search below entirely when it matches.
    _known_exact_paths = [
        "/content/drive/MyDrive/subjects",
    ]
    for p in _known_exact_paths:
        if os.path.isdir(p) and os.path.isdir(os.path.join(p, "died")) and os.path.isdir(os.path.join(p, "survived")):
            return p
    if IN_KAGGLE:
        # Depth-capped AND stops at the first match -- without both of these, this can hang
        # for a very long time on a dataset with many files: os.walk with no depth limit
        # will also descend INTO a matched died/survived folder to enumerate its (possibly
        # thousands of) files, and Kaggle's dataset mount can have real per-file latency on
        # first access. Neither is needed just to confirm the folder exists.
        for root, dirs, files in os.walk("/kaggle/input"):
            depth = root[len("/kaggle/input"):].count(os.sep)
            if depth > 4:
                dirs[:] = []
                continue
            if "died" in dirs and "survived" in dirs:
                candidates.append(root)
                dirs[:] = []   # don't descend into died/survived's own contents
                break          # found it -- stop searching entirely
    if IN_COLAB:
        # Common places people drop an uploaded/extracted dataset on Drive -- searched
        # shallowly (max depth ~4) since walking all of MyDrive can be slow if it's large.
        drive_roots = ["/content/drive/MyDrive", "/content/drive/MyDrive/INSPIRE",
                       "/content/drive/MyDrive/inspire", "/content/drive/MyDrive/data",
                       "/content/drive/MyDrive/subjects", "/content"]
        for base in drive_roots:
            if not os.path.isdir(base):
                continue
            for root, dirs, files in os.walk(base):
                depth = root[len(base):].count(os.sep)
                if depth > 4:
                    dirs[:] = []   # don't descend further from here
                    continue
                if "died" in dirs and "survived" in dirs:
                    candidates.append(root)
    for p in ["/mnt/user-data/uploads/inspire_subjects_small",
              "/home/claude/work/inspire_subjects_small/inspire_subjects_small",
              "./inspire_subjects_small", "./subjects"]:
        if os.path.isdir(p) and os.path.isdir(os.path.join(p, "died")):
            candidates.append(p)
    return candidates[0] if candidates else None

def _find_codes_dir():
    if IN_KAGGLE:
        for root, dirs, files in os.walk("/kaggle/input"):
            depth = root[len("/kaggle/input"):].count(os.sep)
            if depth > 4:
                dirs[:] = []
                continue
            if "icd10.json.gz" in files or any(f.startswith("WHO_ATC") for f in files):
                return root
    if IN_COLAB and os.path.isdir("/content/drive/MyDrive"):
        for root, dirs, files in os.walk("/content/drive/MyDrive"):
            depth = root[len("/content/drive/MyDrive"):].count(os.sep)
            if depth > 4:
                dirs[:] = []
                continue
            if "icd10.json.gz" in files or any(f.startswith("WHO_ATC") for f in files):
                return root
    for p in ["/home/claude/work/inspire-analysis/inspire-analysis-thrisha-main/codes", "./codes"]:
        if os.path.isdir(p):
            return p
    return None

if CONFIG["SUBJECTS_DIR"] is None:
    CONFIG["SUBJECTS_DIR"] = _find_subjects_dir()
if CONFIG["CODES_DIR"] is None:
    CONFIG["CODES_DIR"] = _find_codes_dir()
if CONFIG["CHECKPOINT_DIR"] is None:
    if IN_COLAB and os.path.isdir("/content/drive/MyDrive"):
        CONFIG["CHECKPOINT_DIR"] = "/content/drive/MyDrive/inspire_checkpoints"
    else:
        CONFIG["CHECKPOINT_DIR"] = "./inspire_checkpoints"   # Kaggle/local: ephemeral, but Kaggle sessions are less disconnect-prone
os.makedirs(CONFIG["CHECKPOINT_DIR"], exist_ok=True)
if CONFIG["CACHE_DIR"] is None:
    if IN_COLAB and os.path.isdir("/content/drive/MyDrive"):
        CONFIG["CACHE_DIR"] = "/content/drive/MyDrive/inspire_parsed_cache"
    else:
        CONFIG["CACHE_DIR"] = "./inspire_parsed_cache"
os.makedirs(CONFIG["CACHE_DIR"], exist_ok=True)
if CONFIG["OUTPUT_DIR"] is None:
    if IN_COLAB and os.path.isdir("/content/drive/MyDrive"):
        CONFIG["OUTPUT_DIR"] = "/content/drive/MyDrive/inspire_outputs"
    else:
        CONFIG["OUTPUT_DIR"] = "./inspire_outputs"   # on Kaggle this is /kaggle/working -> downloadable
FIGURE_DIR = os.environ.get("INSPIRE_FIG_DIR") or os.path.join(CONFIG["OUTPUT_DIR"], "figures")
MODEL_DIR = os.path.join(CONFIG["OUTPUT_DIR"], "models")
SNAPSHOT_DIR = os.path.join(CONFIG["OUTPUT_DIR"], "snapshots")
TABLE_DIR = os.path.join(CONFIG["OUTPUT_DIR"], "tables")
for _d in (FIGURE_DIR, MODEL_DIR, SNAPSHOT_DIR, TABLE_DIR):
    os.makedirs(_d, exist_ok=True)

print("SUBJECTS_DIR   ->", CONFIG["SUBJECTS_DIR"])
print("CODES_DIR      ->", CONFIG["CODES_DIR"], "(optional — used only for richer ICD-10/ATC descriptions)")
print("CHECKPOINT_DIR ->", CONFIG["CHECKPOINT_DIR"],
      "(on Drive -- survives a Colab disconnect)" if CONFIG["CHECKPOINT_DIR"].startswith("/content/drive") else
      "(local/ephemeral -- will NOT survive a Colab disconnect; mount Drive if you're on Colab)")
print("OUTPUT_DIR     ->", CONFIG["OUTPUT_DIR"], "(figures/, models/, snapshots/, tables/)")
print("CACHE_DIR      ->", CONFIG["CACHE_DIR"],
      "(on Drive -- parsed data survives a Colab disconnect, only parsed once ever)" if CONFIG["CACHE_DIR"].startswith("/content/drive") else
      "(local/ephemeral -- re-parses every fresh session)")
assert CONFIG["SUBJECTS_DIR"] is not None, (
    "Could not auto-find a subjects folder (must contain died/ and survived/ subfolders of JSON). "
    "Set CONFIG['SUBJECTS_DIR'] manually -- e.g. on Colab, '/content/drive/MyDrive/<wherever you put it>'."
)

### Run-time tracking

Every major Part below records how long it actually took, on *your* hardware and *your*
config -- not an estimate. Part 12 prints the full breakdown at the end, so every run
tells you directly whether it hit the 20-30 minute target, and exactly which Part to
adjust first if it didn't.

In [ ]:
RUN_TIMER = {"checkpoints": [("notebook start", time.time())]}

def record_checkpoint(label):
    RUN_TIMER["checkpoints"].append((label, time.time()))

def print_timing_summary():
    cps = RUN_TIMER["checkpoints"]
    print(f"{'Part':35s} {'elapsed':>10s}")
    print("-" * 47)
    for i in range(1, len(cps)):
        label, t = cps[i]
        prev_t = cps[i - 1][1]
        print(f"{label:35s} {t - prev_t:8.1f}s")
    total = cps[-1][1] - cps[0][1]
    print("-" * 47)
    print(f"{'TOTAL':35s} {total:8.1f}s  ({total/60:.1f} min)")

print("Run-time tracking initialised.")

### Saving every figure (v4)

Every `plt.show()` below also writes the figure as a PNG into `OUTPUT_DIR/figures/`,
numbered in order and named after its title — so a full run leaves a folder of every
plot, ready for the paper or a supervisor meeting. Headless runs (`run_pipeline.py`)
save and close instead of displaying.

In [ ]:
import re as _re
HEADLESS = bool(os.environ.get("INSPIRE_HEADLESS"))
SAVED_FIGURES = []

def _figure_title(fig):
    if getattr(fig, "_suptitle", None) is not None and fig._suptitle.get_text():
        return fig._suptitle.get_text()
    for ax in fig.axes:
        if ax.get_title():
            return ax.get_title()
    return "figure"

if (CONFIG["SAVE_FIGURES"] or HEADLESS) and not getattr(plt.show, "_inspire_saver", False):
    _ORIGINAL_SHOW = plt.show

    def _show_and_save(*args, **kwargs):
        for num in (plt.get_fignums() if CONFIG["SAVE_FIGURES"] else []):
            fig = plt.figure(num)
            slug = _re.sub(r"[^a-z0-9]+", "_", _figure_title(fig).lower()).strip("_")[:60] or "figure"
            path = os.path.join(FIGURE_DIR, f"fig_{len(SAVED_FIGURES) + 1:03d}_{slug}.png")
            try:
                fig.savefig(path, dpi=150, bbox_inches="tight")
                SAVED_FIGURES.append(path)
            except Exception as e:   # a figure that cannot be saved must never stop the run
                print(f"(could not save figure {slug}: {e})")
        if HEADLESS:
            plt.close("all")
        else:
            _ORIGINAL_SHOW(*args, **kwargs)
    _show_and_save._inspire_saver = True
    plt.show = _show_and_save
    print(f"Figures will be saved to {FIGURE_DIR}")

# Mixed precision (v4): bf16 autocast for training on GPUs that support it. Off on CPU.
USE_AMP = bool(CONFIG["MIXED_PRECISION"] and DEVICE.type == "cuda" and torch.cuda.is_bf16_supported())
print(f"Mixed precision (bf16) during training: {USE_AMP}")

# Part 3 -- Data loading

Each patient is one JSON file (in `died/` or `survived/`) with six lists: `labs`, `vitals`
(intra-op), `ward_vitals`, `operations`, `diagnoses`, `medications`. This section parses
every file **exactly once** in a single streaming pass, building all six downstream tables
at the same time -- rather than parsing all patients into one big in-memory dict first and
then re-scanning that dict once per table, which is wasted work and wasted memory at full
scale. This pattern (one file open, one parse, extract everything needed, discard, move on)
matches the proven-fast approach already used in this project's other Colab notebook on the
full 99,886-patient cohort.

**Note on folder labels (§1.6.3):** the `died`/`survived` folder is used only to know which
files to open -- the actual mortality label used for training is **recomputed from
`operations`** in §5.2, not trusted from the folder name, for the reason explained there.

**Caching (new).** The parse above is the single most expensive CPU step in this notebook
at full scale -- tens of thousands of small JSON files, each opened and parsed individually.
The first run does that parse once, then saves the resulting tables as Parquet to
`CONFIG['CACHE_DIR']`. Every later run (this session or a future one, if `CACHE_DIR` is on
a mounted Drive) checks for that cache first and, if found, loads it directly -- seconds
instead of minutes-to-tens-of-minutes. The cache is keyed to `SUBJECTS_DIR`,
`MAX_SUBJECTS_PER_CLASS`, and `TIME_WINDOW` together, so a smoke-test run and a full run (or
a pre-op vs. peri-op run) never collide or silently load the wrong cache.

In [ ]:
def _safe_float(x):
    try:
        if x is None or x == "":
            return None
        return float(x)
    except (TypeError, ValueError):
        return None

def _safe_int(x):
    try:
        if x is None or x == "":
            return None
        return int(float(x))
    except (TypeError, ValueError):
        return None

import hashlib

def _cache_key():
    src = repr((CONFIG["SUBJECTS_DIR"], CONFIG["MAX_SUBJECTS_PER_CLASS"], CONFIG["TIME_WINDOW"]))
    return hashlib.sha256(src.encode()).hexdigest()[:12]

CACHE_KEY = _cache_key()
CACHE_SUBDIR = os.path.join(CONFIG["CACHE_DIR"], f"parsed_{CACHE_KEY}")
_CACHE_TABLES = ["labs", "vitals", "ward_vitals", "operations", "diagnoses", "medications", "folder_label"]

def _cache_complete(subdir):
    return os.path.isdir(subdir) and all(
        os.path.exists(os.path.join(subdir, f"{name}.parquet")) for name in _CACHE_TABLES
    )

def _load_from_cache(subdir):
    dfs = {name: pd.read_parquet(os.path.join(subdir, f"{name}.parquet")) for name in _CACHE_TABLES}
    folder_label = dict(zip(dfs["folder_label"]["subject_id"], dfs["folder_label"]["folder"]))
    return dfs, folder_label

# NOTE: no separate _save_to_cache function -- stream_parse_subjects (below) writes each
# chunk straight to CACHE_SUBDIR as it parses, which is the whole point of chunking (never
# holding the full cohort in memory to save "all at once" afterward). USE_CACHE=False only
# means "don't trust a pre-existing cache as valid input" on a future run -- it does not
# turn off the chunked writes themselves, since those are what keep memory bounded during
# parsing regardless of whether a cache is wanted afterward.

In [ ]:
def stream_parse_subjects(subjects_dir, max_per_class, time_window, cache_subdir, chunk_size, seed=SEED):
    # Parses in CHUNKS, flushed to Parquet incrementally, instead of holding every row of
    # every table as a Python dict for the WHOLE cohort in memory at once. Two separate
    # memory wins stack here: (1) within each chunk, values are accumulated in parallel
    # per-column lists (ints/floats/strings), not one dict object per row -- roughly 6x
    # lighter per row, measured directly (a dict-of-4-fields costs ~192 bytes/row in
    # CPython; four parallel lists holding the same data cost ~32 bytes/row); (2) only
    # ever holding `chunk_size` patients' worth of rows at once, then writing to disk and
    # freeing them, means peak memory no longer scales with total cohort size at all --
    # it's bounded by chunk_size regardless of whether the cohort is 30 patients or
    # 99,886. Categorical dtypes are applied ONCE, after every chunk has been written and
    # is read back as a single table -- not per-chunk -- since a categorical column's
    # dictionary can legitimately differ chunk to chunk (different item_names observed in
    # different patients), which is safer to reconcile in one pass than to guess is
    # compatible mid-stream.
    import pyarrow as pa
    import pyarrow.parquet as pq

    rng = random.Random(seed)
    filepaths = []
    for folder in ["died", "survived"]:
        folder_path = os.path.join(subjects_dir, folder)
        if not os.path.isdir(folder_path):
            continue
        fps = sorted(glob.glob(os.path.join(folder_path, "*.json")))
        if max_per_class is not None and len(fps) > max_per_class:
            fps = rng.sample(fps, max_per_class)
            fps.sort()
        filepaths.extend((fp, folder) for fp in fps)

    print(f"{len(filepaths)} subject files to parse, in chunks of {chunk_size}...")
    load_vitals = (time_window == "peri_op")

    os.makedirs(cache_subdir, exist_ok=True)
    table_names = ["labs", "vitals", "ward_vitals", "operations", "diagnoses", "medications"]
    writers = {name: None for name in table_names}   # opened lazily on each table's first non-empty chunk
    folder_label = {}

    # Fixed, explicit column lists for every table -- rows are built with .get(), never a
    # raw dict(record) copy. This guarantees every row in every chunk has EXACTLY the same
    # set of columns, so pyarrow's schema can never drift between chunks because one
    # chunk's records happened to include/omit a key that another chunk's didn't (a real
    # risk at ~99,886 patients that a 30-patient dev sample is too uniform to expose).
    OPERATIONS_COLS = ["op_id", "subject_id", "hadm_id", "case_id", "opdate", "age", "sex",
                        "weight", "height", "race", "asa", "emop", "department", "antype",
                        "icd10_pcs", "orin_time", "orout_time", "opstart_time", "opend_time",
                        "admission_time", "discharge_time", "anstart_time", "anend_time",
                        "cpbon_time", "cpboff_time", "icuin_time", "icuout_time",
                        "inhosp_death_time", "allcause_death_time"]
    MEDICATIONS_COLS = ["subject_id", "chart_time", "drug_name", "route", "drug_name2",
                         "drug_name3", "atc_code", "atc_code2", "atc_code3"]
    OPERATIONS_STRING_COLS = ["op_id", "subject_id", "hadm_id", "case_id", "opdate", "sex",
                               "race", "department", "antype", "icd10_pcs", "cpbon_time", "cpboff_time"]
    MEDICATIONS_STRING_COLS = ["subject_id", "drug_name", "route", "drug_name2", "drug_name3",
                                "atc_code", "atc_code2", "atc_code3"]

    def _flush_chunk(name, rows, int_cols=("chart_time",), float_cols=("value",), string_cols=("subject_id",)):
        # rows: list of dicts for this chunk only (small -- at most chunk_size patients'
        # worth) -- the one place per chunk we pay the dict-object cost, not for the whole run.
        if not rows:
            return
        df = pd.DataFrame(rows)
        # Nullable "Int32" (capital I, pandas' extension dtype), not plain int32/int64: a
        # plain numpy int column cannot hold NaN at all, so pandas silently upcasts a
        # column to float64 in any chunk that happens to contain one missing value, while
        # a chunk with no gaps stays int64 -- two chunks then disagree on dtype, and
        # ParquetWriter correctly refuses to append a table whose schema doesn't match the
        # one it was opened with. Nullable Int32 holds missing values natively, so every
        # chunk gets the identical dtype regardless of whether that particular chunk has
        # any gaps.
        for c in int_cols:
            if c in df.columns:
                df[c] = pd.array(df[c], dtype="Int32")
        for c in float_cols:
            if c in df.columns:
                df[c] = df[c].astype("float32")
        # Same idea for string columns: pandas' nullable "string" dtype (not plain object)
        # so a chunk where a column happens to be entirely missing/empty still gets typed
        # as string, not inferred as pyarrow's separate "null" type -- which would also
        # collide with a later chunk's genuinely-string-typed version of the same column.
        for c in string_cols:
            if c in df.columns:
                df[c] = df[c].astype("string")
        table = pa.Table.from_pandas(df, preserve_index=False)
        if writers[name] is None:
            writers[name] = pq.ParquetWriter(os.path.join(cache_subdir, f"{name}.parquet"), table.schema)
        writers[name].write_table(table)

    t0 = time.time()
    n_in_chunk = 0
    labs_rows, vitals_rows, ward_vitals_rows = [], [], []
    op_rows, dx_rows, med_rows = [], [], []

    for i, (fp, folder) in enumerate(filepaths):
        with open(fp) as f:
            d = json.load(f)
        sid = str(d["subject_id"])
        folder_label[sid] = folder

        for r in d.get("labs", []):
            labs_rows.append({
                "subject_id": sid,
                "chart_time": _safe_int(r.get("chart_time")),
                "item_name": r.get("item_name"),
                "value": _safe_float(r.get("value")),
            })

        if load_vitals:
            for r in d.get("vitals", []):
                vitals_rows.append({
                    "subject_id": sid,
                    "op_id": r.get("op_id"),
                    "chart_time": _safe_int(r.get("chart_time")),
                    "item_name": r.get("item_name"),
                    "value": _safe_float(r.get("value")),
                })

        for r in d.get("ward_vitals", []):
            ward_vitals_rows.append({
                "subject_id": sid,
                "chart_time": _safe_int(r.get("chart_time")),
                "item_name": r.get("item_name"),
                "value": _safe_float(r.get("value")),
            })

        for op in d.get("operations", []):
            row = {col: op.get(col) for col in OPERATIONS_COLS}   # fixed schema, see note above
            row["subject_id"] = sid
            for tcol in ["orin_time", "orout_time", "opstart_time", "opend_time",
                         "admission_time", "discharge_time", "anstart_time", "anend_time",
                         "icuin_time", "icuout_time"]:
                row[tcol] = _safe_int(row.get(tcol))
            for icol in ["age", "asa", "emop"]:
                row[icol] = _safe_int(row.get(icol))
            for fcol in ["weight", "height"]:
                row[fcol] = _safe_float(row.get(fcol))
            row["inhosp_death_time"] = _safe_int(row.get("inhosp_death_time"))
            row["allcause_death_time"] = _safe_int(row.get("allcause_death_time"))
            op_rows.append(row)

        for dx in d.get("diagnoses", []):
            dx_rows.append({"subject_id": sid, "chart_time": _safe_int(dx.get("chart_time")),
                             "icd10_cm": dx.get("icd10_cm")})

        for m in d.get("medications", []):
            row = {col: m.get(col) for col in MEDICATIONS_COLS}   # fixed schema, see note above
            row["subject_id"] = sid
            row["chart_time"] = _safe_int(m.get("chart_time"))
            med_rows.append(row)

        del d
        n_in_chunk += 1

        if n_in_chunk >= chunk_size or i == len(filepaths) - 1:
            _flush_chunk("labs", labs_rows, string_cols=("subject_id", "item_name"))
            _flush_chunk("vitals", vitals_rows, string_cols=("subject_id", "item_name", "op_id"))
            _flush_chunk("ward_vitals", ward_vitals_rows, string_cols=("subject_id", "item_name"))
            _flush_chunk("operations", op_rows,
                         int_cols=("orin_time", "orout_time", "opstart_time", "opend_time",
                                   "admission_time", "discharge_time", "anstart_time", "anend_time",
                                   "icuin_time", "icuout_time", "age", "asa", "emop",
                                   "inhosp_death_time", "allcause_death_time"),
                         float_cols=("weight", "height"),
                         string_cols=OPERATIONS_STRING_COLS)
            _flush_chunk("diagnoses", dx_rows, int_cols=("chart_time",), float_cols=(),
                         string_cols=("subject_id", "icd10_cm"))
            _flush_chunk("medications", med_rows, int_cols=("chart_time",), float_cols=(),
                         string_cols=MEDICATIONS_STRING_COLS)
            labs_rows, vitals_rows, ward_vitals_rows = [], [], []
            op_rows, dx_rows, med_rows = [], [], []
            n_in_chunk = 0
            print(f"  ...{i+1}/{len(filepaths)} ({time.time()-t0:.0f}s elapsed, chunk flushed to disk)")

    for w in writers.values():
        if w is not None:
            w.close()

    # If a table had zero rows in EVERY chunk (e.g. "vitals" whenever TIME_WINDOW=='pre_op'),
    # its writer was never opened and its Parquet file was never created -- write an empty
    # placeholder with the right columns so the cache-completeness check (which expects
    # every table's file to exist) doesn't wrongly conclude the cache is broken/partial on
    # every future run.
    _empty_schemas = {
        "labs": ["subject_id", "chart_time", "item_name", "value"],
        "vitals": ["subject_id", "op_id", "chart_time", "item_name", "value"],
        "ward_vitals": ["subject_id", "chart_time", "item_name", "value"],
        "operations": ["subject_id"],
        "diagnoses": ["subject_id", "chart_time", "icd10_cm"],
        "medications": ["subject_id", "chart_time"],
    }
    for name in table_names:
        if writers[name] is None:
            pd.DataFrame(columns=_empty_schemas[name]).to_parquet(os.path.join(cache_subdir, f"{name}.parquet"))

    vitals_note = "peri_op -- intra-op vitals included" if load_vitals else "pre_op -- intra-op vitals skipped entirely, not just unused"
    print(f"Parsed {len(filepaths)} subjects in {time.time()-t0:.1f}s (chunked streaming, {vitals_note})")

    # Read each table back as ONE table now that every chunk is on disk, and apply
    # categorical dtypes exactly once, on the complete data -- see docstring above for why
    # this is deferred rather than done per-chunk.
    cat_cols_by_table = {
        "labs": ["subject_id", "item_name"],
        "vitals": ["subject_id", "item_name", "op_id"],
        "ward_vitals": ["subject_id", "item_name"],
        "operations": [],
        "diagnoses": ["subject_id", "icd10_cm"],
        "medications": ["subject_id", "drug_name", "route", "atc_code", "drug_name2", "drug_name3", "atc_code2", "atc_code3"],
    }
    dfs = {}
    for name in table_names:
        path = os.path.join(cache_subdir, f"{name}.parquet")
        if os.path.exists(path):
            df = pd.read_parquet(path)
            for c in cat_cols_by_table[name]:
                if c in df.columns:
                    df[c] = df[c].astype("category")
            dfs[name] = df
        else:
            dfs[name] = pd.DataFrame(columns=["subject_id", "op_id", "chart_time", "item_name", "value"]) \
                        if name == "vitals" else pd.DataFrame()

    pd.DataFrame({"subject_id": list(folder_label.keys()), "folder": list(folder_label.values())}) \
        .to_parquet(os.path.join(cache_subdir, "folder_label.parquet"))

    return dfs, folder_label

In [ ]:
if CONFIG["USE_CACHE"] and _cache_complete(CACHE_SUBDIR):
    print(f"Found a matching cache at {CACHE_SUBDIR} -- loading from Parquet (fast path).")
    _t0 = time.time()
    DFS, FOLDER_LABEL = _load_from_cache(CACHE_SUBDIR)
    print(f"Loaded from cache in {time.time()-_t0:.1f}s.")
else:
    if CONFIG["USE_CACHE"]:
        print(f"No cache found at {CACHE_SUBDIR} -- parsing from scratch (this happens once; "
              f"chunks are written to disk as they're parsed, see Part 3's note on memory).")
    DFS, FOLDER_LABEL = stream_parse_subjects(
        CONFIG["SUBJECTS_DIR"], CONFIG["MAX_SUBJECTS_PER_CLASS"], CONFIG["TIME_WINDOW"],
        CACHE_SUBDIR, CONFIG["PARSE_CHUNK_SIZE"]
    )
    # No separate save step needed -- stream_parse_subjects already wrote every chunk to
    # CACHE_SUBDIR as it went, which is the whole point of chunking in the first place.

labs_df        = DFS["labs"]
vitals_df      = DFS["vitals"]
ward_vitals_df = DFS["ward_vitals"]
operations_df  = DFS["operations"]
diagnoses_df   = DFS["diagnoses"]
medications_df = DFS["medications"]

print(f"\nLoaded {len(FOLDER_LABEL)} subjects "
      f"({sum(v=='died' for v in FOLDER_LABEL.values())} in died/, "
      f"{sum(v=='survived' for v in FOLDER_LABEL.values())} in survived/ — folder-provided, unverified)")
if CONFIG["MAX_SUBJECTS_PER_CLASS"] is not None:
    _n_died_loaded = sum(v == "died" for v in FOLDER_LABEL.values())
    _n_survived_loaded = sum(v == "survived" for v in FOLDER_LABEL.values())
    print(f"\nDOWNSAMPLING REPORT (CONFIG['MAX_SUBJECTS_PER_CLASS']={CONFIG['MAX_SUBJECTS_PER_CLASS']}):")
    print(f"  died:     {_n_died_loaded} loaded -- ALL real deaths are always kept, this cap never touches them")
    print(f"  survived: {_n_survived_loaded} loaded (capped from the full cohort's ~99,417 survivors, "
          f"randomly sampled per-file before parsing -- see load_all_subjects)")
    print(f"  This is a real, intentional target run size, not a placeholder smoke test -- "
          f"set MAX_SUBJECTS_PER_CLASS=None only once you have the time budget for the true "
          f"full ~99,886-patient cohort (expect data loading alone to take several minutes more).")
if CONFIG["TIME_WINDOW"] != "peri_op":
    print("TIME_WINDOW='pre_op' -- intra-op vitals table NOT parsed at all (not just unused), "
          "saving both time and memory. Switch to 'peri_op' in Part 2 if you need it.")

print("\nlabs_df        ", labs_df.shape)
print("vitals_df      ", vitals_df.shape, " (intra-op)")
print("ward_vitals_df ", ward_vitals_df.shape)
print("operations_df  ", operations_df.shape)
print("diagnoses_df   ", diagnoses_df.shape)
print("medications_df ", medications_df.shape)

print("\nActual memory used by each table (post dtype-optimization) -- watch this number, "
      "not a guess extrapolated from a different-sized run:")
_total_mb = 0
for _name, _df in [("labs_df", labs_df), ("vitals_df", vitals_df), ("ward_vitals_df", ward_vitals_df),
                    ("operations_df", operations_df), ("diagnoses_df", diagnoses_df), ("medications_df", medications_df)]:
    _mb = _df.memory_usage(deep=True).sum() / 1e6
    _total_mb += _mb
    print(f"  {_name:16s} {_mb:8.1f} MB")
print(f"  {'TOTAL':16s} {_total_mb:8.1f} MB")
_n_loaded = max(len(FOLDER_LABEL), 1)
print(f"\nMedication administrations per loaded patient: {len(medications_df) / _n_loaded:.0f} "
      f"(the source repo's README gives ~99/patient for the full cohort). A run that keeps every "
      f"death but caps survivors is sicker and denser than the full cohort, so scaling this "
      f"run's memory linearly to ~99,886 patients will tend to OVERESTIMATE the real footprint.")

In [ ]:
# Sanity check: exactly one operation row expected per (subject, op_id); most patients
# have one operation in this dev subset, some have several (relevant to §6.10/§6.11).
ops_per_subject = operations_df.groupby("subject_id")["op_id"].nunique()
print(ops_per_subject.value_counts().sort_index().rename("n_subjects_with_this_many_ops"))
operations_df.head(3)

### A note on memory and speed strategy

If the memory printout above is tight, or a fresh (uncached) parse is taking a long time,
the order of things to try is:

1. **Confirm `TIME_WINDOW='pre_op'`** (the default) — this now skips *parsing* intra-op
   `vitals` entirely, not just using it, which saves both time and memory versus the
   earlier version of this notebook.
2. **Let the cache do its job.** The first parse of a given `SUBJECTS_DIR` is the expensive
   one; every subsequent run (this session or later, if `CACHE_DIR` is on Drive) should be
   loading Parquet in seconds. If it isn't, check `CONFIG['USE_CACHE']` is `True` and that
   `CACHE_DIR` actually points somewhere persistent.
3. **Run a `MAX_SUBJECTS_PER_CLASS` smoke test first** (e.g. 2,000–5,000) and look at the
   real memory printout, rather than extrapolating from this dev subset — which, per the
   note above, is likely denser-than-average and will overestimate the real number.
4. **Only if still constrained, downsample — but the training `survived` split only, never
   `died`, and never validation/test** (Part 8.4 implements this, off by default). Two
   real costs of downsampling `survived` patients that are worth knowing before you turn
   it on: it shrinks the unlabelled pool the Part 9.4 autoencoder pre-training specifically
   relies on (a design decision made explicitly *because* splitting into six systems
   already splits the scarce mortality labels six ways), and if it ever touched
   validation/test it would corrupt the true ~0.47% prevalence that makes AUPRC meaningful
   — which is why it's restricted to the training split only.
5. **If you do downsample, stratify by `department`, not by ICD-10 "organ system."**
   Department is a single, always-populated categorical field per patient — clean to
   stratify on. ICD-10 diagnoses are multi-label (a patient's diagnoses can span several
   organ-system chapters at once), so there's no single unambiguous rule for assigning one
   patient to one "organ-system stratum" without inventing a new, separate convention just
   for this — and department is already the stratification key `§8.2`'s grouped SMOTENC
   uses, so reusing it keeps one consistent clinical-neighbourhood definition through the
   whole pipeline instead of two different ones.

In [ ]:
record_checkpoint("Part 3 -- data loading")
print(f"Part 3 complete. Elapsed so far: {time.time() - RUN_TIMER['checkpoints'][0][1]:.1f}s")

# Part 4 — Item-name inventory, the 30-day mortality label, and the cohort table

Three things happen here, each directly implementing a decision from Part 1:

1. **§4.1** — list every distinct `item_name` in `labs` / `vitals` / `ward_vitals`. This is
   the programmatic check behind §1.2.3's claim that GI and MSK have no dedicated
   lab/vital panel — run it and see for yourself rather than taking the claim on faith.
2. **§4.2** — recompute the 30-day mortality label directly from `operations`, **not**
   from the `died`/`survived` folder name (§1.6.3's label-bug note).
3. **§4.3** — assemble one row per patient: label, static fields, multi-operation counts.

## 4.1 Item-name inventory — confirming §1.2.3 programmatically

In [ ]:
def item_name_inventory(*dfs_and_names):
    inv = {}
    for df, name in dfs_and_names:
        inv[name] = sorted(df["item_name"].dropna().unique().tolist()) if "item_name" in df.columns and len(df) else []
    return inv

INVENTORY = item_name_inventory((labs_df, "labs"), (vitals_df, "vitals (intra-op)"), (ward_vitals_df, "ward_vitals"))
for name, items in INVENTORY.items():
    print(f"{name}: {len(items)} distinct item_names")

# GI/MSK check: do any of these look like GI- or MSK-specific measurements?
gi_keywords = ["bowel", "stool", "gi_", "gastro", "bilirubin"]  # bilirubin included to show it's hepatic, see §1.2.3
msk_keywords = ["mobility", "joint", "rom", "muscle", "ortho"]
all_items = set(sum(INVENTORY.values(), []))
print("\nItems matching GI-ish keywords:", [i for i in all_items if any(k in i.lower() for k in gi_keywords)])
print("Items matching MSK-ish keywords:", [i for i in all_items if any(k in i.lower() for k in msk_keywords)])
print("\n-> Confirms §1.2.3: no dedicated GI or MSK signal in labs/vitals/ward_vitals.")
print("   GI and MSK must be built from ICD-10 diagnoses + department (done in §6.7-6.8).")

## 4.2 The 30-day mortality label — recomputed from `operations`, not the folder name

In [ ]:
MINUTES_PER_DAY = 24 * 60

def compute_labels(operations_df):
    # One row per subject. Mirrors subject.py's inhosp_death_30day() logic:
    # died = inhosp_death_time is set AND inhosp_death_time < orout_time(last op) + 30 days.
    # 'last op' = operation with the max orin_time for that subject (current repo convention;
    # see §1.6.3/CONFIG note below for why this is a genuine open question, not a settled one).
    rows = []
    for sid, g in operations_df.groupby("subject_id"):
        g = g.sort_values("orin_time")
        first_op = g.iloc[0]
        last_op = g.iloc[-1]
        n_ops = len(g)

        inhosp_death_time = first_op["inhosp_death_time"]   # same across all ops for a subject, per source repo's note
        allcause_death_time = first_op["allcause_death_time"]

        # NOTE: pandas stores our None sentinels as NaN once a column mixes numbers and
        # missing values, so `x is not None` silently fails here -- must use pd.notna().
        has_inhosp_death = pd.notna(inhosp_death_time)

        died_30day_from_last = False
        died_30day_from_first = False
        if has_inhosp_death:
            if pd.notna(last_op["orout_time"]):
                died_30day_from_last = inhosp_death_time < (last_op["orout_time"] + 30 * MINUTES_PER_DAY)
            if pd.notna(first_op["orout_time"]):
                died_30day_from_first = inhosp_death_time < (first_op["orout_time"] + 30 * MINUTES_PER_DAY)

        died_ever = has_inhosp_death

        rows.append({
            "subject_id": sid,
            "n_operations": n_ops,
            "age": last_op["age"],
            "sex": last_op["sex"],
            "asa": last_op["asa"],
            "emop": last_op["emop"],
            "department": last_op["department"],
            "antype": last_op.get("antype"),
            "weight": last_op["weight"],
            "height": last_op["height"],
            "op_id_last": last_op["op_id"],
            "orin_time_last": last_op["orin_time"],
            "orout_time_last": last_op["orout_time"],
            "admission_time_last": last_op["admission_time"],
            "discharge_time_last": last_op["discharge_time"],
            "died_ever": died_ever,                                   # NOT the training label - see §1.6.3
            "died_30day_from_last_op": died_30day_from_last,          # default training label (Path C = 'last operation', see §1.6.4/§11.7)
            "died_30day_from_first_op": died_30day_from_first,        # sensitivity-analysis alternative
        })
    return pd.DataFrame(rows)

cohort_df = compute_labels(operations_df)
cohort_df = cohort_df.merge(
    pd.Series(FOLDER_LABEL, name="folder_label").rename_axis("subject_id").reset_index(),
    on="subject_id", how="left"
)

# Cross-check against the folder label, exactly the check §1.6.3 says not to skip.
cohort_df["folder_says_died"] = cohort_df["folder_label"].eq("died")
mismatch = cohort_df[cohort_df["folder_says_died"] != cohort_df["died_30day_from_last_op"]]
print(f"Cohort: {len(cohort_df)} patients.")
print(f"  died_ever (all-cause, any time)      : {cohort_df['died_ever'].sum()}")
print(f"  died_30day_from_last_op (TRAINING LABEL) : {cohort_df['died_30day_from_last_op'].sum()}")
print(f"  died_30day_from_first_op (sensitivity)   : {cohort_df['died_30day_from_first_op'].sum()}")
print(f"  folder-label says 'died'             : {cohort_df['folder_says_died'].sum()}")
print(f"  Rows where folder label != recomputed 30-day label: {len(mismatch)}  "
      f"(non-zero here is expected and is exactly the §1.6.3 bug the recompute avoids inheriting)")
mismatch[["subject_id", "folder_says_died", "died_ever", "died_30day_from_last_op"]]

## 4.3 Multi-operation counts by department/system — feeds §6.10/§6.11

In [ ]:
# Per-subject operation history (all ops, in order) — used later for the "operations in the
# same area" feature (§6.10) and the post-cardiac-surgery exception flag (§6.11).
OPS_HISTORY = {}
for sid, g in operations_df.groupby("subject_id"):
    g = g.sort_values("orin_time")
    OPS_HISTORY[sid] = g.to_dict("records")

print("Departments observed:", sorted(operations_df["department"].dropna().unique().tolist()))
print("\nMortality (30-day, last-op label) by department:")
tmp = cohort_df.groupby("department")["died_30day_from_last_op"].agg(["mean", "count"])
print(tmp.sort_values("mean", ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cohort_df["n_operations"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#0a7d6e")
axes[0].set_title("Operations per patient"); axes[0].set_xlabel("n_operations"); axes[0].set_ylabel("n_patients")

dep_mort = cohort_df.groupby("department")["died_30day_from_last_op"].mean().sort_values(ascending=False)
dep_mort.plot(kind="bar", ax=axes[1], color="#c0392b")
axes[1].set_title("30-day mortality rate by department"); axes[1].set_ylabel("mortality rate")
plt.tight_layout(); plt.show()

# Part 5 — Exploratory data analysis

A cohort overview, then targeted checks that surface things the modelling choices in
Part 6 onward need to account for.

In [ ]:
# 5.0 Cohort overview -- at-a-glance picture of who's actually in this run.
n_total = len(cohort_df)
n_died = int(cohort_df["died_30day_from_last_op"].sum())
n_survived = n_total - n_died
print(f"Cohort: {n_total} patients -- {n_died} died within 30 days ({n_died/n_total:.1%}), "
      f"{n_survived} survived ({n_survived/n_total:.1%})")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# died/survived counts
axes[0].bar(["died", "survived"], [n_died, n_survived], color=["#c0392b", "#2980b9"])
axes[0].set_title("Cohort composition"); axes[0].set_ylabel("n patients")
for i, v in enumerate([n_died, n_survived]):
    axes[0].text(i, v, str(v), ha="center", va="bottom")

# age distribution by outcome
for outcome, color, label in [(1.0, "#c0392b", "died"), (0.0, "#2980b9", "survived")]:
    ages = cohort_df.loc[cohort_df["died_30day_from_last_op"] == outcome, "age"].dropna()
    if len(ages):
        axes[1].hist(ages, bins=15, alpha=0.6, color=color, label=label, density=True)
axes[1].set_title("Age distribution by outcome"); axes[1].set_xlabel("age"); axes[1].legend()

# department distribution
dept_counts = cohort_df["department"].value_counts()
axes[2].bar(dept_counts.index, dept_counts.values, color="#8e44ad")
axes[2].set_title("Patients per department"); axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout(); plt.show()

In [ ]:
# 5.1 ASA vs mortality -- sanity check that the data behaves the way the literature predicts
# (higher ASA class -> higher mortality is a well-established, monotonic clinical relationship).
asa_mort = cohort_df.groupby("asa")["died_30day_from_last_op"].agg(["mean", "count"])
print(asa_mort)

fig, ax = plt.subplots(figsize=(5, 4))
asa_mort["mean"].plot(kind="bar", ax=ax, color="#2c3e50")
ax.set_title("30-day mortality rate by ASA class"); ax.set_xlabel("ASA class"); ax.set_ylabel("mortality rate")
plt.tight_layout(); plt.show()

In [ ]:
# 5.2 Missingness -- which pre-op labs/vitals are actually available per patient?
# This is the concrete evidence behind §1.3's MNAR discussion: a feature with very low
# pre-op coverage is either genuinely unmeasured or (per §1.3) not ordered because the
# clinician judged it unnecessary -- worth seeing the real numbers before choosing §7's
# imputation strategy.
def coverage_table(df, cohort_df, time_col="chart_time", window_minutes=None, op_time_lookup=None):
    # Fraction of patients with >=1 observation of each item_name, optionally restricted
    # to a pre-op window (window_minutes before orin_time).
    if window_minutes is not None:
        keep_rows = []
        for sid, g in df.groupby("subject_id"):
            orin = op_time_lookup.get(sid)
            if orin is None:
                continue
            lo = orin - window_minutes
            keep_rows.append(g[(g[time_col] >= lo) & (g[time_col] <= orin)])
        df = pd.concat(keep_rows) if keep_rows else df.iloc[0:0]
    n_patients = cohort_df["subject_id"].nunique()
    cov = df.groupby("item_name")["subject_id"].nunique() / n_patients
    return cov.sort_values(ascending=False)

orin_lookup = cohort_df.set_index("subject_id")["orin_time_last"].to_dict()
window = CONFIG["PRE_OP_DAYS"] * 24 * 60

labs_cov_preop = coverage_table(labs_df, cohort_df, window_minutes=window, op_time_lookup=orin_lookup)
wv_cov_preop   = coverage_table(ward_vitals_df, cohort_df, window_minutes=window, op_time_lookup=orin_lookup)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
labs_cov_preop.plot(kind="bar", ax=axes[0], color="#2980b9"); axes[0].set_title(f"Pre-op ({CONFIG['PRE_OP_DAYS']}d) lab coverage")
wv_cov_preop.plot(kind="bar", ax=axes[1], color="#8e44ad"); axes[1].set_title(f"Pre-op ({CONFIG['PRE_OP_DAYS']}d) ward-vital coverage")
for ax in axes:
    ax.set_ylabel("fraction of patients with >=1 value"); ax.tick_params(axis='x', labelsize=7)
plt.tight_layout(); plt.show()

print("Lowest-coverage pre-op labs (candidates for the MNAR discussion in §1.3):")
print(labs_cov_preop.tail(8))

In [ ]:
# 5.3 ICD-10 chapter distribution -- which chapters actually show up, and how do they
# relate to mortality? Directly relevant to §1.2.3 (GI = chapter XI, MSK = chapter XIII).

ICD10_CHAPTERS = [
    ('I', 'A00-B99', 'Certain infectious and parasitic diseases'),
    ('II', 'C00-D48', 'Neoplasms'),
    ('III', 'D50-D89', 'Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism'),
    ('IV', 'E00-E90', 'Endocrine, nutritional and metabolic diseases'),
    ('V', 'F00-F99', 'Mental and behavioural disorders'),
    ('VI', 'G00-G99', 'Diseases of the nervous system'),
    ('VII', 'H00-H59', 'Diseases of the eye and adnexia'),
    ('VIII', 'H60-H95', 'Diseases of the ear and mastoid process'),
    ('IX', 'I00-I99', 'Diseases of the circulatory system'),
    ('X', 'J00-J99', 'Diseases of the respiratory system'),
    ('XI', 'K00-K93', 'Diseases of the digestive system'),                       # <- GI
    ('XII', 'L00-L99', 'Diseases of the skin and subcutaneous tissue'),
    ('XIII', 'M00-M99', 'Diseases of the musculoskeletal system and connective tissue'),  # <- MSK
    ('XIV', 'N00-N99', 'Diseases of the genitourinary system'),
    ('XV', 'O00-O99', 'Pregnancy, childbirth and the puerperium'),
    ('XVI', 'P00-P96', 'Certain conditions originating in the perinatal period'),
    ('XVII', 'Q00-Q99', 'Congenital malformations, deformations and chromosomal abnormalities'),
    ('XVIII', 'R00-R99', 'Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified'),
    ('XIX', 'S00-T98', 'Injury, poisoning and certain other consequences of external causes'),
    ('XX', 'V01-Y98', 'External causes of morbidity and mortality'),
    ('XXI', 'Z00-Z99', 'Factors influencing health status and contact with health services'),
    ('XXII', 'U00-U99', 'Codes for special purposes'),
]

def _in_block(code3, block):
    # code3: first 3 chars of an ICD-10-CM code, e.g. 'N18'. block: 'A00-B99'.
    if not code3 or len(code3) < 3:
        return False
    alpha = ord(code3[0].upper())
    try:
        numeric = int(code3[1:3])
    except ValueError:
        return False
    sblock, eblock = block.split('-')
    salpha, snum = ord(sblock[0]), int(sblock[1:])
    ealpha, enum = ord(eblock[0]), int(eblock[1:])
    return salpha <= alpha <= ealpha and snum <= numeric <= enum

def icd10_chapter(code):
    if not isinstance(code, str) or len(code) < 3:
        return None
    code3 = code[:3].upper()
    for numeral, block, desc in ICD10_CHAPTERS:
        if _in_block(code3, block):
            return numeral
    return None

diagnoses_df["chapter"] = diagnoses_df["icd10_cm"].apply(icd10_chapter)
chapter_desc = {c: d for c, b, d in ICD10_CHAPTERS}

chapter_counts = diagnoses_df["chapter"].value_counts()
print(chapter_counts)

# mortality rate among patients who have >=1 diagnosis in each chapter
chap_mort = {}
for chap in chapter_counts.index:
    sids = set(diagnoses_df.loc[diagnoses_df["chapter"] == chap, "subject_id"])
    sub = cohort_df[cohort_df["subject_id"].isin(sids)]
    if len(sub):
        chap_mort[chap] = (sub["died_30day_from_last_op"].mean(), len(sub))
chap_mort_df = pd.DataFrame(chap_mort, index=["mortality_rate", "n_patients"]).T.sort_values("mortality_rate", ascending=False)
chap_mort_df["description"] = [chapter_desc.get(c, "") for c in chap_mort_df.index]
chap_mort_df

# Part 6 — Organ-system feature engineering

This is the section that turns the theory in §1.1–§1.6 into actual feature tables. Order
of build, each tied back to a Part 1 section:

- §6.1 the organ-system → raw-signal grouping table (the routing map)
- §6.2 pre-op / peri-op time-window extraction per patient (§1.6.3)
- §6.3–§6.5 the four "genuine time series" systems: Renal, Respiratory, Metabolic/hepatic, Haematology
- §6.9 Cardiovascular, built *jointly* with Renal because of the coupling in §1.6.2
- §6.6 ICD-10 features: chapter flags/counts + HFRS (§1.2)
- §6.7–§6.8 GI and MSK (diagnosis/department-driven, §1.2.3)
- §6.10 operation-count-in-same-area features (§1.6.4)
- §6.11 the 6-month post-cardiac-surgery exception flag (§1.6.4, rule-based on purpose)
- §6.12 static features (age, sex, ASA, emop, department, weight, height) (§1.5.1, §1.2.2)
- §6.13 medications → ATC-level aggregate features (§1.5.3)
- §6.14 assembling everything into the final per-patient, per-system feature bundle

## 6.1 The organ-system → raw-signal routing map

This extends the six-system grouping already sketched in the source repo's own
`docs/roadmap_and_architecture.md` §4.1 with the two new systems from the meeting notes,
and is filtered to the item-names actually observed in §4.1's inventory (so nothing below
references a feature this dataset doesn't have).

In [ ]:
# --- Time-series systems: item_name -> system, split by source table ---
# Kept as an explicit, editable dict (not auto-derived) so the organ-system assignment is
# a reviewable, documented decision -- exactly the kind of thing you said you want to be
# able to interrogate and change yourself.

SYSTEM_LABS = {
    "renal":          ["bun", "calcium", "chloride", "creatinine", "ica", "phosphorus", "potassium", "sodium"],
    "cardiovascular":  ["ck", "ckmb", "troponin_i"],   # troponin_t not present in this subset's inventory (§4.1)
    "respiratory":    ["be", "hco3", "paco2", "pao2", "ph", "sao2"],
    "metabolic_hepatic": ["albumin", "alp", "alt", "ast", "glucose", "hba1c", "lacate", "total_bilirubin", "total_protein"],
    "haematology":    ["aptt", "crp", "fibrinogen", "hb", "hct", "lymphocyte", "platelet", "ptinr", "seg", "wbc"],
    "neurological":   [],   # no dedicated neuro lab in this dataset
}

SYSTEM_WARD_VITALS = {
    "renal":          ["crrt", "uo"],
    "cardiovascular":  ["hr", "nibp_sbp", "nibp_dbp", "nibp_mbp", "iabp"],
    "respiratory":    ["fio2", "rr", "spo2", "vent", "ecmo"],
    "metabolic_hepatic": ["bt"],
    "haematology":    [],
    "neurological":   ["gcs_e", "gcs_m", "gcs_v"],
}

SYSTEM_INTRAOP_VITALS = {   # only used when CONFIG['TIME_WINDOW'] == 'peri_op'
    "renal":          ["uo"],
    "cardiovascular":  ["hr", "art_sbp", "art_dbp", "art_mbp", "ci", "cvp", "svi",
                        "nibp_sbp", "nibp_dbp", "nibp_mbp", "pap_sbp", "pap_dbp", "pap_mbp"],
    "respiratory":    ["etco2", "fio2", "spo2", "peep", "pip", "pplat", "rr", "minvol", "vt"],
    "metabolic_hepatic": ["bt"],
    "haematology":    ["ebl", "rbc", "ffp"],
    "neurological":   ["bis"],
}

TIME_SERIES_SYSTEMS = ["renal", "cardiovascular", "respiratory", "metabolic_hepatic", "haematology", "neurological"]

# Sanity-check every listed item_name actually exists in this dataset's inventory (§4.1).
for sysdict, inv_key in [(SYSTEM_LABS, "labs"), (SYSTEM_WARD_VITALS, "ward_vitals"), (SYSTEM_INTRAOP_VITALS, "vitals (intra-op)")]:
    known = set(INVENTORY[inv_key])
    for system, items in sysdict.items():
        unknown = [i for i in items if i not in known]
        if unknown:
            print(f"WARNING: {inv_key}/{system} references unseen item_names: {unknown}")
print("Routing map validated against §4.1 inventory (no warnings above = every referenced item_name exists).")

## 6.2 Pre-op / peri-op window extraction

Implements §1.6.3's `CONFIG['TIME_WINDOW']` toggle. `'pre_op'` keeps only records in
`[orin_time - PRE_OP_DAYS*24*60, orin_time]` for the **last** operation (matching the label
definition in §4.2 — Path C, "from last operation"; §11.7 revisits first-operation and
exclude-multi-op as a sensitivity check). `'peri_op'` extends the upper bound to
`orout_time` and additionally pulls in intra-op `vitals`.

In [ ]:
def get_window(subject_id, cohort_row):
    orin = cohort_row["orin_time_last"]
    orout = cohort_row["orout_time_last"]
    lo = orin - CONFIG["PRE_OP_DAYS"] * 24 * 60
    if CONFIG["TIME_WINDOW"] == "pre_op":
        hi = orin
    elif CONFIG["TIME_WINDOW"] == "peri_op":
        hi = orout if pd.notna(orout) else orin
    else:
        raise ValueError(f"Unknown TIME_WINDOW {CONFIG['TIME_WINDOW']!r}")
    return lo, hi

COHORT_INDEXED = cohort_df.set_index("subject_id")

# --- Fast per-patient/feature lookup (replaces a full-table scan per call) -----------
# The naive version of windowed_series() -- filter the whole DataFrame by subject_id AND
# item_name AND time range, on every single call -- is fine on the 30-patient dev subset
# but does NOT scale: this notebook calls it roughly 60-70 times per patient across
# Part 6, so at the full ~99,886-patient cohort that's several million calls, each an
# O(table_size) scan over tables with tens of millions of rows. Realistically that's days,
# not hours -- it would exceed any Colab/Kaggle session limit long before the GPU or RAM
# did. The fix below groups each table ONCE into a plain Python dict keyed by
# (subject_id, item_name) -> (chart_time array, value array); every lookup after that is a
# dict hash lookup plus a small in-memory numpy filter, both essentially O(1) with respect
# to the table's total size. (An earlier version of this fix used a sorted pandas
# MultiIndex + .loc[] instead -- measured at only ~20-90x faster than the naive scan on
# benchmark tables, versus >4,000x for the plain-dict version below: pandas' .loc[] call
# overhead dominates for small per-key results, which is exactly this notebook's access
# pattern. Benchmarked below rather than assumed.) Output is identical either way -- this
# only changes speed, verified by confirming every printed count/statistic in this
# notebook is unchanged after the switch.
_GROUPED_ARRAYS_CACHE = {}

def _get_grouped_arrays(df):
    key = id(df)
    if key not in _GROUPED_ARRAYS_CACHE:
        grouped = {}
        if len(df) and "item_name" in df.columns:
            for gkey, g in df.groupby(["subject_id", "item_name"], observed=True, sort=False):
                grouped[gkey] = (g["chart_time"].to_numpy(), g["value"].to_numpy())
        _GROUPED_ARRAYS_CACHE[key] = grouped
    return _GROUPED_ARRAYS_CACHE[key]

def windowed_series(df, subject_id, item_name, lo, hi, subj_col="subject_id"):
    # Returns a sorted (chart_time -> value) dict for one patient/feature within [lo, hi].
    if len(df) == 0 or "item_name" not in df.columns:
        return {}
    grouped = _get_grouped_arrays(df)
    arrs = grouped.get((subject_id, item_name))
    if arrs is None:
        return {}
    times, vals = arrs
    mask = (times >= lo) & (times <= hi) & ~np.isnan(vals.astype(float))
    if not mask.any():
        return {}
    t_sel, v_sel = times[mask], vals[mask]
    order = np.argsort(t_sel)
    return dict(zip(t_sel[order], v_sel[order]))

print(f"TIME_WINDOW = {CONFIG['TIME_WINDOW']!r}  (see §1.6.3 for what each option represents)")

### A note on why this matters more than it looks

The grouped-lookup rewrite above is the single highest-impact change in this notebook for
runnability at full scale — more than any of the memory optimizations. A self-contained
timing comparison on synthetic data of a similar shape to the full cohort (run once,
below, to make this concrete rather than asserted) shows the real speedup, split into the
one-time index-build cost (paid once per table, however large the cohort) and the
marginal per-lookup cost (paid millions of times, so this is the number that actually
determines whether a full run finishes in minutes/hours or days).

In [ ]:
import time

def _make_synthetic_table(n_subjects, avg_rows_per_subject, n_item_names=40):
    rng = np.random.default_rng(0)
    n_rows = n_subjects * avg_rows_per_subject
    subject_ids = rng.integers(0, n_subjects, size=n_rows).astype(str)
    item_names = rng.choice([f"item_{i}" for i in range(n_item_names)], size=n_rows)
    chart_times = rng.integers(-10000, 10000, size=n_rows)
    values = rng.normal(size=n_rows).astype(np.float32)
    df = pd.DataFrame({"subject_id": subject_ids, "item_name": item_names,
                        "chart_time": chart_times, "value": values})
    df["subject_id"] = df["subject_id"].astype("category")
    df["item_name"] = df["item_name"].astype("category")
    return df

_SYN_N_SUBJECTS = 2000
_synthetic_df = _make_synthetic_table(_SYN_N_SUBJECTS, avg_rows_per_subject=200)
print(f"Synthetic benchmark table: {len(_synthetic_df):,} rows, {_SYN_N_SUBJECTS:,} subjects "
      f"(a small slice of the full cohort's scale, kept small here so the OLD method's demo "
      f"below finishes quickly -- the gap only widens at the real ~99,886-patient size)")

def _old_windowed_series(df, subject_id, item_name, lo, hi):
    sub = df[(df["subject_id"] == subject_id) & (df["item_name"] == item_name) &
             (df["chart_time"] >= lo) & (df["chart_time"] <= hi)]
    return dict(zip(sub["chart_time"], sub["value"]))

_sample_subject_ids = np.random.default_rng(1).choice(_synthetic_df["subject_id"].cat.categories, size=150, replace=True)
_sample_items = [f"item_{i}" for i in range(6)]
n_calls = len(_sample_subject_ids) * len(_sample_items)

_t0 = time.time()
for sid in _sample_subject_ids:
    for item in _sample_items:
        _old_windowed_series(_synthetic_df, sid, item, -10000, 10000)
_old_elapsed = time.time() - _t0

_GROUPED_ARRAYS_CACHE.clear()
_t0 = time.time()
_ = windowed_series(_synthetic_df, _sample_subject_ids[0], _sample_items[0], -10000, 10000)   # triggers the one-time group build
_build_elapsed = time.time() - _t0

_t0 = time.time()
for sid in _sample_subject_ids:
    for item in _sample_items:
        windowed_series(_synthetic_df, sid, item, -10000, 10000)
_new_lookups_elapsed = time.time() - _t0

old_ms_per_call = _old_elapsed / n_calls * 1000
new_ms_per_call = _new_lookups_elapsed / n_calls * 1000
print(f"\n{n_calls:,} lookups against a {len(_synthetic_df):,}-row table:")
print(f"  old (full-scan every call):    {_old_elapsed:.2f}s total  ({old_ms_per_call:.3f} ms/call)")
print(f"  new -- one-time group build:   {_build_elapsed:.2f}s (paid ONCE per table for the whole notebook run)")
print(f"  new -- {n_calls:,} grouped lookups: {_new_lookups_elapsed:.4f}s  ({new_ms_per_call:.5f} ms/call)")
print(f"  marginal per-call speedup:     {old_ms_per_call/max(new_ms_per_call,1e-9):.0f}x  "
      f"(this is the number that matters at full scale, where millions of calls share one build)")
print(f"\nThe one-time build cost scales with table size (more rows/groups to organise), so at "
      f"the real full-cohort scale expect the build itself to take real minutes, not seconds -- "
      f"that is an acceptable, one-time cost given it replaces what would otherwise be a "
      f"multi-day total runtime under the old per-call scan.")

del _synthetic_df
_GROUPED_ARRAYS_CACHE.clear()   # drop the benchmark's groups; the real tables get grouped fresh on first real use

## 6.3 Per-system time-series extraction, resampling, and the missingness mask

For each patient and each of the six time-series systems (§6.1), build a
`[TARGET_SEQ_LEN, n_features_in_system]` array plus a same-shaped **mask** array
(1 = observed/interpolated-from-real-data, 0 = no data at all for that patient/feature —
see §1.3's MNAR discussion for why the mask is kept as its own feature rather than
discarded once a value is filled in). The actual *value*-filling strategy is deliberately
left as a placeholder here (`_TODO_impute`) and implemented properly in §7, once all the
methods have been introduced together — this section only handles resampling onto a
common time grid, which is method-independent.

In [ ]:
def resample_to_grid(chart_time2value, lo, hi, n_points):
    # Evenly-spaced grid of n_points between lo and hi; each grid point takes the NEAREST
    # observation, but only if it is within one grid step -- otherwise NaN, left for Part 7's
    # imputation. v2: vectorised with numpy (identical output to the v1 per-point loop, which
    # was one of the slowest steps at full-cohort scale). Ties go to the later observation,
    # exactly as before.
    grid = np.linspace(lo, hi, n_points)
    out = np.full(n_points, np.nan)
    if len(chart_time2value) == 0:
        return grid, out
    keys = sorted(chart_time2value.keys())
    times = np.array(keys, dtype=float)
    values = np.array([chart_time2value[t] for t in keys], dtype=float)
    idx = np.searchsorted(times, grid)
    right = np.clip(idx, 0, len(times) - 1)
    left = np.clip(idx - 1, 0, len(times) - 1)
    d_right = np.where(idx < len(times), np.abs(times[right] - grid), np.inf)
    d_left = np.where(idx > 0, np.abs(times[left] - grid), np.inf)
    use_right = d_right <= d_left
    best_d = np.where(use_right, d_right, d_left)
    best_v = np.where(use_right, values[right], values[left])
    step = (hi - lo) / max(n_points - 1, 1)
    ok = best_d <= max(step, 1e-6)
    out[ok] = best_v[ok]
    return grid, out

def extract_system_tensor(subject_id, system, lo, hi):
    # Returns (raw_values [T,F] with NaN for gaps, mask [T,F], feature_names) for one
    # patient/system, pooling labs + ward_vitals (+ intra-op vitals if peri_op).
    feature_names = []
    columns = []
    sources = [(labs_df, SYSTEM_LABS[system]), (ward_vitals_df, SYSTEM_WARD_VITALS[system])]
    if CONFIG["TIME_WINDOW"] == "peri_op":
        sources.append((vitals_df, SYSTEM_INTRAOP_VITALS[system]))
    for df, items in sources:
        for item in items:
            series = windowed_series(df, subject_id, item, lo, hi)
            grid, raw = resample_to_grid(series, lo, hi, CONFIG["TARGET_SEQ_LEN"])
            feature_names.append(item)
            columns.append(raw)
    if not columns:
        return (np.zeros((CONFIG["TARGET_SEQ_LEN"], 0)),
                np.zeros((CONFIG["TARGET_SEQ_LEN"], 0)),
                [])
    raw_values = np.stack(columns, axis=1)          # [T, F]
    mask = (~np.isnan(raw_values)).astype(np.float32)
    return raw_values, mask, feature_names

# Build the raw (pre-imputation) tensors for every patient x system, kept in a dict so §7
# can impute in place without re-doing the windowing/resampling work.
RAW_SYSTEM_TENSORS = {}   # (subject_id, system) -> dict(raw=[T,F], mask=[T,F], feature_names=[...])
for sid in cohort_df["subject_id"]:
    row = COHORT_INDEXED.loc[sid]
    lo, hi = get_window(sid, row)
    for system in TIME_SERIES_SYSTEMS:
        raw, mask, fnames = extract_system_tensor(sid, system, lo, hi)
        RAW_SYSTEM_TENSORS[(sid, system)] = {"raw": raw, "mask": mask, "feature_names": fnames}

# Quick coverage summary: for each system, what fraction of (patient, feature) cells have
# at least one real observation in the window?
for system in TIME_SERIES_SYSTEMS:
    fnames = RAW_SYSTEM_TENSORS[(cohort_df["subject_id"].iloc[0], system)]["feature_names"]
    if not fnames:
        print(f"{system:18s}: no features assigned (see §6.1)")
        continue
    total, observed = 0, 0
    for sid in cohort_df["subject_id"]:
        m = RAW_SYSTEM_TENSORS[(sid, system)]["mask"]
        total += m.shape[0] * m.shape[1]
        observed += m.sum()
    print(f"{system:18s}: {len(fnames):2d} features, {observed/total:5.1%} of (patient,timepoint,feature) cells observed in-window")

## 6.9 Cardiovascular ↔ Renal coupling (§1.6.2)

Two concrete implementations of the meeting note "renal directly proportionate to
cardiovascular," matching the two mechanisms described in §1.6.2:

1. **Architectural**: a compact cardiovascular summary (mean heart rate, mean arterial
   pressure deviation from normal, presence of an IABP flag) is computed here and appended
   to the *renal* branch's static side-input in §9 — so the renal encoder sees
   cardiovascular context directly, not just renal labs.
2. **Hand-crafted interaction feature**: `renal_cardiac_interaction`, a creatinine ×
   blood-pressure-deviation product, added to the static feature table (§6.14) as a
   cheap, sample-efficient prior (§1.6.2 explains why this is added *alongside* the
   learned coupling, not instead of it).

`CONFIG['SYMMETRIC_CARDIORENAL_COUPLING']` toggles whether the *cardiovascular* branch
symmetrically receives a renal summary back — off by default, per the §1.6.2 reasoning.

In [ ]:
NORMAL_MAP_MMHG = 93.0   # a commonly used "normal" mean arterial pressure reference point

def cardiac_summary_for_patient(sid, lo, hi):
    # A small, fixed-size cardiovascular summary vector -- fed into the renal branch (§9)
    # as the architectural half of the cardiorenal coupling, and used below to build the
    # hand-crafted interaction feature.
    hr_series = windowed_series(ward_vitals_df, sid, "hr", lo, hi)
    map_series = windowed_series(ward_vitals_df, sid, "nibp_mbp", lo, hi)
    iabp_series = windowed_series(ward_vitals_df, sid, "iabp", lo, hi)

    mean_hr = float(np.mean(list(hr_series.values()))) if hr_series else np.nan
    mean_map = float(np.mean(list(map_series.values()))) if map_series else np.nan
    map_deviation = (mean_map - NORMAL_MAP_MMHG) if not np.isnan(mean_map) else np.nan
    has_iabp = float(len(iabp_series) > 0 and any(v > 0 for v in iabp_series.values()))

    return {"cardio_mean_hr": mean_hr, "cardio_map_deviation": map_deviation, "cardio_has_iabp": has_iabp}

CARDIAC_SUMMARY = {}
for sid in cohort_df["subject_id"]:
    row = COHORT_INDEXED.loc[sid]
    lo, hi = get_window(sid, row)
    CARDIAC_SUMMARY[sid] = cardiac_summary_for_patient(sid, lo, hi)

cardiac_summary_df = pd.DataFrame(CARDIAC_SUMMARY).T
cardiac_summary_df.index.name = "subject_id"
print(f"Cardiovascular summary built for {len(cardiac_summary_df)} patients "
      f"(NaNs below reflect real missingness -- filled properly in §7, not here).")
cardiac_summary_df.describe()

In [ ]:
def renal_cardiac_interaction_for_patient(sid, lo, hi):
    # Hand-crafted §1.6.2 interaction feature: creatinine x |MAP deviation|.
    # NaN-safe -- returns NaN if either side is unavailable, filled like any other
    # static feature in §7.
    creat_series = windowed_series(labs_df, sid, "creatinine", lo, hi)
    mean_creat = float(np.mean(list(creat_series.values()))) if creat_series else np.nan
    map_dev = CARDIAC_SUMMARY[sid]["cardio_map_deviation"]
    if np.isnan(mean_creat) or (map_dev is None) or np.isnan(map_dev):
        return np.nan
    return mean_creat * abs(map_dev)

RENAL_CARDIAC_INTERACTION = {
    sid: renal_cardiac_interaction_for_patient(sid, *get_window(sid, COHORT_INDEXED.loc[sid]))
    for sid in cohort_df["subject_id"]
}
pd.Series(RENAL_CARDIAC_INTERACTION, name="renal_cardiac_interaction").describe()

## 6.6 / 6.7 / 6.8 — ICD-10 chapter features, HFRS, GI and MSK systems, infection flag

All built together because they share the same underlying computation (diagnosis chapter
membership within a lookback window) — GI and MSK (§1.2.3, §1.6.1) are simply the two
chapters the meeting notes called out by name, exposed as their own system entries rather
than folded into a generic "diagnosis count" feature.

- **GI** = ICD-10 chapter **XI** (`K00-K93`) flag/count + `department == 'GS'` (general
  surgery, the closest department proxy for GI surgical burden in this dataset).
- **MSK** = ICD-10 chapter **XIII** (`M00-M99`) flag/count + `department == 'OS'`
  (orthopaedic surgery).
- **HFRS** (Hospital Frailty Risk Score, Gilbert et al. 2018) — a validated, externally
  weighted score over 109 specific ICD-10 codes, restricted here to the published lookback
  window (`CONFIG['HFRS_LOOKBACK_YEARS']`, default 2, and only meaningfully applied for
  patients 75+ per the original paper — see §1.2.1). **This revision uses the full 109-code
  weights table**, extracted directly from the source repo's `frailty_hfrs.py` (an earlier
  revision of this notebook shipped a ~30-code representative subset — that limitation,
  flagged in `Multimodal_Notebook_Summary.md` §7, is now closed).
- **Infection/inflammation flag** — the cross-cutting signal `roadmap_and_architecture.md`
  §4.1a designed but had not yet implemented (until this revision). Per SOFA (Vincent et
  al. 1996) and Sepsis-3 (Singer et al. 2016), infection is modelled as a **modifier
  layered across systems**, not a competing 7th/9th organ system — implemented here as a
  small set of static features (chapter-I diagnosis flag, the six specific high-mortality
  codes already flagged in the source repo's EDA, fever, abnormal WBC, and a data-driven
  "elevated CRP" flag — see the code cell for why CRP uses a relative, not absolute,
  threshold) plus a 0–5 composite count, fed into the static branch alongside GI/MSK/HFRS.

In [ ]:
# v2 SPEED FIX: group diagnoses and medications by patient ONCE. The v1 code filtered the
# whole table for every patient (df[df.subject_id == sid]), which is fine at 10k patients
# but grows roughly with (patients x rows) -- hours at the full ~99,886-patient cohort.
# Same rows, same order within a patient, so every feature below is unchanged.
def _group_by_subject(df):
    if len(df) == 0:
        return {}
    return {k: g for k, g in df.groupby("subject_id", observed=True, sort=False)}

_DX_BY_SUBJECT = _group_by_subject(diagnoses_df)
_MEDS_BY_SUBJECT = _group_by_subject(medications_df)
_EMPTY_DX = diagnoses_df.iloc[0:0]
_EMPTY_MEDS = medications_df.iloc[0:0]

def dx_in_window(sid, lo, hi):
    g = _DX_BY_SUBJECT.get(sid, _EMPTY_DX)
    return g[(g["chart_time"] >= lo) & (g["chart_time"] <= hi)]

def meds_in_window(sid, lo, hi):
    g = _MEDS_BY_SUBJECT.get(sid, _EMPTY_MEDS)
    return g[(g["chart_time"] >= lo) & (g["chart_time"] <= hi)]

# Full Hospital Frailty Risk Score weights table (Gilbert et al. 2018, Table A2, 109
# ICD-10-CM clusters), extracted directly from the source repo's frailty_hfrs.py so this
# notebook stays self-contained/Kaggle-portable without importing that module.
HFRS_WEIGHTS = {
    'A04': 1.1, 'A09': 1.1, 'A41': 1.6, 'B95': 1.7, 'B96': 2.9, 'D64': 0.4,
    'E05': 0.9, 'E16': 1.4, 'E53': 1.9, 'E55': 1.0, 'E83': 0.4, 'E86': 2.3,
    'E87': 2.3, 'F00': 7.1, 'F01': 2.0, 'F03': 2.1, 'F05': 3.2, 'F10': 0.7,
    'F32': 0.5, 'G20': 1.8, 'G30': 4.0, 'G31': 1.2, 'G40': 1.5, 'G45': 1.2,
    'G81': 4.4, 'H54': 1.9, 'H91': 0.9, 'I63': 0.8, 'I67': 2.6, 'I69': 3.7,
    'I95': 1.6, 'J18': 1.1, 'J22': 0.7, 'J69': 1.0, 'J96': 1.5, 'K26': 1.6,
    'K52': 0.3, 'K59': 1.8, 'K92': 0.8, 'L03': 2.0, 'L08': 0.4, 'L89': 1.7,
    'L97': 1.6, 'M15': 0.4, 'M19': 1.5, 'M25': 2.3, 'M41': 0.9, 'M48': 0.5,
    'M79': 1.1, 'M80': 0.8, 'M81': 1.4, 'N17': 1.8, 'N18': 1.4, 'N19': 1.6,
    'N20': 0.7, 'N28': 1.3, 'N39': 3.2, 'R00': 0.7, 'R02': 1.0, 'R11': 0.3,
    'R13': 0.8, 'R26': 2.6, 'R29': 3.6, 'R31': 3.0, 'R32': 1.2, 'R33': 1.3,
    'R40': 2.5, 'R41': 2.7, 'R44': 1.6, 'R45': 1.2, 'R47': 1.0, 'R50': 0.1,
    'R54': 2.2, 'R55': 1.8, 'R56': 2.6, 'R63': 0.9, 'R69': 1.3, 'R79': 0.6,
    'R94': 1.4, 'S00': 3.2, 'S01': 1.1, 'S06': 2.4, 'S09': 1.2, 'S22': 1.8,
    'S32': 1.4, 'S42': 2.3, 'S51': 0.5, 'S72': 1.4, 'S80': 2.0, 'T83': 2.4,
    'U80': 0.8, 'W01': 0.9, 'W06': 1.1, 'W10': 0.9, 'W18': 2.1, 'W19': 3.2,
    'X59': 1.5, 'Y84': 0.7, 'Y95': 1.2, 'Z22': 1.7, 'Z50': 2.1, 'Z60': 1.8,
    'Z73': 0.6, 'Z74': 1.1, 'Z75': 2.0, 'Z87': 1.5, 'Z91': 0.5, 'Z93': 1.0,
    'Z99': 0.8,
}
assert len(HFRS_WEIGHTS) == 109, f"Expected 109 HFRS codes (Gilbert et al. Table A2), got {len(HFRS_WEIGHTS)}"

def compute_hfrs(sid, lo, hi, age):
    # Sum of HFRS weights for matching diagnosis codes within the lookback window,
    # restricted to patients aged 75+ (per Gilbert et al.'s validated population -- Sec 1.2.1).
    # Returns 0.0 (not NaN) for younger patients: HFRS is defined as not-applicable, not
    # missing, below the validated age range.
    if pd.isna(age) or age < 75:
        return 0.0
    lookback_minutes = CONFIG["HFRS_LOOKBACK_YEARS"] * 365 * 24 * 60
    dx = dx_in_window(sid, lo - lookback_minutes, hi)
    score = 0.0
    for code in dx["icd10_cm"].dropna():
        code3 = str(code)[:3].upper()
        if code3 in HFRS_WEIGHTS:
            score += HFRS_WEIGHTS[code3]
    return score

# v3: every organ system gets diagnosis-code features from its own ICD-10 chapter, so all
# eight systems are built the same way (GI and MSK are no longer a special case -- they are
# simply the two systems with no time-series measurements in this dataset).
ORGAN_SYSTEM_ICD10_CHAPTERS = {
    "renal": ["XIV"],              # genitourinary
    "cardiovascular": ["IX"],      # circulatory
    "respiratory": ["X"],          # respiratory
    "metabolic_hepatic": ["IV"],   # endocrine, nutritional, metabolic
    "haematology": ["III"],        # blood and immune
    "neurological": ["VI"],        # nervous system
    "gi": ["XI"],                  # digestive
    "msk": ["XIII"],               # musculoskeletal
}
ORGAN_SYSTEMS = list(ORGAN_SYSTEM_ICD10_CHAPTERS)   # the eight organ systems, one order everywhere

def organ_system_dx_features(sid, lo, hi, department):
    dx = dx_in_window(sid, lo, hi)
    chapters = dx["icd10_cm"].dropna().apply(icd10_chapter)
    n_chapters = chapters.value_counts()
    feats = {}
    for system, chs in ORGAN_SYSTEM_ICD10_CHAPTERS.items():
        n = int(sum(n_chapters.get(c, 0) for c in chs))
        feats[f"sysdx_{system}_count"] = n
        feats[f"sysdx_{system}_flag"] = float(n > 0)
    # department flags kept from v1 (surgical specialty treating the system)
    feats["gi_department_flag"] = float(department == "GS")
    feats["msk_department_flag"] = float(department == "OS")
    feats["n_diagnoses_in_window"] = int(len(dx))
    feats["n_distinct_chapters_in_window"] = int(chapters.nunique())
    return feats

ICD10_FEATURES = {}
for sid in cohort_df["subject_id"]:
    row = COHORT_INDEXED.loc[sid]
    lo, hi = get_window(sid, row)
    feats = organ_system_dx_features(sid, lo, hi, row["department"])
    if CONFIG["INCLUDE_HFRS"]:
        feats["hfrs"] = compute_hfrs(sid, lo, hi, row["age"])
    ICD10_FEATURES[sid] = feats

icd10_features_df = pd.DataFrame(ICD10_FEATURES).T
icd10_features_df.index.name = "subject_id"
print("Patients with >=1 diagnosis in each organ system's ICD-10 chapter (pre-op window):")
for _s in ORGAN_SYSTEMS:
    print(f"  {_s:18s} {int(icd10_features_df[f'sysdx_{_s}_flag'].sum()):6d} / {len(icd10_features_df)}")
if CONFIG["INCLUDE_HFRS"]:
    n_scored = int((icd10_features_df["hfrs"] > 0).sum())
    print(f"Patients with HFRS > 0 (i.e. aged 75+ with a matching code): {n_scored} / {len(icd10_features_df)}")
icd10_features_df.describe()

### Infection / inflammation cross-cutting flag (`roadmap_and_architecture.md` §4.1a)

Six components, each a real, cheap-to-compute signal rather than a learned sub-model —
consistent with the design note that this is a *modifier*, not a competing organ system.
The CRP threshold is deliberately **data-driven (this cohort's own 75th percentile)**
rather than an absolute clinical cutoff (e.g. "CRP > 100 mg/L") -- this notebook never
loaded `parameters.csv`, so CRP's exact reporting unit for this INSPIRE export isn't
independently confirmed here, and a wrong absolute threshold is worse than an honestly
relative one. Fever and abnormal-WBC thresholds use standard, unit-unambiguous clinical
cutoffs (°C and count/volume respectively), so those stay absolute.

In [ ]:
HIGH_RISK_INFECTION_CODES = {"D65", "I46", "R57", "J80", "K72", "A41"}   # from the source repo's own EDA (docs/eda_findings.md)
FEVER_THRESHOLD_C = 38.0
WBC_NORMAL_RANGE = (4.0, 11.0)   # x10^9/L, standard adult reference range

# Data-driven CRP threshold: computed once from the training-eligible population (here,
# the whole cohort at bundle-build time, since this runs before the train/val/test split
# in Part 8 -- acceptable for a threshold that's a description of the cohort's own
# distribution, not a fitted statistic that could leak label information).
_all_crp_values = labs_df.loc[labs_df["item_name"] == "crp", "value"].dropna()
CRP_ELEVATED_THRESHOLD = float(_all_crp_values.quantile(0.75)) if len(_all_crp_values) else None
print(f"CRP 'elevated' threshold (75th percentile of all observed CRP in this cohort): "
      f"{CRP_ELEVATED_THRESHOLD}")

def infection_inflammation_features(sid, lo, hi):
    dx = dx_in_window(sid, lo, hi)
    codes = set(dx["icd10_cm"].dropna().astype(str))
    chapter_i_flag = float(any(icd10_chapter(c) == "I" for c in codes))
    high_risk_flag = float(len(codes & HIGH_RISK_INFECTION_CODES) > 0)

    bt_series = windowed_series(ward_vitals_df, sid, "bt", lo, hi)
    fever_flag = float(len(bt_series) > 0 and max(bt_series.values()) >= FEVER_THRESHOLD_C)

    wbc_series = windowed_series(labs_df, sid, "wbc", lo, hi)
    wbc_abnormal_flag = 0.0
    if wbc_series:
        wbc_vals = list(wbc_series.values())
        wbc_abnormal_flag = float(any(v < WBC_NORMAL_RANGE[0] or v > WBC_NORMAL_RANGE[1] for v in wbc_vals))

    crp_series = windowed_series(labs_df, sid, "crp", lo, hi)
    crp_elevated_flag = 0.0
    if crp_series and CRP_ELEVATED_THRESHOLD is not None:
        crp_elevated_flag = float(max(crp_series.values()) >= CRP_ELEVATED_THRESHOLD)

    composite = chapter_i_flag + high_risk_flag + fever_flag + wbc_abnormal_flag + crp_elevated_flag

    return {
        "infection_chapter_i_flag": chapter_i_flag,
        "infection_high_risk_code_flag": high_risk_flag,
        "infection_fever_flag": fever_flag,
        "infection_wbc_abnormal_flag": wbc_abnormal_flag,
        "infection_crp_elevated_flag": crp_elevated_flag,
        "infection_composite_score": composite,   # 0-5, a simple count of the above
    }

INFECTION_FEATURES = {}
for sid in cohort_df["subject_id"]:
    row = COHORT_INDEXED.loc[sid]
    lo, hi = get_window(sid, row)
    INFECTION_FEATURES[sid] = infection_inflammation_features(sid, lo, hi)

infection_features_df = pd.DataFrame(INFECTION_FEATURES).T
infection_features_df.index.name = "subject_id"
print(f"\nPatients with infection_composite_score > 0: "
      f"{(infection_features_df['infection_composite_score'] > 0).sum()} / {len(infection_features_df)}")
infection_features_df.describe()

## 6.10 Operation-count-in-same-area features (§1.6.4)

For the **last** operation (the one the label is computed relative to, §4.2), count how
many *prior* operations that patient had in the **same department** and compute the time
gap in days since the most recent one. This is offered to the model as a feature, on
purpose — unlike the exception in §6.11 below, whether repeat-same-area surgery predicts
higher or lower risk is exactly the kind of question this notebook's own framing (§1.6.4)
says should be *learned*, not assumed.

In [ ]:
def operation_count_features(sid):
    history = OPS_HISTORY[sid]   # sorted by orin_time, built in §4.3
    if len(history) <= 1:
        return {"n_prior_ops_same_dept": 0, "days_since_last_op_same_dept": np.nan,
                "n_prior_ops_any_dept": 0, "days_since_last_op_any_dept": np.nan}
    last_op = history[-1]
    prior_ops = history[:-1]
    same_dept = [op for op in prior_ops if op.get("department") == last_op.get("department")]

    def days_since(ops):
        if not ops:
            return np.nan
        most_recent = max(op["orin_time"] for op in ops if op.get("orin_time") is not None)
        return (last_op["orin_time"] - most_recent) / (24 * 60)

    return {
        "n_prior_ops_same_dept": len(same_dept),
        "days_since_last_op_same_dept": days_since(same_dept),
        "n_prior_ops_any_dept": len(prior_ops),
        "days_since_last_op_any_dept": days_since(prior_ops),
    }

OP_COUNT_FEATURES = {sid: operation_count_features(sid) for sid in cohort_df["subject_id"]}
op_count_df = pd.DataFrame(OP_COUNT_FEATURES).T
op_count_df.index.name = "subject_id"
print(f"Patients with >=1 prior operation in the same department as their last: "
      f"{(op_count_df['n_prior_ops_same_dept'] > 0).sum()} / {len(op_count_df)}")
op_count_df.describe()

## 6.11 The 6-month post-cardiovascular-surgery exception (§1.6.4)

Implemented as a **rule**, deliberately not a learned feature, for the reasons argued in
§1.6.4 (sample size + actionability). `CTS` (cardiothoracic surgery) is used as this
dataset's department proxy for "cardiovascular surgery" — check this mapping against your
site's actual department coding before trusting the flag on a new cohort.

In [ ]:
CARDIAC_DEPARTMENTS = {"CTS"}       # adjust if your site's department coding differs
MIN_RECOVERY_DAYS_AFTER_CARDIAC_SURGERY = 180   # ~6 months, the clinical protocol window

def cardiac_recovery_exception_flag(sid):
    # 1.0 if the patient's last operation occurred less than 180 days after a PRIOR
    # cardiovascular-department operation -- i.e. the protocol window was violated (or the
    # two operations are causally linked, e.g. a staged/urgent re-intervention). This is a
    # flag for the model and for manual review, not an automatic exclusion -- see §11.6 for
    # the follow-up qualitative check this notebook recommends running once the model exists.
    history = OPS_HISTORY[sid]
    if len(history) <= 1:
        return 0.0
    last_op = history[-1]
    prior_cardiac_ops = [op for op in history[:-1] if op.get("department") in CARDIAC_DEPARTMENTS]
    if not prior_cardiac_ops:
        return 0.0
    most_recent_cardiac = max(op["orin_time"] for op in prior_cardiac_ops if op.get("orin_time") is not None)
    gap_days = (last_op["orin_time"] - most_recent_cardiac) / (24 * 60)
    return float(gap_days < MIN_RECOVERY_DAYS_AFTER_CARDIAC_SURGERY)

CARDIAC_EXCEPTION_FLAG = {sid: cardiac_recovery_exception_flag(sid) for sid in cohort_df["subject_id"]}
n_flagged = sum(CARDIAC_EXCEPTION_FLAG.values())
print(f"Patients flagged (last op within 6 months of a prior CTS op): {int(n_flagged)} / {len(CARDIAC_EXCEPTION_FLAG)}")
print("(Low or zero counts are expected on this 30-patient dev subset -- re-run on the full "
      "cohort, where cardiothoracic re-intervention is far more likely to appear.)")

## 6.12 Static features (§1.5.1, §1.2.2)

Age, sex, ASA, `emop` (emergency-operation flag), department (one-hot), weight, height —
100% coverage, cheap, and (per §1.2.2/§1.6.1) currently unused by the DNN in the source
repo despite ASA being one of the strongest established predictors in the literature.
Included here by default via `CONFIG['INCLUDE_STATIC_ASA']`.

In [ ]:
def static_features_row(row):
    return {
        "age": row["age"],
        "sex_F": float(row["sex"] == "F"),
        "asa": row["asa"] if CONFIG["INCLUDE_STATIC_ASA"] else np.nan,
        "emop": row["emop"],
        "weight": row["weight"],
        "height": row["height"],
        "n_operations_total": row["n_operations"],
    }

static_df = cohort_df.set_index("subject_id").apply(static_features_row, axis=1, result_type="expand")
dept_onehot = pd.get_dummies(cohort_df.set_index("subject_id")["department"], prefix="dept").astype(float)
static_df = pd.concat([static_df, dept_onehot], axis=1)
if CONFIG["USE_ANTYPE"] and "antype" in cohort_df.columns:
    # v4: anaesthesia type of the last operation (planned before surgery) -- part of the
    # surgical context. Missing -> its own "missing" column rather than a guess.
    _antype = cohort_df.set_index("subject_id")["antype"].fillna("missing").astype(str).str.strip().str.lower()
    _antype = _antype.str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_").replace("", "missing")
    antype_onehot = pd.get_dummies(_antype, prefix="antype").astype(float)
    static_df = pd.concat([static_df, antype_onehot], axis=1)
    print("Anaesthesia types:", _antype.value_counts().to_dict())
static_df.describe()

## 6.13 Medications → ATC-level aggregate features (§1.5.3)

Per §1.5.3's decision, this notebook aggregates at the **WHO ATC level-2** (e.g. the first
3 characters of the ATC code, roughly the "therapeutic subgroup" level) rather than
per-drug — far fewer categories, each with enough examples in a small cohort to be a
usable count feature rather than a near-empty one-hot. A learned per-class embedding
(rather than a raw count) is a natural upgrade once the full cohort is available — flagged
as an extension, not built by default here, per §1.5.3's reasoning about label-starved
high-cardinality embeddings.

In [ ]:
def atc_level2(code):
    if not isinstance(code, str) or len(code) < 3:
        return None
    return code[:3].upper()

def medication_features(sid, lo, hi):
    meds = meds_in_window(sid, lo, hi)
    atc2 = meds["atc_code"].dropna().apply(atc_level2)
    return {
        "n_medication_administrations": int(len(meds)),
        "n_distinct_atc2_classes": int(atc2.nunique()),
        "n_distinct_drug_names": int(meds["drug_name"].nunique()) if "drug_name" in meds else 0,
    }

MED_FEATURES = {}
for sid in cohort_df["subject_id"]:
    row = COHORT_INDEXED.loc[sid]
    lo, hi = get_window(sid, row)
    MED_FEATURES[sid] = medication_features(sid, lo, hi)

med_features_df = pd.DataFrame(MED_FEATURES).T
med_features_df.index.name = "subject_id"
med_features_df.describe()

## 6.14b Organ-system content from procedure codes and medication families (v4)

In this dataset there is no blood test or bedside measurement that belongs only to the gut
(GI) or to bones and muscles (MSK), so in v3 those two systems learned from their diagnosis
codes alone. Two sources already in the data were unused and give every organ system —
GI and MSK most of all — something real to read:

- **What the operation is on.** The 2nd character of an ICD-10-PCS medical-surgical code
  (section `0`) is the body system operated on: `D` gastrointestinal, `F` hepatobiliary &
  pancreas, `K`–`S` muscles, tendons, ligaments, bones and joints, `2`–`6` heart and
  vessels, `T` urinary, `B` respiratory, `0`–`1` nervous system, `7` lymphatic/blood,
  `G` endocrine. For each organ system: is the LAST operation (the one being predicted,
  planned before surgery) on it, and how many EARLIER operations were.
- **Which medicine families were given in the pre-op window.** WHO ATC groups: `A`
  alimentary tract (except `A10` diabetes drugs → metabolism), `M` musculo-skeletal, `C`
  cardiovascular, `R` respiratory, `N` nervous system, `B` blood, `H` hormones and `A10`
  → metabolism, `G04` urologicals → kidneys. Count of administrations per system (log1p).

Each new column is named `proc_<system>_...` / `atc_<system>_...` and is owned by that
system in §9.6 — every system is still built the same way. Mapping printed for sign-off.
**Caveat to check:** for emergency operations the recorded procedure code is what was
actually done, which a surgeon may only fully know once the operation starts.

In [ ]:
PCS_BODY_SYSTEM_TO_ORGAN = {
    "0": "neurological", "1": "neurological",
    "2": "cardiovascular", "3": "cardiovascular", "4": "cardiovascular", "5": "cardiovascular", "6": "cardiovascular",
    "7": "haematology", "B": "respiratory", "D": "gi", "F": "gi", "G": "metabolic_hepatic",
    "K": "msk", "L": "msk", "M": "msk", "N": "msk", "P": "msk", "Q": "msk", "R": "msk", "S": "msk",
    "T": "renal",
}

def pcs_organ_system(code):
    # ICD-10-PCS: char 1 = section ('0' = medical & surgical), char 2 = body system.
    if not isinstance(code, str):
        return None
    c = code.strip().upper()
    if len(c) < 2 or c[0] != "0":
        return None
    return PCS_BODY_SYSTEM_TO_ORGAN.get(c[1])

def atc_organ_system(code):
    if not isinstance(code, str) or not code.strip():
        return None
    c = code.strip().upper()
    if c.startswith("A10") or c.startswith("H"):
        return "metabolic_hepatic"
    if c.startswith("G04"):
        return "renal"
    return {"A": "gi", "M": "msk", "C": "cardiovascular", "R": "respiratory", "N": "neurological",
            "B": "haematology"}.get(c[0])

def organ_content_features(sid, lo, hi):
    feats = {}
    if CONFIG["USE_PROCEDURE_CODES"]:
        history = OPS_HISTORY[sid]
        last_sys = pcs_organ_system(history[-1].get("icd10_pcs")) if history else None
        prior = Counter(pcs_organ_system(op.get("icd10_pcs")) for op in history[:-1])
        for s in ORGAN_SYSTEMS:
            feats[f"proc_{s}_last_op"] = float(last_sys == s)
            feats[f"proc_{s}_n_prior_ops"] = float(prior.get(s, 0))
    if CONFIG["USE_ATC_SYSTEM_GROUPS"]:
        meds = meds_in_window(sid, lo, hi)
        counts = Counter(atc_organ_system(c) for c in meds["atc_code"].dropna()) if len(meds) else Counter()
        for s in ORGAN_SYSTEMS:
            feats[f"atc_{s}_log_n_admin"] = float(np.log1p(counts.get(s, 0)))
    return feats

ORGAN_CONTENT_FEATURES = {}
for sid in cohort_df["subject_id"]:
    _lo, _hi = get_window(sid, COHORT_INDEXED.loc[sid])
    ORGAN_CONTENT_FEATURES[sid] = organ_content_features(sid, _lo, _hi)
organ_content_df = pd.DataFrame(ORGAN_CONTENT_FEATURES).T
organ_content_df.index.name = "subject_id"

if len(organ_content_df.columns):
    _y = COHORT_INDEXED.loc[organ_content_df.index, "died_30day_from_last_op"].astype(float)
    _rows = []
    for s in ORGAN_SYSTEMS:
        r = {"organ_system": s}
        if f"proc_{s}_last_op" in organ_content_df:
            on = organ_content_df[f"proc_{s}_last_op"] > 0
            r.update({"last_op_on_this_system": int(on.sum()),
                      "death_rate_if_yes": float(_y[on].mean()) if on.any() else np.nan,
                      "death_rate_if_no": float(_y[~on].mean()) if (~on).any() else np.nan})
        if f"atc_{s}_log_n_admin" in organ_content_df:
            r["patients_with_any_medicine_in_family"] = int((organ_content_df[f"atc_{s}_log_n_admin"] > 0).sum())
        _rows.append(r)
    ORGAN_CONTENT_SUMMARY = pd.DataFrame(_rows).set_index("organ_system")
    print("v4 organ-system content from procedure codes + medicine families (descriptive only):")
    print(ORGAN_CONTENT_SUMMARY.round(4).to_string())
    _n_pcs_unmapped = sum(pcs_organ_system(op.get("icd10_pcs")) is None for h in OPS_HISTORY.values() for op in h[-1:])
    print(f"\nLast operations whose procedure code maps to no organ system (eye, ear, skin, reproductive, "
          f"general regions, non-surgical sections or missing): {_n_pcs_unmapped} / {len(OPS_HISTORY)}")
else:
    print("v4 procedure/medicine-family features switched off.")

## 6.15 Aggregated time-series-derived static features (Part C §4 of the imbalance/imputation reference)

The second external review's most concretely useful suggestion: give the static branch
richer clinical summaries of the time series, rather than only demographics/ICD-10/
operation-history. Two benefits at once — it enriches what the DNN's static branch has to
work with, **and** it gives §8's grouped SMOTENC more informative features to interpolate
over. For each patient, within the same window as §6.2 (`get_window`):

- **Mean / min / max / std** of six clinically load-bearing signals: `creatinine`,
  `potassium`, `glucose`, `wbc`, `lactate`, `hb` (labs) and `hr`, `nibp_mbp`, `spo2`
  (ward/intra-op vitals) — 9 signals × 4 statistics = 36 features.
- **Severe-hypotension reading count** — number of mean-arterial-pressure readings below
  65 mmHg in the window. Stated precisely as a **reading count, not a duration**: INSPIRE's
  vitals are irregularly sampled, so "minutes spent hypotensive" would need an assumption
  about how long each reading represents, which this notebook avoids asserting.
- **Vasopressor administration count** and **high-alert medication administration count**
  — INSPIRE's `medications` table has **no dose field** (checked directly against the
  schema in §3), so the reviewed suggestion's "cumulative mcg dose" isn't computable as
  asked. This substitutes an **administration count** for a fixed drug-name/ATC keyword
  list — a real, honestly-weaker proxy (it can't distinguish one large dose from several
  small ones, or detect a true *escalation* in dose over time), stated here rather than
  silently implied to be the same thing.

In [ ]:
AGG_LABS = ["creatinine", "potassium", "glucose", "wbc", "lacate", "hb"]     # 'lacate' matches this dataset's item_name spelling (see §4.1)
AGG_WARD_VITALS = ["hr", "nibp_mbp", "spo2"]
SEVERE_HYPOTENSION_MAP_THRESHOLD = 65.0   # mmHg, a standard clinical cutoff for organ-perfusion risk

# Fixed keyword lists rather than a formal drug ontology lookup -- simple, auditable, and
# easy to extend; matched case-insensitively against drug_name (route/ATC not required).
VASOPRESSOR_KEYWORDS = ["norepinephrine", "noradrenaline", "epinephrine", "adrenaline",
                         "dopamine", "dobutamine", "vasopressin", "phenylephrine"]
HIGH_ALERT_KEYWORDS = VASOPRESSOR_KEYWORDS + ["fentanyl", "propofol", "midazolam", "heparin", "insulin"]

def _series_stats(chart_time2value, prefix):
    if not chart_time2value:
        return {f"{prefix}_mean": np.nan, f"{prefix}_min": np.nan, f"{prefix}_max": np.nan, f"{prefix}_std": np.nan}
    vals = np.array(list(chart_time2value.values()), dtype=float)
    return {
        f"{prefix}_mean": float(np.mean(vals)),
        f"{prefix}_min": float(np.min(vals)),
        f"{prefix}_max": float(np.max(vals)),
        f"{prefix}_std": float(np.std(vals)) if len(vals) > 1 else 0.0,
    }

def aggregate_features_for_patient(sid, lo, hi):
    feats = {}
    for lab in AGG_LABS:
        series = windowed_series(labs_df, sid, lab, lo, hi)
        feats.update(_series_stats(series, f"agg_{lab}"))
    for vital in AGG_WARD_VITALS:
        series = windowed_series(ward_vitals_df, sid, vital, lo, hi)
        feats.update(_series_stats(series, f"agg_{vital}"))
        if CONFIG["TIME_WINDOW"] == "peri_op" and vital in SYSTEM_INTRAOP_VITALS.get("cardiovascular", []) + SYSTEM_INTRAOP_VITALS.get("respiratory", []):
            iv_series = windowed_series(vitals_df, sid, vital, lo, hi)
            for t, v in iv_series.items():
                series.setdefault(t, v)

    map_series = windowed_series(ward_vitals_df, sid, "nibp_mbp", lo, hi)
    feats["severe_hypotension_reading_count"] = int(sum(1 for v in map_series.values() if v < SEVERE_HYPOTENSION_MAP_THRESHOLD))

    meds = meds_in_window(sid, lo, hi)
    if len(meds) == 0:
        feats["vasopressor_administration_count"] = 0
        feats["high_alert_med_administration_count"] = 0
    else:
        # .astype(object) (not .astype(str)) deliberately -- on some pandas versions,
        # .astype(str) on an EMPTY series produces the newer pandas "str" extension dtype,
        # whose .sum() concatenates strings ('') instead of summing booleans numerically.
        # object dtype sidesteps that entirely and behaves the same either way.
        drug_names_lower = meds["drug_name"].astype(object).str.lower()
        feats["vasopressor_administration_count"] = int(drug_names_lower.apply(lambda n: any(k in n for k in VASOPRESSOR_KEYWORDS)).astype(bool).sum())
        feats["high_alert_med_administration_count"] = int(drug_names_lower.apply(lambda n: any(k in n for k in HIGH_ALERT_KEYWORDS)).astype(bool).sum())

    return feats

AGG_FEATURES = {}
for sid in cohort_df["subject_id"]:
    row = COHORT_INDEXED.loc[sid]
    lo, hi = get_window(sid, row)
    AGG_FEATURES[sid] = aggregate_features_for_patient(sid, lo, hi)

agg_features_df = pd.DataFrame(AGG_FEATURES).T
agg_features_df.index.name = "subject_id"
print(f"Aggregated features built: {agg_features_df.shape[1]} new static features "
      f"(36 mean/min/max/std + 3 count-based)")
print(f"Patients with >=1 severe-hypotension reading: "
      f"{(agg_features_df['severe_hypotension_reading_count'] > 0).sum()} / {len(agg_features_df)}")
print(f"Patients with >=1 vasopressor administration in-window: "
      f"{(agg_features_df['vasopressor_administration_count'] > 0).sum()} / {len(agg_features_df)}")
agg_features_df.describe().T[["mean", "std", "min", "max"]].head(10)

## 6.16 NEWS2 early-warning score (v2) — computed, verified, then included

NEWS2 (Royal College of Physicians, 2017) is built here from the **pre-op ward vitals
already loaded** (RR, SpO2, FiO2/ventilator for "air or oxygen", SBP, HR, temperature,
GCS for consciousness) — same window as every other feature, so no leakage from surgery
or after it. It is added to the static vector **before** SMOTENC, so synthetic patients
get an interpolated NEWS2 like any other continuous feature; Part 8.2b and Part 9.1b then
check NEWS2 **before vs. after** each augmentation step.

"Only include it if it is right" is enforced mechanically: the scoring function is
self-tested against every band edge of the RCP chart below, and if any check fails
NEWS2 is **left out of the model** (with the failures printed) rather than included.

Documented approximations (for clinical sign-off): SpO2 Scale 1 for everyone; AVPU is
not recorded so GCS stands in (all observed components at maximum = Alert); missing
FiO2/ventilator = room air; a set needs ≥ `NEWS2_MIN_CORE_VITALS` of the five measured
vitals or it gets no score (never a falsely reassuring low one).

In [ ]:
# --- inlined from src-dnn/inspire_dnn/news2.py (edit it there, then rebuild) ---
# NEWS2 -- National Early Warning Score 2 (Royal College of Physicians, 2017).
#
# Vectorised numpy implementation: every band function takes a scalar OR an array and
# returns float scores with NaN wherever the input is missing, so one code path serves
# single readings, whole observation sets, and the resampled model grids used by the
# augmentation check.
#
# Scoring table implemented (RCP NEWS2 chart, SpO2 Scale 1):
#   Respiration rate   <=8:3 | 9-11:1 | 12-20:0 | 21-24:2 | >=25:3
#   SpO2 (Scale 1)     <=91:3 | 92-93:2 | 94-95:1 | >=96:0
#   Air or oxygen      oxygen:2 | air:0
#   Systolic BP        <=90:3 | 91-100:2 | 101-110:1 | 111-219:0 | >=220:3
#   Pulse              <=40:3 | 41-50:1 | 51-90:0 | 91-110:1 | 111-130:2 | >=131:3
#   Consciousness      Alert:0 | new confusion / V / P / U:3
#   Temperature        <=35.0:3 | 35.1-36.0:1 | 36.1-38.0:0 | 38.1-39.0:1 | >=39.1:2
# Aggregate 0-20. Clinical response bands: 0-4 low, any single parameter scoring 3
# "low-medium" (urgent ward review), 5-6 medium, >=7 high.
#
# Documented approximations for THIS dataset (flagged for clinical sign-off):
#   * Scale 1 is used for everyone -- Scale 2 needs a clinician-set hypercapnic target
#     range that the dataset does not record.
#   * Consciousness: AVPU is not recorded, so GCS is used -- "Alert" only if every
#     observed GCS component is at its maximum (E4, M6, V5); anything lower scores 3.
#   * Oxygen: "on oxygen" = FiO2 above room air (>21%, either % or fraction units) or a
#     ventilator flag. Missing FiO2 is treated as room air only if
#     assume_air_if_missing=True (pre-op ward patients are usually on air) -- the
#     alternative is to leave that component unscored.
import numpy as np

NEWS2_PARAMS = ("rr", "spo2", "on_oxygen", "sbp", "hr", "alert", "temp_c")
NEWS2_CORE_VITALS = ("rr", "spo2", "sbp", "hr", "temp_c")   # the five measured vitals


def _arr(x):
    return np.asarray(x, dtype=float)


def _banded(x, conditions_scores):
    # conditions_scores: list of (boolean_array, score); first match wins; NaN stays NaN.
    x = _arr(x)
    out = np.full(x.shape, np.nan)
    assigned = np.isnan(x)
    for cond, score in conditions_scores:
        take = cond & ~assigned
        out[take] = score
        assigned |= take
    return out


def score_respiration_rate(rr):
    rr = _arr(rr)
    return _banded(rr, [(rr <= 8, 3), (rr < 12, 1), (rr <= 20, 0), (rr <= 24, 2), (rr > 24, 3)])


def score_spo2_scale1(spo2):
    s = _arr(spo2)
    return _banded(s, [(s <= 91, 3), (s < 94, 2), (s < 96, 1), (s >= 96, 0)])


def score_air_or_oxygen(on_oxygen):
    o = _arr(on_oxygen)
    return _banded(o, [(o > 0.5, 2), (o <= 0.5, 0)])


def score_systolic_bp(sbp):
    b = _arr(sbp)
    return _banded(b, [(b <= 90, 3), (b <= 100, 2), (b <= 110, 1), (b < 220, 0), (b >= 220, 3)])


def score_pulse(hr):
    h = _arr(hr)
    return _banded(h, [(h <= 40, 3), (h <= 50, 1), (h <= 90, 0), (h <= 110, 1), (h <= 130, 2), (h > 130, 3)])


def score_consciousness(alert):
    a = _arr(alert)
    return _banded(a, [(a > 0.5, 0), (a <= 0.5, 3)])


def score_temperature(temp_c):
    t = _arr(temp_c)
    return _banded(t, [(t <= 35.0, 3), (t <= 36.0, 1), (t <= 38.0, 0), (t <= 39.0, 1), (t > 39.0, 2)])


SCORERS = {
    "rr": score_respiration_rate, "spo2": score_spo2_scale1, "on_oxygen": score_air_or_oxygen,
    "sbp": score_systolic_bp, "hr": score_pulse, "alert": score_consciousness, "temp_c": score_temperature,
}


def on_supplemental_oxygen(fio2=np.nan, vent=np.nan, assume_air_if_missing=True):
    # FiO2 may be recorded as a percentage (21-100) or a fraction (0.21-1.0): anything
    # <= 1.0 is read as a fraction. "Above room air" uses a small tolerance (21.5%) so a
    # charted 21% is air, not oxygen.
    f = _arr(fio2)
    v = _arr(vent)
    f_pct = np.where(f <= 1.0, f * 100.0, f)
    on = np.where(np.isnan(f_pct), np.nan, (f_pct > 21.5).astype(float))
    on = np.where((~np.isnan(v)) & (v > 0), 1.0, on)
    if assume_air_if_missing:
        on = np.where(np.isnan(on), 0.0, on)
    return on


def alert_from_gcs(gcs_e=np.nan, gcs_m=np.nan, gcs_v=np.nan):
    # 1.0 = Alert, 0.0 = not alert, NaN = no GCS component observed.
    e, m, v = _arr(gcs_e), _arr(gcs_m), _arr(gcs_v)
    below_max = ((~np.isnan(e)) & (e < 4)) | ((~np.isnan(m)) & (m < 6)) | ((~np.isnan(v)) & (v < 5))
    any_obs = ~(np.isnan(e) & np.isnan(m) & np.isnan(v))
    return np.where(~any_obs, np.nan, np.where(below_max, 0.0, 1.0))


def news2_components(rr=np.nan, spo2=np.nan, on_oxygen=np.nan, sbp=np.nan, hr=np.nan,
                     alert=np.nan, temp_c=np.nan):
    vals = dict(rr=rr, spo2=spo2, on_oxygen=on_oxygen, sbp=sbp, hr=hr, alert=alert, temp_c=temp_c)
    return {p: SCORERS[p](vals[p]) for p in NEWS2_PARAMS}


def news2_total(components, assume_alert_if_missing=True, min_core_vitals=3):
    # Returns (total, n_core_vitals_observed, any_single_3). A missing component
    # contributes 0 -- which is why a total is only reported (not NaN) when at least
    # `min_core_vitals` of the five measured vitals were actually observed.
    stacked = np.stack([_arr(components[p]) for p in NEWS2_PARAMS])
    if assume_alert_if_missing:
        i = NEWS2_PARAMS.index("alert")
        stacked[i] = np.where(np.isnan(stacked[i]), 0.0, stacked[i])
    total = np.nansum(stacked, axis=0)
    n_core = np.sum(~np.isnan(np.stack([_arr(components[p]) for p in NEWS2_CORE_VITALS])), axis=0)
    any3 = np.any(stacked == 3, axis=0)
    total = np.where(n_core >= min_core_vitals, total, np.nan)
    return total, n_core, any3


def news2_risk_band(total, any_single_3):
    # 0 = low (0-4), 1 = low-medium (single parameter 3, total <5), 2 = medium (5-6),
    # 3 = high (>=7); NaN where total is NaN.
    t = _arr(total)
    a = np.asarray(any_single_3, dtype=bool)
    band = np.where(t >= 7, 3, np.where(t >= 5, 2, np.where(a, 1, 0))).astype(float)
    return np.where(np.isnan(t), np.nan, band)


def news2_observation_sets(series_by_param, bucket_minutes=60, carry_forward_minutes=240,
                           assume_air_if_missing=True, assume_alert_if_missing=True,
                           min_core_vitals=3):
    # series_by_param: {raw_item_name: {chart_time: value}} using the dataset's ward-vital
    # names: rr, spo2, fio2, vent, bt, nibp_sbp, hr, gcs_e, gcs_m, gcs_v.
    # Ward vitals are charted as a "set" at roughly the same time; each set is
    # reconstructed by bucketing chart_times into `bucket_minutes` bins, and a parameter
    # missing from a bin is carried forward from its last reading only if that reading is
    # within `carry_forward_minutes` (bounded LOCF -- never across a long gap).
    # Returns a dict of equal-length arrays, one entry per bucket that had any reading.
    all_times = sorted({t for s in series_by_param.values() for t in s})
    if not all_times:
        return {"time": np.array([]), "total": np.array([]), "n_core": np.array([]),
                "any3": np.array([], dtype=bool), "band": np.array([])}
    t0 = all_times[0]
    bucket_of = lambda t: int((t - t0) // bucket_minutes)
    buckets = sorted({bucket_of(t) for t in all_times})
    bucket_end = np.array([t0 + (b + 1) * bucket_minutes for b in buckets], dtype=float)

    def value_at_buckets(item):
        s = series_by_param.get(item) or {}
        out = np.full(len(buckets), np.nan)
        if not s:
            return out
        times = np.array(sorted(s), dtype=float)
        vals = np.array([s[t] for t in sorted(s)], dtype=float)
        for i, end in enumerate(bucket_end):
            j = np.searchsorted(times, end, side="left") - 1   # last reading before bucket end
            if j >= 0 and (end - times[j]) <= carry_forward_minutes + bucket_minutes:
                out[i] = vals[j]
        return out

    rr, spo2, sbp, hr, bt = (value_at_buckets(k) for k in ("rr", "spo2", "nibp_sbp", "hr", "bt"))
    on_o2 = on_supplemental_oxygen(value_at_buckets("fio2"), value_at_buckets("vent"),
                                   assume_air_if_missing=assume_air_if_missing)
    alert = alert_from_gcs(value_at_buckets("gcs_e"), value_at_buckets("gcs_m"), value_at_buckets("gcs_v"))
    comps = news2_components(rr=rr, spo2=spo2, on_oxygen=on_o2, sbp=sbp, hr=hr, alert=alert, temp_c=bt)
    total, n_core, any3 = news2_total(comps, assume_alert_if_missing, min_core_vitals)
    return {"time": bucket_end, "total": total, "n_core": n_core, "any3": any3,
            "band": news2_risk_band(total, any3)}


def news2_summary_features(sets):
    # Static per-patient summary of the observation sets. NaN where no valid set exists
    # (filled by the normal static-feature imputation), with news2_available recording
    # whether a real score existed -- the same observed/imputed discipline as the masks.
    valid = ~np.isnan(sets["total"]) if len(sets["total"]) else np.array([], dtype=bool)
    if not valid.any():
        return {"news2_max": np.nan, "news2_last": np.nan, "news2_mean": np.nan,
                "news2_n_sets": 0.0, "news2_high_flag": 0.0, "news2_red_flag": 0.0,
                "news2_available": 0.0}
    tot = sets["total"][valid]
    return {
        "news2_max": float(np.max(tot)),
        "news2_last": float(tot[-1]),
        "news2_mean": float(np.mean(tot)),
        "news2_n_sets": float(len(tot)),
        "news2_high_flag": float(np.any(tot >= 7)),                             # RCP 'high' band ever reached
        "news2_red_flag": float(np.any(sets["any3"][valid]) or np.any(tot >= 5)),  # medium-or-worse ever
        "news2_available": 1.0,
    }


NEWS2_FEATURE_NAMES = ("news2_max", "news2_last", "news2_mean", "news2_n_sets",
                       "news2_high_flag", "news2_red_flag", "news2_available")
NEWS2_BINARY_FEATURES = ("news2_high_flag", "news2_red_flag", "news2_available")


def news2_self_test():
    # Checks every band edge of the official RCP table plus two worked totals. Returns
    # (passed: bool, failures: list[str]). The pipeline only includes NEWS2 as a model
    # feature if this passes -- "only include it if it is right", made mechanical.
    cases = [
        (score_respiration_rate, [(8, 3), (9, 1), (11, 1), (12, 0), (20, 0), (21, 2), (24, 2), (25, 3), (5, 3), (40, 3)]),
        (score_spo2_scale1, [(91, 3), (92, 2), (93, 2), (94, 1), (95, 1), (96, 0), (100, 0), (85, 3)]),
        (score_air_or_oxygen, [(1, 2), (0, 0)]),
        (score_systolic_bp, [(90, 3), (91, 2), (100, 2), (101, 1), (110, 1), (111, 0), (219, 0), (220, 3), (70, 3)]),
        (score_pulse, [(40, 3), (41, 1), (50, 1), (51, 0), (90, 0), (91, 1), (110, 1), (111, 2), (130, 2), (131, 3)]),
        (score_consciousness, [(1, 0), (0, 3)]),
        (score_temperature, [(35.0, 3), (35.1, 1), (36.0, 1), (36.1, 0), (38.0, 0), (38.1, 1), (39.0, 1), (39.1, 2), (34.0, 3)]),
    ]
    failures = []
    for fn, pairs in cases:
        for x, want in pairs:
            got = float(fn(x))
            if got != want:
                failures.append(f"{fn.__name__}({x}) = {got}, expected {want}")
    if not np.isnan(float(score_pulse(np.nan))):
        failures.append("missing input must stay NaN")
    # Worked example 1 -- everything normal on air, alert: total 0, low band.
    c = news2_components(rr=16, spo2=98, on_oxygen=0, sbp=125, hr=72, alert=1, temp_c=36.8)
    t, n, a = news2_total(c)
    if float(t) != 0 or float(news2_risk_band(t, a)) != 0:
        failures.append(f"normal patient total {float(t)} (expected 0, band low)")
    # Worked example 2 -- RR 22 (2), SpO2 93 (2), O2 (2), SBP 105 (1), HR 115 (2), alert (0), 38.5 (1) = 10, high.
    c = news2_components(rr=22, spo2=93, on_oxygen=1, sbp=105, hr=115, alert=1, temp_c=38.5)
    t, n, a = news2_total(c)
    if float(t) != 10 or float(news2_risk_band(t, a)) != 3:
        failures.append(f"sick patient total {float(t)} (expected 10, band high)")
    # Single red score with a low total -> low-medium band.
    c = news2_components(rr=16, spo2=98, on_oxygen=0, sbp=125, hr=72, alert=0, temp_c=36.8)
    t, n, a = news2_total(c)
    if float(t) != 3 or float(news2_risk_band(t, a)) != 1:
        failures.append(f"single-red-score patient total {float(t)} band {float(news2_risk_band(t, a))} (expected 3, low-medium)")
    # Unit handling and GCS mapping.
    if float(on_supplemental_oxygen(0.21)) != 0 or float(on_supplemental_oxygen(40)) != 1 or float(on_supplemental_oxygen(np.nan, 1)) != 1:
        failures.append("FiO2/vent -> oxygen mapping")
    if float(alert_from_gcs(4, 6, 5)) != 1 or float(alert_from_gcs(4, 6, 4)) != 0 or not np.isnan(float(alert_from_gcs())):
        failures.append("GCS -> alert mapping")
    # Too few observed vitals -> no score rather than a falsely reassuring low one.
    c = news2_components(hr=72)
    t, n, a = news2_total(c, min_core_vitals=3)
    if not np.isnan(float(t)):
        failures.append("a set with only 1 core vital must not produce a score")
    return (len(failures) == 0), failures


def compare_before_after(before, after, name="news2", plausible_range=(0.0, 20.0), min_n=20):
    # Distribution check between a "before" sample (real patients) and an "after" sample
    # (synthetic or augmented patients). Returns a plain dict for printing/logging.
    b = _arr(before); b = b[~np.isnan(b)]
    a = _arr(after); a = a[~np.isnan(a)]
    res = {"feature": name, "n_before": int(len(b)), "n_after": int(len(a))}
    if len(b) == 0 or len(a) == 0:
        res["verdict"] = "not enough data"
        return res
    lo, hi = plausible_range
    res.update({
        "median_before": float(np.median(b)), "median_after": float(np.median(a)),
        "mean_before": float(np.mean(b)), "mean_after": float(np.mean(a)),
        "pct_high_before": float(np.mean(b >= 7)), "pct_high_after": float(np.mean(a >= 7)),
        "n_outside_range_after": int(np.sum((a < lo) | (a > hi))),
    })
    try:
        from scipy.stats import ks_2samp
        ks = ks_2samp(b, a)
        res["ks_stat"], res["ks_p"] = float(ks.statistic), float(ks.pvalue)
    except Exception:
        res["ks_stat"], res["ks_p"] = np.nan, np.nan
    # Verdict: implausible values are a hard fail; otherwise judge the SIZE of the shift
    # (KS statistic), not just its p-value -- with hundreds of synthetic rows a clinically
    # trivial shift is "significant".
    if res["n_outside_range_after"] > 0:
        res["verdict"] = "FAIL: values outside the possible NEWS2 range"
    elif len(a) < min_n:
        res["verdict"] = f"TOO FEW to judge the shape (n_after={len(a)} < {min_n}); range check passed"
    elif not np.isnan(res["ks_stat"]) and res["ks_stat"] > 0.2:
        res["verdict"] = "REVIEW: distribution shifted (KS > 0.2)"
    else:
        res["verdict"] = "PASS"
    return res

In [ ]:
NEWS2_VALIDATED, _news2_failures = news2_self_test()
print(f"NEWS2 scoring self-test against the RCP 2017 chart: {'PASSED' if NEWS2_VALIDATED else 'FAILED'}")
for _f in _news2_failures:
    print("   ", _f)
USE_NEWS2 = bool(CONFIG["INCLUDE_NEWS2"] and NEWS2_VALIDATED)
if CONFIG["INCLUDE_NEWS2"] and not NEWS2_VALIDATED:
    print("NEWS2 will NOT be added to the model this run -- fix the failures above first.")
elif not CONFIG["INCLUDE_NEWS2"]:
    print("CONFIG['INCLUDE_NEWS2']=False -- NEWS2 computed for reference only, not used as a feature.")

In [ ]:
NEWS2_INPUT_ITEMS = ["rr", "spo2", "fio2", "vent", "bt", "nibp_sbp", "hr", "gcs_e", "gcs_m", "gcs_v"]
_missing_items = [i for i in NEWS2_INPUT_ITEMS if i not in set(INVENTORY["ward_vitals"])]
if _missing_items:
    print(f"Ward-vital items not present in this cohort (scored as missing): {_missing_items}")

NEWS2_SETS, NEWS2_FEATURES = {}, {}
_t0 = time.time()
for sid in cohort_df["subject_id"]:
    lo, hi = get_window(sid, COHORT_INDEXED.loc[sid])
    series = {item: windowed_series(ward_vitals_df, sid, item, lo, hi) for item in NEWS2_INPUT_ITEMS}
    sets = news2_observation_sets(
        series, bucket_minutes=CONFIG["NEWS2_BUCKET_MINUTES"],
        carry_forward_minutes=CONFIG["NEWS2_CARRY_FORWARD_MINUTES"],
        assume_air_if_missing=CONFIG["NEWS2_ASSUME_AIR_IF_MISSING"],
        assume_alert_if_missing=CONFIG["NEWS2_ASSUME_ALERT_IF_MISSING"],
        min_core_vitals=CONFIG["NEWS2_MIN_CORE_VITALS"])
    NEWS2_SETS[sid] = sets
    NEWS2_FEATURES[sid] = news2_summary_features(sets)
news2_features_df = pd.DataFrame(NEWS2_FEATURES).T
news2_features_df.index.name = "subject_id"
print(f"NEWS2 computed for {len(news2_features_df)} patients in {time.time() - _t0:.1f}s")
print(f"  patients with >=1 scorable observation set: {int(news2_features_df['news2_available'].sum())} "
      f"({news2_features_df['news2_available'].mean():.1%})")
_all_n_core = np.concatenate([s["n_core"] for s in NEWS2_SETS.values() if len(s["n_core"])]) \
    if any(len(s["n_core"]) for s in NEWS2_SETS.values()) else np.array([])
if len(_all_n_core):
    print(f"  core vitals observed per set (of 5): mean {_all_n_core.mean():.2f}; "
          f"sets with all 5: {np.mean(_all_n_core == 5):.1%} -- the rest had a component carried "
          f"forward or unscored (see the approximation note above)")
news2_features_df.describe().T[["mean", "std", "min", "max"]]

In [ ]:
# Construct-validity check (descriptive, like §5.1's ASA check): if NEWS2 is being computed
# correctly, 30-day mortality should rise with the worst pre-op NEWS2 band.
_v = news2_features_df.join(COHORT_INDEXED[["died_30day_from_last_op"]])
_v = _v[_v["news2_available"] == 1]
NEWS2_MORTALITY_MONOTONE = None
if len(_v):
    _v["worst_band"] = pd.cut(_v["news2_max"], bins=[-0.1, 4, 6, 20], labels=["low (0-4)", "medium (5-6)", "high (7+)"])
    news2_validity = _v.groupby("worst_band", observed=False)["died_30day_from_last_op"].agg(["mean", "count"])
    news2_validity.columns = ["mortality_rate", "n_patients"]
    print(news2_validity)
    _rates = news2_validity["mortality_rate"].dropna().values
    NEWS2_MORTALITY_MONOTONE = bool(len(_rates) >= 2 and np.all(np.diff(_rates) >= 0))
    print(f"\nMortality rises across NEWS2 bands: {'YES' if NEWS2_MORTALITY_MONOTONE else 'NO -- check before trusting the feature'}"
          f" (a sanity check on the implementation, not a model result)")
    fig, ax = plt.subplots(figsize=(5, 3.5))
    news2_validity["mortality_rate"].plot(kind="bar", ax=ax, color="#c0392b")
    ax.set_title("30-day mortality by worst pre-op NEWS2 band"); ax.set_ylabel("mortality rate"); ax.set_xlabel("")
    plt.tight_layout(); plt.show()

## 6.14 Assembling the final per-patient feature bundle

Pulls everything from §6.1–§6.13 into one `PATIENT_BUNDLE` dict keyed by `subject_id`,
containing:

- `ts[system]` → `{"raw": [T,F] with NaN gaps, "mask": [T,F], "feature_names": [...]}`
  for each of the six time-series systems (renal/cardiovascular/respiratory/
  metabolic_hepatic/haematology/neurological) — still **unimputed**, exactly as built in
  §6.3, ready for §7.
- `static` → one flat vector: demographics/ASA/department (§6.12), GI/MSK/HFRS (§6.6–6.8),
  operation-count features (§6.10), the cardiac-recovery exception flag (§6.11), the
  cardiovascular summary + renal-cardiac interaction (§6.9), and medication aggregates
  (§6.13).
- `label` → the 30-day mortality target from §4.2.

This is the single object every later section (§7 imputation, §8 split/sampling, §9 model)
reads from.

In [ ]:
def build_static_vector(sid):
    parts = {}
    parts.update(static_df.loc[sid].to_dict())
    parts.update(icd10_features_df.loc[sid].to_dict())
    parts.update(infection_features_df.loc[sid].to_dict())   # §6.6, roadmap §4.1a -- new this revision
    parts.update(op_count_df.loc[sid].to_dict())
    parts["cardiac_recovery_exception"] = CARDIAC_EXCEPTION_FLAG[sid]
    parts.update(cardiac_summary_df.loc[sid].to_dict())
    # v3: the hand-crafted creatinine x |MAP deviation| interaction (RENAL_CARDIAC_INTERACTION,
    # still computed above for reference) is NOT a model input: inter-system relationships
    # are learned by the coupling layer, never defined by hand.
    parts.update(med_features_df.loc[sid].to_dict())
    if len(organ_content_df.columns):  # v4: procedure body-system + medicine-family features per organ system
        parts.update(organ_content_df.loc[sid].to_dict())
    parts.update(AGG_FEATURES[sid])   # §6.15, Part C §4 of the imbalance/imputation reference
    if USE_NEWS2:                      # §6.16 (v2) -- only if the self-test passed
        parts.update(news2_features_df.loc[sid].to_dict())
    # v2 FIX: return the features in SORTED name order. STATIC_FEATURE_NAMES (below) is sorted,
    # and every later step that picks static columns BY NAME (SMOTENC's categorical columns,
    # the GI/MSK branches, NEWS2 checks, donor matching) indexes into that sorted list. In v1
    # the values stayed in build order, so names and values were silently misaligned (e.g. the
    # columns taken as 'gi_*' actually held white-cell-count statistics).
    return {k: parts[k] for k in sorted(parts)}

PATIENT_BUNDLE = {}
for sid in cohort_df["subject_id"]:
    ts = {system: RAW_SYSTEM_TENSORS[(sid, system)] for system in TIME_SERIES_SYSTEMS}
    PATIENT_BUNDLE[sid] = {
        "ts": ts,
        "static": build_static_vector(sid),
        "label": float(COHORT_INDEXED.loc[sid, "died_30day_from_last_op"]),
    }

STATIC_FEATURE_NAMES = sorted(next(iter(PATIENT_BUNDLE.values()))["static"].keys())
print(f"Bundle built for {len(PATIENT_BUNDLE)} patients.")
print(f"Static feature vector length: {len(STATIC_FEATURE_NAMES)}")
print(f"Static features: {STATIC_FEATURE_NAMES}")
for system in TIME_SERIES_SYSTEMS:
    fnames = PATIENT_BUNDLE[cohort_df['subject_id'].iloc[0]]['ts'][system]['feature_names']
    print(f"  {system:18s} time-series features ({len(fnames)}): {fnames}")

In [ ]:
# Free most raw long-format tables now that PATIENT_BUNDLE holds everything §7-§12 need --
# medications_df and diagnoses_df specifically, since nothing downstream reads them again.
# labs_df/vitals_df/ward_vitals_df are DELIBERATELY KEPT ALIVE (unlike an earlier revision
# of this cell, which freed all five): §11.8's risk-trajectory analysis needs to re-query
# these at custom time cutoffs after the model is trained, which isn't possible once
# they're gone. This keeps a modest amount of memory alive at full scale in exchange for
# that analysis working -- worth it now that Part 3's chunked parsing (not these already
# fairly compact flattened tables) is the fix that solved the real memory bottleneck.
import gc

_freed_mb = sum(df.memory_usage(deep=True).sum() for df in [medications_df, diagnoses_df]) / 1e6
del medications_df, diagnoses_df
RAW_SYSTEM_TENSORS.clear()   # PATIENT_BUNDLE['ts'][system] already holds these dicts directly -- see §6.3's note
gc.collect()
print(f"Freed medications_df/diagnoses_df (~{_freed_mb:.1f} MB on this dev subset). "
      f"labs_df/vitals_df/ward_vitals_df are kept alive -- needed by §11.8's risk-trajectory "
      f"analysis, which re-queries them at custom time cutoffs after training.")

In [ ]:
# Quick label sanity check on the assembled bundle before moving to §7.
labels = np.array([PATIENT_BUNDLE[sid]["label"] for sid in cohort_df["subject_id"]])
print(f"Positive rate in bundle: {labels.mean():.1%}  ({int(labels.sum())} / {len(labels)})")
print(f"For reference, the full cohort's rate is ~0.5-0.9% (§1.4). A run that keeps every death "
      f"but caps survivors (MAX_SUBJECTS_PER_CLASS={CONFIG['MAX_SUBJECTS_PER_CLASS']}) has a higher rate "
      f"than deployment -- AUPRC and calibration here are not deployment numbers.")
print(f"NEWS2 in the static vector: {USE_NEWS2}  |  renal_cardiac_interaction (manual coupling feature) "
      f"included: {'renal_cardiac_interaction' in STATIC_FEATURE_NAMES}")

In [ ]:
record_checkpoint("Parts 4-6 -- labelling + organ-system feature engineering")

# Part 7 — Missing data: implementing the §1.3 methods

Every method from §1.3.1's table is implemented here as a function with the same
signature, switched on by `CONFIG['IMPUTATION_STRATEGY']`. §7.5 runs a **side-by-side
comparison** of all four on the same data so you can see concretely what each one does to
the same gaps — this is the "experiment on it yourself" piece you asked for, made runnable
rather than theoretical.

Two things apply regardless of which strategy is chosen, per §1.3:

1. **The mask (§6.3) is never touched.** Whatever value-fill method runs, the mask feature
   stays exactly as observed — it's the model's only way to recover MNAR signal.
2. **Statistics (medians, KNN neighbours, MICE models) are always fit on the training split
   only** (computed in §8, referenced here) — fitting on the full dataset before splitting
   would leak test-set information into imputed training values, a common and serious
   mistake in exactly this kind of pipeline.

## 7.1 Whole-feature-missing fallback (applies before any of the strategies below)

In [ ]:
def population_stats(train_ids):
    # Per (system, feature) mean/median across the TRAINING split only -- used whenever a
    # patient has zero observations of a feature anywhere in their window (§1.3.2's first
    # branch: nothing else is possible in that case).
    stats = {}
    for system in TIME_SERIES_SYSTEMS:
        fnames = PATIENT_BUNDLE[train_ids[0]]["ts"][system]["feature_names"]
        if not fnames:
            continue
        all_vals = []
        for sid in train_ids:
            raw = PATIENT_BUNDLE[sid]["ts"][system]["raw"]
            all_vals.append(raw)
        stacked = np.concatenate(all_vals, axis=0) if all_vals else np.zeros((0, len(fnames)))
        for j, fname in enumerate(fnames):
            col = stacked[:, j]
            col = col[~np.isnan(col)]
            stats[(system, fname)] = {
                "mean": float(np.mean(col)) if len(col) else 0.0,
                "median": float(np.median(col)) if len(col) else 0.0,
            }
    return stats

## 7.2 The four imputation strategies

In [ ]:
def impute_median(raw, feature_names, system, stats):
    # §1.3.1 row 1/2: fill every NaN with the training-set median for that feature.
    out = raw.copy()
    for j, fname in enumerate(feature_names):
        med = stats[(system, fname)]["median"]
        col = out[:, j]
        col[np.isnan(col)] = med
    return out

def _fade_to_mean_interpolate(times_observed, values_observed, grid, mean_val, decay=8.0):
    # The source repo's smooth_fade_to_mean_interpolator idea (§1.3.1 row 3), reimplemented
    # standalone: linear interpolation between observed points, fading toward the population
    # mean the further a grid point is from any observation.
    if len(times_observed) == 0:
        return np.full(len(grid), mean_val)
    if len(times_observed) == 1:
        # can't interpolate a slope from one point -- fall back to nearest + fade
        t0, v0 = times_observed[0], values_observed[0]
        out = np.array([v0 * math.exp(-abs(g - t0) / decay) + mean_val * (1 - math.exp(-abs(g - t0) / decay)) for g in grid])
        return out
    interp = np.interp(grid, times_observed, values_observed)
    t_min, t_max = times_observed.min(), times_observed.max()
    out = interp.copy()
    for i, g in enumerate(grid):
        if g < t_min:
            w = math.exp(-abs(t_min - g) / decay)
            out[i] = w * interp[i] + (1 - w) * mean_val
        elif g > t_max:
            w = math.exp(-abs(g - t_max) / decay)
            out[i] = w * interp[i] + (1 - w) * mean_val
    return out

def impute_interpolate(raw, feature_names, system, stats, grid):
    # §1.3.1 row 3: within-patient interpolation, fading to the population mean outside
    # the observed range -- the default for fast-changing vitals per §1.3.2's decision tree.
    out = raw.copy()
    n_t = raw.shape[0]
    for j, fname in enumerate(feature_names):
        col = out[:, j]
        obs_idx = ~np.isnan(col)
        mean_val = stats[(system, fname)]["mean"]
        if obs_idx.sum() == 0:
            out[:, j] = mean_val
        else:
            out[:, j] = _fade_to_mean_interpolate(grid[obs_idx], col[obs_idx], grid, mean_val)
    return out

def impute_forward_fill(raw, feature_names, system, stats):
    # §1.3.1 row 4 (LOCF): repeat the last observed value forward; before any observation,
    # fall back to the training-set median (there's nothing to carry forward from yet).
    out = raw.copy()
    n_t = raw.shape[0]
    for j in range(out.shape[1]):
        last_val = None
        med = stats[(system, feature_names[j])]["median"]
        for t in range(n_t):
            if np.isnan(out[t, j]):
                out[t, j] = last_val if last_val is not None else med
            else:
                last_val = out[t, j]
    return out

def impute_knn(raw_matrix_all_patients, k):
    # §1.3.1 row 5: KNN across patients, one call per (system) on a [n_patients, T*F]
    # flattened matrix -- used at the whole-cohort level in §7.3, not per-patient like the
    # other three, since KNN needs other patients to find neighbours from.
    imputer = KNNImputer(n_neighbors=min(k, max(raw_matrix_all_patients.shape[0] - 1, 1)))
    return imputer.fit_transform(raw_matrix_all_patients)

## 7.3 The default decision-tree strategy (§1.3.2), and static-feature imputation

Fast-changing intra-op/ward vitals (`hr`, `nibp_*`, `spo2`, `rr`, `fio2`, `bt`, ...) use
interpolate-and-fade; slower-changing labs use forward-fill; anything with zero
observations for a patient falls back to the training median — exactly the tree in §1.3.2.
Which item names count as "fast-changing vitals" vs. "slow-changing labs" is defined
explicitly below (editable) rather than inferred, so the classification is a visible,
checkable decision.

In [ ]:
FAST_CHANGING_ITEMS = {
    "hr", "nibp_sbp", "nibp_dbp", "nibp_mbp", "spo2", "rr", "fio2", "bt", "uo",
    "art_sbp", "art_dbp", "art_mbp", "etco2", "peep", "pip", "pplat", "gcs_e", "gcs_m", "gcs_v",
}
# Everything else in the routing map (labs like creatinine, albumin, wbc, ...) is treated
# as slow-changing -> forward-fill, per §1.3.2.

def impute_patient_system(sid, system, strategy, stats):
    bundle = PATIENT_BUNDLE[sid]["ts"][system]
    raw, fnames = bundle["raw"], bundle["feature_names"]
    if raw.shape[1] == 0:
        return raw.copy()
    row = COHORT_INDEXED.loc[sid]
    lo, hi = get_window(sid, row)
    grid = np.linspace(lo, hi, CONFIG["TARGET_SEQ_LEN"])

    if strategy == "median":
        return impute_median(raw, fnames, system, stats)
    elif strategy == "interpolate":
        return impute_interpolate(raw, fnames, system, stats, grid)
    elif strategy == "decision_tree":
        out = raw.copy()
        fast_idx = [j for j, f in enumerate(fnames) if f in FAST_CHANGING_ITEMS]
        slow_idx = [j for j, f in enumerate(fnames) if f not in FAST_CHANGING_ITEMS]
        if fast_idx:
            fast_names = [fnames[j] for j in fast_idx]
            out[:, fast_idx] = impute_interpolate(raw[:, fast_idx], fast_names, system, stats, grid)
        if slow_idx:
            slow_names = [fnames[j] for j in slow_idx]
            out[:, slow_idx] = impute_forward_fill(raw[:, slow_idx], slow_names, system, stats)
        return out
    else:
        raise ValueError(f"strategy {strategy!r} handled elsewhere (knn) or unknown")

def impute_static_vector(sid, train_static_df):
    # Static/tabular features (§6.14): median-fill per column, fit on the training split
    # only (passed in) -- same leakage discipline as the time-series imputers above.
    vec = pd.Series(PATIENT_BUNDLE[sid]["static"])
    filled = vec.fillna(train_static_df.median(numeric_only=True))
    return filled

## 7.4 Running the chosen strategy end-to-end (train-fit, applied to everyone)

`CONFIG['IMPUTATION_STRATEGY']` decides which of the four runs here. Statistics are always
computed from the **training split** (§8.1, run once below to get train IDs before this
cell, then re-used identically in §8) to avoid leakage, as promised in the Part 7 header.
KNN is handled specially (§7.2) since it needs the whole training matrix, not one patient
at a time.

In [ ]:
# A first, lightweight split done here (before §8's full split/sampling section) purely to
# get TRAIN_IDS for leakage-safe statistics -- §8 re-derives and USES this same split for
# training; nothing here duplicates or conflicts with it.
ALL_IDS = cohort_df["subject_id"].tolist()
ALL_LABELS = np.array([PATIENT_BUNDLE[sid]["label"] for sid in ALL_IDS])
TRAIN_IDS_FOR_STATS, _HOLD_IDS = train_test_split(
    ALL_IDS, test_size=(CONFIG["VAL_FRACTION"] + CONFIG["TEST_FRACTION"]),
    stratify=ALL_LABELS, random_state=SEED
)

STATS = population_stats(TRAIN_IDS_FOR_STATS)
train_static_df = pd.DataFrame({sid: PATIENT_BUNDLE[sid]["static"] for sid in TRAIN_IDS_FOR_STATS}).T

IMPUTED_BUNDLE = {}
strategy = CONFIG["IMPUTATION_STRATEGY"]

if strategy == "knn":
    # Build one big matrix per system: [n_patients, T*F], impute across patients, reshape back.
    for system in TIME_SERIES_SYSTEMS:
        fnames = PATIENT_BUNDLE[ALL_IDS[0]]["ts"][system]["feature_names"]
        if not fnames:
            continue
        mat = np.stack([PATIENT_BUNDLE[sid]["ts"][system]["raw"].reshape(-1) for sid in ALL_IDS])
        mat_imputed = impute_knn(mat, CONFIG["KNN_NEIGHBORS"])
        for i, sid in enumerate(ALL_IDS):
            IMPUTED_BUNDLE.setdefault(sid, {})[system] = mat_imputed[i].reshape(CONFIG["TARGET_SEQ_LEN"], len(fnames))
else:
    for sid in ALL_IDS:
        IMPUTED_BUNDLE[sid] = {}
        for system in TIME_SERIES_SYSTEMS:
            IMPUTED_BUNDLE[sid][system] = impute_patient_system(sid, system, strategy, STATS)

for sid in ALL_IDS:
    IMPUTED_BUNDLE[sid]["static"] = impute_static_vector(sid, train_static_df)

# v2 guard: the stored static vector must be in exactly the order of STATIC_FEATURE_NAMES,
# because later steps select static columns by position looked up from that list.
assert list(IMPUTED_BUNDLE[ALL_IDS[0]]["static"].index) == STATIC_FEATURE_NAMES, \
    "static feature values are not in STATIC_FEATURE_NAMES order -- column selection by name would be wrong"
print(f"Imputation strategy used: {strategy!r}")
remaining_nans = sum(np.isnan(IMPUTED_BUNDLE[sid][system]).sum()
                      for sid in ALL_IDS for system in TIME_SERIES_SYSTEMS
                      if IMPUTED_BUNDLE[sid][system].size)
print(f"Remaining NaNs after imputation (should be 0): {remaining_nans}")

## 7.5 Side-by-side comparison of all four strategies on one real gap

In [ ]:
# Pick one patient/system/feature with a genuine, visible gap and show what each strategy
# does to it -- exactly the kind of comparison you said you want to be able to run yourself.
example_sid = ALL_IDS[0]
example_system = "renal"
example_feature_idx = 0
row = COHORT_INDEXED.loc[example_sid]
lo, hi = get_window(example_sid, row)
grid = np.linspace(lo, hi, CONFIG["TARGET_SEQ_LEN"])
raw = PATIENT_BUNDLE[example_sid]["ts"][example_system]["raw"]
fnames = PATIENT_BUNDLE[example_sid]["ts"][example_system]["feature_names"]

fig, ax = plt.subplots(figsize=(9, 4.5))
strategies_to_show = ["median", "interpolate", "decision_tree"]
colors = {"median": "#e67e22", "interpolate": "#2980b9", "decision_tree": "#27ae60"}
for strat in strategies_to_show:
    filled = impute_patient_system(example_sid, example_system, strat, STATS)
    ax.plot(grid, filled[:, example_feature_idx], label=strat, color=colors[strat], alpha=0.8)
obs_mask = ~np.isnan(raw[:, example_feature_idx])
ax.scatter(grid[obs_mask], raw[obs_mask, example_feature_idx], color="black", zorder=5, label="observed", s=40)
ax.set_title(f"Imputation comparison: patient {example_sid}, {example_system}/{fnames[example_feature_idx]}")
ax.set_xlabel("chart_time (minutes)"); ax.set_ylabel("value"); ax.legend()
plt.tight_layout(); plt.show()
print("This is exactly the comparison to re-run (swap example_sid/example_system/example_feature_idx) "
      "as you experiment with §1.3's strategies on your own features of interest.")

## 7.6 Which imputation method is actually most accurate, per system? A held-out masking benchmark

Section 7.5 showed what each strategy's fill *looks like* on one gap. This section
measures it: for each organ system, a fraction of genuinely **observed** values are
artificially hidden, each strategy fills them in blind, and the result is compared against
the real (held-out) value that was actually there. This gives an honest, per-system answer
to "which method is most accurate" — not a visual impression, an average error in the
same clinical units the feature is measured in (e.g. mg/dL, beats/min), so it's directly
interpretable rather than an abstract score.

**A limitation worth stating plainly:** this only measures accuracy on values that
*happened to be observed close in time to other observations* -- masking a real point and
then trying to reconstruct it is inherently easier than the true task of filling a gap
where nothing nearby was ever measured. Treat this as a relative comparison between
methods, not an absolute accuracy guarantee for every gap in the dataset.

In [ ]:
def imputation_accuracy_benchmark(systems, patient_ids, mask_fraction=0.2, seed=SEED, max_patients=300):
    rng = np.random.default_rng(seed)
    bench_ids = list(patient_ids)
    if len(bench_ids) > max_patients:
        bench_ids = list(rng.choice(bench_ids, size=max_patients, replace=False))

    rows = []
    for system in TIME_SERIES_SYSTEMS:
        fnames = PATIENT_BUNDLE[bench_ids[0]]["ts"][system]["feature_names"]
        if not fnames:
            continue
        for strategy in ["median", "interpolate", "decision_tree"]:
            all_true, all_pred = [], []
            for sid in bench_ids:
                row_info = COHORT_INDEXED.loc[sid]
                lo, hi = get_window(sid, row_info)
                grid = np.linspace(lo, hi, CONFIG["TARGET_SEQ_LEN"])
                raw_orig = PATIENT_BUNDLE[sid]["ts"][system]["raw"]
                mask_orig = PATIENT_BUNDLE[sid]["ts"][system]["mask"]
                obs_positions = np.argwhere(mask_orig == 1)
                if len(obs_positions) == 0:
                    continue
                n_mask = max(1, int(len(obs_positions) * mask_fraction))
                n_mask = min(n_mask, len(obs_positions))
                chosen_idx = rng.choice(len(obs_positions), size=n_mask, replace=False)
                chosen = obs_positions[chosen_idx]

                raw_masked = raw_orig.copy()
                true_vals = []
                for (t_idx, f_idx) in chosen:
                    true_vals.append(raw_orig[t_idx, f_idx])
                    raw_masked[t_idx, f_idx] = np.nan

                if strategy == "median":
                    imputed = impute_median(raw_masked, fnames, system, STATS)
                elif strategy == "interpolate":
                    imputed = impute_interpolate(raw_masked, fnames, system, STATS, grid)
                else:  # decision_tree
                    imputed = raw_masked.copy()
                    fast_idx = [j for j, f in enumerate(fnames) if f in FAST_CHANGING_ITEMS]
                    slow_idx = [j for j, f in enumerate(fnames) if f not in FAST_CHANGING_ITEMS]
                    if fast_idx:
                        imputed[:, fast_idx] = impute_interpolate(raw_masked[:, fast_idx], [fnames[j] for j in fast_idx], system, STATS, grid)
                    if slow_idx:
                        imputed[:, slow_idx] = impute_forward_fill(raw_masked[:, slow_idx], [fnames[j] for j in slow_idx], system, STATS)

                for (t_idx, f_idx), true_v in zip(chosen, true_vals):
                    all_true.append(true_v)
                    all_pred.append(imputed[t_idx, f_idx])

            if all_true:
                all_true_arr, all_pred_arr = np.array(all_true), np.array(all_pred)
                mae = float(np.mean(np.abs(all_pred_arr - all_true_arr)))
                rows.append({"system": system, "strategy": strategy, "mae": mae, "n_masked_points": len(all_true)})

    return pd.DataFrame(rows)

if CONFIG["SKIP_IMPUTATION_ACCURACY_BENCHMARKS"]:
    print("CONFIG['SKIP_IMPUTATION_ACCURACY_BENCHMARKS']=True -- skipping this benchmark to "
          "keep total runtime inside the 20-30 minute budget. The function above is still "
          "defined -- set the flag to False and re-run this cell to get real numbers.")
    IMPUTATION_BENCHMARK = pd.DataFrame(columns=["system", "strategy", "mae", "n_masked_points"])
else:
    IMPUTATION_BENCHMARK = imputation_accuracy_benchmark(TIME_SERIES_SYSTEMS, TRAIN_IDS_FOR_STATS)
    print(f"Benchmark built on {IMPUTATION_BENCHMARK['n_masked_points'].sum() if len(IMPUTATION_BENCHMARK) else 0} "
          f"held-out (masked) real data points across all systems.")
IMPUTATION_BENCHMARK

In [ ]:
# Small multiples -- one subplot per system, each on its own scale (a shared axis across
# systems would make e.g. glucose's MAE look "worse" than creatinine's purely because of
# unit scale, not because the method is worse -- see the note above this section).
_systems_with_data = IMPUTATION_BENCHMARK["system"].unique().tolist() if len(IMPUTATION_BENCHMARK) else []
if _systems_with_data:
    n_sys = len(_systems_with_data)
    fig, axes = plt.subplots(1, n_sys, figsize=(3.2 * n_sys, 4), sharey=False)
    if n_sys == 1:
        axes = [axes]
    strategy_colors = {"median": "#95a5a6", "interpolate": "#2980b9", "decision_tree": "#27ae60"}
    for ax, system in zip(axes, _systems_with_data):
        sub = IMPUTATION_BENCHMARK[IMPUTATION_BENCHMARK["system"] == system].set_index("strategy")
        strategies = ["median", "interpolate", "decision_tree"]
        maes = [sub.loc[s, "mae"] if s in sub.index else 0 for s in strategies]
        colors = [strategy_colors[s] for s in strategies]
        ax.bar(strategies, maes, color=colors)
        best = strategies[int(np.argmin(maes))] if any(m > 0 for m in maes) else None
        ax.set_title(system.replace("_", "\n"), fontsize=10)
        ax.set_ylabel("mean absolute error\n(original clinical units)")
        ax.tick_params(axis="x", rotation=30)
        if best:
            ax.set_xlabel(f"best: {best}", fontsize=9, color="#27ae60")
    plt.suptitle("Imputation accuracy by system -- lower is better (held-out masking benchmark, §7.6)")
    plt.tight_layout()
    plt.show()

    print("\nBest strategy per system (lowest held-out error):")
    for system in _systems_with_data:
        sub = IMPUTATION_BENCHMARK[IMPUTATION_BENCHMARK["system"] == system]
        best_row = sub.loc[sub["mae"].idxmin()]
        print(f"  {system:18s} -> {best_row['strategy']:14s} (MAE={best_row['mae']:.3f}, n={int(best_row['n_masked_points'])} held-out points)")
else:
    if CONFIG["SKIP_IMPUTATION_ACCURACY_BENCHMARKS"]:
        print("Skipped (CONFIG['SKIP_IMPUTATION_ACCURACY_BENCHMARKS']=True) -- not a data problem.")
    else:
        print("No systems had enough observed data to benchmark on this sample -- re-run at larger scale.")

## 7.7 Regression-based imputation (MICE / `IterativeImputer`) -- tested, not assumed

Sections 7.1-7.6 use fixed *rules* (interpolate, carry-forward, population average). This
section asks a genuinely different question: can a small **regression model** fill a gap
better by using everything else known about that same patient at that moment (e.g.
"given this patient's sodium, potassium, and BUN right now, what's their creatinine
probably doing"), instead of only looking at that one feature's own history?

**How this differs from every method in Section 7.6:** those methods only ever look
*within one feature's own timeline*. `IterativeImputer` looks *across features*, learning
relationships between them from the whole training population, then uses those
relationships to fill a gap in one feature using the other features observed at the same
time. This is the standard MICE (Multiple Imputation by Chained Equations) approach --
Van Buuren & Groothuis-Oudshoorn 2011 -- fit once per organ system, pooling every training
patient's timesteps together so it has enough data to learn real cross-feature structure
(fitting it per-patient, on ~24 timesteps alone, would be far too little data to learn
anything).

**Why this wasn't the default from the start, restated plainly:** MICE is normally used on
data that's mostly present with occasional gaps. This dataset's pre-op coverage is closer
to 90% *missing* per feature -- a genuinely harder regime for a method that needs some
signal in the other columns to predict from. This section measures whether it still helps
here, rather than assuming either way.

In [ ]:
def build_regression_imputer(system, train_ids, fnames):
    # Pools every (patient, timestep) row across the given patients into one big matrix
    # -- IterativeImputer needs real cross-patient volume to learn feature relationships,
    # which a single patient's ~24 timesteps could never provide on their own.
    rows = []
    for sid in train_ids:
        raw = PATIENT_BUNDLE[sid]["ts"][system]["raw"]
        for t in range(raw.shape[0]):
            rows.append(raw[t, :])
    X = np.array(rows)
    imputer = IterativeImputer(max_iter=10, random_state=SEED, sample_posterior=False)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")   # IterativeImputer is verbose about near-constant/all-NaN columns on sparse data -- expected here, not a bug
        imputer.fit(X)
    return imputer

def impute_regression(raw, imputer):
    # raw: [T, F] for ONE patient, with NaN gaps (including any artificially held-out
    # points from the benchmark below). Uses the population-fit imputer from above.
    with warnings.catch_warnings():
        simplefilter = warnings.simplefilter
        simplefilter("ignore")
        return imputer.transform(raw)

print("Regression (MICE) imputation defined -- benchmarked against the rule-based methods below.")

### Head-to-head: regression imputation vs. the rule-based methods, same benchmark as §7.6

Same held-out masking test as before (hide real values, predict them blind, measure the
real error) -- now with `IterativeImputer` added as a fourth competitor, so the comparison
is apples-to-apples on identical held-out points.

In [ ]:
def imputation_accuracy_benchmark_with_regression(systems, patient_ids, mask_fraction=0.2, seed=SEED, max_patients=300):
    rng = np.random.default_rng(seed)
    bench_ids = list(patient_ids)
    if len(bench_ids) > max_patients:
        bench_ids = list(rng.choice(bench_ids, size=max_patients, replace=False))

    rows = []
    for system in TIME_SERIES_SYSTEMS:
        fnames = PATIENT_BUNDLE[bench_ids[0]]["ts"][system]["feature_names"]
        if not fnames or len(fnames) < 2:   # regression imputation needs >=2 features to have anything to regress from
            continue

        # Decide the held-out mask ONCE per system, shared across all four methods, so
        # this is a fair identical-conditions comparison, not different gaps per method.
        masked_raw_by_patient, true_by_patient = {}, {}
        for sid in bench_ids:
            raw_orig = PATIENT_BUNDLE[sid]["ts"][system]["raw"]
            mask_orig = PATIENT_BUNDLE[sid]["ts"][system]["mask"]
            obs_positions = np.argwhere(mask_orig == 1)
            if len(obs_positions) == 0:
                continue
            n_mask = max(1, int(len(obs_positions) * mask_fraction))
            n_mask = min(n_mask, len(obs_positions))
            chosen = obs_positions[rng.choice(len(obs_positions), size=n_mask, replace=False)]
            raw_masked = raw_orig.copy()
            true_vals = {}
            for (t_idx, f_idx) in chosen:
                true_vals[(t_idx, f_idx)] = raw_orig[t_idx, f_idx]
                raw_masked[t_idx, f_idx] = np.nan
            masked_raw_by_patient[sid] = raw_masked
            true_by_patient[sid] = true_vals

        if not masked_raw_by_patient:
            continue

        # Fit the regression imputer on the SAME masked training data (not leaking the
        # held-out true values into its own fit).
        try:
            reg_imputer = build_regression_imputer(system, list(masked_raw_by_patient.keys()), fnames)
        except Exception as e:
            reg_imputer = None
            print(f"  {system}: regression imputer failed to fit ({e}) -- skipping regression for this system")

        for strategy in ["median", "interpolate", "decision_tree", "regression"]:
            if strategy == "regression" and reg_imputer is None:
                continue
            all_true, all_pred = [], []
            for sid, raw_masked in masked_raw_by_patient.items():
                row_info = COHORT_INDEXED.loc[sid]
                lo, hi = get_window(sid, row_info)
                grid = np.linspace(lo, hi, CONFIG["TARGET_SEQ_LEN"])

                if strategy == "median":
                    imputed = impute_median(raw_masked, fnames, system, STATS)
                elif strategy == "interpolate":
                    imputed = impute_interpolate(raw_masked, fnames, system, STATS, grid)
                elif strategy == "decision_tree":
                    imputed = raw_masked.copy()
                    fast_idx = [j for j, f in enumerate(fnames) if f in FAST_CHANGING_ITEMS]
                    slow_idx = [j for j, f in enumerate(fnames) if f not in FAST_CHANGING_ITEMS]
                    if fast_idx:
                        imputed[:, fast_idx] = impute_interpolate(raw_masked[:, fast_idx], [fnames[j] for j in fast_idx], system, STATS, grid)
                    if slow_idx:
                        imputed[:, slow_idx] = impute_forward_fill(raw_masked[:, slow_idx], [fnames[j] for j in slow_idx], system, STATS)
                else:  # regression
                    imputed = impute_regression(raw_masked, reg_imputer)

                for (t_idx, f_idx), true_v in true_by_patient[sid].items():
                    all_true.append(true_v)
                    all_pred.append(imputed[t_idx, f_idx])

            if all_true:
                mae = float(np.mean(np.abs(np.array(all_pred) - np.array(all_true))))
                rows.append({"system": system, "strategy": strategy, "mae": mae, "n_masked_points": len(all_true)})

    return pd.DataFrame(rows)

if CONFIG["SKIP_IMPUTATION_ACCURACY_BENCHMARKS"]:
    print("CONFIG['SKIP_IMPUTATION_ACCURACY_BENCHMARKS']=True -- skipping this benchmark. "
          "This is the single most expensive optional section in the notebook (IterativeImputer/"
          "MICE fitting is iterative) -- set the flag to False and re-run once you've confirmed "
          "your session's timing headroom.")
    IMPUTATION_BENCHMARK_V2 = pd.DataFrame(columns=["system", "strategy", "mae", "n_masked_points"])
else:
    IMPUTATION_BENCHMARK_V2 = imputation_accuracy_benchmark_with_regression(TIME_SERIES_SYSTEMS, TRAIN_IDS_FOR_STATS)
IMPUTATION_BENCHMARK_V2

In [ ]:
_systems_v2 = IMPUTATION_BENCHMARK_V2["system"].unique().tolist() if len(IMPUTATION_BENCHMARK_V2) else []
if _systems_v2:
    n_sys = len(_systems_v2)
    fig, axes = plt.subplots(1, n_sys, figsize=(3.4 * n_sys, 4), sharey=False)
    if n_sys == 1:
        axes = [axes]
    strategy_colors = {"median": "#95a5a6", "interpolate": "#2980b9", "decision_tree": "#27ae60", "regression": "#e74c3c"}
    strategies_order = ["median", "interpolate", "decision_tree", "regression"]
    for ax, system in zip(axes, _systems_v2):
        sub = IMPUTATION_BENCHMARK_V2[IMPUTATION_BENCHMARK_V2["system"] == system].set_index("strategy")
        present = [s for s in strategies_order if s in sub.index]
        maes = [sub.loc[s, "mae"] for s in present]
        colors = [strategy_colors[s] for s in present]
        ax.bar(present, maes, color=colors)
        best = present[int(np.argmin(maes))] if maes else None
        ax.set_title(system.replace("_", "\n"), fontsize=10)
        ax.set_ylabel("MAE (clinical units)")
        ax.tick_params(axis="x", rotation=35)
        if best:
            ax.set_xlabel(f"best: {best}", fontsize=9, color="#27ae60")
    plt.suptitle("Imputation accuracy, rule-based methods vs. regression (MICE) -- lower is better (§7.7)")
    plt.tight_layout()
    plt.show()

    n_regression_wins = 0
    for system in _systems_v2:
        sub = IMPUTATION_BENCHMARK_V2[IMPUTATION_BENCHMARK_V2["system"] == system]
        best_row = sub.loc[sub["mae"].idxmin()]
        if best_row["strategy"] == "regression":
            n_regression_wins += 1
        print(f"  {system:18s} -> best: {best_row['strategy']:14s} (MAE={best_row['mae']:.3f})")
    print(f"\nRegression imputation won on {n_regression_wins}/{len(_systems_v2)} systems on this sample.")
    print("Re-run this once trained on the larger cohort -- MICE specifically benefits from more "
          "rows to learn cross-feature relationships from, more than the rule-based methods do.")
else:
    if CONFIG["SKIP_IMPUTATION_ACCURACY_BENCHMARKS"]:
        print("Skipped (CONFIG['SKIP_IMPUTATION_ACCURACY_BENCHMARKS']=True) -- not a data problem.")
    else:
        print("No systems had enough features/data to benchmark regression imputation on this sample.")

## 7.8 Standardisation (z-score, fit on training data only)

One deliberately non-optional step, separate from the §1.3 imputation choice: raw clinical
values live on wildly different scales (`glucose` in the hundreds, `potassium` around 4,
`wbc` in the thousands per µL) — feeding that directly into a neural network makes large-
magnitude features dominate the loss for reasons that have nothing to do with clinical
importance, and destabilises the autoencoder pre-training in §10.1 in particular (its loss
is a raw MSE, which is scale-sensitive in exactly this way). Mean/std are computed **once,
from `TRAIN_IDS_FOR_STATS` only** (§7.4's split, reused here — never refit on val/test, for
the same leakage reason given throughout Part 7), then applied to every patient.

In [ ]:
def compute_train_std(train_ids):
    std_stats = {}
    for system in TIME_SERIES_SYSTEMS:
        fnames = PATIENT_BUNDLE[train_ids[0]]["ts"][system]["feature_names"]
        if not fnames:
            continue
        stacked = np.stack([IMPUTED_BUNDLE[sid][system] for sid in train_ids])   # [N, T, F]
        for j, fname in enumerate(fnames):
            col = stacked[:, :, j].reshape(-1)
            std_stats[(system, fname)] = float(col.std() + 1e-6)
    return std_stats

TRAIN_STD = compute_train_std(TRAIN_IDS_FOR_STATS)

def standardize_system_tensor(raw, feature_names, system):
    out = raw.copy()
    for j, fname in enumerate(feature_names):
        mean = STATS[(system, fname)]["mean"]
        std = TRAIN_STD[(system, fname)]
        out[:, j] = (out[:, j] - mean) / std
    return out

for sid in ALL_IDS:
    for system in TIME_SERIES_SYSTEMS:
        fnames = PATIENT_BUNDLE[sid]["ts"][system]["feature_names"]
        if fnames:
            IMPUTED_BUNDLE[sid][system] = standardize_system_tensor(IMPUTED_BUNDLE[sid][system], fnames, system)

# Static features: same idea, via sklearn's StandardScaler fit on the training static table.
static_scaler = StandardScaler()
static_scaler.fit(train_static_df.fillna(train_static_df.median(numeric_only=True)).values)

for sid in ALL_IDS:
    vec = IMPUTED_BUNDLE[sid]["static"]
    scaled = static_scaler.transform(vec.values.reshape(1, -1))[0]
    IMPUTED_BUNDLE[sid]["static"] = pd.Series(scaled, index=vec.index)

print("Standardisation applied (train-fit z-score) to all time-series and static features.")
example_sid = ALL_IDS[0]
print(f"Example, patient {example_sid}, renal system, post-standardisation value range: "
      f"[{IMPUTED_BUNDLE[example_sid]['renal'].min():.2f}, {IMPUTED_BUNDLE[example_sid]['renal'].max():.2f}] "
      f"(should now be roughly within a few units of 0, not raw clinical units)")

In [ ]:
record_checkpoint("Part 7 -- missing-data imputation + standardisation")

# Part 8 -- Train/val/test split and class-imbalance sampling

This section implements the combined pipeline agreed on after reviewing external advice
against this notebook's actual multi-branch architecture (full writeup:
`Data_Imbalance_and_Imputation_Reference.md`, Part C). In order:

- **8.1** stratified split (non-negotiable, unchanged from the original design)
- **8.2** the new default strategy -- **grouped SMOTENC + Tomek-link cleanup**, applied
  only to the static branch, only within clinical strata (department x ASA), targeting a
  1:10 positive:negative ratio rather than full 1:1 balance
- **8.3** sequence-branch augmentation (jitter + time-masking) for real minority training
  patients -- the sequence branch never receives SMOTE-style synthetic data (Part C's own
  reasoning: interpolating between two raw physiological trajectories in different
  patients' time coordinates isn't a well-defined operation)
- **8.4** the older, simpler strategies (plain SMOTE/ADASYN, random over/undersample,
  class-weight-only) are kept available via `CONFIG['SAMPLING_STRATEGY']` for direct
  comparison, but are no longer the default

## 8.1 Stratified split

Re-derives the same split as Part 7's `TRAIN_IDS_FOR_STATS` / `_HOLD_IDS` (same `SEED`,
same call) and further splits the holdout into validation and test. With only 30 patients
in this dev subset, don't over-read exact percentages here -- the code is written to hold
correctly once the full cohort is loaded, where `VAL_FRACTION`/`TEST_FRACTION` will
produce far more stable folds.

In [ ]:
TRAIN_IDS = TRAIN_IDS_FOR_STATS   # identical split object reused from Part 7, not recomputed
HOLD_LABELS = np.array([PATIENT_BUNDLE[sid]["label"] for sid in _HOLD_IDS])
val_frac_of_hold = CONFIG["VAL_FRACTION"] / (CONFIG["VAL_FRACTION"] + CONFIG["TEST_FRACTION"])

VAL_IDS, TEST_IDS = train_test_split(
    _HOLD_IDS, test_size=(1 - val_frac_of_hold), stratify=HOLD_LABELS, random_state=SEED
)

for name, ids in [("train", TRAIN_IDS), ("val", VAL_IDS), ("test", TEST_IDS)]:
    labels = [PATIENT_BUNDLE[sid]["label"] for sid in ids]
    print(f"{name:5s}: n={len(ids):3d}  positives={int(sum(labels)):2d}  rate={np.mean(labels):.1%}")

## 8.1b Optional: department-stratified downsample of TRAINING survived patients only

**Off by default** (`CONFIG['DOWNSAMPLE_TRAIN_SURVIVED_TO'] = None`). This is a
memory/compute lever, not a statistical recommendation — see Part 3's "note on memory
strategy" for the full reasoning on why this should be the last resort, not the first
move, and why it only ever touches training-split `survived` patients:

- **Never touches `died` patients** — they're already the scarce class; downsampling them
  further would be actively counterproductive.
- **Never touches `VAL_IDS`/`TEST_IDS`** — validation and test must keep the true
  prevalence for AUPRC/calibration to mean anything (Part C step 9).
- **Stratified by department** (not "organ-system ICD" — see Part 3's note on why) so the
  downsampled training pool's department mix still resembles the real cohort's, rather
  than randomly dropping an entire department's worth of survived patients by chance.

In [ ]:
def downsample_train_survived(train_ids, target_n_survived, strata_col="department", seed=SEED):
    labels_by_id = {sid: PATIENT_BUNDLE[sid]["label"] for sid in train_ids}
    survived_ids = [sid for sid in train_ids if labels_by_id[sid] == 0.0]
    died_ids = [sid for sid in train_ids if labels_by_id[sid] == 1.0]

    if target_n_survived >= len(survived_ids):
        print(f"DOWNSAMPLE_TRAIN_SURVIVED_TO={target_n_survived} >= current {len(survived_ids)} "
              f"survived training patients -- nothing to do.")
        return train_ids

    strata = COHORT_INDEXED.loc[survived_ids, strata_col].fillna("missing").astype(str)
    rng = np.random.default_rng(seed)
    kept_survived_ids = []
    # Proportional allocation per stratum, rounded, with a final random top-up/trim to hit
    # the exact target -- keeps the department mix close to the original without needing
    # every stratum to divide evenly.
    frac = target_n_survived / len(survived_ids)
    for stratum_key, group in strata.groupby(strata):
        ids_in_stratum = group.index.tolist()
        n_keep = max(1, round(len(ids_in_stratum) * frac))
        n_keep = min(n_keep, len(ids_in_stratum))
        kept_survived_ids.extend(rng.choice(ids_in_stratum, size=n_keep, replace=False).tolist())

    if len(kept_survived_ids) > target_n_survived:
        kept_survived_ids = rng.choice(kept_survived_ids, size=target_n_survived, replace=False).tolist()

    print(f"Downsampled training 'survived' patients: {len(survived_ids)} -> {len(kept_survived_ids)} "
          f"(stratified by {strata_col!r}); training 'died' patients unchanged at {len(died_ids)}.")
    return died_ids + kept_survived_ids

if CONFIG["DOWNSAMPLE_TRAIN_SURVIVED_TO"] is not None:
    TRAIN_IDS = downsample_train_survived(TRAIN_IDS, CONFIG["DOWNSAMPLE_TRAIN_SURVIVED_TO"])
    labels = [PATIENT_BUNDLE[sid]["label"] for sid in TRAIN_IDS]
    print(f"train (post-downsample): n={len(TRAIN_IDS)}  positives={int(sum(labels))}  rate={np.mean(labels):.1%}")
    print("NOTE: Part 7's imputation/standardization statistics (STATS, TRAIN_STD) were "
          "already fit on the FULL pre-downsample training pool (TRAIN_IDS_FOR_STATS) -- "
          "intentional, not stale: more patients gives more robust statistics, and only the "
          "set of patients actually used for gradient updates needs to shrink here.")
else:
    print("DOWNSAMPLE_TRAIN_SURVIVED_TO is None -- no downsampling applied (recommended default).")

## 8.2 Grouped SMOTENC + Tomek-link cleanup (new default)

**Why grouped:** plain SMOTE/SMOTENC would happily blend a cardiothoracic ASA-5 patient
with an outpatient orthopedic ASA-1 patient purely because both happened to die -- that's
not a clinically meaningful neighborhood. Splitting the minority cohort into strata by
`(department, asa)` and running SMOTENC **within** each stratum keeps every synthetic
patient's interpolation partners clinically comparable to each other.

**Why SMOTENC, not plain SMOTE:** the static vector mixes continuous features (age, the
36 aggregated lab/vital statistics from Section 6.15) with one-hot department columns and
binary flags (`sex_F`, `cardiac_recovery_exception`, the GI/MSK ICD-10 flags, ...). Plain
SMOTE would linearly interpolate those binary/one-hot columns into meaningless fractional
values (e.g. `dept_GS = 0.6`). SMOTENC is told exactly which columns are categorical and
handles them by majority vote among neighbors instead of interpolation.

**Why a 1:10 target, not full 1:1 balance:** at the full cohort's ~469 real deaths,
synthesizing up to 46,900+ to match ~99,417 survivors would force >99x amplification of
the same ~469 real points -- pure noise-filling well past where SMOTE's own literature
says new information stops being added. 1:10 (~4,700 synthetic positives against the full
survivor pool) is the conservative end of the two external reviews' suggested 1:10-1:4
range.

**The Tomek-link cleanup step, and how synthetic vs. real rows are tracked correctly:**
after SMOTENC runs per stratum, a global Tomek-link pass identifies and removes majority
(survived) points that are each a synthetic minority point's nearest opposite-class
neighbor -- this widens the decision margin around the newly-added synthetic points. Row
identity through this step is tracked via each sampler's own `sample_indices_` attribute
(the array of retained input-row positions that `imbalanced-learn` documents and
guarantees), **not** by assuming synthetic rows land at a particular position in the
output array -- the earlier draft of this pipeline (reviewed in
`Data_Imbalance_and_Imputation_Reference.md` Part A.1) relied on exactly that unstable
assumption; this version doesn't.

In [ ]:
from imblearn.over_sampling import SMOTE, ADASYN, SMOTENC, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler, TomekLinks

def compute_pos_weight_from_counts(n_pos, n_neg):
    return float(n_neg / max(n_pos, 1))

def compute_pos_weight(train_ids):
    labels = np.array([PATIENT_BUNDLE[sid]["label"] for sid in train_ids])
    return compute_pos_weight_from_counts(labels.sum(), len(labels) - labels.sum())

# Which static columns are categorical -> passed to SMOTENC as a boolean mask, in the
# same column order as STATIC_FEATURE_NAMES (built in Section 6.14).
_BINARY_STATIC_COLS = {
    "sex_F", "emop", "gi_department_flag",
    "msk_department_flag", "cardiac_recovery_exception", "cardio_has_iabp", "asa",
    "infection_chapter_i_flag", "infection_high_risk_code_flag", "infection_fever_flag",
    "infection_wbc_abnormal_flag", "infection_crp_elevated_flag",
    "news2_high_flag", "news2_red_flag", "news2_available",   # v2 -- binary NEWS2 flags
}
CATEGORICAL_STATIC_MASK = np.array([
    name.startswith("dept_") or name in _BINARY_STATIC_COLS
    or (name.startswith("sysdx_") and name.endswith("_flag"))   # v3: per-organ-system diagnosis flags
    or name.startswith("antype_")                                # v4: anaesthesia type one-hot
    or (name.startswith("proc_") and name.endswith("_last_op"))  # v4: is the operation on this system?
    for name in STATIC_FEATURE_NAMES
])
# v2 FIX: imbalanced-learn 0.14.x rejects a boolean mask for `categorical_features`
# ("The truth value of an array ... is ambiguous"). Integer column indices are accepted by
# every imblearn version, so they are what gets passed to SMOTENC below.
CATEGORICAL_STATIC_IDX = np.where(CATEGORICAL_STATIC_MASK)[0].tolist()
print(f"Static features flagged categorical for SMOTENC: {int(CATEGORICAL_STATIC_MASK.sum())} "
      f"/ {len(STATIC_FEATURE_NAMES)} ({[n for n, c in zip(STATIC_FEATURE_NAMES, CATEGORICAL_STATIC_MASK) if c]})")

def _static_matrix(ids):
    return np.stack([IMPUTED_BUNDLE[sid]["static"].values.astype(float) for sid in ids])

def grouped_smotenc(train_ids, target_ratio, min_stratum_minority, strata_cols):
    labels_by_id = {sid: PATIENT_BUNDLE[sid]["label"] for sid in train_ids}
    # NOTE: .astype(str) on a float column with NaN leaves the NaN as an actual float NaN
    # on modern pandas (its "str" dtype is still nullable) rather than the string "nan" --
    # .fillna() first, per column, avoids that trap before building the join key.
    strata_df = COHORT_INDEXED.loc[train_ids, strata_cols].copy()
    for col in strata_cols:
        strata_df[col] = strata_df[col].fillna("missing").astype(str)
    strata = strata_df.agg("_".join, axis=1)

    all_synthetic_rows = []   # list of np.ndarray, one per synthetic patient
    all_synthetic_strata = [] # v2: which stratum each synthetic row came from (for §8.2c donors)
    n_strata_synthesized, n_strata_skipped, n_strata_errored = 0, 0, 0

    for stratum_key, group in strata.groupby(strata):
        stratum_ids = group.index.tolist()
        pos_ids = [sid for sid in stratum_ids if labels_by_id[sid] == 1.0]
        neg_ids = [sid for sid in stratum_ids if labels_by_id[sid] == 0.0]
        n_pos, n_neg = len(pos_ids), len(neg_ids)

        if n_pos < min_stratum_minority or n_neg == 0:
            n_strata_skipped += 1
            continue
        desired_pos = int(n_neg * target_ratio)
        if CONFIG["SMOTE_MAX_AMPLIFICATION"] is not None:   # v3: cap per-stratum amplification
            desired_pos = min(desired_pos, n_pos * (1 + CONFIG["SMOTE_MAX_AMPLIFICATION"]))
        if desired_pos <= n_pos:
            n_strata_skipped += 1
            continue

        X_stratum = _static_matrix(pos_ids + neg_ids)
        y_stratum = np.array([1.0] * n_pos + [0.0] * n_neg)
        k_neighbors = max(1, min(5, n_pos - 1))
        try:
            smotenc = SMOTENC(categorical_features=CATEGORICAL_STATIC_IDX,   # v2: indices, not the mask
                               sampling_strategy={1: desired_pos},
                               k_neighbors=k_neighbors, random_state=SEED)
            X_res, y_res = smotenc.fit_resample(X_stratum, y_stratum)
        except ValueError as e:
            # v2: an ERROR is not the same as "too few positives" (handled above) -- count it
            # separately so a library/version problem can never again pass as a quiet skip.
            print(f"  stratum {stratum_key!r}: SMOTENC ERROR ({e})")
            n_strata_errored += 1
            continue

        n_synthetic_this_stratum = len(X_res) - len(X_stratum)
        # SMOTENC/SMOTE append generated samples after the originals, in the fixed order
        # the input was given (X_stratum = pos_ids + neg_ids) -- documented behaviour, not
        # an assumption about arbitrary internal ordering. Sanity-checked below regardless.
        synthetic_rows_this_stratum = X_res[len(X_stratum):]
        synthetic_labels_this_stratum = y_res[len(X_stratum):]
        assert np.all(synthetic_labels_this_stratum == 1.0), (
            "Expected all synthesized rows to be minority-class -- ordering assumption violated, "
            "do not trust this stratum's synthetic rows."
        )
        all_synthetic_rows.extend(list(synthetic_rows_this_stratum))
        all_synthetic_strata.extend([stratum_key] * len(synthetic_rows_this_stratum))
        n_strata_synthesized += 1

    print(f"Grouped SMOTENC: {n_strata_synthesized} strata synthesized "
          f"({len(all_synthetic_rows)} synthetic patients), {n_strata_skipped} skipped "
          f"(too few real positives, or already at/above target ratio), "
          f"{n_strata_errored} ERRORED")
    if n_strata_errored:
        msg = (f"{n_strata_errored} strata raised an error inside SMOTENC -- this is a code/library "
               f"problem, not a data-size skip. Fix it before trusting this run's results.")
        if CONFIG["STRICT_SAMPLING"]:
            raise RuntimeError(msg + " (set CONFIG['STRICT_SAMPLING']=False to continue anyway)")
        print("WARNING: " + msg)
    GROUPED_SMOTENC_SUMMARY.update(synthesized_strata=n_strata_synthesized, skipped_strata=n_strata_skipped,
                                   errored_strata=n_strata_errored, n_synthetic=len(all_synthetic_rows))
    GROUPED_SMOTENC_STRATA.clear()
    GROUPED_SMOTENC_STRATA.extend(all_synthetic_strata)
    return all_synthetic_rows

GROUPED_SMOTENC_STRATA = []   # filled by grouped_smotenc(), aligned with its returned rows
GROUPED_SMOTENC_SUMMARY = {}  # counts, read by the Part 12 verification summary

def tomek_cleanup(train_ids, synthetic_rows):
    real_X = _static_matrix(train_ids)
    real_y = np.array([PATIENT_BUNDLE[sid]["label"] for sid in train_ids])
    synth_X = np.stack(synthetic_rows) if synthetic_rows else np.zeros((0, real_X.shape[1]))
    synth_y = np.ones(len(synthetic_rows))

    combined_X = np.concatenate([real_X, synth_X], axis=0)
    combined_y = np.concatenate([real_y, synth_y], axis=0)
    combined_keys = list(train_ids) + [f"synthetic_{i}" for i in range(len(synthetic_rows))]

    if len(synthetic_rows) == 0:
        return train_ids, []   # nothing to clean up around

    tl = TomekLinks(sampling_strategy="majority")
    X_clean, y_clean = tl.fit_resample(combined_X, combined_y)
    try:
        kept_positions = tl.sample_indices_
    except AttributeError:
        print("WARNING: TomekLinks.sample_indices_ unavailable in this imblearn version -- "
              "falling back to keeping everything (no Tomek cleanup applied this run).")
        kept_positions = np.arange(len(combined_keys))

    kept_keys = [combined_keys[i] for i in kept_positions]
    kept_real_ids = [k for k in kept_keys if not str(k).startswith("synthetic_")]
    kept_synthetic_positions = [int(k.split("_")[1]) for k in kept_keys if str(k).startswith("synthetic_")]
    kept_synthetic_rows = [synthetic_rows[i] for i in kept_synthetic_positions]

    n_removed_real = len(train_ids) - len(kept_real_ids)
    n_removed_synthetic = len(synthetic_rows) - len(kept_synthetic_rows)
    print(f"Tomek cleanup: removed {n_removed_real} real (majority) patients, "
          f"{n_removed_synthetic} synthetic patients as ambiguous boundary pairs")
    TOMEK_KEPT_SYNTHETIC_POSITIONS[:] = kept_synthetic_positions
    return kept_real_ids, kept_synthetic_rows

TOMEK_KEPT_SYNTHETIC_POSITIONS = []   # which of the SMOTENC rows survived Tomek (v2, for donor/stratum bookkeeping)

In [ ]:
def apply_sampling_strategy(train_ids, strategy):
    # Returns (effective_train_ids, pos_weight, synthetic_static_rows_or_None).
    pos_weight = compute_pos_weight(train_ids)

    if strategy in ("class_weight", "none"):
        return train_ids, pos_weight, None

    if strategy == "focal_loss":
        return train_ids, pos_weight, None   # focal loss handled in the loss function itself, Part 9.5

    labels = np.array([PATIENT_BUNDLE[sid]["label"] for sid in train_ids])

    if strategy == "grouped_smotenc_tomek":
        synthetic_rows = grouped_smotenc(train_ids, CONFIG["SMOTE_TARGET_RATIO"],
                                          CONFIG["SMOTE_MIN_STRATUM_MINORITY"], CONFIG["SMOTE_STRATA_COLS"])
        if not synthetic_rows:
            print("No strata met the synthesis criteria on this sample -- falling back to class_weight only. "
                  "Expected on a very small dev subset; re-run at full scale.")
            return train_ids, pos_weight, None
        kept_real_ids, kept_synthetic_rows = tomek_cleanup(train_ids, synthetic_rows)
        synthetic_pairs = [(row, 1.0) for row in kept_synthetic_rows]
        # v2 FIX: the v1 line subtracted the number of Tomek-removed MAJORITY patients from the
        # POSITIVE count (never reached in v1 because SMOTENC produced nothing). Tomek with
        # sampling_strategy="majority" never removes real positives, so:
        n_real_pos = int(labels.sum())
        n_pos_final = n_real_pos + len(kept_synthetic_rows)
        n_neg_final = len(kept_real_ids) - n_real_pos
        final_pos_weight = compute_pos_weight_from_counts(n_pos_final, max(n_neg_final, 1))
        print(f"Final effective training set: {len(kept_real_ids)} real + {len(kept_synthetic_rows)} synthetic "
              f"patients. Residual pos_weight after sampling: {final_pos_weight:.2f} "
              f"(sampling narrowed the gap; loss-weighting finishes it, per Part C step 8)")
        return kept_real_ids, final_pos_weight, synthetic_pairs

    ids_arr = np.array(train_ids).reshape(-1, 1)   # sklearn resamplers need a 2D "X"

    if strategy == "random_oversample":
        res_ids, res_labels = RandomOverSampler(random_state=SEED).fit_resample(ids_arr, labels)
        return res_ids.ravel().tolist(), 1.0, None

    if strategy == "random_undersample":
        res_ids, res_labels = RandomUnderSampler(random_state=SEED).fit_resample(ids_arr, labels)
        return res_ids.ravel().tolist(), 1.0, None

    if strategy in ("smote", "adasyn"):
        X = _static_matrix(train_ids)
        y = labels
        n_minority = int(y.sum())
        if n_minority < 2:
            print(f"WARNING: only {n_minority} positive training examples -- SMOTE/ADASYN need "
                  f">=2 to find neighbours; falling back to class_weight for this run.")
            return train_ids, pos_weight, None
        k_neighbors = max(1, min(5, n_minority - 1))
        Sampler = SMOTE if strategy == "smote" else ADASYN
        try:
            X_res, y_res = Sampler(random_state=SEED, k_neighbors=k_neighbors).fit_resample(X, y)
        except ValueError as e:
            print(f"WARNING: {strategy} failed ({e}); falling back to class_weight.")
            return train_ids, pos_weight, None
        n_synthetic = len(X_res) - len(X)
        synthetic_rows = [(X_res[len(X) + i], float(y_res[len(X) + i])) for i in range(n_synthetic)]
        print(f"{strategy} (unconstrained, no clinical-neighborhood grouping): generated {n_synthetic} "
              f"synthetic minority rows -- prefer 'grouped_smotenc_tomek' unless comparing directly.")
        return train_ids, 1.0, synthetic_rows

    raise ValueError(f"Unknown SAMPLING_STRATEGY {strategy!r}")

EFFECTIVE_TRAIN_IDS, POS_WEIGHT, SYNTHETIC_STATIC_ROWS = apply_sampling_strategy(TRAIN_IDS, CONFIG["SAMPLING_STRATEGY"])
print(f"\nSAMPLING_STRATEGY = {CONFIG['SAMPLING_STRATEGY']!r}")
print(f"Effective training set size: {len(EFFECTIVE_TRAIN_IDS)} real patients "
      f"+ {len(SYNTHETIC_STATIC_ROWS or [])} synthetic")
print(f"pos_weight passed to the loss function: {POS_WEIGHT:.2f}")

### 8.2b NEWS2 before vs. after SMOTENC (v2)

SMOTENC interpolates NEWS2 like any continuous feature, so the question is whether the
synthetic "died" patients still carry clinically possible NEWS2 values that look like
the real "died" patients they were made from. **Before** = real died patients in the
training split; **after** = the synthetic ones that survived Tomek cleanup. Hard fail
if any value falls outside 0-20; REVIEW if the distribution shifts materially (KS > 0.2).
Interpolated scores are non-integer by construction — expected for a continuous
feature, reported so it isn't a surprise.

In [ ]:
NEWS2_AUGMENTATION_CHECKS = []
if not USE_NEWS2:
    print("NEWS2 not in the static vector this run -- nothing to check.")
elif not SYNTHETIC_STATIC_ROWS:
    print("No synthetic patients this run -- nothing to compare against.")
else:
    _idx = {n: STATIC_FEATURE_NAMES.index(n) for n in ("news2_max", "news2_mean", "news2_available")}
    def _raw_static(rows):   # back to clinical units (inverse of Part 7.8's StandardScaler)
        return static_scaler.inverse_transform(np.stack(rows))
    _real_pos_ids = [sid for sid in EFFECTIVE_TRAIN_IDS if PATIENT_BUNDLE[sid]["label"] == 1.0]
    _real_neg_ids = [sid for sid in EFFECTIVE_TRAIN_IDS if PATIENT_BUNDLE[sid]["label"] == 0.0]
    _before = _raw_static([IMPUTED_BUNDLE[s]["static"].values.astype(float) for s in _real_pos_ids])
    _after = _raw_static([row for row, _ in SYNTHETIC_STATIC_ROWS])
    _neg = _raw_static([IMPUTED_BUNDLE[s]["static"].values.astype(float) for s in _real_neg_ids])
    # compare only patients who genuinely had a NEWS2 (available flag ~1 before scaling)
    _b_ok = _before[:, _idx["news2_available"]] > 0.5
    _a_ok = _after[:, _idx["news2_available"]] > 0.5
    for feat in ("news2_max", "news2_mean"):
        res = compare_before_after(_before[_b_ok, _idx[feat]], _after[_a_ok, _idx[feat]], name=f"{feat} (SMOTENC)")
        res["pct_non_integer_after"] = float(np.mean(np.abs(_after[_a_ok, _idx[feat]] - np.round(_after[_a_ok, _idx[feat]])) > 1e-6)) \
            if _a_ok.any() else np.nan
        NEWS2_AUGMENTATION_CHECKS.append(res)
    print(pd.DataFrame(NEWS2_AUGMENTATION_CHECKS).set_index("feature").T)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    _bins = np.arange(0, 21, 1)
    ax.hist(_neg[_neg[:, _idx["news2_available"]] > 0.5, _idx["news2_max"]], bins=_bins, alpha=0.35, density=True, label="real survived", color="#2980b9")
    ax.hist(_before[_b_ok, _idx["news2_max"]], bins=_bins, alpha=0.6, density=True, label="real died (before)", color="#c0392b")
    ax.hist(_after[_a_ok, _idx["news2_max"]], bins=_bins, alpha=0.5, density=True, label="synthetic died (after)", color="#f39c12", histtype="step", linewidth=2)
    ax.set_xlabel("worst pre-op NEWS2"); ax.set_ylabel("density"); ax.legend()
    ax.set_title("NEWS2 before vs. after SMOTENC")
    plt.tight_layout(); plt.show()

### 8.2c Time series for synthetic patients (v2)

SMOTENC only creates a synthetic **static** vector. In v1 every synthetic patient got a
fully empty, fully masked time series — and every synthetic patient is labelled "died",
so "no observations anywhere" became a perfect death signal inside training data that
does not exist in real test patients. With `SYNTHETIC_TS_MODE='nearest_real_donor'` each
synthetic patient instead borrows the time series of the nearest real died patient from
the **same department × ASA stratum** (distance on continuous static features), with
fresh jitter every epoch (Part 9.1) so it is never an exact copy.

In [ ]:
SYNTHETIC_TS_DONORS = []
if SYNTHETIC_STATIC_ROWS and CONFIG["SYNTHETIC_TS_MODE"] == "nearest_real_donor":
    _cont_idx = [i for i, c in enumerate(CATEGORICAL_STATIC_MASK) if not c]
    _strata_df = COHORT_INDEXED.loc[EFFECTIVE_TRAIN_IDS, CONFIG["SMOTE_STRATA_COLS"]].copy()
    for _c in CONFIG["SMOTE_STRATA_COLS"]:
        _strata_df[_c] = _strata_df[_c].fillna("missing").astype(str)
    _stratum_of = _strata_df.agg("_".join, axis=1).to_dict()
    _pos_by_stratum = defaultdict(list)
    for _sid in EFFECTIVE_TRAIN_IDS:
        if PATIENT_BUNDLE[_sid]["label"] == 1.0:
            _pos_by_stratum[_stratum_of[_sid]].append(_sid)
    _all_pos = [s for v in _pos_by_stratum.values() for s in v]
    _kept_strata = [GROUPED_SMOTENC_STRATA[p] for p in TOMEK_KEPT_SYNTHETIC_POSITIONS] \
        if len(TOMEK_KEPT_SYNTHETIC_POSITIONS) == len(SYNTHETIC_STATIC_ROWS) else [None] * len(SYNTHETIC_STATIC_ROWS)
    for (row, _), stratum in zip(SYNTHETIC_STATIC_ROWS, _kept_strata):
        candidates = _pos_by_stratum.get(stratum) or _all_pos
        cand_X = _static_matrix(candidates)[:, _cont_idx]
        d = np.linalg.norm(cand_X - np.asarray(row)[_cont_idx], axis=1)
        SYNTHETIC_TS_DONORS.append(candidates[int(np.argmin(d))])
    _uses = Counter(SYNTHETIC_TS_DONORS)
    print(f"Assigned time-series donors to {len(SYNTHETIC_TS_DONORS)} synthetic patients "
          f"from {len(_uses)} distinct real died patients (max reuse of one donor: {max(_uses.values())}).")
elif SYNTHETIC_STATIC_ROWS:
    print("SYNTHETIC_TS_MODE='empty' -- synthetic patients get an all-unobserved time series (v1 behaviour).")

## 8.3 Sequence-branch augmentation for real minority patients (jitter + time-mask)

Applies **only to real positive training patients** (never to survived patients, never to
synthetic static rows, which have no real sequence to begin with -- Section 9.1 already
fills those with zeros on the standardized scale). Two augmentations, chosen to match
what's actually buildable on this notebook's fixed-length resampled time grid (Section
6.3's `TARGET_SEQ_LEN` points per system) rather than raw irregular timestamps:

- **Magnitude jitter**: adds small Gaussian noise (`CONFIG['JITTER_SIGMA']`, on the
  already-standardized z-score scale) to observed values only -- imputed/masked positions
  are left alone so jitter doesn't compound with the imputation strategy's own guesswork.
- **Time-masking**: randomly zeroes out a short contiguous span of timesteps (and their
  mask), forcing the encoder to tolerate a missing stretch it hasn't seen at exactly that
  position before -- a practical, honestly-labelled stand-in for the "window slicing /
  cropping" augmentation discussed in the external review, adapted for a fixed-length grid
  rather than literal random-interval cropping of raw timestamps.

**DTW Barycentric Averaging and embedding-space SMOTE are deliberately not implemented
here** -- both were flagged in `Data_Imbalance_and_Imputation_Reference.md` Part A.2 as
higher-effort, later-pass items (DBA needs a proper aligned-averaging implementation;
embedding-space SMOTE is only valid after the Section 9.4 pre-training phase completes,
which hasn't happened yet at this point in the notebook).

In [ ]:
def build_augment_items(effective_train_ids, use_augmentation, copies_per_patient):
    if not use_augmentation:
        return []
    real_positive_ids = [sid for sid in effective_train_ids
                          if not str(sid).startswith("synthetic_") and PATIENT_BUNDLE[sid]["label"] == 1.0]
    items = []
    for sid in real_positive_ids:
        for copy_idx in range(copies_per_patient):
            items.append((sid, copy_idx))   # copy_idx also seeds the augmentation's randomness, Section 9.1
    return items

AUGMENT_ITEMS = build_augment_items(EFFECTIVE_TRAIN_IDS, CONFIG["USE_SEQUENCE_AUGMENTATION"],
                                     CONFIG["SEQUENCE_AUGMENTATION_COPIES"])
print(f"Sequence augmentation: {len(AUGMENT_ITEMS)} augmented copies "
      f"({CONFIG['SEQUENCE_AUGMENTATION_COPIES']} per real positive training patient, "
      f"jitter_sigma={CONFIG['JITTER_SIGMA']})")

In [ ]:
record_checkpoint("Part 8 -- class-imbalance sampling")

# Part 9 — The multimodal organ-system DNN

This is where every theory decision from Part 1 becomes a `torch.nn.Module`. Build order:

- §9.1 the `INSPIREDataset` — turns `IMPUTED_BUNDLE` into model-ready tensors
- §9.2 `SystemEncoder` — one per organ system, transformer over time (§1.5.1) with the
  §6.9 cardiovascular→renal coupling wired in via an extra side-input
- §9.3 `StaticEncoder` — the small MLP for static/GI/MSK/HFRS/operation-count features
  (§1.5.1, §1.6.1)
- §9.4 the autoencoder pre-training head (§1.5.4, phase 1 of "jointly learned")
- §9.5 the loss functions — `pos_weight`-scaled BCE and focal loss (§1.4.1)
- §9.6 `MortalityModel` — the full network: six system encoders + static encoder → fusion
- §9.7 the fusion head itself — NAM-style additive (default) or plain concatenation
  (§1.5.2), switchable via `CONFIG['FUSION_STRATEGY']`

## 9.1 `INSPIREDataset` — bundle → tensors

One `__getitem__` call returns everything the model needs for one patient: six
`[T, F_system]` tensors + six masks, the static feature vector, and the label. Three kinds
of item now feed the training dataset, each flagged in the returned dict so later analysis
(§11's per-patient qualitative checks) can tell them apart:

- **Real patients** — straight from `IMPUTED_BUNDLE`.
- **Synthetic static rows** (§8.2, grouped SMOTENC + Tomek) — a real static vector, but no
  real time series exists for them: every system's sequence is filled with zeros (the
  population mean, on the standardized scale — §7.6) and the mask is all-zero, honestly
  recording that nothing was actually observed.
- **Augmented real minority patients** (§8.3) — a *real* patient's *real* static vector
  and diagnosis history, but with jitter and/or a time-mask applied to their sequence
  branch on the fly, a different random perturbation each epoch (`copy_idx` combined with
  the current epoch seeds the perturbation, so repeated passes over the same augmented
  item don't produce an identical duplicate every time).

In [ ]:
import zlib

def _stable_seed(sid, copy_idx, epoch_seed):
    # v2: Python's built-in hash() of a string is randomised per process (PYTHONHASHSEED),
    # so v1's augmentation noise was not reproducible between sessions. crc32 is.
    return zlib.crc32(f"{sid}|{copy_idx}|{epoch_seed}".encode()) & 0xFFFFFFFF

def _jitter_cols(system):
    # 1.0 for features that may be jittered, 0.0 for discrete/ordinal/binary ones (v2)
    fnames = PATIENT_BUNDLE[TRAIN_IDS[0]]["ts"][system]["feature_names"]
    return np.array([0.0 if f in CONFIG["JITTER_EXCLUDE_ITEMS"] else 1.0 for f in fnames])

JITTER_COLS = {s: _jitter_cols(s) for s in TIME_SERIES_SYSTEMS}

def _jitter_and_mask_sequence(raw, mask, sigma, rng, jitter_cols=None):
    # raw, mask: [T, F] numpy arrays, already imputed + standardized (Part 7).
    # Jitter only OBSERVED positions (mask==1) so noise doesn't compound with imputation.
    # Time-mask: zero out a short contiguous span of timesteps (all features at once) --
    # the fixed-length-grid stand-in for window slicing/cropping, see Part 8.3.
    if raw.shape[1] == 0:
        return raw.copy(), mask.copy()
    out_raw = raw.copy()
    out_mask = mask.copy()

    noise = rng.normal(loc=0.0, scale=sigma, size=out_raw.shape)
    if jitter_cols is not None:
        noise = noise * jitter_cols[None, :]   # v2: leave discrete items (GCS, device flags) untouched
    out_raw = out_raw + noise * out_mask   # only perturb where mask==1 (real observations)

    T = out_raw.shape[0]
    if T >= 4:
        span = max(1, T // 6)
        start = rng.integers(0, max(T - span, 1))
        out_raw[start:start + span, :] = 0.0
        out_mask[start:start + span, :] = 0.0

    return out_raw, out_mask

class INSPIREDataset(Dataset):
    def __init__(self, subject_ids, synthetic_static_rows=None, augment_items=None, epoch_seed=0,
                 synthetic_ts_donors=None):
        self.subject_ids = list(subject_ids)
        self.synthetic_rows = synthetic_static_rows or []
        self.synthetic_ts_donors = synthetic_ts_donors or []   # v2: aligned with synthetic_rows (Part 8.2c)
        self.augment_items = augment_items or []   # list of (sid, copy_idx)
        self.n_real = len(self.subject_ids)
        self.n_synthetic = len(self.synthetic_rows)
        self.epoch_seed = epoch_seed   # bump this between epochs for fresh augmentation noise

    def __len__(self):
        return self.n_real + self.n_synthetic + len(self.augment_items)

    def __getitem__(self, idx):
        if idx < self.n_real:
            sid = self.subject_ids[idx]
            item = {"is_synthetic": False, "is_augmented": False, "subject_id": sid}
            for system in TIME_SERIES_SYSTEMS:
                raw = IMPUTED_BUNDLE[sid][system]
                mask = PATIENT_BUNDLE[sid]["ts"][system]["mask"]
                item[f"{system}_x"] = torch.tensor(raw, dtype=torch.float32)
                item[f"{system}_mask"] = torch.tensor(mask, dtype=torch.float32)
            item["static"] = torch.tensor(IMPUTED_BUNDLE[sid]["static"].values.astype(float), dtype=torch.float32)
            item["label"] = torch.tensor(PATIENT_BUNDLE[sid]["label"], dtype=torch.float32)
            return item

        idx -= self.n_real
        if idx < self.n_synthetic:
            static_vec, label = self.synthetic_rows[idx]
            item = {"is_synthetic": True, "is_augmented": False, "subject_id": f"synthetic_{idx}"}
            if self.synthetic_ts_donors:
                # v2: borrow the nearest real died patient's series, freshly jittered each epoch
                donor = self.synthetic_ts_donors[idx]
                rng = np.random.default_rng(_stable_seed(f"synthetic_{idx}", 0, self.epoch_seed))
                for system in TIME_SERIES_SYSTEMS:
                    aug_raw, aug_mask = _jitter_and_mask_sequence(
                        IMPUTED_BUNDLE[donor][system], PATIENT_BUNDLE[donor]["ts"][system]["mask"],
                        CONFIG["JITTER_SIGMA"], rng, JITTER_COLS[system])
                    item[f"{system}_x"] = torch.tensor(aug_raw, dtype=torch.float32)
                    item[f"{system}_mask"] = torch.tensor(aug_mask, dtype=torch.float32)
                item["static"] = torch.tensor(static_vec, dtype=torch.float32)
                item["label"] = torch.tensor(label, dtype=torch.float32)
                return item
            for system in TIME_SERIES_SYSTEMS:
                fnames = PATIENT_BUNDLE[self.subject_ids[0]]["ts"][system]["feature_names"]
                # NOTE: IMPUTED_BUNDLE is standardised (§7.6, z-score, mean 0) by the time this
                # runs, so "population mean" for a synthetic patient is simply 0 in this space
                # -- not the raw STATS mean, which is still in original clinical units.
                filled = np.zeros((CONFIG["TARGET_SEQ_LEN"], len(fnames))) if fnames else np.zeros((CONFIG["TARGET_SEQ_LEN"], 0))
                item[f"{system}_x"] = torch.tensor(filled, dtype=torch.float32)
                item[f"{system}_mask"] = torch.zeros(filled.shape, dtype=torch.float32)   # honestly: nothing was observed
            item["static"] = torch.tensor(static_vec, dtype=torch.float32)
            item["label"] = torch.tensor(label, dtype=torch.float32)
            return item

        idx -= self.n_synthetic
        sid, copy_idx = self.augment_items[idx]
        rng = np.random.default_rng(_stable_seed(sid, copy_idx, self.epoch_seed))
        item = {"is_synthetic": False, "is_augmented": True, "subject_id": f"{sid}_aug{copy_idx}"}
        for system in TIME_SERIES_SYSTEMS:
            raw = IMPUTED_BUNDLE[sid][system]
            mask = PATIENT_BUNDLE[sid]["ts"][system]["mask"]
            aug_raw, aug_mask = _jitter_and_mask_sequence(raw, mask, CONFIG["JITTER_SIGMA"], rng, JITTER_COLS[system])
            item[f"{system}_x"] = torch.tensor(aug_raw, dtype=torch.float32)
            item[f"{system}_mask"] = torch.tensor(aug_mask, dtype=torch.float32)
        item["static"] = torch.tensor(IMPUTED_BUNDLE[sid]["static"].values.astype(float), dtype=torch.float32)
        item["label"] = torch.tensor(PATIENT_BUNDLE[sid]["label"], dtype=torch.float32)
        return item

def collate_bundle(batch):
    out = {}
    for key in batch[0]:
        if key in ("is_synthetic", "is_augmented", "subject_id"):
            out[key] = [b[key] for b in batch]
        else:
            out[key] = torch.stack([b[key] for b in batch])
    return out

train_dataset = INSPIREDataset(EFFECTIVE_TRAIN_IDS, SYNTHETIC_STATIC_ROWS, AUGMENT_ITEMS,
                               synthetic_ts_donors=SYNTHETIC_TS_DONORS)
val_dataset   = INSPIREDataset(VAL_IDS)
test_dataset  = INSPIREDataset(TEST_IDS)

from torch.utils.data import WeightedRandomSampler

def item_labels(ds):
    # label of every dataset item: real patients, then synthetic (all 1), then augmented copies (all 1)
    return np.array([PATIENT_BUNDLE[s]["label"] for s in ds.subject_ids] + [1.0] * ds.n_synthetic
                    + [1.0] * len(ds.augment_items))

# v4: worker processes + pinned memory for big runs (CONFIG['NUM_WORKERS'] > 0, e.g. Isambard-AI).
LOADER_KW = {"num_workers": CONFIG["NUM_WORKERS"], "pin_memory": DEVICE.type == "cuda"}
if CONFIG["NUM_WORKERS"] > 0:
    LOADER_KW["persistent_workers"] = True

def make_train_loader(ds, seed=SEED):
    # v3: balanced batches. At ~1:200 a plain shuffled batch of 256 has ~1.25 deaths on
    # average and ~29% of batches have none. The sampler draws each epoch's items so that
    # ~SAMPLER_TARGET_POSITIVE_RATE of them are deaths (real, synthetic or augmented).
    bs = min(CONFIG["BATCH_SIZE"], len(ds))
    if not CONFIG["BALANCED_SAMPLER"]:
        return DataLoader(ds, batch_size=bs, shuffle=True, collate_fn=collate_bundle, **LOADER_KW)
    y = item_labels(ds)
    p = CONFIG["SAMPLER_TARGET_POSITIVE_RATE"]
    n_pos, n_neg = max(y.sum(), 1), max(len(y) - y.sum(), 1)
    w = np.where(y == 1, p / n_pos, (1 - p) / n_neg)
    sampler = WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=len(ds), replacement=True,
                                    generator=torch.Generator().manual_seed(seed))
    return DataLoader(ds, batch_size=bs, sampler=sampler, collate_fn=collate_bundle, **LOADER_KW)

train_loader = make_train_loader(train_dataset)
# Pre-training is label-free, so it uses REAL training patients only, in their natural mix
# (synthetic/augmented copies of deaths would only distort what "normal" looks like).
pretrain_dataset = INSPIREDataset(EFFECTIVE_TRAIN_IDS)
pretrain_loader = DataLoader(pretrain_dataset, batch_size=min(CONFIG["BATCH_SIZE"], len(pretrain_dataset)),
                             shuffle=True, collate_fn=collate_bundle, **LOADER_KW)
if CONFIG["BALANCED_SAMPLER"]:
    # Batches are now ~10% deaths, so the loss weight that finishes the balancing is the
    # batch-level ratio, not the raw training ratio (recalibration in 11.1c maps the scores
    # back to the true risk scale).
    POS_WEIGHT = (1 - CONFIG["SAMPLER_TARGET_POSITIVE_RATE"]) / CONFIG["SAMPLER_TARGET_POSITIVE_RATE"]
    print(f"Balanced sampler on: ~{CONFIG['SAMPLER_TARGET_POSITIVE_RATE']:.0%} deaths per batch; pos_weight set to {POS_WEIGHT:.1f}")
val_loader   = DataLoader(val_dataset, batch_size=min(CONFIG["BATCH_SIZE"], max(len(val_dataset), 1)),
                           shuffle=False, collate_fn=collate_bundle, **LOADER_KW)
test_loader  = DataLoader(test_dataset, batch_size=min(CONFIG["BATCH_SIZE"], max(len(test_dataset), 1)),
                           shuffle=False, collate_fn=collate_bundle, **LOADER_KW)

print(f"train_dataset: {len(train_dataset)} items "
      f"({train_dataset.n_real} real, {train_dataset.n_synthetic} synthetic, {len(AUGMENT_ITEMS)} augmented)")
print(f"val_dataset:   {len(val_dataset)} items")
print(f"test_dataset:  {len(test_dataset)} items")

_batch = next(iter(train_loader))
print("\nExample batch shapes:")
for k, v in _batch.items():
    if torch.is_tensor(v):
        print(f"  {k:20s} {tuple(v.shape)}")

### 9.1b NEWS2 before vs. after sequence augmentation (v2)

Jitter (σ = `JITTER_SIGMA` on the z-scored scale) and time-masking are applied to real
died patients' sequences. **Before** = NEWS2 recomputed from each patient's original
model grid; **after** = recomputed from the augmented grid, at the same time points
(values converted back to clinical units first). A good augmentation perturbs values a
little without routinely pushing patients into a different NEWS2 band.

In [ ]:
def _grid_news2(sid, xs_by_system, masks_by_system):
    # NEWS2 per grid time point from a patient's (standardised) model tensors.
    vals = {}
    for system, item in [("respiratory", "rr"), ("respiratory", "spo2"), ("respiratory", "fio2"),
                         ("respiratory", "vent"), ("metabolic_hepatic", "bt"), ("cardiovascular", "nibp_sbp"),
                         ("cardiovascular", "hr"), ("neurological", "gcs_e"), ("neurological", "gcs_m"),
                         ("neurological", "gcs_v")]:
        fnames = PATIENT_BUNDLE[sid]["ts"][system]["feature_names"]
        if item not in fnames:
            vals[item] = np.full(CONFIG["TARGET_SEQ_LEN"], np.nan); continue
        j = fnames.index(item)
        x = xs_by_system[system][:, j] * TRAIN_STD[(system, item)] + STATS[(system, item)]["mean"]
        # round to charting precision (temp 0.1 C, everything else whole numbers): NEWS2 bands
        # are defined on charted values, so 95.95 vs 96 is not a real band change
        x = np.round(x, 1) if item == "bt" else np.round(x)
        vals[item] = np.where(masks_by_system[system][:, j] > 0, x, np.nan)   # observed points only
    comps = news2_components(rr=vals["rr"], spo2=vals["spo2"],
                             on_oxygen=on_supplemental_oxygen(vals["fio2"], vals["vent"], CONFIG["NEWS2_ASSUME_AIR_IF_MISSING"]),
                             sbp=vals["nibp_sbp"], hr=vals["hr"],
                             alert=alert_from_gcs(vals["gcs_e"], vals["gcs_m"], vals["gcs_v"]), temp_c=vals["bt"])
    total, _, any3 = news2_total(comps, CONFIG["NEWS2_ASSUME_ALERT_IF_MISSING"], CONFIG["NEWS2_MIN_CORE_VITALS"])
    return total, news2_risk_band(total, any3)

if not NEWS2_VALIDATED:
    print("NEWS2 self-test failed -- skipping the augmentation check.")
elif not AUGMENT_ITEMS:
    print("No sequence-augmented copies this run -- nothing to check.")
else:
    _b_tot, _a_tot, _b_band, _a_band, _n_masked, _n_points = [], [], [], [], 0, 0
    for sid, copy_idx in AUGMENT_ITEMS:
        rng = np.random.default_rng(_stable_seed(sid, copy_idx, 0))
        xs0 = {s: IMPUTED_BUNDLE[sid][s] for s in TIME_SERIES_SYSTEMS}
        ms0 = {s: PATIENT_BUNDLE[sid]["ts"][s]["mask"] for s in TIME_SERIES_SYSTEMS}
        xs1, ms1 = {}, {}
        for s in TIME_SERIES_SYSTEMS:   # same draw order as INSPIREDataset.__getitem__
            xs1[s], ms1[s] = _jitter_and_mask_sequence(xs0[s], ms0[s], CONFIG["JITTER_SIGMA"], rng, JITTER_COLS[s])
        t0, b0 = _grid_news2(sid, xs0, ms0)
        t1, b1 = _grid_news2(sid, xs1, ms1)
        both = ~np.isnan(t0) & ~np.isnan(t1)
        _n_points += int((~np.isnan(t0)).sum()); _n_masked += int((~np.isnan(t0) & np.isnan(t1)).sum())
        _b_tot.extend(t0[both]); _a_tot.extend(t1[both]); _b_band.extend(b0[both]); _a_band.extend(b1[both])
    _b_tot, _a_tot = np.array(_b_tot), np.array(_a_tot)
    res = compare_before_after(_b_tot, _a_tot, name="news2 per time point (jitter + time-mask)")
    res["mean_abs_change"] = float(np.mean(np.abs(_a_tot - _b_tot))) if len(_b_tot) else np.nan
    res["pct_band_changed"] = float(np.mean(np.array(_b_band) != np.array(_a_band))) if len(_b_band) else np.nan
    res["pct_points_hidden_by_time_mask"] = _n_masked / max(_n_points, 1)
    NEWS2_AUGMENTATION_CHECKS.append(res)
    print(pd.Series(res))
    print(f"\nReading this: jitter moved NEWS2 by {res['mean_abs_change']:.2f} points on average and changed "
          f"the risk band at {res['pct_band_changed']:.1%} of scorable time points. Under ~10% band changes "
          f"is a gentle augmentation; if it is much higher, lower CONFIG['JITTER_SIGMA'].")

## 9.2–9.4 Model building blocks (v3)

Inlined from `src-dnn/inspire_dnn/layers.py` (unit-tested there). In one line each:

- **Eight organ systems, all built the same way.** Each owns its measurements over time
  (none for GI and MSK in this dataset) and its own static features (its ICD-10 chapter,
  its summary statistics — mapping printed in 9.6 for sign-off).
- **`OrganSystemEncoder`** — ONE shared transformer body for all eight systems, a small
  input adapter per system and a "which system am I" tag. ~8x fewer body parameters than a
  network per system, and small systems borrow what data-rich systems teach the body.
- **`WholePatientLayer`** — reads every feature at once. It supports each system network
  by re-scaling that system's summary (0.5x–1.5x per channel: it can change how strongly a
  system's own evidence counts, not add another system's values), and it has its own
  whole-patient term in the risk. Starts as "no adjustment".
- **`SystemCouplingLayer`** — the inter-system layer. Each system reads the other seven;
  nothing hand-defined. **Pre-trained label-free** by hiding one system per patient and
  predicting it from the other seven (so it learns from every patient, not just deaths),
  then fine-tuned for mortality.
- **`SystemArbitrationLayer`** — how much say each system's answer gets for this patient,
  read from the whole-patient summary. Average say is 1; starts equal.
- **`NAMFusion`** — risk logit = bias + whole-patient term + Σ say × answer. Exact.

Guard-rails against memorising, all in `CONFIG`: shared body, label-free pre-training,
gentle fine-tuning (encoder learning rate × 0.1), AdamW weight decay, organ-system
dropout, capped SMOTENC, balanced batches. Part 11.2d (leakage check) and 11.4 (learning
curve, train/val gap, ensemble stability) test whether they worked.

In [ ]:
# --- inlined from src-dnn/inspire_dnn/layers.py (edit it there, then rebuild) ---
# Model building blocks for the INSPIRE organ-system DNN (v3). Pure PyTorch, no dataset
# globals: everything is passed in explicitly, so every piece is unit-tested with random
# tensors (tests/test_layers.py) and inlined into the notebooks unchanged.
#
#   8 organ systems (renal, cardiovascular, respiratory, metabolic/hepatic, haematology,
#   neurological, GI, MSK) -- all built the SAME way:
#       measurements over time (empty for GI/MSK in this dataset) + own static features
#       (own ICD-10 chapter, own summary stats)
#            |  OrganSystemEncoder: ONE shared transformer body for all 8, a small
#            |  per-system input adapter, and a "which system am I" tag
#            v
#   WholePatientLayer: reads ALL features; gently re-scales each system's summary (a narrow,
#            |  scale-only bottleneck) and gives its own whole-patient term
#            v
#   SystemCouplingLayer: each system reads the OTHER systems (learned attention; nothing
#            |  hand-defined). Pre-trained label-free by predicting a hidden system from the
#            |  other seven (masked-system task), then fine-tuned for mortality.
#            v
#   NAM answers f_s  x  SystemArbitrationLayer say w_s
#   logit = bias + f_whole_patient + f_surgical_context + sum_s w_s * f_s   (exact additive breakdown)
#
#   v4: an optional SURGICAL-CONTEXT term (emergency vs scheduled, ASA, department,
#   anaesthesia type) with its own risk points, so "how much did the surgical situation add"
#   is reported on its own instead of being mixed into the whole-patient term.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

ENCODER_SIZE_PRESETS = {
    # name: (embed_dim, n_heads, n_layers, ff_multiplier)
    "small":  (16, 2, 1, 2),
    "medium": (32, 4, 2, 4),
    "large":  (64, 4, 2, 4),
    # v4: big-data presets for the full ~99,886-patient cohort on large GPUs (e.g. Isambard-AI)
    "xlarge":  (128, 8, 3, 4),
    "xxlarge": (256, 8, 4, 4),
}


class OrganSystemEncoder(nn.Module):
    # One encoder for all eight organ systems. Each system has its own small input adapters
    # (its measurements and its static features differ in number), then everything runs
    # through ONE shared transformer body. Sharing means ~8x fewer body parameters and lets
    # small systems (GI, MSK) benefit from what the data-rich systems teach the body.
    #
    # Token layout per system: [summary token = own static features] + [T time tokens].
    # A system with no measurements (GI/MSK here) is just the summary token.
    def __init__(self, system_specs, embed_dim, n_heads, n_layers, ff_multiplier, target_seq_len,
                 dropout=0.1, share_body=True):
        super().__init__()
        self.systems = list(system_specs)
        self.specs = {s: tuple(v) for s, v in system_specs.items()}   # s -> (n_ts_features, n_static_features)
        self.T = target_seq_len
        self.ts_in = nn.ModuleDict({s: nn.Linear(2 * f, embed_dim) for s, (f, _) in self.specs.items() if f > 0})
        self.static_in = nn.ModuleDict({s: nn.Sequential(nn.Linear(k, embed_dim), nn.ReLU(), nn.Linear(embed_dim, embed_dim))
                                        for s, (_, k) in self.specs.items() if k > 0})
        self.empty_static = nn.Parameter(torch.zeros(embed_dim))
        self.system_tag = nn.Parameter(torch.randn(len(self.systems), embed_dim) * 0.02)
        self.pos_embedding = nn.Parameter(torch.randn(1, target_seq_len + 1, embed_dim) * 0.02)

        def make_body():
            layer = nn.TransformerEncoderLayer(embed_dim, n_heads, embed_dim * ff_multiplier, dropout,
                                               batch_first=True, norm_first=True)
            return nn.TransformerEncoder(layer, num_layers=n_layers, enable_nested_tensor=False)
        self.share_body = share_body
        if share_body:
            self.body = make_body()
        else:
            self.bodies = nn.ModuleDict({s: make_body() for s in self.systems})
        self.out_norm = nn.LayerNorm(embed_dim)

    def _body(self, s):
        return self.body if self.share_body else self.bodies[s]

    def tokens(self, s, x, mask, static_s, batch_size):
        tag = self.system_tag[self.systems.index(s)]
        if s in self.static_in:
            summary = self.static_in[s](static_s)
        else:
            summary = self.empty_static.unsqueeze(0).expand(batch_size, -1)
        seq = (summary + tag).unsqueeze(1)
        if s in self.ts_in:
            seq = torch.cat([seq, self.ts_in[s](torch.cat([x, mask], dim=-1)) + tag], dim=1)
        return seq + self.pos_embedding[:, : seq.shape[1]]

    def forward(self, s, x, mask, static_s, batch_size):
        h = self.out_norm(self._body(s)(self.tokens(s, x, mask, static_s, batch_size)))
        return h.mean(dim=1), (h[:, 1:] if s in self.ts_in else None)

    @torch.no_grad()
    def attention_map(self, s, x, mask, static_s):
        # First-layer self-attention for one system (averaged over heads), pre-norm applied --
        # the audit hook for Part 11.3. Token 0 = summary token, 1..T = time points.
        seq = self.tokens(s, x, mask, static_s, x.shape[0] if x is not None else static_s.shape[0])
        layer0 = self._body(s).layers[0]
        h = layer0.norm1(seq)
        _, w = layer0.self_attn(h, h, h, need_weights=True, average_attn_weights=True)
        return w


class ReconstructionHead(nn.Module):
    def __init__(self, embed_dim, n_out, target_seq_len=None):
        super().__init__()
        self.n_out, self.T = n_out, target_seq_len
        if n_out == 0:
            return
        size = n_out * (target_seq_len or 1)
        self.decoder = nn.Sequential(nn.Linear(embed_dim, embed_dim * 2), nn.ReLU(), nn.Linear(embed_dim * 2, size))

    def forward(self, e):
        if self.n_out == 0:
            return None
        out = self.decoder(e)
        return out.view(-1, self.T, self.n_out) if self.T else out


def masked_reconstruction_loss(pred, target, mask=None):
    if pred is None:
        return None
    if mask is None:
        return ((pred - target) ** 2).mean()
    return ((pred - target) ** 2 * mask).sum() / mask.sum().clamp(min=1.0)


class WholePatientLayer(nn.Module):
    # Reads EVERY feature (full static vector + a per-feature summary of every time series)
    # and returns (g, scales). g is the whole-patient summary (its own NAM term, and the input
    # to the arbitration layer). scales re-weight each organ system's summary channel by
    # channel within [0.5, 1.5] -- a deliberately narrow, scale-only bottleneck: it can change
    # how strongly a system's own evidence counts, but cannot add another system's values.
    # The scale projection is zero-initialised, so it starts as "no adjustment".
    def __init__(self, n_static, ts_feature_counts, system_names, embed_dim, dropout=0.1, static_keep_idx=None):
        super().__init__()
        self.ts_systems = [s for s, f in ts_feature_counts.items() if f > 0]
        # v4: static_keep_idx lets the surgical-context term own its columns exclusively
        # (None = read every static column, the v3 behaviour).
        self.static_keep_idx = None if static_keep_idx is None else list(static_keep_idx)
        n_in_static = n_static if self.static_keep_idx is None else len(self.static_keep_idx)
        in_dim = n_in_static + sum(2 * ts_feature_counts[s] for s in self.ts_systems)
        self.system_names = list(system_names)
        self.net = nn.Sequential(nn.Linear(in_dim, embed_dim * 2), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(embed_dim * 2, embed_dim))
        self.film = nn.Linear(embed_dim, len(self.system_names) * embed_dim)
        nn.init.zeros_(self.film.weight)
        nn.init.zeros_(self.film.bias)
        self.embed_dim = embed_dim

    def forward(self, batch):
        st = batch["static"]
        parts = [st if self.static_keep_idx is None else st[:, self.static_keep_idx]]
        for s in self.ts_systems:
            parts += [batch[f"{s}_x"].mean(dim=1), batch[f"{s}_mask"].mean(dim=1)]
        g = self.net(torch.cat(parts, dim=-1))
        scales = 1.0 + 0.5 * torch.tanh(self.film(g)).view(-1, len(self.system_names), self.embed_dim)
        return g, scales


class SystemCouplingLayer(nn.Module):
    # Learned inter-system coupling. Each system (query) attends over the OTHER systems
    # (its own token is masked out), so the message it receives is information from elsewhere
    # in the body.   e_s' = e_s + g_s * message_s   (g_s a learnable gate, starts at 0.1).
    # forward() also returns the raw messages, which masked-system pre-training uses to
    # predict a hidden system from the other seven.
    def __init__(self, system_names, embed_dim, n_heads=2, prior_links=(), prior_strength=1.0,
                 dropout=0.1, gate_init=0.1):
        super().__init__()
        assert embed_dim % n_heads == 0, "embed_dim must be divisible by coupling heads"
        self.system_names = list(system_names)
        S = len(self.system_names)
        self.n_heads, self.head_dim = n_heads, embed_dim // n_heads
        self.identity = nn.Parameter(torch.randn(S, embed_dim) * 0.02)
        self.q = nn.Linear(embed_dim, embed_dim)
        self.k = nn.Linear(embed_dim, embed_dim)
        self.v = nn.Linear(embed_dim, embed_dim)
        self.o = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.gate = nn.Parameter(torch.full((S,), float(gate_init)))
        prior = torch.zeros(S, S)
        for sender, receiver in prior_links:
            if sender in self.system_names and receiver in self.system_names:
                prior[self.system_names.index(receiver), self.system_names.index(sender)] = prior_strength
        self.prior_bias = nn.Parameter(prior)
        self.register_buffer("self_mask", torch.eye(S, dtype=torch.bool))

    def forward(self, E, return_messages=False):
        B, S, D = E.shape
        tagged = E + self.identity.unsqueeze(0)
        q = self.q(tagged).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k(tagged).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v(E).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / math.sqrt(self.head_dim) + self.prior_bias
        attn = torch.softmax(scores.masked_fill(self.self_mask, float("-inf")), dim=-1)
        msg = self.o((self.dropout(attn) @ v).transpose(1, 2).reshape(B, S, D))
        out = E + self.gate.view(1, S, 1) * msg
        report = {"weights": attn.mean(dim=1), "gate": self.gate.detach()}   # weights: [B, receiver, sender]
        return (out, report, msg) if return_messages else (out, report)


class SystemArbitrationLayer(nn.Module):
    # How much "say" each organ system's answer gets for THIS patient:
    #   w = n_systems * softmax(a(x))  -> average say is exactly 1, starts equal (zero-init).
    def __init__(self, system_names, input_dim, hidden_dim=32):
        super().__init__()
        self.system_names = list(system_names)
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(),
                                 nn.Linear(hidden_dim, len(self.system_names)))
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, x):
        w = len(self.system_names) * torch.softmax(self.net(x), dim=-1)
        return w, (torch.log(w.clamp_min(1e-6)) ** 2).mean()


class NAMFusion(nn.Module):
    def __init__(self, term_names, embed_dim):
        super().__init__()
        self.system_names = list(term_names)
        hidden = max(embed_dim // 2, 4)
        self.shape_functions = nn.ModuleDict({
            n: nn.Sequential(nn.Linear(embed_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))
            for n in self.system_names})
        self.bias = nn.Parameter(torch.zeros(1))

    def answers(self, emb):
        return {n: self.shape_functions[n](emb[n]).squeeze(-1) for n in self.system_names}

    def forward(self, emb, say=None):
        answers = self.answers(emb)
        contributions = {n: (answers[n] * say[n] if (say is not None and n in say) else answers[n])
                         for n in self.system_names}
        return sum(contributions.values()) + self.bias, contributions, answers


class ConcatFusion(nn.Module):
    def __init__(self, term_names, embed_dim):
        super().__init__()
        self.system_names = list(term_names)
        self.classifier = nn.Sequential(nn.Linear(embed_dim * len(self.system_names), embed_dim),
                                        nn.ReLU(), nn.Linear(embed_dim, 1))

    def forward(self, emb, say=None):
        return self.classifier(torch.cat([emb[n] for n in self.system_names], dim=-1)).squeeze(-1), None, None


FUSION_CLASSES = {"nam": NAMFusion, "concat": ConcatFusion}


class MortalityModel(nn.Module):
    # ts_feature_counts: {organ_system: n_time_series_features} for ALL 8 systems (0 = none).
    # system_static_idx:  {organ_system: [columns of the static vector owned by that system]}
    # context_idx:        columns of the static vector that belong to no single system
    #                     (age, ASA, department, NEWS2, frailty, ...).
    # surgical_context_idx (v4, optional): columns for the surgical-context term (emergency
    #                     flag, ASA, department, anaesthesia type). Empty/None = no such term.
    # surgical_context_exclusive: if True those columns are removed from the whole-patient
    #                     layer's input, so their effect on risk is reported ONLY by the
    #                     surgical-context term (plus how they change each system's say).
    def __init__(self, ts_feature_counts, system_static_idx, context_idx, n_static,
                 embed_dim=32, n_heads=4, n_layers=2, ff_multiplier=4, target_seq_len=24,
                 fusion_strategy="nam", share_encoder=True,
                 coupling_mode="learned", coupling_heads=2, coupling_prior_links=(),
                 coupling_prior_strength=1.0, coupling_gate_init=0.1,
                 use_arbitration=True, use_whole_patient=True, system_dropout=0.0, dropout=0.1,
                 surgical_context_idx=None, surgical_context_exclusive=True):
        super().__init__()
        self.organ_systems = list(ts_feature_counts)
        self.ts_systems = [s for s in self.organ_systems if ts_feature_counts[s] > 0]
        self.system_names = self.ts_systems   # compatibility with older analysis cells
        self.system_static_idx = {s: list(system_static_idx.get(s, [])) for s in self.organ_systems}
        self.context_idx = list(context_idx)
        self.coupling_mode = coupling_mode
        self.use_manual = False                # v3: no hand-defined inter-system input exists any more
        self.system_dropout = system_dropout

        specs = {s: (ts_feature_counts[s], len(self.system_static_idx[s])) for s in self.organ_systems}
        self.encoder = OrganSystemEncoder(specs, embed_dim, n_heads, n_layers, ff_multiplier,
                                          target_seq_len, dropout, share_body=share_encoder)
        self.recon_ts = nn.ModuleDict({s: ReconstructionHead(embed_dim, ts_feature_counts[s], target_seq_len)
                                       for s in self.ts_systems})
        self.recon_static = nn.ModuleDict({s: ReconstructionHead(embed_dim, len(self.system_static_idx[s]))
                                           for s in self.organ_systems if self.system_static_idx[s]})
        self.mask_token = nn.Parameter(torch.zeros(embed_dim))   # stands in for a hidden / dropped system

        self.surgical_context_idx = list(surgical_context_idx or [])
        self.has_surgical_context = len(self.surgical_context_idx) > 0
        if self.has_surgical_context:
            self.surgical_encoder = nn.Sequential(nn.Linear(len(self.surgical_context_idx), embed_dim), nn.ReLU(),
                                                  nn.Dropout(dropout), nn.Linear(embed_dim, embed_dim))
        else:
            self.surgical_encoder = None
        _sc = set(self.surgical_context_idx) if (self.has_surgical_context and surgical_context_exclusive) else set()
        wp_keep = [i for i in range(n_static) if i not in _sc] if _sc else None
        if _sc:   # the same exclusivity for the no-whole-patient ablation path
            self.context_idx = [i for i in self.context_idx if i not in _sc]

        self.use_whole_patient = use_whole_patient
        if use_whole_patient:
            self.whole_patient = WholePatientLayer(n_static, ts_feature_counts, self.organ_systems, embed_dim, dropout,
                                                   static_keep_idx=wp_keep)
            self.context_term = "whole_patient"
        else:
            self.whole_patient = None
            self.context_encoder = nn.Sequential(nn.Linear(max(len(self.context_idx), 1), embed_dim * 2), nn.ReLU(),
                                                 nn.Dropout(dropout), nn.Linear(embed_dim * 2, embed_dim))
            self.context_term = "static"
        self.coupling = None
        if coupling_mode == "learned":
            self.coupling = SystemCouplingLayer(self.organ_systems, embed_dim, coupling_heads,
                                                coupling_prior_links, coupling_prior_strength, dropout,
                                                gate_init=coupling_gate_init)
        self.term_names = self.organ_systems + [self.context_term] + (["surgical_context"] if self.has_surgical_context else [])
        self.fusion_strategy = fusion_strategy
        self.fusion = FUSION_CLASSES[fusion_strategy](self.term_names, embed_dim)
        # the arbitration layer reads the whole-patient summary (and, v4, the surgical-context
        # summary -- e.g. an emergency operation may change how much each system should count)
        arb_in = embed_dim * (2 if self.has_surgical_context else 1)
        self.arbitration = SystemArbitrationLayer(self.organ_systems, arb_in) \
            if (use_arbitration and fusion_strategy == "nam") else None

    # ---- pieces -------------------------------------------------------------------------
    def encoder_parameters(self):
        return list(self.encoder.parameters()) + list(self.recon_ts.parameters()) + list(self.recon_static.parameters())

    def _static_s(self, static, s):
        idx = self.system_static_idx[s]
        return static[:, idx] if idx else None

    def encode_systems(self, batch):
        static = batch["static"]
        B = static.shape[0]
        emb, per_t = {}, {}
        for s in self.organ_systems:
            x = batch.get(f"{s}_x"); m = batch.get(f"{s}_mask")
            emb[s], per_t[s] = self.encoder(s, x, m, self._static_s(static, s), B)
        return emb, per_t

    # ---- phase 1: label-free pre-training ----------------------------------------------
    def pretrain_loss(self, batch, masked_system_weight=1.0):
        # (a) each system reconstructs its OWN data from its own summary (autoencoder);
        # (b) masked-system task: one system per patient is hidden and must be predicted
        #     from the OTHER systems through the coupling layer -- this is how the coupling
        #     layer learns real inter-system relationships from every patient, no labels.
        emb, _ = self.encode_systems(batch)
        static = batch["static"]
        losses = []
        for s in self.ts_systems:
            l = masked_reconstruction_loss(self.recon_ts[s](emb[s]), batch[f"{s}_x"], batch[f"{s}_mask"])
            losses.append(l)
        for s in self.recon_static:
            losses.append(masked_reconstruction_loss(self.recon_static[s](emb[s]), self._static_s(static, s)))
        own = torch.stack(losses).mean()
        if self.coupling is None or masked_system_weight == 0:
            return own, {"own": float(own), "masked_system": 0.0}
        B, S = static.shape[0], len(self.organ_systems)
        E = torch.stack([emb[s] for s in self.organ_systems], dim=1)
        hidden = torch.randint(0, S, (B,), device=E.device)
        onehot = F.one_hot(hidden, S).bool().unsqueeze(-1)
        E_masked = torch.where(onehot, self.mask_token.view(1, 1, -1).expand_as(E), E)
        _, _, msg = self.coupling(E_masked, return_messages=True)
        m_losses = []
        for i, s in enumerate(self.organ_systems):
            sel = hidden == i
            if not sel.any():
                continue
            pred_in = self.mask_token + msg[sel, i]
            if s in self.recon_ts:
                m_losses.append(masked_reconstruction_loss(self.recon_ts[s](pred_in), batch[f"{s}_x"][sel], batch[f"{s}_mask"][sel]))
            if s in self.recon_static:
                m_losses.append(masked_reconstruction_loss(self.recon_static[s](pred_in), self._static_s(static, s)[sel]))
        ms = torch.stack(m_losses).mean() if m_losses else torch.zeros((), device=E.device)
        return own + masked_system_weight * ms, {"own": float(own), "masked_system": float(ms)}

    # ---- full forward (phase 2 + inference) ------------------------------------------------
    def forward(self, batch):
        static = batch["static"]
        emb, per_t = self.encode_systems(batch)
        own_emb = dict(emb)
        if self.whole_patient is not None:
            g, scales = self.whole_patient(batch)
            for i, s in enumerate(self.organ_systems):
                emb[s] = emb[s] * scales[:, i]
        else:
            g = self.context_encoder(static[:, self.context_idx] if self.context_idx else torch.zeros_like(static[:, :1]))
            scales = None
        if self.training and self.system_dropout > 0:
            for s in self.organ_systems:   # organ-system dropout: the model can't lean on one system
                drop = (torch.rand(static.shape[0], 1, device=static.device) < self.system_dropout)
                emb[s] = torch.where(drop, self.mask_token.expand_as(emb[s]), emb[s])
        pre_coupling = {s: emb[s] for s in self.organ_systems}
        coupling_report = None
        if self.coupling is not None:
            E2, coupling_report = self.coupling(torch.stack([emb[s] for s in self.organ_systems], dim=1))
            for i, s in enumerate(self.organ_systems):
                emb[s] = E2[:, i]
        emb[self.context_term] = g
        sc = None
        if self.has_surgical_context:
            sc = self.surgical_encoder(static[:, self.surgical_context_idx])
            emb["surgical_context"] = sc
        say, aux = None, torch.zeros((), device=static.device)
        if self.arbitration is not None:
            w, aux = self.arbitration(g if sc is None else torch.cat([g, sc], dim=-1))
            say = {s: w[:, i] for i, s in enumerate(self.organ_systems)}
        logit, contributions, answers = self.fusion(emb, say)
        return {
            "logit": logit,
            "embeddings": {s: emb[s] for s in self.organ_systems},
            "embeddings_own": own_emb,
            "embeddings_pre_coupling": pre_coupling,
            "static_embedding": g,
            "surgical_context_embedding": sc,
            "whole_patient_scales": scales,
            "per_system_signal": contributions,
            "system_answers": answers,
            "system_say": say,
            "coupling_report": coupling_report,
            "aux_loss": aux,
            "per_timestep": per_t,
        }


def coupling_effect(model, out):
    # Per patient: f_s(after coupling) - f_s(before coupling), in logit units.
    if model.coupling is None or out["system_answers"] is None:
        return None
    with torch.no_grad():
        extra = {model.context_term: out["static_embedding"]}
        if out.get("surgical_context_embedding") is not None:
            extra["surgical_context"] = out["surgical_context_embedding"]
        before = model.fusion.answers({**out["embeddings_pre_coupling"], **extra})
    return {s: out["system_answers"][s] - before[s] for s in model.organ_systems}


@torch.no_grad()
def own_data_share(model, batch, n_repeats=3, seed=0):
    # Leakage check. For each organ system s: shuffle s's OWN inputs across patients and
    # measure how much s's answer moves; then shuffle EVERYTHING ELSE and measure again.
    # own_share = own / (own + other). Near 1 = the system's answer is driven by its own
    # data (what a surgeon would assume); low = it is mostly echoing other systems.
    model.eval()
    gen = torch.Generator(device="cpu").manual_seed(seed)
    base = model(batch)["system_answers"]
    if base is None:
        return None
    B = batch["static"].shape[0]
    res = {}
    for s in model.organ_systems:
        own_cols = model.system_static_idx[s]
        d_own, d_other = 0.0, 0.0
        for _ in range(n_repeats):
            perm = torch.randperm(B, generator=gen).to(batch["static"].device)
            b1 = dict(batch); st = batch["static"].clone()
            if own_cols:
                st[:, own_cols] = batch["static"][perm][:, own_cols]
            b1["static"] = st
            if s in model.ts_systems:
                b1[f"{s}_x"], b1[f"{s}_mask"] = batch[f"{s}_x"][perm], batch[f"{s}_mask"][perm]
            d_own += float((model(b1)["system_answers"][s] - base[s]).abs().mean())
            b2 = {k: v for k, v in batch.items()}
            st2 = batch["static"][perm].clone()
            if own_cols:
                st2[:, own_cols] = batch["static"][:, own_cols]
            b2["static"] = st2
            for t in model.ts_systems:
                if t != s:
                    b2[f"{t}_x"], b2[f"{t}_mask"] = batch[f"{t}_x"][perm], batch[f"{t}_mask"][perm]
            d_other += float((model(b2)["system_answers"][s] - base[s]).abs().mean())
        res[s] = {"change_own": d_own / n_repeats, "change_other": d_other / n_repeats,
                  "own_share": d_own / max(d_own + d_other, 1e-12)}
    return res

## 9.5 Loss functions — `pos_weight` BCE and focal loss (§1.4.1)

Both take raw logits (no sigmoid applied beforehand) for numerical stability, matching
`torch.nn.BCEWithLogitsLoss` conventions. `CONFIG['USE_FOCAL_LOSS']` switches between them
independently of `CONFIG['SAMPLING_STRATEGY']` — the two are complementary, not
mutually exclusive, per §1.4.1's note that focal loss "complements class weighting rather
than replacing it."

In [ ]:
def focal_loss_with_logits(logits, targets, pos_weight, gamma=2.0):
    # Lin et al. 2017 focal loss, generalised with a pos_weight term so it still respects
    # class imbalance (gamma alone down-weights EASY examples of either class; pos_weight
    # additionally up-weights the rarer class specifically -- combining both per §1.4.1).
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p = torch.sigmoid(logits)
    p_t = p * targets + (1 - p) * (1 - targets)
    focal_term = (1 - p_t) ** gamma
    weight = torch.where(targets == 1, torch.tensor(pos_weight, device=logits.device), torch.tensor(1.0, device=logits.device))
    return (weight * focal_term * bce).mean()

def mortality_loss(logits, targets, pos_weight, use_focal, gamma):
    if use_focal:
        return focal_loss_with_logits(logits, targets, pos_weight, gamma)
    return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=torch.tensor(pos_weight, device=logits.device))

print(f"Loss configured: {'focal loss' if CONFIG['USE_FOCAL_LOSS'] else 'pos_weight BCE'}, pos_weight={POS_WEIGHT:.2f}")

## 9.6 Assembling the model — which static features belong to which organ system

Every static column is assigned to exactly one organ system or to "patient context".
The table is printed and saved (`organ_system_feature_map.csv`) for clinical sign-off —
a few placements are judgement calls (e.g. vasopressor use under the heart). Context
(age, ASA, department, NEWS2, frailty, infection flags, operation history, medication
counts) belongs to no single system and is read by the whole-patient layer.

In [ ]:
def assign_static_feature(name):
    if name.startswith("sysdx_"):
        for s in ORGAN_SYSTEMS:
            if name.startswith(f"sysdx_{s}_"):
                return s
    for s in ORGAN_SYSTEMS:   # v4: procedure body-system and medicine-family features (§6.14b)
        if name.startswith((f"proc_{s}_", f"atc_{s}_")):
            return s
    if CONFIG["USE_SURGICAL_CONTEXT_TERM"] and (name in ("emop", "asa") or name.startswith(("dept_", "antype_"))):
        return "surgical_context"   # v4: read by the surgical-context term (§9.6b)
    rules = [
        ("gi", ("gi_department_flag",)),
        ("msk", ("msk_department_flag",)),
        ("cardiovascular", ("cardio_", "agg_hr_", "agg_nibp_mbp_", "severe_hypotension", "vasopressor_")),
        ("renal", ("agg_creatinine_", "agg_potassium_")),
        ("respiratory", ("agg_spo2_",)),
        ("metabolic_hepatic", ("agg_glucose_", "agg_lacate_")),
        ("haematology", ("agg_wbc_", "agg_hb_")),
    ]
    for system, prefixes in rules:
        if name.startswith(prefixes):
            return system
    return "context"

STATIC_OWNER = {n: assign_static_feature(n) for n in STATIC_FEATURE_NAMES}
SYSTEM_STATIC_IDX = {s: [i for i, n in enumerate(STATIC_FEATURE_NAMES) if STATIC_OWNER[n] == s] for s in ORGAN_SYSTEMS}
CONTEXT_IDX = [i for i, n in enumerate(STATIC_FEATURE_NAMES) if STATIC_OWNER[n] in ("context", "surgical_context")]
SURGICAL_CONTEXT_IDX = [i for i, n in enumerate(STATIC_FEATURE_NAMES) if STATIC_OWNER[n] == "surgical_context"]
TS_FEATURE_COUNTS = {s: (len(PATIENT_BUNDLE[TRAIN_IDS[0]]["ts"][s]["feature_names"]) if s in TIME_SERIES_SYSTEMS else 0)
                     for s in ORGAN_SYSTEMS}
SYSTEM_FEATURE_COUNTS = {s: TS_FEATURE_COUNTS[s] for s in TIME_SERIES_SYSTEMS}   # kept for older cells
N_STATIC_FEATURES = len(STATIC_FEATURE_NAMES)

organ_system_feature_map = pd.DataFrame([
    {"organ_system": s,
     "measurements_over_time": ", ".join(PATIENT_BUNDLE[TRAIN_IDS[0]]["ts"][s]["feature_names"]) if s in TIME_SERIES_SYSTEMS else "(none in this dataset)",
     "static_features": ", ".join(STATIC_FEATURE_NAMES[i] for i in SYSTEM_STATIC_IDX[s])}
    for s in ORGAN_SYSTEMS] + [{"organ_system": "surgical context (v4)", "measurements_over_time": "-",
                                "static_features": ", ".join(STATIC_FEATURE_NAMES[i] for i in SURGICAL_CONTEXT_IDX)},
                               {"organ_system": "patient context", "measurements_over_time": "-",
                                "static_features": ", ".join(STATIC_FEATURE_NAMES[i] for i in CONTEXT_IDX
                                                             if i not in SURGICAL_CONTEXT_IDX)}])
organ_system_feature_map.to_csv(os.path.join(CONFIG["CHECKPOINT_DIR"], "organ_system_feature_map.csv"), index=False)
organ_system_feature_map.to_csv(os.path.join(TABLE_DIR, "organ_system_feature_map.csv"), index=False)
for _, r in organ_system_feature_map.iterrows():
    print(f"{r['organ_system']:18s} | over time: {r['measurements_over_time'][:70]}")
    print(f"{'':18s} | static:    {r['static_features'][:110]}")

def build_model(fusion_strategy=None, coupling_mode=None, use_arbitration=None, use_whole_patient=None,
                share_encoder=None, seed=SEED, use_surgical_context=None, drop_systems=(), encoder_size=None):
    # v4 options -- use_surgical_context: the separate surgical-context risk term;
    # drop_systems: e.g. ("gi", "msk") for the "6 systems vs 8" test (their static columns
    # then go to the whole-patient layer, nothing is thrown away); encoder_size: a preset
    # name for the embedding-size sweep (Part 13.9).
    torch.manual_seed(seed)
    use_sc = CONFIG["USE_SURGICAL_CONTEXT_TERM"] if use_surgical_context is None else use_surgical_context
    keep = [s for s in ORGAN_SYSTEMS if s not in set(drop_systems)]
    dropped_cols = [i for s in drop_systems for i in SYSTEM_STATIC_IDX.get(s, [])]
    if encoder_size is not None:
        ed, nh, nl, ff = _ENCODER_PRESETS[encoder_size]
    else:
        ed, nh, nl, ff = CONFIG["EMBED_DIM"], CONFIG["TRANSFORMER_HEADS"], CONFIG["TRANSFORMER_LAYERS"], CONFIG["FF_MULTIPLIER"]
    return MortalityModel(
        ts_feature_counts={s: TS_FEATURE_COUNTS[s] for s in keep},
        system_static_idx={s: SYSTEM_STATIC_IDX[s] for s in keep},
        context_idx=sorted(CONTEXT_IDX + dropped_cols),
        n_static=N_STATIC_FEATURES,
        embed_dim=ed, n_heads=nh, n_layers=nl,
        ff_multiplier=ff, target_seq_len=CONFIG["TARGET_SEQ_LEN"],
        fusion_strategy=fusion_strategy or CONFIG["FUSION_STRATEGY"],
        share_encoder=CONFIG["SHARE_ENCODER"] if share_encoder is None else share_encoder,
        coupling_mode=coupling_mode or CONFIG["COUPLING_MODE"], coupling_heads=CONFIG["COUPLING_HEADS"],
        coupling_prior_links=[tuple(l) for l in CONFIG["COUPLING_PRIOR_LINKS"]],
        coupling_prior_strength=CONFIG["COUPLING_PRIOR_STRENGTH"], coupling_gate_init=CONFIG["COUPLING_GATE_INIT"],
        use_arbitration=CONFIG["USE_ARBITRATION_LAYER"] if use_arbitration is None else use_arbitration,
        use_whole_patient=CONFIG["USE_WHOLE_PATIENT_LAYER"] if use_whole_patient is None else use_whole_patient,
        system_dropout=CONFIG["SYSTEM_DROPOUT"], dropout=CONFIG["DROPOUT"],
        surgical_context_idx=SURGICAL_CONTEXT_IDX if use_sc else None,
        surgical_context_exclusive=CONFIG["SURGICAL_CONTEXT_EXCLUSIVE"],
    ).to(DEVICE)

model = build_model()

def _count(module):
    return sum(p.numel() for p in module.parameters()) if module is not None else 0
n_params = _count(model)
_n_enc = _count(model.encoder)
print(f"\nMortalityModel built: {n_params:,} parameters in total")
print(f"  organ-system encoder ({'one shared body' if CONFIG['SHARE_ENCODER'] else 'a body per system'}): {_n_enc:,}")
print(f"  whole-patient layer: {_count(model.whole_patient):,}   coupling layer: {_count(model.coupling):,}   "
      f"arbitration layer: {_count(model.arbitration):,}   surgical-context term: {_count(model.surgical_encoder):,}")
print(f"Terms in the risk (each reported separately): {model.term_names}")

import hashlib
_fingerprint_source = repr((
    TS_FEATURE_COUNTS, N_STATIC_FEATURES, SYSTEM_STATIC_IDX, CONFIG["EMBED_DIM"], CONFIG["TRANSFORMER_HEADS"],
    CONFIG["TRANSFORMER_LAYERS"], CONFIG["FF_MULTIPLIER"], CONFIG["TARGET_SEQ_LEN"], CONFIG["FUSION_STRATEGY"],
    CONFIG["SHARE_ENCODER"], CONFIG["COUPLING_MODE"], CONFIG["COUPLING_HEADS"], CONFIG["USE_ARBITRATION_LAYER"],
    CONFIG["USE_WHOLE_PATIENT_LAYER"], CONFIG["USE_SURGICAL_CONTEXT_TERM"], CONFIG["SURGICAL_CONTEXT_EXCLUSIVE"],
    tuple(SURGICAL_CONTEXT_IDX), "v4",
))
ARCHITECTURE_FINGERPRINT = hashlib.sha256(_fingerprint_source.encode()).hexdigest()[:10]
print(f"Architecture fingerprint: {ARCHITECTURE_FINGERPRINT} (checkpoints are tagged with this)")

_batch_dev = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in _batch.items()}
model.eval()
with torch.no_grad():
    _out = model(_batch_dev)
print(f"\nSmoke test forward pass OK. logit shape: {tuple(_out['logit'].shape)}")
if _out["per_system_signal"] is not None:
    _recon = sum(_out["per_system_signal"].values()) + model.fusion.bias
    print(f"Terms sum exactly to the logit: {torch.allclose(_recon, _out['logit'], atol=1e-4)}")

## 9.8 Layer-by-layer sizes (v4) — the table for the paper

Like the layer tables in the reference paper James shared: what goes into each part of the
model and what comes out. `B` = patients in a batch, `T` = time points (24), `F` = a
system's measurements, `K` = a system's own static features, `D` = embedding size, `S` =
number of organ systems. Saved to `OUTPUT_DIR/tables/layer_sizes.csv`.

In [ ]:
def layer_size_table(m):
    D = m.encoder.pos_embedding.shape[-1]; T = CONFIG["TARGET_SEQ_LEN"]; S = len(m.organ_systems)
    body = m.encoder.body if m.encoder.share_body else next(iter(m.encoder.bodies.values()))
    L = len(body.layers); H = body.layers[0].self_attn.num_heads; FF = body.layers[0].linear1.out_features
    rows = []
    for s in m.organ_systems:
        F_s, K_s = m.encoder.specs[s]
        if F_s:
            rows.append((f"{s}: input over time", f"[B, {T}, {F_s}] values + [B, {T}, {F_s}] mask", "", "the 24-hole egg carton + 'real or filled' stickers"))
            rows.append((f"{s}: time-point adapter (Linear)", f"[B, {T}, {2 * F_s}]", f"[B, {T}, {D}]", "one token per time point"))
        if K_s:
            rows.append((f"{s}: own-facts adapter (MLP)", f"[B, {K_s}]", f"[B, {D}]", "summary token: own diagnoses, procedures, medicines, stats"))
    rows += [
        ("+ system tag + position code", f"[B, {T}+1, {D}]", f"[B, {T}+1, {D}]", "which system, which time point"),
        (f"shared transformer body ({L} layers, {H} heads)", f"[B, {T}+1, {D}]", f"[B, {T}+1, {D}]", f"attention; feed-forward width {FF}"),
        ("average over tokens + LayerNorm", f"[B, {T}+1, {D}]", f"[B, {D}]", "THE EMBEDDING of one organ system"),
    ]
    if m.whole_patient is not None:
        n_in = m.whole_patient.net[0].in_features
        rows += [("whole-patient layer (MLP)", f"[B, {n_in}]", f"[B, {D}]", "every feature at once -> whole-patient summary g"),
                 ("whole-patient volume knobs", f"[B, {D}]", f"[B, {S}, {D}]", "x0.5 - x1.5 per system and channel")]
    if m.coupling is not None:
        rows.append(("coupling layer (attention across systems)", f"[B, {S}, {D}]", f"[B, {S}, {D}]", "each system listens to the others"))
    if m.has_surgical_context:
        rows.append(("surgical-context encoder (MLP)", f"[B, {len(m.surgical_context_idx)}]", f"[B, {D}]", "emergency, ASA, department, anaesthesia"))
    if m.arbitration is not None:
        rows.append(("arbitration layer (MLP + softmax)", f"[B, {m.arbitration.net[0].in_features}]", f"[B, {S}]", "how much say each system gets (average 1)"))
    rows.append((f"NAM shape functions (one per term, {len(m.term_names)} terms)", f"[B, {D}]", "[B, 1] each", "risk points per term"))
    rows.append(("sum of terms + bias", f"{len(m.term_names)} x [B]", "[B]", "logit -> sigmoid -> chance of dying"))
    return pd.DataFrame(rows, columns=["layer", "input size", "output size", "notes"])

LAYER_SIZE_TABLE = layer_size_table(model)
LAYER_SIZE_TABLE.to_csv(os.path.join(TABLE_DIR, "layer_sizes.csv"), index=False)
print(LAYER_SIZE_TABLE.to_string(index=False))

# Part 10 — Training: label-free pre-training, then gentle fine-tuning (v3)

**Phase 1 — pre-training, no labels, real training patients only.** Two tasks at once:
every organ system rebuilds its own data from its own summary (as before), and the
**masked-system task** hides one system per patient and rebuilds it from the other seven
through the coupling layer. That is how the inter-system layer learns real relationships
from every patient instead of from a few hundred deaths.

**Phase 2 — fine-tuning for 30-day mortality.** AdamW with weight decay; the pre-trained
encoder moves 10x more gently than the new parts (`FINETUNE_ENCODER_LR_MULT`); balanced
batches; organ-system dropout; best epoch chosen on validation AUPRC. Training AUPRC on a
fixed set of real training patients is tracked alongside validation — a widening gap is
the clearest sign of memorising.

Both phases checkpoint every few epochs and resume automatically if the cell is re-run.
`train_full_model()` (bottom) repeats the exact same recipe for the ensemble, ablations
and learning curve in Part 11.

**v4 — the model's diary.** While it trains, the notebook now also writes down:
- every epoch: validation AUROC/AUPRC for ALL patients, and separately for EMERGENCY and
  SCHEDULED patients (`EPOCH_LOG`), starting from *before any training*;
- at chosen epochs: a **snapshot** of every snapshot patient's embeddings (the 32-number
  notes per organ system), their risk points and their predicted risk (`SNAPSHOTS`, also
  saved to `OUTPUT_DIR/snapshots/`) — Part 13 turns these into the before/after maps;
- at the end: the **final model**, saved with everything needed to reload it, and a
  reload check proving the saved file gives identical predictions.

In [ ]:
def save_checkpoint(path, **state):
    torch.save(state, path)

def load_checkpoint_if_exists(path):
    if os.path.exists(path):
        try:
            return torch.load(path, map_location=DEVICE, weights_only=False)
        except Exception as e:
            print(f"Found a checkpoint at {path} but failed to load it ({e}) -- starting fresh.")
    return None

def _to_device(batch):
    return {k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}

import contextlib

def _autocast():
    # v4: bf16 mixed precision for TRAINING only (evaluation stays in full precision).
    return torch.autocast("cuda", dtype=torch.bfloat16) if USE_AMP else contextlib.nullcontext()

def run_pretrain_epoch(model, loader, optimizer):
    model.train()
    tot, own, ms, n = 0.0, 0.0, 0.0, 0
    for batch in loader:
        batch = _to_device(batch)
        optimizer.zero_grad()
        with _autocast():
            loss, parts = model.pretrain_loss(batch, CONFIG["MASKED_SYSTEM_WEIGHT"])
        loss.backward()
        optimizer.step()
        tot += float(loss); own += parts["own"]; ms += parts["masked_system"]; n += 1
    return tot / max(n, 1), own / max(n, 1), ms / max(n, 1)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for batch in loader:
        out = model(_to_device(batch))
        all_logits.append(out["logit"].cpu().numpy())
        all_labels.append(batch["label"].numpy())
    logits, labels = np.concatenate(all_logits), np.concatenate(all_labels)
    probs = 1 / (1 + np.exp(-logits))
    result = {"probs": probs, "labels": labels, "logits": logits}
    both = len(np.unique(labels)) > 1
    result["auroc"] = roc_auc_score(labels, probs) if both else float("nan")
    result["auprc"] = average_precision_score(labels, probs) if both else float("nan")
    return result

def run_finetune_epoch(model, loader, optimizer, pos_weight, use_focal, gamma):
    model.train()
    total, n = 0.0, 0
    for batch in loader:
        batch = _to_device(batch)
        optimizer.zero_grad()
        with _autocast():
            out = model(batch)
            loss = mortality_loss(out["logit"].float(), batch["label"], pos_weight, use_focal, gamma)
            loss = loss + CONFIG["ARBITRATION_REG"] * out["aux_loss"].float()
        loss.backward()
        optimizer.step()
        total += float(loss); n += 1
    return total / max(n, 1)

def make_pretrain_optimizer(model):
    return torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"])

def make_finetune_optimizer(model):
    enc_ids = {id(p) for p in model.encoder_parameters()}
    enc = [p for p in model.parameters() if id(p) in enc_ids]
    rest = [p for p in model.parameters() if id(p) not in enc_ids]
    base = CONFIG["LR"] * 0.5
    return torch.optim.AdamW([{"params": enc, "lr": base * CONFIG["FINETUNE_ENCODER_LR_MULT"]},
                              {"params": rest, "lr": base}], weight_decay=CONFIG["WEIGHT_DECAY"])

_rng_tm = np.random.default_rng(SEED)
_real_train = [s for s in EFFECTIVE_TRAIN_IDS]
TRAIN_METRIC_IDS = list(_rng_tm.choice(_real_train, size=min(CONFIG["TRAIN_METRIC_SUBSAMPLE"], len(_real_train)), replace=False))
train_metric_loader = DataLoader(INSPIREDataset(TRAIN_METRIC_IDS), batch_size=CONFIG["BATCH_SIZE"],
                                 shuffle=False, collate_fn=collate_bundle, **LOADER_KW)
print(f"Training-set AUPRC will be tracked on {len(TRAIN_METRIC_IDS)} real training patients "
      f"({int(sum(PATIENT_BUNDLE[s]['label'] for s in TRAIN_METRIC_IDS))} deaths).")

# ---- v4: the model's diary -- per-epoch metrics by emergency/scheduled, and snapshots ----
def _emop_of(ids):
    return np.array([float(COHORT_INDEXED.loc[s, "emop"]) if pd.notna(COHORT_INDEXED.loc[s, "emop"]) else 0.0 for s in ids])

VAL_EMOP = _emop_of(VAL_IDS)
EPOCH_LOG = []   # one dict per logged step: phase, epoch, val metrics (all / emergency / scheduled), losses

def _group_metrics(labels, probs, emop, prefix="val"):
    out = {}
    for name, sel in (("all", np.ones(len(labels), bool)), ("emergency", emop > 0.5), ("scheduled", emop <= 0.5)):
        y, p = labels[sel], probs[sel]
        ok = sel.any() and len(np.unique(y)) > 1
        out[f"{prefix}_auroc_{name}"] = float(roc_auc_score(y, p)) if ok else np.nan
        out[f"{prefix}_auprc_{name}"] = float(average_precision_score(y, p)) if ok else np.nan
        out[f"{prefix}_n_deaths_{name}"] = int(y.sum()) if sel.any() else 0
    return out

def log_epoch(phase, epoch, val_res=None, **extra):
    row = {"phase": phase, "epoch": epoch, **extra}
    if val_res is not None:
        row.update(_group_metrics(val_res["labels"], val_res["probs"], VAL_EMOP))
    EPOCH_LOG.append(row)
    return row

# Snapshot patients: every validation death + a random sample of validation survivors.
_rng_snap = np.random.default_rng(SEED + 7)
_val_dead = [s for s in VAL_IDS if PATIENT_BUNDLE[s]["label"] == 1]
_val_alive = [s for s in VAL_IDS if PATIENT_BUNDLE[s]["label"] == 0]
_n_alive = max(0, min(len(_val_alive), CONFIG["SNAPSHOT_N_PATIENTS"] - len(_val_dead)))
SNAPSHOT_IDS = _val_dead + list(_rng_snap.choice(_val_alive, size=_n_alive, replace=False)) if _n_alive else list(_val_dead)
snapshot_loader = DataLoader(INSPIREDataset(SNAPSHOT_IDS), batch_size=CONFIG["BATCH_SIZE"], shuffle=False,
                             collate_fn=collate_bundle, **LOADER_KW)
SNAPSHOTS = []   # list of dicts (tag, phase, epoch, arrays) -- also written to OUTPUT_DIR/snapshots/*.npz

@torch.no_grad()
def take_snapshot(m, tag, phase, epoch):
    # Records, for every snapshot patient: each organ system's embedding BEFORE and AFTER the
    # whole-patient + coupling layers, the whole-patient summary, the surgical-context summary,
    # every term's risk points, and the predicted risk. float16 on disk to keep files small.
    try:
        m.eval()
        own, final, g, sc, contrib, logit = [], [], [], [], [], []
        for batch in snapshot_loader:
            out = m(_to_device(batch))
            own.append(torch.stack([out["embeddings_own"][s] for s in m.organ_systems], 1).float().cpu().numpy())
            final.append(torch.stack([out["embeddings"][s] for s in m.organ_systems], 1).float().cpu().numpy())
            g.append(out["static_embedding"].float().cpu().numpy())
            if out.get("surgical_context_embedding") is not None:
                sc.append(out["surgical_context_embedding"].float().cpu().numpy())
            if out["per_system_signal"] is not None:
                contrib.append(torch.stack([out["per_system_signal"][t] for t in m.term_names], 1).float().cpu().numpy())
            logit.append(out["logit"].float().cpu().numpy())
        cat = lambda xs: np.concatenate(xs) if xs else None
        snap = {"tag": tag, "phase": phase, "epoch": epoch, "systems_own": cat(own), "systems": cat(final),
                "whole_patient": cat(g), "surgical_context": cat(sc), "contrib": cat(contrib), "logit": cat(logit),
                "term_names": list(m.term_names), "organ_systems": list(m.organ_systems)}
        SNAPSHOTS.append(snap)
        np.savez_compressed(os.path.join(SNAPSHOT_DIR, f"snapshot_{len(SNAPSHOTS):02d}_{tag}.npz"),
                            **{k: (v.astype(np.float16) if isinstance(v, np.ndarray) else np.array(v))
                               for k, v in snap.items() if v is not None},
                            subject_id=np.array(SNAPSHOT_IDS))
        return snap
    except Exception as e:   # a diary entry must never stop training -- but it is reported loudly
        print(f"!! SNAPSHOT '{tag}' FAILED ({type(e).__name__}: {e}) -- training continues; Part 13 will have one fewer picture.")
        return None

print(f"Snapshots will follow {len(SNAPSHOT_IDS)} validation patients ({len(_val_dead)} deaths, "
      f"{int(_emop_of(_val_dead).sum()) if _val_dead else 0} of them emergency).")

PRETRAIN_CKPT_PATH = os.path.join(CONFIG["CHECKPOINT_DIR"], f"pretrain_checkpoint_{ARCHITECTURE_FINGERPRINT}.pt")
FINETUNE_CKPT_PATH = os.path.join(CONFIG["CHECKPOINT_DIR"], f"finetune_checkpoint_{ARCHITECTURE_FINGERPRINT}.pt")

In [ ]:
pretrain_optimizer = make_pretrain_optimizer(model)
pretrain_history = {"total": [], "own": [], "masked_system": []}
start_epoch_pretrain = 0
_ckpt = load_checkpoint_if_exists(PRETRAIN_CKPT_PATH)
if _ckpt is not None and _ckpt.get("phase") == "pretrain":
    try:
        model.load_state_dict(_ckpt["model_state"]); pretrain_optimizer.load_state_dict(_ckpt["optimizer_state"])
        pretrain_history = _ckpt["history"]; start_epoch_pretrain = _ckpt["epoch"] + 1
        print(f"Resuming pre-training at epoch {start_epoch_pretrain}")
    except (RuntimeError, KeyError, ValueError) as e:
        print(f"Checkpoint does not match this model ({e}) -- starting fresh.")

if start_epoch_pretrain == 0:
    # v4: "before any training" -- random dials. Everything later is compared with this.
    _r0 = evaluate(model, val_loader)
    log_epoch("before training", -1, _r0)
    take_snapshot(model, "before_training", "before training", -1)
    print(f"Before any training: val AUPRC {_r0['auprc']:.3f} (a random model scores about the death rate, "
          f"{_r0['labels'].mean():.3f})")

print("Phase 1: label-free pre-training (own-data reconstruction + masked-system prediction)")
_best, _stale = float("inf"), 0
for epoch in range(start_epoch_pretrain, CONFIG["EPOCHS_PRETRAIN"]):
    tot, own, ms = run_pretrain_epoch(model, pretrain_loader, pretrain_optimizer)
    for k, v in zip(("total", "own", "masked_system"), (tot, own, ms)):
        pretrain_history[k].append(v)
    log_epoch("pre-training", epoch, None, pretrain_own=own, pretrain_masked_system=ms)
    if epoch % max(CONFIG["EPOCHS_PRETRAIN"] // 6, 1) == 0 or epoch == CONFIG["EPOCHS_PRETRAIN"] - 1:
        print(f"  epoch {epoch:3d}  own-data={own:.4f}  masked-system={ms:.4f}")
    if (epoch + 1) % CONFIG["CHECKPOINT_EVERY_N_EPOCHS"] == 0 or epoch == CONFIG["EPOCHS_PRETRAIN"] - 1:
        save_checkpoint(PRETRAIN_CKPT_PATH, phase="pretrain", epoch=epoch, model_state=model.state_dict(),
                        optimizer_state=pretrain_optimizer.state_dict(), history=pretrain_history)
    if tot < _best - 1e-5:
        _best, _stale = tot, 0
    else:
        _stale += 1
    if _stale >= CONFIG["EARLY_STOPPING_PATIENCE"]:
        print(f"  early stopping at epoch {epoch}")
        break

_r_pt = evaluate(model, val_loader)
log_epoch("after pre-training", -1, _r_pt)
take_snapshot(model, "after_pretraining", "after pre-training", -1)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(pretrain_history["own"], color="#0a7d6e", label="own-data reconstruction")
ax.plot(pretrain_history["masked_system"], color="#8e44ad", label="masked-system prediction")
ax.set_title("Phase 1: label-free pre-training"); ax.set_xlabel("epoch"); ax.set_ylabel("masked MSE"); ax.legend()
plt.tight_layout(); plt.show()
print("The masked-system curve falling means the coupling layer is learning to predict a hidden organ "
      "system from the others -- i.e. it is finding real inter-system relationships before it ever sees a label.")

In [ ]:
finetune_optimizer = make_finetune_optimizer(model)
history = {"train_loss": [], "val_auroc": [], "val_auprc": [], "train_auprc": []}
_SEL = CONFIG["MODEL_SELECTION_METRIC"]
best_val_auroc, best_state, start_epoch_finetune = -1.0, None, 0   # name kept for checkpoint compatibility
_ckpt = load_checkpoint_if_exists(FINETUNE_CKPT_PATH)
if _ckpt is not None and _ckpt.get("phase") == "finetune":
    try:
        model.load_state_dict(_ckpt["model_state"]); finetune_optimizer.load_state_dict(_ckpt["optimizer_state"])
        history, best_val_auroc, best_state = _ckpt["history"], _ckpt["best_val_auroc"], _ckpt["best_state"]
        start_epoch_finetune = _ckpt["epoch"] + 1
        print(f"Resuming fine-tuning at epoch {start_epoch_finetune} (best val {_SEL.upper()} so far {best_val_auroc:.3f})")
    except (RuntimeError, KeyError, ValueError) as e:
        print(f"Checkpoint does not match this model ({e}) -- starting fresh.")

print("\nPhase 2: fine-tuning on 30-day mortality")
_stale = 0
for epoch in range(start_epoch_finetune, CONFIG["EPOCHS_FINETUNE"]):
    train_dataset.epoch_seed = 1000 + epoch
    train_loss = run_finetune_epoch(model, train_loader, finetune_optimizer, POS_WEIGHT, CONFIG["USE_FOCAL_LOSS"], CONFIG["FOCAL_GAMMA"])
    val_result = evaluate(model, val_loader)
    tr_result = evaluate(model, train_metric_loader)
    history["train_loss"].append(train_loss); history["val_auroc"].append(val_result["auroc"])
    history["val_auprc"].append(val_result["auprc"]); history["train_auprc"].append(tr_result["auprc"])
    log_epoch("fine-tuning", epoch, val_result, train_loss=train_loss, train_auprc_all=tr_result["auprc"])
    if epoch in CONFIG["SNAPSHOT_FINETUNE_EPOCHS"]:
        take_snapshot(model, f"finetune_epoch_{epoch:03d}", "fine-tuning", epoch)
    if not math.isnan(val_result[_SEL]) and val_result[_SEL] > best_val_auroc:
        best_val_auroc, best_state, _stale = val_result[_SEL], {k: v.clone() for k, v in model.state_dict().items()}, 0
    else:
        _stale += 1
    if epoch % max(CONFIG["EPOCHS_FINETUNE"] // 8, 1) == 0 or epoch == CONFIG["EPOCHS_FINETUNE"] - 1:
        print(f"  epoch {epoch:3d}  loss={train_loss:.4f}  train_auprc={tr_result['auprc']:.3f}  "
              f"val_auprc={val_result['auprc']:.3f}  val_auroc={val_result['auroc']:.3f}")
    if (epoch + 1) % CONFIG["CHECKPOINT_EVERY_N_EPOCHS"] == 0 or epoch == CONFIG["EPOCHS_FINETUNE"] - 1:
        save_checkpoint(FINETUNE_CKPT_PATH, phase="finetune", epoch=epoch, model_state=model.state_dict(),
                        optimizer_state=finetune_optimizer.state_dict(), history=history,
                        best_val_auroc=best_val_auroc, best_state=best_state)
    if _stale >= CONFIG["EARLY_STOPPING_PATIENCE"]:
        print(f"  early stopping at epoch {epoch} -- val {_SEL.upper()} hasn't improved in {CONFIG['EARLY_STOPPING_PATIENCE']} epochs")
        break

LAST_EPOCH = len(history["val_auprc"]) - 1
if LAST_EPOCH >= 0 and LAST_EPOCH not in CONFIG["SNAPSHOT_FINETUNE_EPOCHS"]:
    take_snapshot(model, f"finetune_epoch_{LAST_EPOCH:03d}_last", "fine-tuning", LAST_EPOCH)

if best_state is not None:
    model.load_state_dict(best_state)
    BEST_EPOCH = int(np.nanargmax(history[f"val_{_SEL}"]))
    print(f"\nRestored best epoch {BEST_EPOCH} (val {_SEL.upper()}={best_val_auroc:.3f}).")
else:
    BEST_EPOCH = len(history["val_auprc"]) - 1
    print("\nNo validation improvement recorded -- using final-epoch weights.")

take_snapshot(model, "chosen_model", "chosen model", BEST_EPOCH)
EPOCH_LOG_DF = pd.DataFrame(EPOCH_LOG)
EPOCH_LOG_DF.to_csv(os.path.join(TABLE_DIR, "epoch_log.csv"), index=False)
print(f"Diary: {len(EPOCH_LOG_DF)} logged steps, {len(SNAPSHOTS)} embedding snapshots "
      f"({', '.join(sn['tag'] for sn in SNAPSHOTS)})")

record_checkpoint("Parts 9-10 -- model build + two-phase training")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], color="#c0392b")
axes[0].set_title("Phase 2: fine-tuning loss"); axes[0].set_xlabel("epoch")
axes[1].plot(history["train_auprc"], label="training AUPRC (real patients)", color="#e67e22")
axes[1].plot(history["val_auprc"], label="validation AUPRC", color="#8e44ad")
axes[1].axvline(BEST_EPOCH, color="gray", linestyle=":", label="chosen epoch")
axes[1].set_title("Learning or memorising? training vs. validation"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()
TRAIN_VAL_GAP = float(history["train_auprc"][BEST_EPOCH] - history["val_auprc"][BEST_EPOCH])
print(f"At the chosen epoch: training AUPRC {history['train_auprc'][BEST_EPOCH]:.3f} vs validation "
      f"{history['val_auprc'][BEST_EPOCH]:.3f} (gap {TRAIN_VAL_GAP:+.3f}). A gap that keeps widening after the "
      f"chosen epoch is memorisation; early stopping is what protects the chosen model from it.")
_n_val_pos = int(sum(PATIENT_BUNDLE[s]["label"] for s in VAL_IDS))
print(f"Validation set: {len(VAL_IDS)} patients, {_n_val_pos} deaths -- read the trend, not single epochs.")

## 10.3 Saving the final model, and proving the saved file works (v4)

The chosen model is saved with everything needed to use it again without re-running the
notebook: the weights, the CONFIG, the feature names and which organ system owns each, the
training-set scaling numbers (so a new patient is put on the same ruler), and the training
diary. Then it is loaded back into a brand-new model and must give **identical**
predictions on the validation patients — otherwise the file is not trustworthy.

In [ ]:
def _model_payload(m):
    return {
        "model_state": {k: v.detach().cpu() for k, v in m.state_dict().items()},
        "architecture_fingerprint": ARCHITECTURE_FINGERPRINT, "version": "v4",
        "config": dict(CONFIG), "run_profile": RUN_PROFILE,
        "static_feature_names": list(STATIC_FEATURE_NAMES),
        "ts_feature_names": {s: PATIENT_BUNDLE[TRAIN_IDS[0]]["ts"][s]["feature_names"] for s in TIME_SERIES_SYSTEMS},
        "ts_feature_counts": dict(TS_FEATURE_COUNTS), "system_static_idx": dict(SYSTEM_STATIC_IDX),
        "context_idx": list(CONTEXT_IDX), "surgical_context_idx": list(SURGICAL_CONTEXT_IDX),
        "static_scaler_mean": static_scaler.mean_, "static_scaler_scale": static_scaler.scale_,
        "ts_means": {f"{k[0]}|{k[1]}": float(v["mean"]) for k, v in STATS.items()},
        "ts_stds": {f"{k[0]}|{k[1]}": float(v) for k, v in TRAIN_STD.items()},
        "best_epoch": BEST_EPOCH, "history": history, "epoch_log": EPOCH_LOG,
        "torch_version": torch.__version__,
    }

def save_final_model(m, name="final_model"):
    path = os.path.join(MODEL_DIR, f"{name}_{ARCHITECTURE_FINGERPRINT}.pt")
    torch.save(_model_payload(m), path)
    return path

def load_inspire_model(path, **build_kwargs):
    # Rebuilds the model with THIS notebook's data layout and loads the saved weights.
    payload = torch.load(path, map_location=DEVICE, weights_only=False)
    assert payload["static_feature_names"] == list(STATIC_FEATURE_NAMES), "static features differ from this run"
    m = build_model(**build_kwargs)
    m.load_state_dict(payload["model_state"])
    m.eval()
    return m, payload

FINAL_MODEL_PATH, MODEL_RELOAD_OK = None, None
if CONFIG["SAVE_FINAL_MODEL"]:
    FINAL_MODEL_PATH = save_final_model(model)
    _reloaded, _ = load_inspire_model(FINAL_MODEL_PATH)
    _a = evaluate(model, val_loader)["logits"]; _b = evaluate(_reloaded, val_loader)["logits"]
    MODEL_RELOAD_OK = bool(np.allclose(_a, _b, atol=1e-5))
    print(f"Saved the chosen model to {FINAL_MODEL_PATH} ({os.path.getsize(FINAL_MODEL_PATH) / 1e6:.1f} MB)")
    print(f"Reload check -- the saved file gives identical validation predictions: {MODEL_RELOAD_OK}")
    del _reloaded
else:
    print("CONFIG['SAVE_FINAL_MODEL']=False -- model not saved.")

In [ ]:
def train_full_model(seed=SEED, train_ds=None, verbose=False, **model_kwargs):
    # The exact recipe above (pre-train -> gentle fine-tune -> best val epoch), without
    # checkpoint files. Used by the ensemble, the ablations and the learning curve.
    m = build_model(seed=seed, **model_kwargs)
    torch.manual_seed(seed)
    ds = train_ds or train_dataset
    real_ids = ds.subject_ids
    pl = DataLoader(INSPIREDataset(real_ids), batch_size=min(CONFIG["BATCH_SIZE"], len(real_ids)), shuffle=True,
                    collate_fn=collate_bundle, generator=torch.Generator().manual_seed(seed), **LOADER_KW)
    opt = make_pretrain_optimizer(m); best, stale = float("inf"), 0
    for ep in range(CONFIG["EPOCHS_PRETRAIN"]):
        tot, _, _ = run_pretrain_epoch(m, pl, opt)
        best, stale = (tot, 0) if tot < best - 1e-5 else (best, stale + 1)
        if stale >= CONFIG["EARLY_STOPPING_PATIENCE"]:
            break
    tl = make_train_loader(ds, seed=seed)
    opt = make_finetune_optimizer(m); best, best_state, stale = -1.0, None, 0
    for ep in range(CONFIG["EPOCHS_FINETUNE"]):
        ds.epoch_seed = 1000 + ep
        run_finetune_epoch(m, tl, opt, POS_WEIGHT, CONFIG["USE_FOCAL_LOSS"], CONFIG["FOCAL_GAMMA"])
        v = evaluate(m, val_loader)[_SEL]
        if not math.isnan(v) and v > best:
            best, best_state, stale = v, {k: t.clone() for k, t in m.state_dict().items()}, 0
        else:
            stale += 1
        if stale >= CONFIG["EARLY_STOPPING_PATIENCE"]:
            break
    if best_state is not None:
        m.load_state_dict(best_state)
    if verbose:
        print(f"  seed {seed}: best val {_SEL.upper()} {best:.3f}")
    return m, best

# Part 11 — Evaluation, calibration, and interpretability

- §11.1 Held-out test metrics (AUROC, AUPRC, Brier score, confusion matrix)
- §11.2 Per-system NAM contribution breakdown — the actual "Renal: CRITICAL, Cardio: OK"
  decomposition promised in §1.1 and §1.5.2
- §11.3 Attention audit — do heads attend to sensible time steps?
- §11.4 Ablation: NAM vs. concat fusion — what does the additive structure cost/buy?
- §11.5 Sensitivity: pre-op-only vs. peri-operative (§1.6.3, Path B)
- §11.6 Cardiac-recovery-exception qualitative check (§1.6.4)
- §11.7 Multi-operation sensitivity: last-op vs. first-op label (§1.6.4, Path C)

All caveats about small-N noise from Parts 8–10 apply throughout — read directions and
relative comparisons, not exact numbers, until this runs on the full cohort.

## 11.1 Held-out test metrics

**AUPRC (PR-AUC) is the metric to trust first, not AUROC** — per Part C step 9 of the
imbalance/imputation reference. At the true full-cohort prevalence (~0.47%, 469/99,886),
AUROC can look deceptively good while precision stays poor, because AUROC's false-positive
rate is measured against an enormous majority class — the same failure mode both external
reviews independently flagged. AUROC is still reported below for reference and comparison
to published benchmarks (e.g. Shickel et al.'s 0.92), but AUPRC is the number to actually
judge this model by.

In [ ]:
def bootstrap_ci(labels, probs, metric, n=None, seed=SEED):
    # 95% percentile interval, resampling test patients with replacement (v3). With ~94
    # test deaths the interval is wide -- that width IS the honest uncertainty.
    n = n or CONFIG["N_BOOTSTRAP"]
    rng = np.random.default_rng(seed); vals = []
    labels, probs = np.asarray(labels), np.asarray(probs)
    for _ in range(n):
        idx = rng.integers(0, len(labels), len(labels))
        if len(np.unique(labels[idx])) > 1:
            vals.append(metric(labels[idx], probs[idx]))
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))) if vals else (np.nan, np.nan)

test_result = evaluate(model, test_loader)
TEST_CI = {}
if len(np.unique(test_result["labels"])) > 1:
    TEST_CI = {"auprc": bootstrap_ci(test_result["labels"], test_result["probs"], average_precision_score),
               "auroc": bootstrap_ci(test_result["labels"], test_result["probs"], roc_auc_score)}
print(f"TEST  AUPRC={test_result['auprc']:.3f} (95% CI {TEST_CI.get('auprc', (np.nan,)*2)[0]:.3f}-{TEST_CI.get('auprc', (np.nan,)*2)[1]:.3f})  "
      f"AUROC={test_result['auroc']:.3f} (95% CI {TEST_CI.get('auroc', (np.nan,)*2)[0]:.3f}-{TEST_CI.get('auroc', (np.nan,)*2)[1]:.3f})  "
      f"(n={len(test_result['labels'])}, deaths={int(test_result['labels'].sum())}; chance AUPRC = {test_result['labels'].mean():.3f})")

if len(np.unique(test_result["labels"])) > 1:
    brier = brier_score_loss(test_result["labels"], test_result["probs"])
    print(f"Brier score: {brier:.3f}  (0 = perfect calibration, 0.25 = a coin flip's worth of miscalibration)")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    fpr, tpr, _ = roc_curve(test_result["labels"], test_result["probs"])
    axes[0].plot(fpr, tpr, color="#2980b9"); axes[0].plot([0, 1], [0, 1], "--", color="gray")
    axes[0].set_title(f"ROC (AUROC={test_result['auroc']:.3f})"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")

    prec, rec, _ = precision_recall_curve(test_result["labels"], test_result["probs"])
    axes[1].plot(rec, prec, color="#c0392b"); axes[1].set_title(f"PR (AUPRC={test_result['auprc']:.3f})")
    axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision")

    # Calibration curve (§1 Part D: "calibration, not just discrimination")
    bins = np.linspace(0, 1, 6)
    bin_ids = np.digitize(test_result["probs"], bins) - 1
    bin_ids = np.clip(bin_ids, 0, len(bins) - 2)
    obs_rate, pred_rate = [], []
    for b in range(len(bins) - 1):
        mask = bin_ids == b
        if mask.sum() > 0:
            obs_rate.append(test_result["labels"][mask].mean())
            pred_rate.append(test_result["probs"][mask].mean())
    axes[2].plot(pred_rate, obs_rate, "o-", color="#8e44ad", label="model")
    axes[2].plot([0, 1], [0, 1], "--", color="gray", label="perfect calibration")
    axes[2].set_title("Calibration"); axes[2].set_xlabel("predicted probability"); axes[2].set_ylabel("observed rate"); axes[2].legend()
    plt.tight_layout(); plt.show()
else:
    print("Only one class present in the test split at this sample size -- AUROC/calibration "
          "undefined this run. Re-run with a different SEED or, better, the full cohort.")

cm_default = confusion_matrix(test_result["labels"], (test_result["probs"] >= 0.5).astype(int))
print("\nConfusion matrix @ threshold 0.5 (rows=true, cols=pred) [[TN, FP], [FN, TP]]:")
print(cm_default)
print("0.5 is an arbitrary default threshold -- see §11.1b below for a threshold actually "
      "chosen from the PR curve, per Part C step 9.")

## 11.1b Choosing an operating threshold — on validation, applied to test (v2)

0.5 is meaningless as a cutoff at this prevalence. The threshold that maximises F1 is a
reasonable default, **but it has to be chosen on the validation set and then applied to
the test set.** v1 chose it on the test set itself, which makes the reported
"76% of deaths caught at 60% precision" slightly optimistic (the threshold was fitted to
the very patients it was scored on). The test-optimal threshold is still printed, for
reference only, so the size of that optimism is visible.

In [ ]:
def best_f1_threshold(labels, probs):
    prec, rec, thr = precision_recall_curve(labels, probs)
    f1 = np.where((prec + rec) > 0, 2 * prec * rec / (prec + rec + 1e-12), 0.0)
    i = int(np.argmax(f1[:-1])) if len(f1) > 1 else 0
    return (float(thr[i]) if len(thr) else 0.5), prec, rec, thr, i

def metrics_at(labels, probs, threshold):
    pred = (probs >= threshold).astype(int)
    tp = int(((pred == 1) & (labels == 1)).sum()); fp = int(((pred == 1) & (labels == 0)).sum())
    fn = int(((pred == 0) & (labels == 1)).sum())
    return {"threshold": threshold, "recall": tp / max(tp + fn, 1), "precision": tp / max(tp + fp, 1),
            "deaths_caught": f"{tp}/{tp + fn}", "false_alarms": fp}

val_result = evaluate(model, val_loader)
if len(np.unique(test_result["labels"])) > 1 and len(np.unique(val_result["labels"])) > 1:
    best_threshold, *_ = best_f1_threshold(val_result["labels"], val_result["probs"])
    test_opt_threshold, prec_arr, rec_arr, thresh_arr, best_idx = best_f1_threshold(test_result["labels"], test_result["probs"])
    threshold_table = pd.DataFrame([
        {"chosen on": "validation (honest)", **metrics_at(test_result["labels"], test_result["probs"], best_threshold)},
        {"chosen on": "test itself (v1 method, optimistic)", **metrics_at(test_result["labels"], test_result["probs"], test_opt_threshold)},
    ]).set_index("chosen on")
    print("Test-set performance at the best-F1 threshold:")
    print(threshold_table)
    cm_tuned = confusion_matrix(test_result["labels"], (test_result["probs"] >= best_threshold).astype(int))
    print(f"\nConfusion matrix on TEST @ validation-chosen threshold {best_threshold:.3f} [[TN, FP], [FN, TP]]:")
    print(cm_tuned)
    print("\nNearby thresholds on the test PR curve, for choosing a clinical trade-off by hand "
          "(then fix it on validation data before quoting it):")
    for i in range(max(0, best_idx - 3), min(len(thresh_arr), best_idx + 4)):
        print(f"  threshold={thresh_arr[i]:.3f}  precision={prec_arr[i]:.2f}  recall={rec_arr[i]:.2f}")
else:
    best_threshold = 0.5
    print("Only one class present in the validation or test split at this sample size -- skipping threshold tuning.")

## 11.1c Recalibration on the validation set (v2)

SMOTENC, sequence augmentation and `pos_weight` all push the model's raw probabilities
upward on purpose (they make deaths "louder" during training), which is why the raw
Brier score is poor even when ranking (AUROC/AUPRC) is good. Platt scaling — a one-feature
logistic regression on the model's logit, fitted on the **validation** set — maps the
scores back to real-world probabilities. It is monotonic, so AUROC and AUPRC are
unchanged; only calibration (Brier, the calibration curve) moves.

In [ ]:
from sklearn.linear_model import LogisticRegression

def _logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))

CALIBRATOR = None
if CONFIG["RECALIBRATE_ON_VAL"] and len(np.unique(val_result["labels"])) > 1 and len(np.unique(test_result["labels"])) > 1:
    CALIBRATOR = LogisticRegression(C=1e6).fit(_logit(val_result["probs"]).reshape(-1, 1), val_result["labels"])
    test_probs_calibrated = CALIBRATOR.predict_proba(_logit(test_result["probs"]).reshape(-1, 1))[:, 1]
    _b_raw = brier_score_loss(test_result["labels"], test_result["probs"])
    _b_cal = brier_score_loss(test_result["labels"], test_probs_calibrated)
    print(f"Test Brier score: raw {_b_raw:.4f} -> recalibrated {_b_cal:.4f}  "
          f"(test prevalence {test_result['labels'].mean():.2%}; mean predicted risk raw "
          f"{test_result['probs'].mean():.2%} -> recalibrated {test_probs_calibrated.mean():.2%})")
    fig, ax = plt.subplots(figsize=(5, 4))
    for probs, name, color in [(test_result["probs"], "raw", "#8e44ad"), (test_probs_calibrated, "recalibrated", "#27ae60")]:
        q = np.unique(np.quantile(probs, np.linspace(0, 1, 11)))
        ids = np.clip(np.digitize(probs, q[1:-1]), 0, len(q) - 2)
        pts = [(probs[ids == b].mean(), test_result["labels"][ids == b].mean()) for b in range(len(q) - 1) if (ids == b).any()]
        ax.plot(*zip(*pts), "o-", color=color, label=name)
    ax.plot([0, 1], [0, 1], "--", color="gray"); ax.set_xscale("symlog", linthresh=0.01); ax.set_yscale("symlog", linthresh=0.01)
    ax.set_xlabel("predicted risk (decile bins)"); ax.set_ylabel("observed death rate"); ax.legend(); ax.set_title("Calibration, test set")
    plt.tight_layout(); plt.show()
else:
    print("Recalibration skipped (flag off, or a split has only one class).")

## 11.1d A simple baseline on the same data (v2)

Listed as an open limitation in the surgeon-facing summary: the DNN had not been
compared with a simple model. This fits a class-weighted logistic regression on the same
standardised static vector plus each time-series feature's mean and "ever observed"
flag — same training patients (real only), same test patients. If the DNN does not beat
this clearly, the extra complexity has not earned its place yet; if it does, that is the
number to quote alongside it.

In [ ]:
def _flat_features(ids):
    rows = []
    for sid in ids:
        parts = [IMPUTED_BUNDLE[sid]["static"].values.astype(float)]
        for system in TIME_SERIES_SYSTEMS:
            x = IMPUTED_BUNDLE[sid][system]
            if x.shape[1]:
                parts += [x.mean(axis=0), PATIENT_BUNDLE[sid]["ts"][system]["mask"].max(axis=0)]
        rows.append(np.concatenate(parts))
    return np.stack(rows)

BASELINE_RESULT, GBM_RESULT = None, None
if CONFIG["RUN_SIMPLE_BASELINE"] and len(np.unique(test_result["labels"])) > 1:
    _train_real = [s for s in TRAIN_IDS]
    _ytr = np.array([PATIENT_BUNDLE[s]["label"] for s in _train_real])
    _lr = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000).fit(_flat_features(_train_real), _ytr)
    _p = _lr.predict_proba(_flat_features(TEST_IDS))[:, 1]
    _yte = np.array([PATIENT_BUNDLE[s]["label"] for s in TEST_IDS])
    BASELINE_RESULT = {"auprc": average_precision_score(_yte, _p), "auroc": roc_auc_score(_yte, _p)}
    _rows = {"logistic regression": BASELINE_RESULT}
    GBM_RESULT = None
    if CONFIG["RUN_GBM_BASELINE"]:
        # v3: gradient boosting -- often the strongest model on tabular clinical data, so it is
        # the bar the DNN has to meet (matching it WITH interpretability is itself a result).
        from sklearn.ensemble import HistGradientBoostingClassifier
        _gbm = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, class_weight="balanced",
                                              early_stopping=True, random_state=SEED)
        _gbm.fit(_flat_features(_train_real), _ytr)
        _pg = _gbm.predict_proba(_flat_features(TEST_IDS))[:, 1]
        GBM_RESULT = {"auprc": average_precision_score(_yte, _pg), "auroc": roc_auc_score(_yte, _pg)}
        _rows["gradient boosting"] = GBM_RESULT
    _rows["organ-system DNN (this run)"] = {"auprc": test_result["auprc"], "auroc": test_result["auroc"]}
    _rows["chance"] = {"auprc": float(_yte.mean()), "auroc": 0.5}
    comparison = pd.DataFrame(_rows).T
    comparison["auprc_95ci"] = [f"{a:.3f}-{b:.3f}" for a, b in [bootstrap_ci(_yte, p, average_precision_score, n=300)
                                for p in ([_p] + ([_pg] if GBM_RESULT else []) + [test_result["probs"]])]] + ["-"]
    print(comparison.round(3))
    print(f"\nTest set: {len(_yte)} patients, {int(_yte.sum())} deaths. With this many deaths a gap "
          f"of a few AUPRC points can be noise -- repeat over seeds (Part 12) before concluding.")
else:
    print("Baseline skipped (flag off, or the test split has only one class).")

## 11.1e Ensemble of models trained from different random starts (v3)

Trains `N_ENSEMBLE - 1` more copies with the exact same recipe and data, then averages
their predicted risks. Averaging several models reduces the variance of any single model
(less memorisation of one random start), usually improves calibration, and gives each
patient a spread that shows how much the models disagree. The same members are reused in
11.4b to test whether the learned inter-system links replicate. This is the number to
quote for the final model; the single model above is kept for the interpretability plots.

In [ ]:
ENSEMBLE_MODELS, ENSEMBLE_RESULT = [model], None
if CONFIG["N_ENSEMBLE"] > 1:
    print(f"Training {CONFIG['N_ENSEMBLE'] - 1} more ensemble members (same recipe, different seeds)...")
    for k in range(1, CONFIG["N_ENSEMBLE"]):
        _t0 = time.time()
        m_k, _ = train_full_model(seed=SEED + 101 * k, verbose=True)
        ENSEMBLE_MODELS.append(m_k)
        print(f"    ({(time.time() - _t0) / 60:.1f} min)")
    _member_test = [evaluate(m, test_loader) for m in ENSEMBLE_MODELS]
    _ens_probs = np.mean([r["probs"] for r in _member_test], axis=0)
    _yt = test_result["labels"]
    if len(np.unique(_yt)) > 1:
        ENSEMBLE_RESULT = {"auprc": average_precision_score(_yt, _ens_probs), "auroc": roc_auc_score(_yt, _ens_probs),
                           "auprc_ci": bootstrap_ci(_yt, _ens_probs, average_precision_score),
                           "auroc_ci": bootstrap_ci(_yt, _ens_probs, roc_auc_score), "probs": _ens_probs}
        members = pd.DataFrame([{"member": i, "test_auprc": r["auprc"], "test_auroc": r["auroc"]} for i, r in enumerate(_member_test)])
        print(members.round(3).to_string(index=False))
        print(f"\nSpread across members: AUPRC {members.test_auprc.min():.3f}-{members.test_auprc.max():.3f} "
              f"(this spread is seed noise -- differences smaller than it are not real)")
        print(f"ENSEMBLE of {len(ENSEMBLE_MODELS)}: AUPRC {ENSEMBLE_RESULT['auprc']:.3f} "
              f"(95% CI {ENSEMBLE_RESULT['auprc_ci'][0]:.3f}-{ENSEMBLE_RESULT['auprc_ci'][1]:.3f}), "
              f"AUROC {ENSEMBLE_RESULT['auroc']:.3f} (95% CI {ENSEMBLE_RESULT['auroc_ci'][0]:.3f}-{ENSEMBLE_RESULT['auroc_ci'][1]:.3f})")
        _sd = np.std([r["probs"] for r in _member_test], axis=0)
        print(f"Per-patient disagreement between members: median SD of predicted risk {np.median(_sd):.3f}, "
              f"90th percentile {np.percentile(_sd, 90):.3f}")
else:
    print("N_ENSEMBLE=1 -- single model only.")

## 11.2 Per-system contribution breakdown — the interpretability payoff

If `FUSION_STRATEGY == 'nam'`, `per_system_signal` from the model's forward pass is the
literal additive decomposition promised in §1.1/§1.5.2 — each system's contribution to the
final logit, individually plottable per patient. This is the concrete version of "Renal:
CRITICAL | Cardio: OK" from the architecture diagram in your meeting notes.

In [ ]:
@torch.no_grad()
def per_system_breakdown(model, loader, n_patients=6):
    model.eval()
    rows = []
    count = 0
    for batch in loader:
        batch_dev = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}
        out = model(batch_dev)
        if out["per_system_signal"] is None:
            print("FUSION_STRATEGY is not 'nam' -- no per-system decomposition available. "
                  "Set CONFIG['FUSION_STRATEGY']='nam' and re-run Part 9-10 to see this.")
            return None
        for i, sid in enumerate(batch["subject_id"]):
            row = {"subject_id": sid, "true_label": float(batch["label"][i]),
                   "predicted_prob": float(torch.sigmoid(out["logit"][i]))}
            for system, contrib in out["per_system_signal"].items():
                row[system] = float(contrib[i])
            rows.append(row)
            count += 1
            if count >= n_patients:
                break
        if count >= n_patients:
            break
    return pd.DataFrame(rows)

breakdown_df = per_system_breakdown(model, test_loader, n_patients=min(8, len(test_dataset)))
if breakdown_df is not None:
    display_cols = [c for c in breakdown_df.columns if c not in ("subject_id", "true_label", "predicted_prob")]
    fig, ax = plt.subplots(figsize=(10, max(3, 0.5 * len(breakdown_df))))
    breakdown_df.set_index("subject_id")[display_cols].plot(kind="barh", stacked=True, ax=ax, colormap="tab10")
    ax.set_title("Per-system additive contribution to the mortality logit (NAM fusion, §9.7)")
    ax.set_xlabel("contribution to logit (positive = pushes toward 'died')")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout(); plt.show()
    breakdown_df

## 11.2b What the coupling layer learned — for surgeon validation (v2)

No inter-system relationship was defined by hand; the layer learned them. This section
turns what it learned into three things a surgeon can check against physiology:

1. **"Who listens to whom" heatmap** — row = the system whose reading is being updated,
   column = the system it takes information from. Each row sums to 100%.
2. **Network diagram** — the strongest learned links as arrows, thickness = strength,
   shown separately for patients who died and who survived.
3. **How much it mattered** — for each system, how far its answer moved (in risk-logit
   units) because of what it heard from the others. A strong-looking link that changes
   nothing is not worth a surgeon's time; this separates the two.

Plus a **validation sheet** (CSV) listing each link in plain words with a bootstrap
confidence interval and blank columns for the surgeon's verdict.

How to read it honestly: attention shows **what the model uses**, not what causes what in
the body. A surgeon's "yes, that's physiological" is evidence the model learned something
sensible; a "no" is a flag to investigate (data artefact, confounding by department).
Links only matter if the gate (how much each system actually uses its message) is
non-trivial — both are shown.

In [ ]:
PLAIN_NAMES = {
    "renal": "Kidneys", "cardiovascular": "Heart & circulation", "respiratory": "Lungs & breathing",
    "metabolic_hepatic": "Metabolism & liver", "haematology": "Blood", "neurological": "Brain & nerves",
    "gi": "Digestive", "msk": "Bones & joints", "static": "Patient context", "whole_patient": "Whole patient",
    "surgical_context": "Surgical context",
}

@torch.no_grad()
def collect_system_outputs(model, ids):
    ds = INSPIREDataset(ids)
    loader = DataLoader(ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, collate_fn=collate_bundle)
    model.eval()
    W, effect, say, answers, contrib, labels, sids = [], [], [], [], [], [], []
    terms = model.term_names
    for batch in loader:
        b = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}
        out = model(b)
        if out["coupling_report"] is not None:
            W.append(out["coupling_report"]["weights"].cpu().numpy())
            eff = coupling_effect(model, out)
            effect.append(np.stack([eff[s].cpu().numpy() for s in model.organ_systems], axis=1))
        if out["system_say"] is not None:
            say.append(np.stack([out["system_say"][s].cpu().numpy() for s in model.organ_systems], axis=1))
        if out["system_answers"] is not None:
            answers.append(np.stack([out["system_answers"][s].cpu().numpy() for s in model.organ_systems], axis=1))
            contrib.append(np.stack([out["per_system_signal"][s].cpu().numpy() for s in terms], axis=1))
        labels.append(batch["label"].numpy()); sids += list(batch["subject_id"])
    cat = lambda xs: np.concatenate(xs) if xs else None
    return {"weights": cat(W), "effect": cat(effect), "say": cat(say), "answers": cat(answers),
            "contrib": cat(contrib), "labels": cat(labels), "subject_id": sids}

# Use val + test patients (never trained on) so the picture is not a memorised one.
_eval_ids = list(VAL_IDS) + list(TEST_IDS)
SYSTEM_OUTPUTS = collect_system_outputs(model, _eval_ids)
_S = model.organ_systems
_names = [PLAIN_NAMES.get(s, s) for s in _S]

if SYSTEM_OUTPUTS["weights"] is None:
    print(f"COUPLING_MODE={CONFIG['COUPLING_MODE']!r} -- no learned coupling layer in this model, nothing to plot.")
    COUPLING_VALIDATION_SHEET = None
else:
    Wt = SYSTEM_OUTPUTS["weights"]                 # [N, receiver, sender]
    y = SYSTEM_OUTPUTS["labels"]
    gate = model.coupling.gate.detach().cpu().numpy()
    mean_W = Wt.mean(axis=0)

    # --- 1. heatmap -------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(8.5, 6.5))
    shown = np.where(np.eye(len(_S), dtype=bool), np.nan, mean_W * 100)
    im = ax.imshow(shown, cmap="Purples", vmin=0, vmax=np.nanmax(shown))
    for i in range(len(_S)):
        for j in range(len(_S)):
            if i != j:
                ax.text(j, i, f"{shown[i, j]:.0f}%", ha="center", va="center", fontsize=8,
                        color="white" if shown[i, j] > 0.6 * np.nanmax(shown) else "black")
    ax.set_xticks(range(len(_S))); ax.set_xticklabels(_names, rotation=40, ha="right")
    ax.set_yticks(range(len(_S)))
    ax.set_yticklabels([f"{n}  (uses msgs: {abs(g):.2f})" for n, g in zip(_names, gate)])
    ax.set_xlabel("...takes information FROM this system"); ax.set_ylabel("This system's reading...")
    ax.set_title("Learned inter-system coupling (val+test patients)\nrow sums to 100%; diagonal blank = a system never reads itself")
    plt.colorbar(im, ax=ax, fraction=0.04, label="% of attention")
    plt.tight_layout(); plt.show()

    # --- 2. network diagram, died vs survived ------------------------------------------
    def _network(ax, M, title, top_k=8):
        n = len(_S)
        ang = np.linspace(np.pi / 2, np.pi / 2 - 2 * np.pi, n, endpoint=False)
        pos = np.c_[np.cos(ang), np.sin(ang)]
        eff = M * np.abs(gate)[:, None]                        # attention x how much the receiver uses it
        np.fill_diagonal(eff, 0)
        idx = np.dstack(np.unravel_index(np.argsort(-eff, axis=None), eff.shape))[0][:top_k]
        vmax = eff.max() if eff.max() > 0 else 1
        for r, snd in idx:
            ax.annotate("", xy=pos[r] * 0.86, xytext=pos[snd] * 0.86,
                        arrowprops=dict(arrowstyle="-|>", color="#8e44ad", alpha=0.85,
                                        lw=0.5 + 5 * eff[r, snd] / vmax, shrinkA=14, shrinkB=14,
                                        connectionstyle="arc3,rad=0.12"))
        for k in range(n):
            ax.scatter(*pos[k], s=900, color="#0a7d6e", zorder=3)   # v3: all eight organ systems drawn alike
            ax.text(*(pos[k] * 1.22), _names[k], ha="center", va="center", fontsize=8)
        ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.5, 1.5); ax.axis("off"); ax.set_title(title, fontsize=10)

    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    for ax, sel, t in [(axes[0], y == 1, "Patients who died"), (axes[1], y == 0, "Patients who survived")]:
        if sel.any():
            _network(ax, Wt[sel].mean(axis=0), f"{t} (n={int(sel.sum())})\narrow = information flow, thickness = strength x use")
    plt.suptitle("Strongest learned links between body systems (top 8)", fontsize=11)
    plt.tight_layout(); plt.show()

    # --- 3. how much the coupling changed each system's answer --------------------------
    E = SYSTEM_OUTPUTS["effect"]                   # [N, S] change in logit units
    fig, ax = plt.subplots(figsize=(9, 4))
    xk = np.arange(len(_S))
    for off, sel, col, lab in [(-0.2, y == 1, "#c0392b", "died"), (0.2, y == 0, "#2980b9", "survived")]:
        if sel.any():
            ax.bar(xk + off, np.abs(E[sel]).mean(axis=0), width=0.4, color=col, label=lab)
    ax.set_xticks(xk); ax.set_xticklabels(_names, rotation=30, ha="right")
    ax.set_ylabel("mean |change in answer|\n(risk-logit units)")
    ax.set_title("How much each system's answer changed because of what it heard from other systems")
    ax.legend(); plt.tight_layout(); plt.show()

    # --- validation sheet ---------------------------------------------------------------
    rng_b = np.random.default_rng(SEED)
    boot = np.stack([Wt[rng_b.integers(0, len(Wt), len(Wt))].mean(axis=0) for _ in range(200)])
    rows = []
    for r in range(len(_S)):
        for snd in range(len(_S)):
            if r == snd:
                continue
            rows.append({
                "receiver": _S[r], "sender": _S[snd],
                "in plain words": f"The model's reading of {_names[r]} draws on {_names[snd]}",
                "attention_%": 100 * mean_W[r, snd],
                "95%_CI": f"{100*np.percentile(boot[:, r, snd], 2.5):.0f}-{100*np.percentile(boot[:, r, snd], 97.5):.0f}%",
                "died_%": 100 * Wt[y == 1][:, r, snd].mean() if (y == 1).any() else np.nan,
                "survived_%": 100 * Wt[y == 0][:, r, snd].mean() if (y == 0).any() else np.nan,
                "receiver_uses_messages": abs(float(gate[r])),
                "effective_strength": mean_W[r, snd] * abs(float(gate[r])),
                "above_even_split": bool(np.percentile(boot[:, r, snd], 2.5) > 1 / (len(_S) - 1)),
                "Surgeon: physiologically plausible? (Y/N/unsure)": "",
                "Surgeon: comment": "",
            })
    COUPLING_VALIDATION_SHEET = pd.DataFrame(rows).sort_values("effective_strength", ascending=False).reset_index(drop=True)
    _csv = os.path.join(CONFIG["CHECKPOINT_DIR"], "coupling_validation_sheet.csv")
    COUPLING_VALIDATION_SHEET.to_csv(_csv, index=False)
    print(f"Even split for a single link = {100/(len(_S)-1):.1f}% (attention spread evenly over the other {len(_S)-1} systems).")
    print("The 95% CI resamples PATIENTS for this one trained model -- it shows how consistent the pattern is "
          "across patients, not whether it would re-appear if the model were retrained. Part 11.4b tests that.")
    if np.abs(gate).max() < 0.05:
        print("\nWARNING: every coupling gate is < 0.05 -- the model is barely using what systems hear from each "
              "other, so the links above have almost no effect on predictions. Not worth a surgeon's review "
              "until a run where the gates are larger (longer training / full cohort).")
    print(COUPLING_VALIDATION_SHEET.head(10)[["in plain words", "attention_%", "95%_CI", "died_%",
                                             "survived_%", "receiver_uses_messages", "above_even_split"]].round(2).to_string())
    _cr = COUPLING_VALIDATION_SHEET[(COUPLING_VALIDATION_SHEET.receiver == "renal") &
                                    (COUPLING_VALIDATION_SHEET.sender == "cardiovascular")]
    if len(_cr):
        print(f"\nCardiorenal check (never given to the model): Kidneys <- Heart link ranks "
              f"#{_cr.index[0] + 1} of {len(COUPLING_VALIDATION_SHEET)} by effective strength, "
              f"{'above' if _cr['above_even_split'].iloc[0] else 'NOT above'} an even split.")
    print(f"\nFull sheet for the surgeon (with blank verdict columns): {_csv}")

## 11.2c How much say each system got (arbitration layer, v2)

Every system gives its own answer; the arbitration layer decides how much that answer
counts for each patient (1.0 = an equal share). Shown by department, because the layer
reads the patient's context — a sensible model should, for example, give the heart more
say in cardiac-surgery patients. That is the clinical check here.

In [ ]:
if SYSTEM_OUTPUTS["say"] is None:
    print("USE_ARBITRATION_LAYER=False (or concat fusion) -- every system has equal say.")
else:
    say_df = pd.DataFrame(SYSTEM_OUTPUTS["say"], columns=_names)
    say_df["department"] = COHORT_INDEXED.loc[SYSTEM_OUTPUTS["subject_id"], "department"].values
    say_by_dept = say_df.groupby("department").mean()
    say_by_dept.insert(0, "n", say_df.groupby("department").size())
    say_by_dept = say_by_dept[say_by_dept["n"] >= 10]
    print("Mean say per system, by department (1.0 = equal share):")
    print(say_by_dept.round(2).to_string())
    if len(say_by_dept):
        fig, ax = plt.subplots(figsize=(9, max(3, 0.45 * len(say_by_dept))))
        im = ax.imshow(say_by_dept[_names].values, cmap="RdBu_r", vmin=0, vmax=2, aspect="auto")
        ax.set_xticks(range(len(_names))); ax.set_xticklabels(_names, rotation=30, ha="right")
        ax.set_yticks(range(len(say_by_dept))); ax.set_yticklabels([f"{d} (n={n})" for d, n in zip(say_by_dept.index, say_by_dept["n"])])
        for i in range(len(say_by_dept)):
            for j, nm in enumerate(_names):
                ax.text(j, i, f"{say_by_dept[nm].iloc[i]:.2f}", ha="center", va="center", fontsize=8)
        plt.colorbar(im, ax=ax, label="say (1 = equal)"); ax.set_title("How much each system's answer counts, by department")
        plt.tight_layout(); plt.show()
    print(f"\nSpread of say across all patients: min {SYSTEM_OUTPUTS['say'].min():.2f}, max {SYSTEM_OUTPUTS['say'].max():.2f}. "
          f"If everything is ~1.0 the layer found no reason to re-weight systems -- a valid result, not a bug.")

## 11.2d Does each organ system answer from its own data? (v3 leakage check + share of risk)

Two checks that protect the "per-system" story once the whole-patient and coupling layers
exist:

- **Own-data share** — shuffle one system's own inputs across patients and see how much
  its answer moves, then shuffle everything else instead. Share near 1 = the kidney answer
  really is about the kidneys; below 0.5 = it mostly echoes other systems (flagged).
- **Share of risk** — how much of each patient's risk comes from the whole-patient term
  versus the eight organ systems. If the whole-patient term takes nearly all of it, the
  organ systems have become decoration.

In [ ]:
LEAKAGE = None
if SYSTEM_OUTPUTS["answers"] is not None:
    _big = DataLoader(INSPIREDataset(list(TEST_IDS)), batch_size=min(1024, len(TEST_IDS)), shuffle=True,
                      collate_fn=collate_bundle, generator=torch.Generator().manual_seed(SEED))
    LEAKAGE = pd.DataFrame(own_data_share(model, _to_device(next(iter(_big))))).T
    LEAKAGE.index = [PLAIN_NAMES.get(s, s) for s in LEAKAGE.index]
    LEAKAGE["verdict"] = np.where(LEAKAGE["own_share"] >= 0.5, "own data", "FLAG: mostly other systems")
    print(LEAKAGE.round(3).to_string())
    _C = np.abs(SYSTEM_OUTPUTS["contrib"])
    _ctx = model.term_names.index(model.context_term)
    _row_tot = np.clip(_C.sum(axis=1), 1e-9, None)
    SHARE_WHOLE_PATIENT = float(np.mean(_C[:, _ctx] / _row_tot))
    SHARE_SURGICAL_CONTEXT = (float(np.mean(_C[:, model.term_names.index("surgical_context")] / _row_tot))
                              if "surgical_context" in model.term_names else None)
    _n_sys = len(model.organ_systems)
    _share_sys_total = float(np.mean(_C[:, :_n_sys].sum(axis=1) / _row_tot))
    print(f"\nShare of risk (mean over val+test patients): {model.context_term} term {SHARE_WHOLE_PATIENT:.0%}"
          + (f", surgical-context term {SHARE_SURGICAL_CONTEXT:.0%}" if SHARE_SURGICAL_CONTEXT is not None else "")
          + f", {_n_sys} organ systems {_share_sys_total:.0%}.")
    _share_terms = pd.Series(_C.mean(axis=0) / _C.sum(axis=1).mean(), index=[PLAIN_NAMES.get(t, t) for t in model.term_names])
    _share_sys = _share_terms.iloc[:_n_sys]
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.bar(list(_share_terms.index), list(_share_terms.values),
           color=["#0a7d6e"] * _n_sys + ["#7f8c8d", "#d35400"][:len(model.term_names) - _n_sys])
    ax.set_ylabel("average share of |risk|"); ax.set_title("Where each prediction comes from")
    ax.tick_params(axis="x", rotation=30); plt.tight_layout(); plt.show()
else:
    SHARE_WHOLE_PATIENT = None
    SHARE_SURGICAL_CONTEXT = None
    print("Concat fusion -- no per-system answers to check.")

## 11.3 Attention audit (§1.1, §3C item 4 of the source repo's roadmap)

Do the transformer heads concentrate on clinically sensible time steps? A lightweight
check: for one system and one patient, plot the mean attention weight the encoder's first
layer places on each time step, alongside the observed/imputed mask — a real audit for a
full run would extend this per-head and cross-reference against the acute-deterioration
ICD-10 codes already flagged in the source repo's EDA (D65, I46, R57, J80, K72, A41).

In [ ]:
@torch.no_grad()
def get_attention_weights(model, batch_item, system):
    # First-layer attention of the shared organ-system encoder for one patient/system.
    # Token 0 = the system's summary token (its static features), tokens 1..T = time points.
    x = batch_item[f"{system}_x"].unsqueeze(0).to(DEVICE) if f"{system}_x" in batch_item else None
    m = batch_item[f"{system}_mask"].unsqueeze(0).to(DEVICE) if f"{system}_mask" in batch_item else None
    st = batch_item["static"].unsqueeze(0).to(DEVICE)
    idx = model.system_static_idx[system]
    if x is None:
        return None
    return model.encoder.attention_map(system, x, m, st[:, idx] if idx else None).squeeze(0).cpu().numpy()

example_item = test_dataset[0]
example_system = "renal"
attn = get_attention_weights(model, example_item, example_system)
if attn is not None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    im = axes[0].imshow(attn, cmap="viridis"); axes[0].set_title(f"{example_system} attention (layer 0, head-avg)")
    axes[0].set_xlabel("attended-to token (0 = summary)"); axes[0].set_ylabel("query token")
    plt.colorbar(im, ax=axes[0], fraction=0.046)
    mask = example_item[f"{example_system}_mask"].numpy()
    axes[1].imshow(mask.T, aspect="auto", cmap="Greys"); axes[1].set_title("observed(black)/imputed(white) mask")
    axes[1].set_xlabel("timestep"); axes[1].set_ylabel("feature index")
    plt.tight_layout(); plt.show()
    # Does attention favour genuinely observed time points? (v3: computed, not just suggested)
    _obs = mask.max(axis=1) > 0
    _col = attn[:, 1:].mean(axis=0)
    if _obs.any() and (~_obs).any():
        print(f"Mean attention on time points with a real observation: {_col[_obs].mean():.4f} vs imputed-only: "
              f"{_col[~_obs].mean():.4f} (ratio {_col[_obs].mean() / max(_col[~_obs].mean(), 1e-9):.2f}; >1 = favours real data)")

## 11.4 Ablation: does each piece earn its place? (v3)

Same data, same split, same sampling — one architecture switch changes per row, so any
difference can be attributed. The full v3 row reuses the main model (no retraining).
Read differences against the ensemble spread from 11.1e: a gap smaller than the
seed-to-seed spread is not a real difference.

In [ ]:
def _test_scores(m):
    r = evaluate(m, test_loader)
    return {"test_auprc": r["auprc"], "test_auroc": r["auroc"], "n_params": sum(p.numel() for p in m.parameters())}

ABLATION_VARIANTS = [
    ("organ systems only (no coupling / arbitration / whole-patient)",
     dict(coupling_mode="none", use_arbitration=False, use_whole_patient=False), None),
    ("+ learned coupling", dict(coupling_mode="learned", use_arbitration=False, use_whole_patient=False), None),
    ("+ arbitration", dict(coupling_mode="learned", use_arbitration=True, use_whole_patient=False), None),
    ("full v3, separate encoder per system", dict(share_encoder=False), None),
    ("full v3, no masked-system pre-training", dict(), {"MASKED_SYSTEM_WEIGHT": 0.0}),
    ("concat fusion (no per-system breakdown)", dict(fusion_strategy="concat", use_arbitration=False), None),
    # v4 -- the two model-changing upgrades, each tested by taking it away:
    ("v4 without the surgical-context term", dict(use_surgical_context=False), None),
    ("v4 surgical context NOT exclusive (whole-patient also reads it)", dict(), {"SURGICAL_CONTEXT_EXCLUSIVE": False}),
    ("6 organ systems (GI and MSK removed -> their features go to whole-patient)", dict(drop_systems=("gi", "msk")), None),
]
if CONFIG["SKIP_FUSION_ABLATION"]:
    print(f"CONFIG['SKIP_FUSION_ABLATION']=True -- skipping the ablation ({len(ABLATION_VARIANTS)} retrainings).")
    ablation_df = pd.DataFrame(columns=["test_auprc", "test_auroc", "n_params"])
else:
    rows = [{"variant": "full v3 (this notebook's model)", **_test_scores(model), "minutes": 0.0}]
    for vname, kw, cfg_patch in ABLATION_VARIANTS:
        _saved = {k: CONFIG[k] for k in (cfg_patch or {})}
        CONFIG.update(cfg_patch or {})
        _t0 = time.time()
        try:
            m_v, _ = train_full_model(seed=SEED, **kw)
            rows.append({"variant": vname, **_test_scores(m_v), "minutes": (time.time() - _t0) / 60})
            del m_v
        finally:
            CONFIG.update(_saved)
        print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in rows[-1].items()})
    ablation_df = pd.DataFrame(rows).set_index("variant")
    print("\n" + ablation_df.round(3).to_string())
ablation_df

### 11.4b Do the learned links replicate? (coupling stability across the ensemble)

Each ensemble member learned its own "who listens to whom" matrix. If the pattern comes
from the data, the members agree; if it comes from the random start, they don't. Rank
correlation over all off-diagonal links, plus top-5 overlap, versus the main model.
> 0.7: individual links are safe to show a surgeon; 0.4-0.7: top links only; < 0.4: not yet.

In [ ]:
@torch.no_grad()
def mean_coupling_matrix(m, ids):
    out = collect_system_outputs(m, ids)
    if out["weights"] is None:
        return None
    return out["weights"].mean(axis=0) * np.abs(m.coupling.gate.detach().cpu().numpy())[:, None]

COUPLING_STABILITY = None
if CONFIG["COUPLING_MODE"] == "learned" and len(ENSEMBLE_MODELS) > 1:
    from scipy.stats import spearmanr
    _off = ~np.eye(len(model.organ_systems), dtype=bool)
    mats = [mean_coupling_matrix(m, _eval_ids) for m in ENSEMBLE_MODELS]
    _top = lambda M: set(np.argsort(-M[_off])[:5])
    COUPLING_STABILITY = pd.DataFrame([{
        "member": k, "spearman_vs_main": spearmanr(mats[0][_off], mats[k][_off]).correlation,
        "top5_overlap": len(_top(mats[0]) & _top(mats[k])) / 5} for k in range(1, len(mats))])
    print(COUPLING_STABILITY.round(2).to_string(index=False))
    _rho = COUPLING_STABILITY["spearman_vs_main"].mean()
    print(f"\nMean rank agreement with the main model: {_rho:.2f}")
    ENSEMBLE_COUPLING = np.mean(mats, axis=0)   # the averaged map: the most stable picture to show
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    vmax = max(mats[0].max(), ENSEMBLE_COUPLING.max())
    for ax, M, t in [(axes[0], mats[0], "main model"), (axes[1], ENSEMBLE_COUPLING, f"average of {len(mats)} models")]:
        im = ax.imshow(np.where(_off, M, np.nan), cmap="Purples", vmin=0, vmax=vmax)
        ax.set_xticks(range(len(_names))); ax.set_xticklabels(_names, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(len(_names))); ax.set_yticklabels(_names, fontsize=8); ax.set_title(t, fontsize=10)
    axes[0].set_ylabel("this system's reading..."); axes[1].set_xlabel("...takes information FROM")
    plt.suptitle("Effective inter-system coupling (attention x gate)", fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print("Coupling stability needs a learned coupling layer and N_ENSEMBLE >= 2.")

### 11.4c Learning curve — is the model starved of data, or memorising? (v3)

Retrains on 25% and 50% of the real training patients (department-stratified, same
validation and test sets) and compares with 100%. Validation AUPRC still climbing steeply
at 100% means more data would help (expected — this is the case for the full cohort);
a flat curve means the architecture, not the data, is the limit. The subsets use real
patients + their jittered copies only (no SMOTENC), so they sit slightly below a
like-for-like line; the shape is what matters.

In [ ]:
LEARNING_CURVE = None
if CONFIG["RUN_LEARNING_CURVE"]:
    _real = [s for s in EFFECTIVE_TRAIN_IDS]
    _lab = np.array([PATIENT_BUNDLE[s]["label"] for s in _real])
    rows = []
    for frac in (0.25, 0.5):
        sub, _ = train_test_split(_real, train_size=frac, stratify=_lab, random_state=SEED)
        ds_sub = INSPIREDataset(sub, None, build_augment_items(sub, CONFIG["USE_SEQUENCE_AUGMENTATION"],
                                                               CONFIG["SEQUENCE_AUGMENTATION_COPIES"]))
        _t0 = time.time()
        m_f, best_val = train_full_model(seed=SEED, train_ds=ds_sub)
        rows.append({"train_fraction": frac, "n_train": len(sub), "train_deaths": int(sum(PATIENT_BUNDLE[s]["label"] for s in sub)),
                     "val_auprc": best_val, "test_auprc": evaluate(m_f, test_loader)["auprc"], "minutes": (time.time() - _t0) / 60})
        del m_f
    rows.append({"train_fraction": 1.0, "n_train": len(_real), "train_deaths": int(_lab.sum()),
                 "val_auprc": best_val_auroc, "test_auprc": test_result["auprc"], "minutes": np.nan})
    LEARNING_CURVE = pd.DataFrame(rows)
    print(LEARNING_CURVE.round(3).to_string(index=False))
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(LEARNING_CURVE.n_train, LEARNING_CURVE.val_auprc, "o-", label="validation AUPRC", color="#8e44ad")
    ax.plot(LEARNING_CURVE.n_train, LEARNING_CURVE.test_auprc, "s--", label="test AUPRC", color="#0a7d6e")
    ax.set_xlabel("training patients"); ax.set_ylabel("AUPRC"); ax.legend(); ax.set_title("Learning curve")
    plt.tight_layout(); plt.show()
else:
    print("RUN_LEARNING_CURVE=False -- skipped.")

### 11.4d Memorisation checks (v3)

Two direct tests. (1) **Train vs. validation gap** at the chosen epoch: a large gap means
the model fits training patients far better than new ones. (2) **Donor check**: real deaths
whose time series were borrowed by synthetic patients (8.2c) are seen many more times in
training. If the model scores them much higher than other real training deaths, it has
memorised those individuals.

In [ ]:
MEMORISATION = {}
_best_epoch = int(np.nanargmax(history["val_auprc"])) if history["val_auprc"] else None
if _best_epoch is not None and history.get("train_auprc"):
    MEMORISATION["train_auprc"] = float(history["train_auprc"][_best_epoch])
    MEMORISATION["val_auprc"] = float(history["val_auprc"][_best_epoch])
    MEMORISATION["gap"] = MEMORISATION["train_auprc"] - MEMORISATION["val_auprc"]
    print(f"At the selected epoch ({_best_epoch}): train AUPRC {MEMORISATION['train_auprc']:.3f} vs "
          f"validation {MEMORISATION['val_auprc']:.3f} -> gap {MEMORISATION['gap']:+.3f}")
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(history["train_auprc"], label="train (real patients)", color="#c0392b")
    ax.plot(history["val_auprc"], label="validation", color="#2980b9")
    ax.axvline(_best_epoch, color="gray", ls=":", label="selected epoch")
    ax.set_xlabel("fine-tuning epoch"); ax.set_ylabel("AUPRC"); ax.legend(); ax.set_title("Train vs. validation")
    plt.tight_layout(); plt.show()
_donors = sorted(set(SYNTHETIC_TS_DONORS))
_train_deaths = [s for s in EFFECTIVE_TRAIN_IDS if PATIENT_BUNDLE[s]["label"] == 1.0]
_others = [s for s in _train_deaths if s not in set(_donors)]
if _donors and _others:
    _pd = evaluate(model, DataLoader(INSPIREDataset(_donors), batch_size=256, collate_fn=collate_bundle))["probs"]
    _po = evaluate(model, DataLoader(INSPIREDataset(_others), batch_size=256, collate_fn=collate_bundle))["probs"]
    MEMORISATION.update(donor_median_risk=float(np.median(_pd)), other_death_median_risk=float(np.median(_po)),
                        n_donors=len(_donors))
    print(f"Donor check: median predicted risk {np.median(_pd):.3f} for {len(_donors)} donor deaths vs "
          f"{np.median(_po):.3f} for {len(_others)} other real training deaths.")
else:
    print("Donor check skipped (no synthetic donors this run).")

## 11.5 Sensitivity: pre-op-only vs. peri-operative (§1.6.3, Path B)

The full re-run this needs (re-window every patient with `TIME_WINDOW='peri_op'`, rebuild
§6-§10 end to end) is expensive to repeat inline for every cell execution, so this section
is a **template you run deliberately** rather than something that fires automatically:
change `CONFIG['TIME_WINDOW']` to `'peri_op'` in Part 2, then **Run All** again, and record
the resulting test AUROC/AUPRC/coverage numbers in the comparison table below by hand (or
extend the automation from §11.4 to also loop over `TIME_WINDOW`, which is the natural next
step once this notebook runs on a machine fast enough to afford two full passes).

| Run | TIME_WINDOW | test AUROC | test AUPRC | mean cardiovascular coverage (§5.2-style) | notes |
|---|---|---|---|---|---|
| 1 | `pre_op` | *(fill in from §11.1)* | | | Decision-support framing (§1.6.3) |
| 2 | `peri_op` | *(re-run and fill in)* | | | Real-time monitoring framing (§1.6.3) |

Per §1.6.3's own reasoning: a higher `peri_op` AUROC is expected partly from genuinely new
physiological information and partly from better measurement *density* (intra-op vitals are
less sparse than ward vitals per §2.7 of the source repo's `Research_Aim.md`) — report the
coverage numbers from §5.2 alongside AUROC for either run so a reader can tell the two
effects apart, exactly as that section recommends.

## 11.6 Cardiac-recovery-exception qualitative check (§1.6.4)

Does the model's own cardiovascular-system contribution (from the NAM breakdown, §11.2)
shift for patients flagged by the rule-based §6.11 exception, versus similar unflagged
patients? A genuine qualitative validation in the spirit of the source repo's own
worked-patient sanity check (Subject 100033460 in its docs) — small-N here, but the
comparison is structured to scale directly once more flagged patients exist in the full
cohort.

In [ ]:
@torch.no_grad()
def cardiac_contribution_by_exception_flag(model, all_ids):
    rows = []
    ds = INSPIREDataset(all_ids)
    loader_all = DataLoader(ds, batch_size=8, shuffle=False, collate_fn=collate_bundle)
    model.eval()
    for batch in loader_all:
        batch_dev = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}
        out = model(batch_dev)
        if out["per_system_signal"] is None:
            return None
        for i, sid in enumerate(batch["subject_id"]):
            rows.append({
                "subject_id": sid,
                "cardiac_recovery_exception": CARDIAC_EXCEPTION_FLAG[sid],
                "cardiovascular_contribution": float(out["per_system_signal"]["cardiovascular"][i]),
                "true_label": float(batch["label"][i]),
            })
    return pd.DataFrame(rows)

exc_df = cardiac_contribution_by_exception_flag(model, ALL_IDS)
if exc_df is not None:
    summary = exc_df.groupby("cardiac_recovery_exception")["cardiovascular_contribution"].agg(["mean", "std", "count"])
    print(summary)
    if (exc_df["cardiac_recovery_exception"] == 1).sum() == 0:
        print("\nNo patients in this dev subset are flagged by the §6.11 rule -- this comparison "
              "is a no-op here. Re-run once the full cohort (with more CTS re-interventions "
              "within the 6-month window) is loaded, where this check becomes informative.")

## 11.7 Multi-operation label sensitivity (§1.6.4, Path C — last-op vs. first-op)

§4.2 already computed `died_30day_from_first_op` alongside the training label
(`died_30day_from_last_op`). A full re-run under the first-op definition needs the same
"change config, Run All" treatment as §11.5 for a genuine model comparison — but the
**label-agreement** check itself doesn't need retraining and is worth doing right here.

In [ ]:
disagreement = cohort_df[cohort_df["died_30day_from_last_op"] != cohort_df["died_30day_from_first_op"]]
print(f"Patients where last-op and first-op 30-day labels disagree: {len(disagreement)} / {len(cohort_df)}")
if len(disagreement):
    print(disagreement[["subject_id", "n_operations", "died_30day_from_last_op", "died_30day_from_first_op"]])
print("\nA non-trivial disagreement rate here is exactly the evidence §1.6.4/Path C says to "
      "collect before picking one definition -- if the disagreement rate is high and "
      "concentrated among multi-operation patients specifically, that's a strong argument "
      "for running (not assuming) the full three-way sensitivity analysis the source repo's "
      "own roadmap already calls for.")

## 11.8 Risk trajectory over time -- watching risk evolve as more data arrives

Everything in Part 11 so far scores a patient using their **complete** pre-op window. This
section instead asks the trained model the same question at **several earlier cutoff
points** for one patient -- re-extracting and re-imputing only the time-series inputs up to
each cutoff (the static features -- demographics, ICD-10, operation history -- are held at
their full-window values for this demo, a simplification worth knowing about if this is
presented further: a fully rigorous version would also restrict static features like
"diagnoses so far" to each cutoff). This is Option 1 from the three timeline-architecture
options discussed earlier in this project ("repeated snapshot") -- the cheapest of the
three to build, since it reuses the already-trained model rather than requiring a new
hazard-curve architecture (Option 2) or a separate deterioration-trend detector (Option 3).

**What this demonstrates:** the model's risk estimate for a patient as a function of how
much of their pre-op window has been observed -- a rising curve as a real deterioration
enters the data is the qualitative pattern worth showing; the exact numbers, like
everywhere else in this notebook at this sample size, are illustrative rather than
validated.

In [ ]:
def predict_risk_at_cutoff(sid, cutoff_time, model, device=DEVICE):
    # Re-extracts and re-imputes only the time-series inputs for one patient, using
    # [lo, cutoff_time] instead of the patient's full pre-op window -- static features are
    # held at their full-window values (see the markdown note above). Reuses the exact
    # imputation/standardisation statistics already fit on the training split, so this is
    # a fair comparison against the model's normal predictions, not a different pipeline.
    row = COHORT_INDEXED.loc[sid]
    full_lo, full_hi = get_window(sid, row)
    lo = full_lo
    hi = min(cutoff_time, full_hi)   # never look beyond what the model was normally given

    batch = {}
    for system in TIME_SERIES_SYSTEMS:
        fnames = PATIENT_BUNDLE[sid]["ts"][system]["feature_names"]
        if not fnames:
            raw = np.zeros((CONFIG["TARGET_SEQ_LEN"], 0))
            mask = np.zeros((CONFIG["TARGET_SEQ_LEN"], 0))
        else:
            raw, mask, _ = extract_system_tensor(sid, system, lo, hi)
            grid = np.linspace(lo, hi, CONFIG["TARGET_SEQ_LEN"])
            fast_idx = [j for j, f in enumerate(fnames) if f in FAST_CHANGING_ITEMS]
            slow_idx = [j for j, f in enumerate(fnames) if f not in FAST_CHANGING_ITEMS]
            imputed = raw.copy()
            if fast_idx:
                imputed[:, fast_idx] = impute_interpolate(raw[:, fast_idx], [fnames[j] for j in fast_idx], system, STATS, grid)
            if slow_idx:
                imputed[:, slow_idx] = impute_forward_fill(raw[:, slow_idx], [fnames[j] for j in slow_idx], system, STATS)
            for j, fname in enumerate(fnames):
                mean, std = STATS[(system, fname)]["mean"], TRAIN_STD[(system, fname)]
                imputed[:, j] = (imputed[:, j] - mean) / std
            raw, mask = imputed, mask
        batch[f"{system}_x"] = torch.tensor(raw, dtype=torch.float32).unsqueeze(0).to(device)
        batch[f"{system}_mask"] = torch.tensor(mask, dtype=torch.float32).unsqueeze(0).to(device)

    static_vec = IMPUTED_BUNDLE[sid]["static"].values.astype(float)   # held at full-window values, see note above
    batch["static"] = torch.tensor(static_vec, dtype=torch.float32).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        out = model(batch)
    prob = float(torch.sigmoid(out["logit"])[0])
    per_system = {k: float(v[0]) for k, v in out["per_system_signal"].items()} if out["per_system_signal"] is not None else None
    return prob, per_system

def plot_risk_trajectory(sid, n_cutoffs=6):
    row = COHORT_INDEXED.loc[sid]
    lo, hi = get_window(sid, row)
    cutoffs = np.linspace(lo + (hi - lo) * 0.15, hi, n_cutoffs)   # skip the very first sliver -- too little data to mean anything
    probs = []
    for c in cutoffs:
        prob, _ = predict_risk_at_cutoff(sid, c, model)
        probs.append(prob)

    true_label = "died" if PATIENT_BUNDLE[sid]["label"] == 1.0 else "survived"
    fig, ax = plt.subplots(figsize=(8, 4.5))
    hours_before_surgery = (cutoffs - hi) / 60.0
    ax.plot(hours_before_surgery, probs, "o-", color="#c0392b" if true_label == "died" else "#2980b9")
    ax.set_xlabel("hours before surgery (0 = end of pre-op window)")
    ax.set_ylabel("predicted 30-day mortality risk")
    ax.set_title(f"Patient {sid} (true outcome: {true_label}) -- risk as more pre-op data arrives")
    ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    return cutoffs, probs

# Pick one real patient who died, to show the pattern this is meant to illustrate.
_died_ids = [sid for sid in ALL_IDS if PATIENT_BUNDLE[sid]["label"] == 1.0]
_example_sid = _died_ids[0] if _died_ids else ALL_IDS[0]
print(f"Plotting risk trajectory for patient {_example_sid}")
_cutoffs, _probs = plot_risk_trajectory(_example_sid)
print(f"Risk at each cutoff: {[f'{p:.2f}' for p in _probs]}")
print(f"\nNOTE: the model behind this trajectory was trained on {len(TRAIN_IDS)} real patients "
      f"({int(sum(PATIENT_BUNDLE[s]['label'] for s in TRAIN_IDS))} deaths) -- it shows what the mechanism "
      f"produces; treat it as illustrative until repeated on the full cohort.")

### Reading this plot for the meeting

This is the mechanism you told James you'd present -- a working demonstration that risk can
be read out **as a function of how much data has arrived**, not just as one end-of-window
number. Two honest framings for presenting it:

- **What's real and presentable:** the pipeline can genuinely produce a risk-over-time
  curve from partial data, using the same trained model, with no architecture change
  needed. That's the actual engineering result.
- **What isn't validated yet:** the specific shape of the curve above, and whether it
  reliably rises before a real deterioration, needs the full cohort to test properly --
  at 18 training patients the network hasn't seen enough real trajectories to have learned
  a trustworthy notion of "getting worse over time" yet.

## 11.9 The clinical story -- risk trajectory alongside the measurements driving it

Section 11.8 showed risk rising over time. This section adds the "why" underneath it: the
actual raw measurements (not standardized, not imputed -- the real observed values) for
whichever organ systems the model's own per-system breakdown (§11.2) says contributed most
to this patient's prediction. The two contributing systems shown are chosen dynamically,
per patient, from the model's own NAM contributions -- this isn't a hand-picked example,
it's read directly from what the model itself flagged as driving the prediction.

In [ ]:
HEADLINE_FEATURE_PER_SYSTEM = {
    "renal": "creatinine", "cardiovascular": "hr", "respiratory": "spo2",
    "metabolic_hepatic": "glucose", "haematology": "hb", "neurological": "gcs_e",
}

def plot_patient_clinical_story(sid, n_cutoffs=6):
    row = COHORT_INDEXED.loc[sid]
    lo, hi = get_window(sid, row)
    cutoffs = np.linspace(lo + (hi - lo) * 0.15, hi, n_cutoffs)

    probs, per_system_at_end = [], None
    for i, c in enumerate(cutoffs):
        prob, per_sys = predict_risk_at_cutoff(sid, c, model)
        probs.append(prob)
        if i == len(cutoffs) - 1:
            per_system_at_end = per_sys

    sys_contribs = {k: v for k, v in (per_system_at_end or {}).items() if k in TIME_SERIES_SYSTEMS}   # v2: gi/msk/static have no time series to plot
    top_systems = sorted(sys_contribs, key=lambda k: -abs(sys_contribs[k]))[:2]
    top_systems = [s for s in top_systems if PATIENT_BUNDLE[sid]["ts"][s]["feature_names"]]   # only systems with real features to plot

    true_label = "died" if PATIENT_BUNDLE[sid]["label"] == 1.0 else "survived"
    n_panels = 1 + len(top_systems)
    fig, axes = plt.subplots(n_panels, 1, figsize=(9, 2.8 * n_panels), sharex=True)
    if n_panels == 1:
        axes = [axes]
    hours_before = (cutoffs - hi) / 60.0

    axes[0].plot(hours_before, probs, "o-", color="#c0392b" if true_label == "died" else "#2980b9", linewidth=2)
    axes[0].set_ylabel("predicted\nmortality risk")
    axes[0].set_title(f"Patient {sid} (true outcome: {true_label}) -- risk and the measurements driving it")
    axes[0].grid(alpha=0.3)
    axes[0].axhline(0.5, color="gray", linestyle=":", linewidth=1)

    for ax_idx, system in enumerate(top_systems, start=1):
        fnames = PATIENT_BUNDLE[sid]["ts"][system]["feature_names"]
        feature = HEADLINE_FEATURE_PER_SYSTEM.get(system)
        if feature not in fnames:
            feature = fnames[0]   # fall back to whatever this system actually has for this patient
        s1 = windowed_series(labs_df, sid, feature, lo, hi)
        s2 = windowed_series(ward_vitals_df, sid, feature, lo, hi)
        series = {**s1, **s2}
        ax = axes[ax_idx]
        contrib = sys_contribs.get(system, 0.0)
        if series:
            times = sorted(series.keys())
            vals = [series[t] for t in times]
            hrs = [(t - hi) / 60.0 for t in times]
            ax.plot(hrs, vals, "o-", color="#8e44ad")
        else:
            ax.text(0.5, 0.5, "no observed values in this window", ha="center", va="center", transform=ax.transAxes, fontsize=9, color="gray")
        ax.set_ylabel(f"{system}\n({feature})")
        ax.set_title(f"NAM contribution: {contrib:+.2f} ({'pushes toward died' if contrib > 0 else 'pushes toward survived'})", fontsize=9)
        ax.grid(alpha=0.3)

    axes[-1].set_xlabel("hours before surgery (0 = end of pre-op window)")
    plt.tight_layout()
    plt.show()
    return top_systems, sys_contribs

print(f"Clinical story for patient {_example_sid} (reusing §11.8's example patient):")
_top_sys, _contribs = plot_patient_clinical_story(_example_sid)
print(f"Top contributing systems for this patient: {_top_sys}")
print(f"Full per-system contribution at the final cutoff: {_contribs}")

### Reading this for the meeting

Top panel: the same risk curve as §11.8. Panels below it: the *actual* clinical
measurements for the one or two organ systems the model's own fusion layer says drove that
prediction, over the same time axis -- so a rise in predicted risk can be pointed at
directly against a real, observed change in a real measurement, not just asserted. This is
the version of "these were the measurements and this was where risk was rising" your
notebook can now produce directly from the trained model, for any patient, not just this
one example -- call `plot_patient_clinical_story(<subject_id>)` for any patient in
`ALL_IDS` to generate the same figure for someone else.

## 11.10 One unified plot: real readings and the risk line, on the same axis

Section 11.9 put the risk curve and the measurements in separate stacked panels. This is
the design actually requested: **one chart**, with a black risk line drawn *through* the
patient's real readings over time -- so you can see directly whether the risk line moves
together with a real clinical change, not just alongside it in a different panel.

**The one real design problem this has to solve, stated plainly:** heart rate lives in the
60-110 range, creatinine lives in the 0.5-1.5 range, and risk is a probability from 0 to 1
-- plotting them on the same raw axis would make the smaller-scale ones invisible. The fix:
every measurement trace is **rescaled to 0-1 using the population's own observed range**
for that feature (not this one patient's range, which would make different patients'
plots incomparable to each other) -- so "near the top of its line" consistently means
"near the highest values seen across the training population," for every trace, on every
patient's plot. Risk is already a 0-1 probability, so it needs no rescaling at all -- it's
the one line already honestly on the shared axis by construction, which is part of why it
reads as the "anchor" line the others are compared against.

**Honest scope note:** this currently spans the pre-op window only (or pre-op + surgery
if `CONFIG['TIME_WINDOW']='peri_op'`) -- extending it into true post-operative recovery
data is not built yet (see the Option B / post-op discussion) and would need the data
window itself extended past `orout_time`, not just a change to this plotting function.

In [ ]:
def compute_population_range(system, feature_idx, patient_ids):
    vals = []
    for sid in patient_ids:
        raw = PATIENT_BUNDLE[sid]["ts"][system]["raw"]
        mask = PATIENT_BUNDLE[sid]["ts"][system]["mask"]
        obs = raw[:, feature_idx][mask[:, feature_idx] == 1]
        vals.extend(obs.tolist())
    if not vals:
        return None, None
    return float(np.min(vals)), float(np.max(vals))

def plot_unified_risk_overlay(sid, n_cutoffs=6, max_traces=3):
    row = COHORT_INDEXED.loc[sid]
    lo, hi = get_window(sid, row)
    cutoffs = np.linspace(lo + (hi - lo) * 0.15, hi, n_cutoffs)

    probs, per_system_at_end = [], None
    for i, c in enumerate(cutoffs):
        prob, per_sys = predict_risk_at_cutoff(sid, c, model)
        probs.append(prob)
        if i == len(cutoffs) - 1:
            per_system_at_end = per_sys

    sys_contribs = {k: v for k, v in (per_system_at_end or {}).items() if k in TIME_SERIES_SYSTEMS}   # v2: gi/msk/static have no time series to plot
    top_systems = sorted(sys_contribs, key=lambda k: -abs(sys_contribs[k]))[:max_traces]
    top_systems = [s for s in top_systems if PATIENT_BUNDLE[sid]["ts"][s]["feature_names"]]

    fig, ax = plt.subplots(figsize=(10, 5.5))
    hours_before = (cutoffs - hi) / 60.0
    ax.plot(hours_before, probs, "-", color="black", linewidth=3, zorder=10, label="predicted mortality risk (0-1)")
    ax.scatter(hours_before, probs, color="black", zorder=11, s=45)

    trace_colors = ["#c0392b", "#2980b9", "#27ae60"]
    for color, system in zip(trace_colors, top_systems):
        fnames = PATIENT_BUNDLE[sid]["ts"][system]["feature_names"]
        feature = HEADLINE_FEATURE_PER_SYSTEM.get(system)
        if feature not in fnames:
            feature = fnames[0]
        f_idx = fnames.index(feature)
        vmin, vmax = compute_population_range(system, f_idx, TRAIN_IDS_FOR_STATS)

        s1 = windowed_series(labs_df, sid, feature, lo, hi)
        s2 = windowed_series(ward_vitals_df, sid, feature, lo, hi)
        series = {**s1, **s2}
        if series and vmin is not None and vmax != vmin:
            times = sorted(series.keys())
            raw_vals = [series[t] for t in times]
            norm_vals = [float(np.clip((v - vmin) / (vmax - vmin), 0, 1)) for v in raw_vals]
            hrs = [(t - hi) / 60.0 for t in times]
            contrib = sys_contribs.get(system, 0.0)
            ax.plot(hrs, norm_vals, "o--", color=color, alpha=0.85,
                     label=f"{system} ({feature}) -- contribution {contrib:+.2f}")

    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("hours before surgery (0 = end of pre-op window)")
    ax.set_ylabel("risk (black, solid) / normalised measurement (coloured, dashed)")
    true_label = "died" if PATIENT_BUNDLE[sid]["label"] == 1.0 else "survived"
    ax.set_title(f"Patient {sid} (true outcome: {true_label}) -- risk overlaid on its own driving measurements")
    ax.legend(loc="upper left", fontsize=8, framealpha=0.9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_unified_risk_overlay(_example_sid)

In [ ]:
record_checkpoint("Part 11 -- evaluation, interpretability, ablations")

# Part 13 — Looking inside the model (v4)

Everything James asked for in the 28-09-2026 meeting, built on the "diary" the training
loop kept in Part 10 (per-epoch metrics + embedding snapshots):

| Section | Question it answers | Like the reference paper's… |
|---|---|---|
| 13.1 | Is everything saved — model, ensemble, predictions, figures? | "save the model / read the model / save the images" |
| 13.2 | How good was the model before training, after pre-training, after each epoch — for emergency and scheduled patients? | training curves |
| 13.3 | What do the patients look like in the model's mind, before vs after training? | Fig. 5 t-SNE, epoch 1 vs epoch 32 |
| 13.4 | Are there natural groups of patients nobody labelled? | "useful features clustering" |
| 13.5 | How alike does the model think patients are, early vs late? | Fig. 4 cosine-similarity heatmaps |
| 13.6 | Does the model work as well for emergency as for scheduled surgery? | results table |
| 13.7 | Does filling missing data, or making synthetic patients, distort NEWS2? | — |
| 13.8 | Inside each organ system, which measurements drove its risk points? (SHAP) | — |
| 13.9 | Does a bigger embedding help? | Fig. 6 score vs hidden size |
| 13.10 | Are the learned embeddings useful for OTHER questions? | "using the same feature vector for other tasks" |

**Plain words.** An *embedding* is the model's short private summary of a patient — a
list of numbers (32 per organ system with the `medium` preset). Patients the model sees as
alike get similar lists. We cannot look at 32 dimensions, so PCA / t-SNE / UMAP squash
them onto a flat map where each dot is a patient. If the dots for patients who died drift
together as training goes on, the model has learned something real about risk.

Every section is run by `run_section(...)`: if one fails, it says so loudly (with the
error) and the rest of Part 13 still runs. Nothing fails silently — the status of every
section is listed at the end (13.11) and in Part 14's verification table.

In [ ]:
# --- inlined from src-dnn/inspire_dnn/analysis.py (edit it there, then rebuild) ---
# Analysis helpers for "looking inside the model" (v4, Part 13). Numpy / scikit-learn /
# matplotlib only -- no PyTorch, no dataset globals -- so every function is unit-tested on
# random arrays (tests/test_analysis.py) and inlined into the notebooks unchanged.
#
# What lives here:
#   * the four patient groups James asked for (emergency / scheduled  x  died / survived)
#   * metrics per group (AUROC, AUPRC with bootstrap CIs, Brier)
#   * squashing embeddings onto a 2-D map (PCA, t-SNE, UMAP if installed)
#   * "do deaths sit near deaths?" -- k-nearest-neighbour death enrichment
#   * unsupervised clustering of the embeddings + a plain-English profile of each cluster
#   * cosine-similarity matrices and a SHAP-style beeswarm plot
import numpy as np
import pandas as pd

GROUP_ORDER = ["Scheduled - survived", "Emergency - survived", "Scheduled - died", "Emergency - died"]
GROUP_COLORS = {"Scheduled - survived": "#6baed6", "Emergency - survived": "#fd8d3c",
                "Scheduled - died": "#08519c", "Emergency - died": "#d94801"}


def group_labels(emop, died):
    """Emergency/scheduled x died/survived label per patient (NaN emop -> 'Scheduled')."""
    emop = np.asarray(emop, dtype=float)
    died = np.asarray(died, dtype=float)
    em = np.where(np.isnan(emop), 0.0, emop) > 0.5
    out = np.empty(len(died), dtype=object)
    for i in range(len(died)):
        out[i] = f"{'Emergency' if em[i] else 'Scheduled'} - {'died' if died[i] > 0.5 else 'survived'}"
    return out


def _safe_metric(fn, y, p):
    y = np.asarray(y); p = np.asarray(p)
    if len(np.unique(y)) < 2:
        return np.nan
    return float(fn(y, p))


def bootstrap_ci(y, p, fn, n=200, seed=0, alpha=0.05):
    y = np.asarray(y); p = np.asarray(p)
    if len(np.unique(y)) < 2 or n <= 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2:
            continue
        vals.append(fn(y[i], p[i]))
    if not vals:
        return (np.nan, np.nan)
    return (float(np.quantile(vals, alpha / 2)), float(np.quantile(vals, 1 - alpha / 2)))


def metrics_by_group(y, p, groups, n_boot=200, seed=0, threshold=None):
    """One row per group (+ 'All'): n, deaths, death rate, AUROC/AUPRC (+95% CI), Brier,
    mean predicted risk, and -- if a threshold is given -- recall (deaths caught) and precision."""
    from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float); groups = np.asarray(groups, dtype=object)
    rows = []
    for g in ["All"] + [x for x in pd.unique(groups)]:
        sel = np.ones(len(y), bool) if g == "All" else (groups == g)
        yy, pp = y[sel], p[sel]
        if len(yy) == 0:
            continue
        row = {"group": g, "n": int(len(yy)), "deaths": int(yy.sum()), "death_rate": float(yy.mean()),
               "mean_predicted_risk": float(pp.mean()),
               "auroc": _safe_metric(roc_auc_score, yy, pp), "auprc": _safe_metric(average_precision_score, yy, pp),
               "brier": float(brier_score_loss(yy, pp)) if len(yy) else np.nan}
        lo, hi = bootstrap_ci(yy, pp, roc_auc_score, n_boot, seed); row["auroc_ci"] = f"{lo:.3f}-{hi:.3f}"
        lo, hi = bootstrap_ci(yy, pp, average_precision_score, n_boot, seed); row["auprc_ci"] = f"{lo:.3f}-{hi:.3f}"
        if threshold is not None:
            pred = pp >= threshold
            tp = int((pred & (yy == 1)).sum()); fn = int((~pred & (yy == 1)).sum()); fp = int((pred & (yy == 0)).sum())
            row["recall_at_threshold"] = tp / max(tp + fn, 1)
            row["precision_at_threshold"] = tp / max(tp + fp, 1)
        rows.append(row)
    return pd.DataFrame(rows).set_index("group")


def project_2d(X, method="tsne", seed=0, perplexity=30.0):
    """Squash an [n, d] embedding onto 2-D. method: 'pca' | 'tsne' | 'umap'.
    Returns None for 'umap' if umap-learn is not installed (reported, never faked)."""
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    X = np.asarray(X, dtype=float)
    X = np.nan_to_num(StandardScaler().fit_transform(X))
    if method == "pca":
        return PCA(n_components=2, random_state=seed).fit_transform(X)
    if method == "tsne":
        from sklearn.manifold import TSNE
        Xr = PCA(n_components=min(50, X.shape[1], X.shape[0] - 1), random_state=seed).fit_transform(X) if X.shape[1] > 50 else X
        perp = float(min(perplexity, max(5.0, (len(X) - 1) / 3.0)))
        return TSNE(n_components=2, perplexity=perp, init="pca", random_state=seed).fit_transform(Xr)
    if method == "umap":
        try:
            import umap
        except ImportError:
            return None
        return umap.UMAP(n_components=2, random_state=seed).fit_transform(X)
    raise ValueError(method)


def knn_death_enrichment(X, died, k=15):
    """Among each patient's k nearest neighbours IN THE FULL EMBEDDING (not the 2-D map),
    what fraction died? Reported for died vs survived patients, and as a ratio to the base
    rate. Ratio >> 1 for died patients = deaths cluster together = the embedding has learned
    something about risk. ~1 = deaths are scattered at random."""
    from sklearn.neighbors import NearestNeighbors
    from sklearn.preprocessing import StandardScaler
    X = np.nan_to_num(StandardScaler().fit_transform(np.asarray(X, dtype=float)))
    died = np.asarray(died, dtype=float)
    k = int(min(k, len(X) - 1))
    if k < 1 or died.sum() == 0:
        return {"k": k, "base_rate": float(died.mean()), "died_neighbour_rate_for_died": np.nan,
                "died_neighbour_rate_for_survived": np.nan, "enrichment_ratio": np.nan}
    nn_ = NearestNeighbors(n_neighbors=k + 1).fit(X)
    _, ind = nn_.kneighbors(X)
    frac = died[ind[:, 1:]].mean(axis=1)
    base = float(died.mean())
    rd = float(frac[died == 1].mean())
    rs = float(frac[died == 0].mean()) if (died == 0).any() else np.nan
    return {"k": k, "base_rate": base, "died_neighbour_rate_for_died": rd,
            "died_neighbour_rate_for_survived": rs, "enrichment_ratio": rd / max(base, 1e-12)}


def cluster_embeddings(X, k_range=(3, 4, 5, 6, 7, 8), seed=0, max_silhouette_n=5000):
    """K-means over the standardised embedding for each k; picks the k with the best
    silhouette (how cleanly separated the clusters are, -1..1). Returns (labels, table)."""
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
    from sklearn.preprocessing import StandardScaler
    X = np.nan_to_num(StandardScaler().fit_transform(np.asarray(X, dtype=float)))
    rng = np.random.default_rng(seed)
    sub = rng.choice(len(X), size=min(max_silhouette_n, len(X)), replace=False)
    best, rows = None, []
    for k in k_range:
        if k >= len(X):
            continue
        lab = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(X)
        if len(np.unique(lab[sub])) < 2:
            continue
        sil = float(silhouette_score(X[sub], lab[sub]))
        rows.append({"k": k, "silhouette": sil})
        if best is None or sil > best[0]:
            best = (sil, k, lab)
    table = pd.DataFrame(rows)
    return (best[2] if best else np.zeros(len(X), int)), table


def cluster_profile(clusters, died, emop=None, extra=None):
    """Plain-English profile of each cluster: size, death rate, emergency share, and the
    most common value / mean of any extra columns (e.g. department, ASA, top system)."""
    df = pd.DataFrame({"cluster": np.asarray(clusters), "died": np.asarray(died, dtype=float)})
    if emop is not None:
        df["emergency"] = np.asarray(emop, dtype=float)
    for name, vals in (extra or {}).items():
        df[name] = np.asarray(vals)
    agg = {"died": ["size", "sum", "mean"]}
    if emop is not None:
        agg["emergency"] = "mean"
    out = df.groupby("cluster").agg(agg)
    out.columns = ["n_patients", "deaths", "death_rate"] + (["emergency_share"] if emop is not None else [])
    for name in (extra or {}):
        col = df[name]
        if pd.api.types.is_numeric_dtype(col):
            out[f"mean_{name}"] = df.groupby("cluster")[name].mean()
        else:
            out[f"most_common_{name}"] = df.groupby("cluster")[name].agg(lambda s: s.value_counts().index[0] if len(s.dropna()) else "")
    out["death_rate_vs_overall"] = out["death_rate"] / max(df["died"].mean(), 1e-12)
    return out.sort_values("death_rate", ascending=False)


def cosine_similarity_matrix(X):
    X = np.asarray(X, dtype=float)
    n = np.linalg.norm(X, axis=1, keepdims=True)
    Xn = X / np.clip(n, 1e-12, None)
    return Xn @ Xn.T


def plot_embedding_map(ax, Z, groups, title="", order=GROUP_ORDER, colors=GROUP_COLORS, s_small=6, s_big=22):
    """Scatter a 2-D map: survivors first (small, faint), deaths on top (bigger), so the rare
    deaths are never hidden under thousands of survivors."""
    groups = np.asarray(groups, dtype=object)
    for g in order:
        sel = groups == g
        if not sel.any():
            continue
        died = g.endswith("died")
        ax.scatter(Z[sel, 0], Z[sel, 1], s=s_big if died else s_small, alpha=0.9 if died else 0.5,
                   c=colors.get(g, "#888888"), label=f"{g} (n={int(sel.sum())})",
                   edgecolors="black" if died else "none", linewidths=0.3 if died else 0)
    ax.set_title(title, fontsize=10); ax.set_xticks([]); ax.set_yticks([])


def plot_categorical_map(ax, Z, values, title="", max_levels=10, s=6):
    values = pd.Series(np.asarray(values, dtype=object)).fillna("missing").astype(str)
    top = values.value_counts().index[:max_levels]
    values = values.where(values.isin(top), "other")
    import matplotlib.pyplot as plt
    cmap = plt.get_cmap("tab10" if len(top) <= 10 else "tab20")
    for i, lev in enumerate(values.value_counts().index):
        sel = (values == lev).values
        ax.scatter(Z[sel, 0], Z[sel, 1], s=s, alpha=0.6, color=cmap(i % cmap.N), label=f"{lev} ({int(sel.sum())})")
    ax.set_title(title, fontsize=10); ax.set_xticks([]); ax.set_yticks([])


def beeswarm(ax, attributions, values, names, top=10, title=""):
    """SHAP-style summary: one row per feature (most important at the top), one dot per
    patient, x = attribution (+ pushes towards 'died'), colour = the feature's own value
    (blue low -> red high, after standardisation)."""
    A = np.asarray(attributions, dtype=float); V = np.asarray(values, dtype=float)
    order = np.argsort(-np.nanmean(np.abs(A), axis=0))[:top][::-1]
    rng = np.random.default_rng(0)
    for row, j in enumerate(order):
        a = A[:, j]; v = V[:, j]
        lo, hi = np.nanpercentile(v, 5) if np.isfinite(v).any() else 0, np.nanpercentile(v, 95) if np.isfinite(v).any() else 1
        c = np.clip((v - lo) / max(hi - lo, 1e-9), 0, 1)
        y = row + rng.uniform(-0.3, 0.3, size=len(a))
        ax.scatter(a, y, c=np.nan_to_num(c, nan=0.5), cmap="coolwarm", s=5, alpha=0.7, vmin=0, vmax=1)
    ax.set_yticks(range(len(order))); ax.set_yticklabels([names[j] for j in order], fontsize=8)
    ax.axvline(0, color="gray", lw=0.8)
    ax.set_xlabel("attribution (points of this term's risk)", fontsize=8); ax.set_title(title, fontsize=10)
    return [names[j] for j in order[::-1]]


def mean_abs_importance(attributions, names):
    A = np.asarray(attributions, dtype=float)
    return pd.Series(np.nanmean(np.abs(A), axis=0), index=list(names)).sort_values(ascending=False)

In [ ]:
# --- inlined from src-dnn/inspire_dnn/explain.py (edit it there, then rebuild) ---
# SHAP-style attribution INSIDE each risk term of the INSPIRE organ-system DNN (v4).
#
# The model already says HOW MUCH each organ system added to a patient's risk (the NAM
# terms). This module answers the next question: WHICH of that system's own measurements
# made it say so ("the kidney points came mostly from creatinine, a little from potassium").
#
# Method: Expected Gradients (Erion et al., 2021) -- the estimator behind shap's
# GradientExplainer. For one risk term f (e.g. the kidney contribution) and one patient x:
#
#     attribution_j  =  E_{baseline b ~ background, alpha ~ U(0,1)} [ (x_j - b_j) * d f(b + alpha (x - b)) / d x_j ]
#
# Only the inputs we are explaining are moved from the baseline patient towards x; every
# other input stays at x's own value. The attributions then add up (approximately, it is a
# Monte-Carlo estimate) to  f(x) - E_b[f(x with those inputs taken from b)]  -- "how far this
# patient's term is from a typical patient's, split across the inputs". `completeness_gap`
# reports how close the sum is, as a sanity check.
#
# Pure PyTorch, no dataset globals, no extra dependency (the `shap` package is not needed).
import torch


def _gather(t, idx):
    return t[idx] if t is not None else None


def expected_gradients(model, batch, background, term, ts_systems=(), static_cols=(),
                       n_samples=32, seed=0, check_completeness=True):
    """Attribute ONE risk term to a chosen set of inputs, per patient.

    model        -- a MortalityModel with NAM fusion (per-term contributions must exist)
    batch        -- dict of tensors for the B patients to explain (already on the model's device)
    background   -- dict of tensors for Nb reference patients (same keys), the "typical patient" pool
    term         -- one of model.term_names, e.g. "renal", "whole_patient", "surgical_context"
    ts_systems   -- organ systems whose time series (values AND missing-pattern mask) are attributed
    static_cols  -- static-vector column indices that are attributed
    Returns {"ts": {system: [B, F] attributions summed over the 24 time points},
             "static": [B, len(static_cols)], "f_x": [B], "f_base_mean": [B], "completeness_gap": float}
    """
    model.eval()
    dev = batch["static"].device
    B = batch["static"].shape[0]
    Nb = background["static"].shape[0]
    gen = torch.Generator(device="cpu").manual_seed(seed)
    static_cols = list(static_cols)
    col_mask = torch.zeros(batch["static"].shape[1], dtype=torch.bool, device=dev)
    if static_cols:
        col_mask[static_cols] = True

    acc_ts = {s: torch.zeros(B, batch[f"{s}_x"].shape[-1], device=dev) for s in ts_systems}
    acc_st = torch.zeros(B, len(static_cols), device=dev)
    f_base_sum = torch.zeros(B, device=dev)

    def term_value(out):
        sig = out["per_system_signal"]
        if sig is None or term not in sig:
            raise ValueError(f"term {term!r} not available (needs NAM fusion; terms: {list(sig or {})})")
        return sig[term]

    for _ in range(n_samples):
        idx = torch.randint(0, Nb, (B,), generator=gen).to(dev)
        alpha = torch.rand(B, generator=gen).to(dev)
        b = dict(batch)
        leaves, diffs = [], []
        for s in ts_systems:
            for kind in ("x", "mask"):
                x = batch[f"{s}_{kind}"]
                base = background[f"{s}_{kind}"][idx]
                a = alpha.view(B, 1, 1)
                leaf = (base + a * (x - base)).detach().requires_grad_(True)
                b[f"{s}_{kind}"] = leaf
                leaves.append(leaf); diffs.append((s, kind, x - base))
        if static_cols:
            x = batch["static"]
            base = background["static"][idx]
            mixed = torch.where(col_mask.view(1, -1), base + alpha.view(B, 1) * (x - base), x)
            st_leaf = mixed.detach().requires_grad_(True)
            b["static"] = st_leaf
            leaves.append(st_leaf); diffs.append(("static", None, x - base))
        f = term_value(model(b))
        grads = torch.autograd.grad(f.sum(), leaves, allow_unused=True)
        for g, (s, kind, d) in zip(grads, diffs):
            if g is None:
                continue
            contrib = g * d
            if s == "static":
                acc_st += contrib[:, static_cols]
            else:
                acc_ts[s] += contrib.sum(dim=1)          # sum over time points -> [B, F]
        if check_completeness:
            with torch.no_grad():
                b0 = dict(batch)
                for s in ts_systems:
                    for kind in ("x", "mask"):
                        b0[f"{s}_{kind}"] = background[f"{s}_{kind}"][idx]
                if static_cols:
                    b0["static"] = torch.where(col_mask.view(1, -1), background["static"][idx], batch["static"])
                f_base_sum += term_value(model(b0))

    res = {"ts": {s: (acc_ts[s] / n_samples).detach() for s in ts_systems},
           "static": (acc_st / n_samples).detach()}
    with torch.no_grad():
        f_x = term_value(model(batch))
    res["f_x"] = f_x.detach()
    if check_completeness:
        f_base = f_base_sum / n_samples
        total = sum(v.sum(dim=1) for v in res["ts"].values()) + res["static"].sum(dim=1) if (res["ts"] or static_cols) \
            else torch.zeros(B, device=dev)
        target = f_x - f_base
        res["f_base_mean"] = f_base.detach()
        denom = target.abs().mean().clamp_min(1e-8)
        res["completeness_gap"] = float((total - target).abs().mean() / denom)
    return res

In [ ]:
import traceback

PART13 = {}            # every result this Part produces, by name
SECTION_STATUS = {}    # section -> "ok" | "skipped: why" | "FAILED: why"

def run_section(name, fn):
    _t0 = time.time()
    try:
        r = fn()
        SECTION_STATUS[name] = r if isinstance(r, str) and r.startswith("skipped") else "ok"
    except Exception as e:
        SECTION_STATUS[name] = f"FAILED: {type(e).__name__}: {e}"
        print(f"\n!! {name} FAILED -- {type(e).__name__}: {e}\n   (the rest of Part 13 continues; full error below)")
        traceback.print_exc()
    print(f"[{name}: {SECTION_STATUS[name]} -- {time.time() - _t0:.1f}s]")

def _plain(t):
    return PLAIN_NAMES.get(t, t)

def _cohort_value(ids, col, default=np.nan):
    return np.array([COHORT_INDEXED.loc[s, col] if col in COHORT_INDEXED.columns else default for s in ids], dtype=object)

SNAP_EMOP = _emop_of(SNAPSHOT_IDS)
SNAP_DIED = np.array([PATIENT_BUNDLE[s]["label"] for s in SNAPSHOT_IDS], dtype=float)
SNAP_GROUPS = group_labels(SNAP_EMOP, SNAP_DIED)

def patient_embedding(snap, which="systems"):
    # One vector per patient: every organ system's embedding side by side + the whole-patient summary.
    parts = [snap[which].reshape(len(snap[which]), -1)]
    if snap.get("whole_patient") is not None:
        parts.append(snap["whole_patient"])
    return np.concatenate(parts, axis=1).astype(np.float32)

def _snap(tag_prefix):
    for sn in SNAPSHOTS:
        if sn["tag"].startswith(tag_prefix):
            return sn
    return None

def key_snapshots():
    # before training, after pre-training, one fine-tuning epoch in the middle, chosen model
    ft = [sn for sn in SNAPSHOTS if sn["phase"] == "fine-tuning"]
    mid = ft[len(ft) // 2] if ft else None
    out = [_snap("before_training"), _snap("after_pretraining"), mid, _snap("chosen_model")]
    seen, keep = set(), []
    for sn in out:
        if sn is not None and sn["tag"] not in seen:
            keep.append(sn); seen.add(sn["tag"])
    return keep

def _snapshot_title(sn):
    return {"before training": "Before any training", "after pre-training": "After pre-training (no labels yet)",
            "chosen model": f"Chosen model (epoch {sn['epoch']})"}.get(sn["phase"], f"Fine-tuning epoch {sn['epoch']}")

def _subsample_keep_deaths(n_max, seed=SEED):
    idx = np.arange(len(SNAPSHOT_IDS))
    if len(idx) <= n_max:
        return idx
    dead = idx[SNAP_DIED == 1]; alive = idx[SNAP_DIED == 0]
    rng = np.random.default_rng(seed)
    return np.sort(np.concatenate([dead, rng.choice(alive, size=max(0, n_max - len(dead)), replace=False)]))

print(f"Part 13 inputs: {len(SNAPSHOTS)} snapshots of {len(SNAPSHOT_IDS)} validation patients "
      f"({int(SNAP_DIED.sum())} deaths; groups: {pd.Series(SNAP_GROUPS).value_counts().to_dict()})")

## 13.1 Save everything (model, ensemble, predictions)

The chosen model was saved and reload-checked in Part 10.3. Here: every ensemble member,
and one CSV with every test patient's prediction and per-term risk points — so any later
analysis can start from files instead of re-training.

In [ ]:
def section_save_everything():
    saved = []
    if CONFIG["SAVE_FINAL_MODEL"] and len(ENSEMBLE_MODELS) > 1:
        for i, m in enumerate(ENSEMBLE_MODELS[1:], start=1):
            saved.append(save_final_model(m, name=f"ensemble_member_{i}"))
    out = collect_system_outputs(model, list(TEST_IDS))
    df = pd.DataFrame({"subject_id": out["subject_id"], "died_30d": out["labels"],
                       "predicted_risk_raw": test_result["probs"]})
    if globals().get("CALIBRATOR") is not None:
        df["predicted_risk_recalibrated"] = test_probs_calibrated
    if ENSEMBLE_RESULT is not None:
        df["predicted_risk_ensemble"] = ENSEMBLE_RESULT["probs"]
    df["emergency"] = _emop_of(df["subject_id"])
    for col in ("department", "asa"):
        df[col] = _cohort_value(df["subject_id"], col)
    if out["contrib"] is not None:
        for j, t in enumerate(model.term_names):
            df[f"points_{t}"] = out["contrib"][:, j]
    path = os.path.join(TABLE_DIR, "test_predictions.csv")
    df.to_csv(path, index=False)
    PART13["test_predictions"] = df
    print(f"Test predictions + per-term risk points -> {path} ({len(df)} patients)")
    print(f"Models in {MODEL_DIR}: {sorted(os.listdir(MODEL_DIR))}")
    print(f"Figures saved so far: {len(SAVED_FIGURES)} in {FIGURE_DIR}")

run_section("13.1 save everything", section_save_everything)

## 13.2 Training curves — before training, after pre-training, and every epoch

The left panel is AUPRC (how well deaths are found without false alarms), the right is
AUROC. Each colour is a patient group: all, emergency, scheduled. The two points on the
far left are **before any training** (random dials) and **after pre-training** (the body
knowledge is learned, but the model has not yet been told who died). Dotted horizontal
lines show what random guessing would score for each group (its death rate, for AUPRC).
The grey line marks the epoch that was kept. Emergency and scheduled curves are computed
on far fewer deaths than "all" — read their trend, not single wiggles.

In [ ]:
def section_training_curves():
    df = pd.DataFrame(EPOCH_LOG)
    if df.empty or "val_auprc_all" not in df:
        return "skipped: no epoch log"
    ft = df[df.phase == "fine-tuning"]
    if ft.empty:
        return "skipped: no fine-tuning epochs logged (resumed from a finished checkpoint?)"
    pre = df[df.phase.isin(["before training", "after pre-training"])]
    xs_pre = {"before training": -3.5, "after pre-training": -1.8}
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    colors = {"all": "#2c3e50", "emergency": "#d94801", "scheduled": "#08519c"}
    yv = np.array([PATIENT_BUNDLE[s]["label"] for s in VAL_IDS], dtype=float)
    for ax, metric in zip(axes, ("auprc", "auroc")):
        for grp, c in colors.items():
            col = f"val_{metric}_{grp}"
            ax.plot(ft["epoch"], ft[col], "-", color=c, label=f"{grp} (val)")
            for _, r in pre.iterrows():
                ax.plot(xs_pre[r["phase"]], r[col], "o", color=c, ms=7)
            if metric == "auprc":
                sel = {"all": np.ones(len(yv), bool), "emergency": VAL_EMOP > 0.5, "scheduled": VAL_EMOP <= 0.5}[grp]
                if sel.any():
                    ax.axhline(yv[sel].mean(), color=c, ls=":", lw=1)
        if metric == "auprc" and "train_auprc_all" in ft:
            ax.plot(ft["epoch"], ft["train_auprc_all"], "--", color="#7f8c8d", label="all (training patients)")
        if metric == "auroc":
            ax.axhline(0.5, color="gray", ls=":", lw=1)
        ax.axvline(BEST_EPOCH, color="gray", lw=1)
        ax.set_xticks([-3.5, -1.8] + list(ax.get_xticks()[ax.get_xticks() >= 0]))
        ax.set_xticklabels(["before\ntraining", "after\npre-train"] + [str(int(t)) for t in ax.get_xticks()[2:]])
        ax.set_xlim(-4.3, max(ft["epoch"].max(), 1) + 1)
        ax.set_xlabel("fine-tuning epoch"); ax.set_ylabel(metric.upper()); ax.legend(fontsize=8)
    axes[0].set_title("How well deaths are found (AUPRC), by group, epoch by epoch")
    axes[1].set_title("How well died vs survived are ranked (AUROC), by group")
    plt.tight_layout(); plt.show()
    rows = []
    for label, sel in (("before training", df.phase == "before training"), ("after pre-training", df.phase == "after pre-training"),
                       ("fine-tuning epoch 0", (df.phase == "fine-tuning") & (df.epoch == 0)),
                       (f"chosen epoch {BEST_EPOCH}", (df.phase == "fine-tuning") & (df.epoch == BEST_EPOCH)),
                       ("last epoch", (df.phase == "fine-tuning") & (df.epoch == ft.epoch.max()))):
        if sel.any():
            r = df[sel].iloc[0]
            rows.append({"moment": label, **{f"{m}_{g}": r[f"val_{m}_{g}"] for m in ("auprc", "auroc")
                                              for g in ("all", "emergency", "scheduled")}})
    tab = pd.DataFrame(rows).set_index("moment")
    PART13["training_moments"] = tab
    tab.to_csv(os.path.join(TABLE_DIR, "training_moments_by_group.csv"))
    print(tab.round(3).to_string())
    n = df.iloc[-1]
    print(f"\nValidation deaths behind these curves: all {int(n.get('val_n_deaths_all', 0))}, emergency "
          f"{int(n.get('val_n_deaths_emergency', 0))}, scheduled {int(n.get('val_n_deaths_scheduled', 0))}.")

run_section("13.2 training curves by group", section_training_curves)

## 13.3 Embedding maps — the patients in the model's mind, before vs after training

One row per method (PCA = a straight-line squash; t-SNE and UMAP = keep near neighbours
near). One column per moment in training. Each dot is a validation patient, coloured by
the four groups James asked for: **emergency / scheduled × died / survived**. Deaths are
drawn larger and on top, because there are so few.

**How to read it.** Before training the colours are mixed — the dials are random. If after
pre-training (no labels used!) deaths already drift towards one region, the body knowledge
itself separates sick from well. After fine-tuning, deaths should gather further. Distance
and axis numbers on t-SNE/UMAP maps mean nothing; only "who is near whom" does — which is
why 13.3b also reports a number computed in the full embedding, not on the flat map.

In [ ]:
def section_embedding_maps():
    snaps = key_snapshots()
    if not snaps:
        return "skipped: no snapshots"
    idx = _subsample_keep_deaths(CONFIG["TSNE_MAX_POINTS"])
    coords = {}
    methods = [m for m in CONFIG["EMBEDDING_METHODS"]]
    for method in methods:
        fig, axes = plt.subplots(1, len(snaps), figsize=(4.6 * len(snaps), 4.6), squeeze=False)
        any_ok = False
        for ax, sn in zip(axes[0], snaps):
            Z = project_2d(patient_embedding(sn)[idx], method=method, seed=SEED)
            if Z is None:
                ax.set_axis_off(); ax.set_title(f"{method}: umap-learn not installed", fontsize=9); continue
            any_ok = True
            coords[(method, sn["tag"])] = Z
            plot_embedding_map(ax, Z, SNAP_GROUPS[idx], _snapshot_title(sn))
        if any_ok:
            h, l = axes[0][-1].get_legend_handles_labels()
            fig.legend(h, l, loc="lower center", ncol=4, fontsize=8, frameon=False)
            fig.suptitle(f"{method.upper()} map of patient embeddings: emergency/scheduled x died/survived", y=1.02)
            plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()
        else:
            plt.close(fig)
            print(f"{method}: not available in this environment -- skipped (install umap-learn to get it).")
    PART13["map_coords"] = coords
    PART13["map_idx"] = idx

run_section("13.3 embedding maps (PCA / t-SNE / UMAP)", section_embedding_maps)

### 13.3b A number for "do deaths sit near deaths?" — at every snapshot

For each patient, look at its 15 nearest neighbours in the FULL embedding (not the flat
map). **Enrichment** = how much more often a died patient's neighbours died than the death
rate overall. 1.0 = deaths scattered at random; 3.0 = a died patient's neighbourhood has
three times the usual death rate. The same is shown for emergency status (how strongly the
embedding encodes "this was an emergency"), and for each organ system's own embedding.

In [ ]:
def section_enrichment():
    rows = []
    for sn in SNAPSHOTS:
        X = patient_embedding(sn)
        d = knn_death_enrichment(X, SNAP_DIED, CONFIG["KNN_ENRICHMENT_K"])
        e = knn_death_enrichment(X, SNAP_EMOP, CONFIG["KNN_ENRICHMENT_K"])
        rows.append({"snapshot": sn["tag"], "phase": sn["phase"], "epoch": sn["epoch"],
                     "death_enrichment": d["enrichment_ratio"], "died_neighbour_rate_for_died": d["died_neighbour_rate_for_died"],
                     "death_rate": d["base_rate"], "emergency_enrichment": e["enrichment_ratio"]})
    tab = pd.DataFrame(rows)
    PART13["enrichment_by_snapshot"] = tab
    tab.to_csv(os.path.join(TABLE_DIR, "embedding_enrichment_by_snapshot.csv"), index=False)
    print(tab.round(3).to_string(index=False))
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
    axes[0].plot(range(len(tab)), tab["death_enrichment"], "o-", color="#c0392b", label="deaths")
    axes[0].plot(range(len(tab)), tab["emergency_enrichment"], "s--", color="#d35400", label="emergency operations")
    axes[0].axhline(1, color="gray", ls=":"); axes[0].set_xticks(range(len(tab)))
    axes[0].set_xticklabels([t.replace("finetune_epoch_", "ep ").replace("_", " ") for t in tab["snapshot"]], rotation=45, ha="right", fontsize=8)
    axes[0].set_ylabel(f"enrichment among {CONFIG['KNN_ENRICHMENT_K']} nearest neighbours")
    axes[0].set_title("Do similar patients share the outcome? (1 = random)"); axes[0].legend()
    sn = _snap("chosen_model") or SNAPSHOTS[-1]
    per_sys = {}
    for j, s in enumerate(sn["organ_systems"]):
        per_sys[_plain(s)] = knn_death_enrichment(sn["systems"][:, j, :], SNAP_DIED, CONFIG["KNN_ENRICHMENT_K"])["enrichment_ratio"]
    per_sys = pd.Series(per_sys).sort_values()
    axes[1].barh(per_sys.index, per_sys.values, color="#0a7d6e"); axes[1].axvline(1, color="gray", ls=":")
    axes[1].set_title("Chosen model: which organ system's own notes group the deaths best?")
    plt.tight_layout(); plt.show()
    PART13["enrichment_by_system"] = per_sys

run_section("13.3b neighbourhood enrichment", section_enrichment)

### 13.3c The chosen model's map, coloured four more ways

Same dots, same positions (the chosen model's t-SNE, or PCA if t-SNE is off): coloured by
department, ASA grade, worst pre-op NEWS2 band, and which risk term pushed that patient's
risk up the most. Then one map per organ system (that system's own notes only).

In [ ]:
def section_map_colourings():
    sn = _snap("chosen_model") or SNAPSHOTS[-1]
    coords, idx = PART13.get("map_coords", {}), PART13.get("map_idx")
    method = next((m for m in ("tsne", "umap", "pca") if (m, sn["tag"]) in coords), None)
    if method is None:
        idx = _subsample_keep_deaths(CONFIG["TSNE_MAX_POINTS"]); method = "pca"
        Z = project_2d(patient_embedding(sn)[idx], "pca", SEED)
    else:
        Z = coords[(method, sn["tag"])]
    ids = [SNAPSHOT_IDS[i] for i in idx]
    news2 = []
    for s in ids:
        v = PATIENT_BUNDLE[s]["static"].get("news2_max", np.nan)
        news2.append("no score" if (v is None or not np.isfinite(v)) else ("high (7+)" if v >= 7 else "medium (5-6)" if v >= 5 else "low (0-4)"))
    top_term = np.array(["-"] * len(idx), dtype=object)
    if sn.get("contrib") is not None:
        top_term = np.array([_plain(sn["term_names"][j]) for j in np.argmax(sn["contrib"][idx], axis=1)], dtype=object)
    colourings = [("Department", _cohort_value(ids, "department")), ("ASA grade", _cohort_value(ids, "asa")),
                  ("Worst pre-op NEWS2 band", np.array(news2, dtype=object)), ("Term adding the most risk", top_term)]
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    for ax, (title, vals) in zip(axes, colourings):
        plot_categorical_map(ax, Z, vals, f"{title} ({method.upper()}, chosen model)")
        ax.legend(fontsize=6, markerscale=2, loc="best")
    plt.tight_layout(); plt.show()
    systems = sn["organ_systems"]
    ncol = 4; nrow = int(np.ceil(len(systems) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.4 * ncol, 4.2 * nrow), squeeze=False)
    for k, s in enumerate(systems):
        ax = axes[k // ncol][k % ncol]
        Zs = project_2d(sn["systems"][idx, k, :], "tsne" if "tsne" in CONFIG["EMBEDDING_METHODS"] else "pca", SEED)
        plot_embedding_map(ax, Zs, SNAP_GROUPS[idx], f"{_plain(s)} -- its own notes")
    for k in range(len(systems), nrow * ncol):
        axes[k // ncol][k % ncol].set_axis_off()
    h, l = axes[0][0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=4, fontsize=8, frameon=False)
    fig.suptitle("Each organ system's embedding, chosen model (emergency/scheduled x died/survived)", y=1.01)
    plt.tight_layout(rect=(0, 0.04, 1, 1)); plt.show()

run_section("13.3c map colourings + per-system maps", section_map_colourings)

## 13.4 Natural patient groups nobody labelled (clustering)

K-means groups the chosen model's embeddings into k clusters (k from 3 to 8, chosen by
the silhouette score: how cleanly separated the groups are, −1 to 1). Each cluster is then
described in plain terms: size, death rate (and how many times the overall rate), share of
emergency operations, typical department and ASA, and which term usually adds the most
risk. A cluster with a high death rate that a surgeon recognises ("emergency general
surgery, ASA 4, kidneys driving the risk") is exactly the qualitative evidence the paper
needs. A cluster nobody recognises is a question, not a finding.

In [ ]:
def section_clustering():
    sn = _snap("chosen_model") or SNAPSHOTS[-1]
    X = patient_embedding(sn)
    labels, sil = cluster_embeddings(X, seed=SEED)
    print("Silhouette by k (higher = cleaner groups):"); print(sil.round(3).to_string(index=False))
    top_term = (np.array([_plain(sn["term_names"][j]) for j in np.argmax(sn["contrib"], axis=1)], dtype=object)
                if sn.get("contrib") is not None else None)
    extra = {"department": _cohort_value(SNAPSHOT_IDS, "department"),
             "asa": pd.to_numeric(pd.Series(_cohort_value(SNAPSHOT_IDS, "asa")), errors="coerce").values,
             "predicted_risk": 1 / (1 + np.exp(-sn["logit"].astype(float)))}
    if top_term is not None:
        extra["top_risk_term"] = top_term
    prof = cluster_profile(labels, SNAP_DIED, SNAP_EMOP, extra)
    PART13["cluster_profile"] = prof
    prof.to_csv(os.path.join(TABLE_DIR, "embedding_clusters.csv"))
    print("\nCluster profiles (sorted by death rate):"); print(prof.round(3).to_string())
    Z = PART13.get("map_coords", {}).get(("tsne", sn["tag"]))
    if Z is not None:
        idx = PART13["map_idx"]
        fig, ax = plt.subplots(figsize=(6.5, 5.5))
        plot_categorical_map(ax, Z, np.array([f"cluster {c}" for c in labels[idx]], dtype=object), "Clusters on the chosen model's t-SNE map")
        dead = SNAP_DIED[idx] == 1
        ax.scatter(Z[dead, 0], Z[dead, 1], s=26, facecolors="none", edgecolors="black", linewidths=0.8, label="died")
        ax.legend(fontsize=7, markerscale=2); plt.tight_layout(); plt.show()

run_section("13.4 clustering", section_clustering)

## 13.5 How alike does the model think patients are? (cosine similarity, early vs late)

Up to 30 patients from each of the four groups, in group order. Each square compares two
patients' embeddings: 1 (dark) = the model sees them as the same kind of patient, 0 = no
relation, negative = opposite. With random dials everything looks alike (a uniformly dark
square). As the model learns, blocks appear — patients within a group resemble each other
more than patients across groups. The numbers printed under the plot say how much.

In [ ]:
def section_similarity():
    rng = np.random.default_rng(SEED)
    pick = []
    for g in GROUP_ORDER:
        members = np.where(SNAP_GROUPS == g)[0]
        if len(members):
            pick += list(rng.choice(members, size=min(30, len(members)), replace=False))
    pick = np.array(pick)
    snaps = [sn for sn in (_snap("before_training"), _snap("after_pretraining"), _snap("chosen_model")) if sn is not None]
    fig, axes = plt.subplots(1, len(snaps), figsize=(5.3 * len(snaps), 4.8), squeeze=False)
    rows = []
    for ax, sn in zip(axes[0], snaps):
        X = patient_embedding(sn)[pick]
        X = X - X.mean(axis=0, keepdims=True)   # centred, so "everyone shares the same offset" does not read as similarity
        C = cosine_similarity_matrix(X)
        im = ax.imshow(C, cmap="RdBu_r", vmin=-1, vmax=1)
        bounds = np.cumsum([int((SNAP_GROUPS[pick] == g).sum()) for g in GROUP_ORDER])[:-1]
        for b in bounds:
            ax.axhline(b - 0.5, color="black", lw=0.6); ax.axvline(b - 0.5, color="black", lw=0.6)
        ax.set_title(_snapshot_title(sn), fontsize=10); ax.set_xticks([]); ax.set_yticks([])
        same = SNAP_GROUPS[pick][:, None] == SNAP_GROUPS[pick][None, :]
        off = ~np.eye(len(pick), dtype=bool)
        rows.append({"snapshot": sn["tag"], "mean_similarity_same_group": float(C[same & off].mean()),
                     "mean_similarity_different_group": float(C[~same].mean())})
    fig.colorbar(im, ax=axes[0].tolist(), shrink=0.8, label="cosine similarity")
    fig.suptitle("Cosine similarity of patient embeddings (blocks: " + " | ".join(GROUP_ORDER) + ")", fontsize=10)
    plt.show()
    tab = pd.DataFrame(rows); tab["gap"] = tab["mean_similarity_same_group"] - tab["mean_similarity_different_group"]
    PART13["similarity"] = tab
    print(tab.round(3).to_string(index=False))
    print("A gap that grows from 'before' to 'chosen' = the model increasingly treats patients in the same group as alike.")

run_section("13.5 cosine similarity heatmaps", section_similarity)

## 13.6 Emergency vs scheduled surgery — results, risk points, and missing data

Test patients only. **Table 1**: for each group, the number of patients and deaths, AUROC
and AUPRC with 95% confidence intervals, the Brier score (lower = better calibrated), the
average predicted risk next to the real death rate, and — at the threshold chosen on
validation in 11.1b — how many deaths are caught (recall) and how often an alarm is right
(precision). Computed for the single model, the recalibrated model and the ensemble where
available. **Table 2**: the average risk points each term adds for each group — this is
where the surgical-context term shows, in one number, how much "emergency" itself adds.
**Table 3**: how much real (not filled-in) data each organ system had, by group — emergency
patients usually arrive with much less pre-op data, and the model can learn that.

In [ ]:
def section_emergency_scheduled():
    ids = list(TEST_IDS)
    y = test_result["labels"]; em = _emop_of(ids)
    grp = np.where(em > 0.5, "Emergency", "Scheduled")
    variants = [("single model (raw)", test_result["probs"])]
    if globals().get("CALIBRATOR") is not None:
        variants.append(("single model (recalibrated)", test_probs_calibrated))
    if ENSEMBLE_RESULT is not None:
        variants.append((f"ensemble of {len(ENSEMBLE_MODELS)}", ENSEMBLE_RESULT["probs"]))
    tabs = []
    for name, p in variants:
        t = metrics_by_group(y, p, grp, n_boot=min(CONFIG["N_BOOTSTRAP"], 500), seed=SEED,
                             threshold=best_threshold if name == "single model (raw)" else None)
        t.insert(0, "model", name); tabs.append(t)
    tab = pd.concat(tabs)
    PART13["emergency_scheduled_metrics"] = tab
    tab.to_csv(os.path.join(TABLE_DIR, "emergency_vs_scheduled_metrics.csv"))
    pd.set_option("display.width", 200)
    print("Table 1 -- performance by group (test set)"); print(tab.round(3).to_string())
    four = pd.Series(group_labels(em, y)).value_counts().reindex(GROUP_ORDER).fillna(0).astype(int)
    print("\nTest patients in the four groups:", four.to_dict())

    out = collect_system_outputs(model, ids)
    if out["contrib"] is not None:
        C = pd.DataFrame(out["contrib"], columns=[_plain(t) for t in model.term_names])
        C["group"] = group_labels(em, y)
        mean_pts = C.groupby("group").mean().reindex([g for g in GROUP_ORDER if g in set(C["group"])])
        PART13["points_by_group"] = mean_pts
        mean_pts.to_csv(os.path.join(TABLE_DIR, "risk_points_by_group.csv"))
        print("\nTable 2 -- average risk points per term (positive = pushes towards 'died')")
        print(mean_pts.round(3).T.to_string())
        fig, ax = plt.subplots(figsize=(12, 4))
        mean_pts.T.plot(kind="bar", ax=ax, color=[GROUP_COLORS[g] for g in mean_pts.index])
        ax.axhline(0, color="gray", lw=0.8); ax.set_ylabel("average risk points (logit)")
        ax.set_title("Where the risk comes from: emergency/scheduled x died/survived (test set)")
        ax.tick_params(axis="x", rotation=30); plt.tight_layout(); plt.show()

    obs = {}
    for s in TIME_SERIES_SYSTEMS:
        frac = np.array([PATIENT_BUNDLE[i]["ts"][s]["mask"].mean() if PATIENT_BUNDLE[i]["ts"][s]["mask"].size else np.nan for i in ids])
        obs[_plain(s)] = {"Emergency": np.nanmean(frac[em > 0.5]) if (em > 0.5).any() else np.nan,
                          "Scheduled": np.nanmean(frac[em <= 0.5]) if (em <= 0.5).any() else np.nan}
    obs = pd.DataFrame(obs).T
    PART13["observed_share_by_group"] = obs
    obs.to_csv(os.path.join(TABLE_DIR, "observed_data_share_by_group.csv"))
    print("\nTable 3 -- share of the 24 time points that hold a REAL reading (rest are filled in)")
    print(obs.round(3).to_string())

run_section("13.6 emergency vs scheduled", section_emergency_scheduled)

## 13.7 NEWS2 after filling missing data, and synthetic-patient consistency

**(a) After filling.** NEWS2 is scored at every one of the 24 time points twice: once using
only real readings (as every earlier check did), once using the filled-in timeline the model
actually sees. If filling in often moves a time point into a different NEWS2 band, the
filling method is inventing deterioration — or false reassurance. Reported: at time points
that already had a real score, how often the band changes; at time points that had NO real
score, what the filled-in scores look like next to real ones; and per patient, the worst
score from real readings vs from the filled timeline.

**(b) Synthetic patients.** A SMOTENC patient's NEWS2 number (in its facts card) is BLENDED
from two real deaths, but its timeline is BORROWED from one real donor. Here the blended
worst-NEWS2 is compared with the worst NEWS2 recomputed from the borrowed timeline. Real
deaths go through the same comparison (their facts-card NEWS2 vs their own timeline) as the
reference: synthetic patients should disagree no more than real ones do.

In [ ]:
def _band(v):
    v = np.asarray(v, dtype=float)
    return np.where(np.isnan(v), np.nan, np.where(v >= 7, 2, np.where(v >= 5, 1, 0)))

def section_news2_checks():
    if not NEWS2_VALIDATED:
        return "skipped: NEWS2 self-test failed"
    ids = (list(VAL_IDS) + list(TEST_IDS))[:3000]
    pt_real, pt_fill, pt_fill_only, pmax_real, pmax_fill = [], [], [], [], []
    for sid in ids:
        xs = {s: IMPUTED_BUNDLE[sid][s] for s in TIME_SERIES_SYSTEMS}
        ms = {s: PATIENT_BUNDLE[sid]["ts"][s]["mask"] for s in TIME_SERIES_SYSTEMS}
        ones = {s: np.ones_like(ms[s]) for s in TIME_SERIES_SYSTEMS}
        t_r, _ = _grid_news2(sid, xs, ms)
        t_f, _ = _grid_news2(sid, xs, ones)
        both = ~np.isnan(t_r) & ~np.isnan(t_f)
        pt_real += list(t_r[both]); pt_fill += list(t_f[both])
        pt_fill_only += list(t_f[np.isnan(t_r) & ~np.isnan(t_f)])
        pmax_real.append(np.nanmax(t_r) if (~np.isnan(t_r)).any() else np.nan)
        pmax_fill.append(np.nanmax(t_f) if (~np.isnan(t_f)).any() else np.nan)
    pt_real, pt_fill, pt_fill_only = map(np.array, (pt_real, pt_fill, pt_fill_only))
    pmax_real, pmax_fill = np.array(pmax_real), np.array(pmax_fill)
    res = {"time_points_with_real_score": int(len(pt_real)),
           "pct_band_changed_where_real_score_existed": float(np.mean(_band(pt_real) != _band(pt_fill))) if len(pt_real) else np.nan,
           "mean_abs_change_where_real_score_existed": float(np.mean(np.abs(pt_fill - pt_real))) if len(pt_real) else np.nan,
           "time_points_scored_only_after_filling": int(len(pt_fill_only)),
           "pct_high_band_real_points": float(np.mean(pt_real >= 7)) if len(pt_real) else np.nan,
           "pct_high_band_filled_only_points": float(np.mean(pt_fill_only >= 7)) if len(pt_fill_only) else np.nan}
    okp = ~np.isnan(pmax_real) & ~np.isnan(pmax_fill)
    res["patients_compared"] = int(okp.sum())
    res["pct_patients_worst_band_changed"] = float(np.mean(_band(pmax_real[okp]) != _band(pmax_fill[okp]))) if okp.any() else np.nan
    PART13["news2_after_filling"] = res
    print("(a) NEWS2 on real readings vs on the filled-in timeline"); print(pd.Series(res).round(4).to_string())
    print("Reading this: a few % band changes where a real score existed is expected (filling supplies the missing "
          "components of a partial set). The filled-only points should look like real ones -- many more 'high' "
          "scores there would mean filling is inventing deterioration.")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    bins = np.arange(-0.5, 21.5, 1)
    if len(pt_real):
        axes[0].hist(pt_real, bins=bins, density=True, alpha=0.6, label="real readings", color="#2980b9")
    if len(pt_fill_only):
        axes[0].hist(pt_fill_only, bins=bins, density=True, alpha=0.6, label="filled-in only", color="#f39c12")
    axes[0].set_xlabel("NEWS2 at a time point"); axes[0].legend(); axes[0].set_title("NEWS2: real vs filled-in time points")
    axes[1].scatter(pmax_real[okp] + np.random.default_rng(0).uniform(-.2, .2, okp.sum()), pmax_fill[okp], s=6, alpha=0.4)
    axes[1].plot([0, 20], [0, 20], "--", color="gray"); axes[1].set_xlabel("worst NEWS2, real readings")
    axes[1].set_ylabel("worst NEWS2, filled timeline"); axes[1].set_title("Per patient: worst score before vs after filling")
    plt.tight_layout(); plt.show()

    if not (USE_NEWS2 and SYNTHETIC_STATIC_ROWS and SYNTHETIC_TS_DONORS):
        print("\n(b) synthetic-patient consistency: skipped -- needs NEWS2 in the static vector and synthetic patients this run.")
        return
    j = STATIC_FEATURE_NAMES.index("news2_max")
    ja = STATIC_FEATURE_NAMES.index("news2_available")
    def grid_max(sid):
        xs = {s: IMPUTED_BUNDLE[sid][s] for s in TIME_SERIES_SYSTEMS}
        ms = {s: PATIENT_BUNDLE[sid]["ts"][s]["mask"] for s in TIME_SERIES_SYSTEMS}
        t, _ = _grid_news2(sid, xs, ms)
        return np.nanmax(t) if (~np.isnan(t)).any() else np.nan
    syn_card, syn_grid = [], []
    for (vec, _lab), donor in zip(SYNTHETIC_STATIC_ROWS, SYNTHETIC_TS_DONORS):
        raw = static_scaler.inverse_transform(np.asarray(vec, dtype=float).reshape(1, -1))[0]
        if raw[ja] > 0.5:
            syn_card.append(raw[j]); syn_grid.append(grid_max(donor))
    real_card, real_grid = [], []
    for sid in TRAIN_IDS:
        if PATIENT_BUNDLE[sid]["label"] == 1 and PATIENT_BUNDLE[sid]["static"].get("news2_available", 0) > 0.5:
            real_card.append(PATIENT_BUNDLE[sid]["static"]["news2_max"]); real_grid.append(grid_max(sid))
    def summ(card, grid, who):
        card, grid = np.array(card, float), np.array(grid, float)
        ok = ~np.isnan(card) & ~np.isnan(grid)
        return {"who": who, "n": int(ok.sum()),
                "mean_abs_difference": float(np.mean(np.abs(card[ok] - grid[ok]))) if ok.any() else np.nan,
                "pct_band_disagree": float(np.mean(_band(card[ok]) != _band(grid[ok]))) if ok.any() else np.nan,
                "correlation": float(np.corrcoef(card[ok], grid[ok])[0, 1]) if ok.sum() > 2 else np.nan}, card[ok], grid[ok]
    s_syn, c1, g1 = summ(syn_card, syn_grid, "synthetic deaths (blended card vs borrowed timeline)")
    s_real, c2, g2 = summ(real_card, real_grid, "real training deaths (card vs own timeline) -- reference")
    tab = pd.DataFrame([s_syn, s_real]).set_index("who")
    PART13["news2_synthetic_consistency"] = tab
    print("\n(b) Synthetic-patient NEWS2 consistency"); print(tab.round(3).to_string())
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ax.scatter(c2, g2, s=12, alpha=0.6, label="real deaths (reference)", color="#2980b9")
    ax.scatter(c1, g1, s=12, alpha=0.6, label="synthetic deaths", color="#f39c12")
    ax.plot([0, 20], [0, 20], "--", color="gray"); ax.set_xlabel("worst NEWS2 on the facts card")
    ax.set_ylabel("worst NEWS2 from the timeline"); ax.legend(); ax.set_title("Do synthetic patients' NEWS2 stories agree?")
    plt.tight_layout(); plt.show()

run_section("13.7 NEWS2 after filling + synthetic consistency", section_news2_checks)

## 13.8 SHAP inside each organ system — which measurements drove its risk points?

The model already reports how many risk points each organ system added. This splits each
system's points across **that system's own inputs** (its measurements over time — value
and whether it was measured — and its own facts: diagnoses, procedure, medicines, summary
stats). Method: expected gradients, the estimator behind SHAP's `GradientExplainer`
(`inspire_dnn/explain.py`), measured against a pool of typical training patients.

**Reading the plots.** One row per input, most important at the top. One dot per test
patient. Dots to the right pushed that system's risk UP; to the left, DOWN. Colour is the
input's own value: red = high, blue = low. "creatinine: red dots on the right" = high
creatinine raises the kidney points — what a clinician would expect. The whole-patient and
surgical-context terms are explained the same way over their facts-card inputs.

This is a check INSIDE each system — the project's main explanation is still the exact
per-system breakdown. `completeness gap` near 0 means the attributions add up to the
system's points (relative to a typical patient), as they should.

In [ ]:
def section_shap():
    if not CONFIG["RUN_SHAP"]:
        return "skipped: CONFIG['RUN_SHAP'] is False"
    if model.fusion_strategy != "nam":
        return "skipped: needs NAM fusion"
    rng = np.random.default_rng(SEED + 13)
    t_dead = [s for s in TEST_IDS if PATIENT_BUNDLE[s]["label"] == 1]
    t_alive = [s for s in TEST_IDS if PATIENT_BUNDLE[s]["label"] == 0]
    n_alive = max(0, min(len(t_alive), CONFIG["SHAP_N_PATIENTS"] - len(t_dead)))
    explain_ids = t_dead + (list(rng.choice(t_alive, size=n_alive, replace=False)) if n_alive else [])
    bg_ids = list(rng.choice(list(TRAIN_IDS), size=min(CONFIG["SHAP_N_BACKGROUND"], len(TRAIN_IDS)), replace=False))
    bg = _to_device(collate_bundle([INSPIREDataset(bg_ids)[i] for i in range(len(bg_ids))]))
    died = np.array([PATIENT_BUNDLE[s]["label"] for s in explain_ids])
    chunk = 128
    batches = [_to_device(collate_bundle([INSPIREDataset(explain_ids[i:i + chunk])[k] for k in range(len(explain_ids[i:i + chunk]))]))
               for i in range(0, len(explain_ids), chunk)]
    ts_names = {s: PATIENT_BUNDLE[TRAIN_IDS[0]]["ts"][s]["feature_names"] for s in TIME_SERIES_SYSTEMS}
    results, long_rows, gaps = {}, [], {}
    for term in model.term_names:
        if term in model.organ_systems:
            ts_sys = [term] if term in model.ts_systems else []
            cols = list(model.system_static_idx[term])
        elif term == "surgical_context":
            ts_sys, cols = [], list(model.surgical_context_idx)
        else:   # whole-patient term: its facts-card inputs
            ts_sys = []
            keep = model.whole_patient.static_keep_idx if model.whole_patient is not None else model.context_idx
            cols = list(range(N_STATIC_FEATURES)) if keep is None else list(keep)
        if not ts_sys and not cols:
            continue
        A_parts, V_parts, gap_list = [], [], []
        for b in batches:
            r = expected_gradients(model, b, bg, term, ts_systems=ts_sys, static_cols=cols,
                                   n_samples=CONFIG["SHAP_N_SAMPLES"], seed=SEED)
            a_cols, v_cols = [], []
            for s in ts_sys:
                a_cols.append(r["ts"][s].cpu().numpy())
                x, m = b[f"{s}_x"].cpu().numpy(), b[f"{s}_mask"].cpu().numpy()
                v_cols.append(np.where(m.sum(1) > 0, (x * m).sum(1) / np.clip(m.sum(1), 1, None), np.nan))
            if cols:
                a_cols.append(r["static"].cpu().numpy()); v_cols.append(b["static"][:, cols].cpu().numpy())
            A_parts.append(np.concatenate(a_cols, axis=1)); V_parts.append(np.concatenate(v_cols, axis=1))
            gap_list.append(r.get("completeness_gap", np.nan))
        names = [f"{f} (over time)" for s in ts_sys for f in ts_names[s]] + [STATIC_FEATURE_NAMES[c] for c in cols]
        A, V = np.concatenate(A_parts), np.concatenate(V_parts)
        results[term] = (A, V, names)
        gaps[_plain(term)] = float(np.nanmean(gap_list))
        imp = mean_abs_importance(A, names)
        for rank, (f, v) in enumerate(imp.items(), start=1):
            jx = names.index(f)
            long_rows.append({"term": _plain(term), "feature": f, "rank": rank, "mean_abs_attribution": v,
                              "mean_attribution_died": float(A[died == 1, jx].mean()) if (died == 1).any() else np.nan,
                              "mean_attribution_survived": float(A[died == 0, jx].mean()) if (died == 0).any() else np.nan})
    PART13["shap"] = results
    tab = pd.DataFrame(long_rows)
    PART13["shap_table"] = tab
    tab.to_csv(os.path.join(TABLE_DIR, "shap_by_system.csv"), index=False)
    PART13["shap_completeness_gap"] = gaps
    print(f"Explained {len(explain_ids)} test patients ({int(died.sum())} deaths) against {len(bg_ids)} background patients.")
    print("Completeness gap per term (0 = attributions add up exactly; < 0.1 is good for a Monte-Carlo estimate):")
    print(pd.Series(gaps).round(3).to_string())
    terms = list(results)
    ncol = 2; nrow = int(np.ceil(len(terms) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(15, 3.6 * nrow), squeeze=False)
    for k, term in enumerate(terms):
        A, V, names = results[term]
        beeswarm(axes[k // ncol][k % ncol], A, V, names, top=10, title=f"{_plain(term)} -- top inputs")
    for k in range(len(terms), nrow * ncol):
        axes[k // ncol][k % ncol].set_axis_off()
    fig.suptitle("SHAP inside each risk term (dot = patient; right = raises that term's risk; red = high value)", y=1.0)
    plt.tight_layout(); plt.show()
    top3 = tab[tab["rank"] <= 3].groupby("term")["feature"].apply(lambda s: ", ".join(s))
    print("\nTop 3 inputs per term:"); print(top3.to_string())

run_section("13.8 SHAP inside each system", section_shap)

## 13.9 Does a bigger embedding help? (the "score vs hidden size" figure)

Retrains the full recipe at several embedding sizes (`CONFIG['SIZE_SWEEP']`, e.g. 16, 32,
64, 128 numbers per organ system), with `SIZE_SWEEP_SEEDS` random starts each, and plots
test AUPRC against size with a band showing the spread across seeds — like Fig. 6 of the
reference paper. Pre-training learns from every patient, so a bigger encoder has plenty to
learn from; but the mortality part learns from only a few hundred deaths, so bigger is not
automatically better. This figure is how that question gets answered with data. Off by
default on Kaggle (it is many retrainings); on in the `isambard` profile.

In [ ]:
def section_size_sweep():
    if not CONFIG["RUN_SIZE_SWEEP"]:
        return "skipped: CONFIG['RUN_SIZE_SWEEP'] is False (on in the isambard profile)"
    rows = []
    for size in CONFIG["SIZE_SWEEP"]:
        for k in range(CONFIG["SIZE_SWEEP_SEEDS"]):
            _t0 = time.time()
            m, best_val = train_full_model(seed=SEED + 1000 * (k + 1), encoder_size=size)
            r = evaluate(m, test_loader)
            rows.append({"encoder_size": size, "embed_dim": _ENCODER_PRESETS[size][0], "seed": k,
                         "n_params": sum(p.numel() for p in m.parameters()), "val_best": best_val,
                         "test_auprc": r["auprc"], "test_auroc": r["auroc"], "minutes": (time.time() - _t0) / 60})
            print({kk: (round(v, 3) if isinstance(v, float) else v) for kk, v in rows[-1].items()})
            del m
    df = pd.DataFrame(rows)
    PART13["size_sweep"] = df
    df.to_csv(os.path.join(TABLE_DIR, "embedding_size_sweep.csv"), index=False)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric in zip(axes, ("test_auprc", "test_auroc")):
        g = df.groupby("embed_dim")[metric]
        mu, lo, hi = g.mean(), g.min(), g.max()
        ax.plot(mu.index, mu.values, "o-", color="#08519c")
        ax.fill_between(mu.index, lo.values, hi.values, alpha=0.25, color="#08519c")
        ax.set_xscale("log", base=2); ax.set_xticks(mu.index); ax.set_xticklabels([str(int(v)) for v in mu.index])
        ax.set_xlabel("embedding size per organ system (numbers)"); ax.set_ylabel(metric.replace("_", " ").upper())
        ax.set_title(f"{metric.replace('test_', '').upper()} vs embedding size (band = spread over seeds)")
    plt.tight_layout(); plt.show()
    print(df.groupby("encoder_size")[["embed_dim", "n_params", "test_auprc", "test_auroc", "minutes"]].mean().round(3).to_string())

run_section("13.9 embedding-size sweep", section_size_sweep)

## 13.10 Reuse test — are the embeddings useful for OTHER questions?

The embeddings are frozen (no retraining) and a deliberately simple model (logistic
regression, 5-fold cross-validation) is asked to predict, from them alone: 30-day death,
**ICU admission after the operation** and **a long hospital stay** (more than 14 days
after the operation, among survivors). The same test is run on the snapshots from before
training and after pre-training. If the after-pre-training embeddings already predict ICU
admission and long stay well — though they were never shown any outcome — the label-free
pre-training learned real physiology. That is the meeting's "using the same feature
vector for other tasks" point, measured.

In [ ]:
def section_reuse_test():
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold, cross_val_predict
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    ids = SNAPSHOT_IDS
    last_ops = [OPS_HISTORY[s][-1] for s in ids]
    icu = np.array([float(op.get("icuin_time") is not None and pd.notna(op.get("icuin_time"))
                          and op["icuin_time"] >= op["orin_time"]) for op in last_ops])
    dis = pd.to_numeric(pd.Series(_cohort_value(ids, "discharge_time_last")), errors="coerce").values
    orout = pd.to_numeric(pd.Series(_cohort_value(ids, "orout_time_last")), errors="coerce").values
    los_days = (dis - orout) / (24 * 60)
    long_stay = np.where(np.isnan(los_days) | (SNAP_DIED == 1), np.nan, (los_days > 14).astype(float))
    targets = {"30-day death": SNAP_DIED, "ICU admission after the operation": icu, "hospital stay > 14 days (survivors)": long_stay}
    rows = []
    for sn in [x for x in (_snap("before_training"), _snap("after_pretraining"), _snap("chosen_model")) if x is not None]:
        X = patient_embedding(sn)
        for tname, y in targets.items():
            ok = ~np.isnan(y)
            yy = y[ok]
            if ok.sum() < 50 or len(np.unique(yy)) < 2 or min(yy.sum(), (1 - yy).sum()) < 5:
                rows.append({"snapshot": _snapshot_title(sn), "target": tname, "auroc": np.nan, "auprc": np.nan,
                             "n": int(ok.sum()), "positives": int(np.nansum(yy))}); continue
            clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000))
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
            p = cross_val_predict(clf, X[ok], yy, cv=cv, method="predict_proba")[:, 1]
            rows.append({"snapshot": _snapshot_title(sn), "target": tname, "auroc": roc_auc_score(yy, p),
                         "auprc": average_precision_score(yy, p), "rate": float(yy.mean()),
                         "n": int(ok.sum()), "positives": int(yy.sum())})
    tab = pd.DataFrame(rows)
    PART13["reuse_test"] = tab
    tab.to_csv(os.path.join(TABLE_DIR, "embedding_reuse_test.csv"), index=False)
    print(tab.round(3).to_string(index=False))
    piv = tab.pivot(index="target", columns="snapshot", values="auroc")
    fig, ax = plt.subplots(figsize=(10, 3.8))
    piv.plot(kind="bar", ax=ax); ax.axhline(0.5, color="gray", ls=":"); ax.set_ylabel("AUROC (5-fold, frozen embeddings)")
    ax.set_title("Reuse test: what else can a simple model read from the frozen embeddings?")
    ax.tick_params(axis="x", rotation=0); plt.tight_layout(); plt.show()

run_section("13.10 reuse test (other outcomes)", section_reuse_test)

## 13.11 Part 13 status and saved files

In [ ]:
PART13_STATUS = pd.Series(SECTION_STATUS, name="status")
print(PART13_STATUS.to_string())
print(f"\nSaved figures: {len(SAVED_FIGURES)} in {FIGURE_DIR}")
print(f"Tables: {sorted(os.listdir(TABLE_DIR))}")
print(f"Snapshots: {len(os.listdir(SNAPSHOT_DIR))} files in {SNAPSHOT_DIR}")
record_checkpoint("Part 13 -- looking inside the model")

# Part 14 — Experiment log, verification, and what to do next

## 14.0 Run-time summary — did this hit the 20-30 minute target?

The actual, measured elapsed time for every major Part of this run, on this hardware,
with this config -- not an estimate.

In [ ]:
record_checkpoint("Part 14 -- wrap-up")
print_timing_summary()
print(f"\nConfig used for this timing: MAX_SUBJECTS_PER_CLASS={CONFIG['MAX_SUBJECTS_PER_CLASS']}, "
      f"BATCH_SIZE={CONFIG['BATCH_SIZE']}, SKIP_IMPUTATION_ACCURACY_BENCHMARKS={CONFIG['SKIP_IMPUTATION_ACCURACY_BENCHMARKS']}, "
      f"SKIP_FUSION_ABLATION={CONFIG['SKIP_FUSION_ABLATION']}")

## 14.1 Experiment log (fill this in as you run variants)

Every `CONFIG` flag in §1.7 is a genuine fork — this table is meant to be copied and
appended to as you try combinations, so the reasoning you asked to preserve stays
attached to actual results, not just intentions. Suggested first sweep once the full
cohort is loaded: hold everything else fixed and vary one flag at a time.

| Run | TIME_WINDOW | IMPUTATION_STRATEGY | SAMPLING_STRATEGY | FUSION_STRATEGY | test AUROC | test AUPRC | notes |
|---|---|---|---|---|---|---|---|
| 1 (this notebook, dev subset) | pre_op | decision_tree | class_weight | nam | *(§11.1)* | *(§11.1)* | 30-patient dev subset — directional only |
| 2 | peri_op | decision_tree | class_weight | nam | | | §11.5 |
| 3 | pre_op | median | class_weight | nam | | | isolates the imputation choice |
| 4 | pre_op | decision_tree | smote | nam | | | needs full cohort — too few positives here (§1.4.2) |
| 5 | pre_op | decision_tree | class_weight | concat | | | §11.4 |

## 14.2 What this notebook deliberately did *not* build (flagged, not silently skipped)

Consistent with keeping every decision visible rather than assumed:

- **Learned ICD-10/ATC embeddings** (§1.2.1, §1.5.3) — chapter flags and ATC level-2
  counts are used instead, because the label-starved small-cohort regime makes a learned
  embedding table under-constrained. Revisit once the full ~99,886-patient cohort is
  available for unsupervised co-occurrence pre-training.
- **POSSUM/P-POSSUM and NEWS2 baselines** (§1.2.2) — the source repo has working NELA
  (`nela.py`) and NEWS2 (`score_models.py`) implementations; wiring them in as comparison
  points alongside this DNN is a natural next cell, not built here to keep this notebook's
  scope to the multimodal architecture itself.
- **Time-to-event / survival reframing** (Dynamic-DeepHit / DySurv-style, per the source
  repo's `Research_Aim.md` §2.8) — a genuinely different output type (a hazard trajectory,
  not a single 30-day probability), flagged as a parallel track rather than folded in here.
- **Symmetric MICE / full multiple imputation** (§1.3.1) — `IterativeImputer` is imported
  and ready; not run by default because, at n=30, its iterative per-feature regressions are
  data-starved in the same way SMOTE is (§1.4.2's caveat applies almost identically here).
- **HFRS's full 109-code table** — §6.6 uses a representative ~30-code subset for
  demonstration; swap in the source repo's complete `frailty_hfrs.py` table for a
  publication-grade run.

## 14.3 Direct next steps, in priority order (v4)

1. **Run the full cohort** (`full` notebook on Kaggle, or `--profile isambard` on
   Isambard-AI) — the subset's death rate is inflated, so its AUPRC and calibration are
   not deployment numbers.
2. **Read Part 13 with a surgeon**: the cluster profiles (13.4), the per-system SHAP top
   inputs (13.8) and the coupling sign-off sheet (11.2b) are the three clinical checks.
3. **Decide GI/MSK with data**: compare the "6 organ systems" ablation row (11.4) with
   the full model. If removing them changes nothing and their risk points stay near zero,
   fold them into the whole-patient term; otherwise keep them.
4. **Embedding size**: use the 13.9 sweep (Isambard) to pick `ENCODER_SIZE` — the
   smallest size whose AUPRC band overlaps the best one.
5. **Peri-operative experiment**: give surgery its own timeline before switching
   `TIME_WINDOW='peri_op'` (with 24 points over 5 days, a 3-hour operation is ~1 point).
6. **Clinical sign-off questions**: CK under heart vs MSK; GCS-based "alert" for NEWS2;
   the procedure-code → organ-system map (§6.14b).

## 14.4 One-paragraph summary, for pasting into a supervisor update

The v4 pipeline predicts 30-day mortality after surgery from the five days BEFORE the
operation (strictly pre-operative), using eight equal organ systems — each reading its own
measurements over time and its own facts (diagnoses, and new in v4, what the operation is
on and which medicine families were given) — through one shared transformer encoder,
label-free pre-training (a hidden organ system is predicted from the other seven), a
learned inter-system layer, a whole-patient layer, and an additive output in which every
prediction splits exactly into per-system risk points plus a whole-patient term and, new
in v4, a separate surgical-context term (emergency vs scheduled, ASA, department,
anaesthesia). Part 13 adds the analyses requested on 28-09-2026: per-epoch performance for
emergency and scheduled patients from before training onwards, before/after embedding maps
(PCA, t-SNE, UMAP) coloured by emergency/scheduled × died/survived, unsupervised clusters,
cosine-similarity heatmaps, SHAP inside each organ system, NEWS2 checks after imputation
and for synthetic patients, an embedding-size sweep, and a reuse test of the frozen
embeddings on ICU admission and length of stay.

## 14.5 Verification summary (v2–v4) — every check from this run in one table

Each row is a check added or fixed in v2, with the value this run actually produced.
"PASS" means the mechanism behaved as intended; it does not mean the model is clinically
validated. Anything marked CHECK needs a look before results are quoted.

In [ ]:
def _v(name, ok, value, meaning):
    return {"check": name, "result": ("PASS" if ok else "CHECK") if ok is not None else "n/a",
            "value": value, "what it means": meaning}

_checks = []
_sm = globals().get("GROUPED_SMOTENC_SUMMARY", {})
if CONFIG["SAMPLING_STRATEGY"] == "grouped_smotenc_tomek":
    _n_syn, _n_err = _sm.get("n_synthetic", 0), _sm.get("errored_strata", 1)
    _checks.append(_v("SMOTENC runs without errors (v1 bug fixed)",
                      (None if (_n_syn == 0 and _n_err == 0) else (_n_err == 0)),
                      f"{_n_syn} synthetic, {_n_err} errored strata"
                      + (" -- no stratum was below the 1:10 target, so nothing to synthesise (expected when a sample's death rate is already high)"
                         if (_n_syn == 0 and _n_err == 0) else ""),
                      "v1: every stratum errored silently"))
_checks.append(_v("pos_weight after sampling is sane", bool(np.isfinite(POS_WEIGHT) and POS_WEIGHT >= 1),
                  f"{POS_WEIGHT:.2f}", "v1 formula subtracted removed survivors from the death count"))
_checks.append(_v("Synthetic patients have real-looking time series",
                  (not SYNTHETIC_STATIC_ROWS) or (len(SYNTHETIC_TS_DONORS) == len(SYNTHETIC_STATIC_ROWS)),
                  f"{len(SYNTHETIC_TS_DONORS)} donors for {len(SYNTHETIC_STATIC_ROWS or [])} synthetic",
                  "no 'empty series => died' shortcut"))
_checks.append(_v("NEWS2 scoring matches the RCP chart", NEWS2_VALIDATED, f"{len(_news2_failures)} failures",
                  "band edges + worked examples"))
_checks.append(_v("NEWS2 included as a feature", USE_NEWS2 if CONFIG["INCLUDE_NEWS2"] else None, USE_NEWS2,
                  "only if the self-test passed"))
_checks.append(_v("Mortality rises with worst pre-op NEWS2 band", NEWS2_MORTALITY_MONOTONE,
                  NEWS2_MORTALITY_MONOTONE, "construct validity of the computed score"))
for r in NEWS2_AUGMENTATION_CHECKS:
    if "pct_band_changed" in r:
        _checks.append(_v("NEWS2 stable under jitter/time-mask", r["pct_band_changed"] < 0.10,
                          f"{r['pct_band_changed']:.1%} of time points changed band, mean |change| {r['mean_abs_change']:.2f}",
                          "augmentation is gentle"))
    else:
        _checks.append(_v(f"NEWS2 before vs after SMOTENC [{r['feature']}]",
                          None if str(r.get("verdict", "")).startswith(("TOO FEW", "not enough")) else r.get("verdict") == "PASS",
                          f"{r.get('verdict')}; median {r.get('median_before', float('nan')):.1f} -> {r.get('median_after', float('nan')):.1f}, "
                          f"KS {r.get('ks_stat', float('nan')):.2f}, outside 0-20: {r.get('n_outside_range_after')}",
                          "synthetic deaths look like real deaths"))
_learned = CONFIG["COUPLING_MODE"] == "learned"
_checks.append(_v("No hand-defined inter-system link", ("renal_cardiac_interaction" not in STATIC_FEATURE_NAMES
                                                       and not CONFIG["COUPLING_PRIOR_LINKS"]) if _learned else None,
                  f"mode={CONFIG['COUPLING_MODE']}, priors={CONFIG['COUPLING_PRIOR_LINKS']}", "the coupling layer learns every link"))
_checks.append(_v("All eight organ systems built the same way", len(model.organ_systems) == 8,
                  ", ".join(model.organ_systems), "GI/MSK are organ systems with no time series, not a special case"))
if CONFIG["SMOTE_MAX_AMPLIFICATION"] is not None and _sm.get("n_synthetic", 0):
    _real_pos = int(sum(PATIENT_BUNDLE[s]["label"] for s in TRAIN_IDS))
    _checks.append(_v("SMOTENC amplification capped", _sm["n_synthetic"] <= CONFIG["SMOTE_MAX_AMPLIFICATION"] * _real_pos,
                      f"{_sm['n_synthetic']} synthetic from {_real_pos} real deaths (cap {CONFIG['SMOTE_MAX_AMPLIFICATION']}x per stratum)",
                      "no stratum inflated from a handful of patients"))
if COUPLING_VALIDATION_SHEET is not None:
    _cr = COUPLING_VALIDATION_SHEET[(COUPLING_VALIDATION_SHEET.receiver == "renal") & (COUPLING_VALIDATION_SHEET.sender == "cardiovascular")]
    _gmax = float(np.abs(model.coupling.gate.detach().cpu().numpy()).max())
    _checks.append(_v("Coupling layer actually used", _gmax > 0.05, f"max gate {_gmax:.2f}", "gate > 0 = systems use messages"))
    _checks.append(_v("Cardiorenal link rediscovered (never given)", bool(_cr["above_even_split"].iloc[0]) if len(_cr) else None,
                      f"rank {_cr.index[0] + 1} of {len(COUPLING_VALIDATION_SHEET)}" if len(_cr) else "-",
                      "a sanity signal, not a requirement -- surgeon reviews the full sheet"))
if COUPLING_STABILITY is not None:
    _rho = float(COUPLING_STABILITY["spearman_vs_main"].mean())
    _checks.append(_v("Learned links replicate across ensemble members", _rho > 0.7,
                      f"mean rank agreement {_rho:.2f}, top-5 overlap {COUPLING_STABILITY['top5_overlap'].mean():.0%}",
                      "> 0.7 = safe to show individual links to a surgeon"))
if LEAKAGE is not None:
    _flag = [n for n, v in LEAKAGE["own_share"].items() if v < 0.5]
    _checks.append(_v("Each system answers mainly from its own data", not _flag,
                      "all systems >= 50% own data" if not _flag else f"flagged: {', '.join(_flag)}",
                      "whole-patient + coupling layers are not leaking other systems' data"))
if SHARE_WHOLE_PATIENT is not None:
    _checks.append(_v("Organ systems carry most of the prediction", SHARE_WHOLE_PATIENT < 0.5,
                      f"{model.context_term} term {SHARE_WHOLE_PATIENT:.0%} of |risk|", "systems are not decoration"))
with torch.no_grad():
    model.eval()
    _o = model({k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in next(iter(test_loader)).items()})
if _o["per_system_signal"] is not None:
    _checks.append(_v("Per-system terms sum exactly to the prediction",
                      bool(torch.allclose(sum(_o["per_system_signal"].values()) + model.fusion.bias, _o["logit"], atol=1e-4)),
                      "exact", "interpretability preserved with coupling + arbitration + whole-patient"))
_checks.append(_v("Threshold chosen on validation, not test", True, f"{best_threshold:.3f}", "removes v1's optimistic bias"))
if "gap" in MEMORISATION:
    _checks.append(_v("Train vs validation AUPRC gap is modest", MEMORISATION["gap"] < 0.15,
                      f"train {MEMORISATION['train_auprc']:.3f} vs val {MEMORISATION['val_auprc']:.3f}", "large gap = memorising"))
if "donor_median_risk" in MEMORISATION:
    _ratio = MEMORISATION["donor_median_risk"] / max(MEMORISATION["other_death_median_risk"], 1e-6)
    _checks.append(_v("SMOTE donors not memorised", _ratio < 1.5,
                      f"donor median risk {MEMORISATION['donor_median_risk']:.3f} vs other deaths {MEMORISATION['other_death_median_risk']:.3f}",
                      "donors reused many times should not be singled out"))
_final = ENSEMBLE_RESULT or {"auprc": test_result["auprc"]}
_label = f"ensemble of {len(ENSEMBLE_MODELS)}" if ENSEMBLE_RESULT else "single DNN"
for _name, _res in [("logistic regression", BASELINE_RESULT), ("gradient boosting", GBM_RESULT)]:
    if _res is not None:
        _checks.append(_v(f"DNN ({_label}) matches or beats {_name} (AUPRC)", _final["auprc"] >= _res["auprc"] - 0.01,
                          f"DNN {_final['auprc']:.3f} vs {_res['auprc']:.3f}", "judge against the CI in 11.1d/11.1e"))
# ---- v4 checks ----
_checks.append(_v("Final model saved and reloads identically", MODEL_RELOAD_OK, FINAL_MODEL_PATH or "-",
                  "the saved file gives the same predictions as the model in memory"))
_checks.append(_v("Embedding snapshots recorded (before / after pre-training / epochs / chosen)",
                  len(SNAPSHOTS) >= 3, f"{len(SNAPSHOTS)} snapshots", "the before-vs-after maps in Part 13 need these"))
_checks.append(_v("Every figure saved to disk", (len(SAVED_FIGURES) > 0) if CONFIG["SAVE_FIGURES"] else None,
                  f"{len(SAVED_FIGURES)} PNGs in {FIGURE_DIR}", "ready for the paper / meeting"))
if "surgical_context" in model.term_names and globals().get("SHARE_SURGICAL_CONTEXT") is not None:
    _checks.append(_v("Surgical-context term reported on its own", True, f"{SHARE_SURGICAL_CONTEXT:.0%} of |risk|",
                      "emergency/scheduled, ASA, department, anaesthesia have their own risk points"))
_ed = PART13.get("enrichment_by_snapshot") if "PART13" in globals() else None
if _ed is not None and len(_ed):
    _first, _last = _ed["death_enrichment"].iloc[0], _ed["death_enrichment"].iloc[-1]
    _checks.append(_v("Deaths gather together in the embedding as training proceeds", bool(_last > max(_first, 1.0)),
                      f"neighbourhood enrichment {_first:.2f} (before training) -> {_last:.2f} (chosen model)",
                      "1 = random; higher = similar patients share the outcome"))
_na = PART13.get("news2_after_filling") if "PART13" in globals() else None
if _na:
    _checks.append(_v("Filling missing data does not distort NEWS2", _na["pct_band_changed_where_real_score_existed"] < 0.10,
                      f"{_na['pct_band_changed_where_real_score_existed']:.1%} of real-score time points changed band",
                      "filling supplies missing components, it should rarely change the band"))
_nsc = PART13.get("news2_synthetic_consistency") if "PART13" in globals() else None
if _nsc is not None and len(_nsc) == 2 and _nsc["n"].min() > 0:
    _s, _r = _nsc["pct_band_disagree"].iloc[0], _nsc["pct_band_disagree"].iloc[1]
    _checks.append(_v("Synthetic patients' NEWS2 story hangs together", bool(_s <= _r + 0.10),
                      f"band disagreement synthetic {_s:.1%} vs real deaths {_r:.1%}", "no worse than real patients (+10 pts)"))
_gap = PART13.get("shap_completeness_gap") if "PART13" in globals() else None
if _gap:
    _worst = max(_gap.values())
    _checks.append(_v("SHAP attributions add up", _worst < 0.15, f"worst completeness gap {_worst:.3f}",
                      "attributions sum to each term's points (Monte-Carlo, so not exactly 0)"))
for _sec, _st in (SECTION_STATUS.items() if "SECTION_STATUS" in globals() else []):
    if str(_st).startswith("FAILED"):
        _checks.append(_v(f"Part {_sec}", False, _st[:80], "a Part 13 analysis failed -- see its error above"))

VERIFICATION_SUMMARY = pd.DataFrame(_checks)
VERIFICATION_SUMMARY.to_csv(os.path.join(TABLE_DIR, "verification_summary.csv"), index=False)
pd.set_option("display.max_colwidth", 90)
print(VERIFICATION_SUMMARY.to_string(index=False))